# St. Hubert Medical Center
## Predictive Model Training and Evaluation

DSA4157 — Business Intelligence
University of Santo Tomas — College of Science

3DSA1 — Group 5

### Objectives

1. Length of Stay (LOS) Prediction — Regression
2. 30-Day Subsequent Visit Prediction — Classification

### Models

- Linear Regression
- Random Forest Regressor
- XGBoost Regressor
- Random Forest Classifier
- XGBoost Classifier

### Methodology

- Dataset validation
- Patient-grouped data splitting
- Target leakage prevention
- Data preprocessing
- Model training and evaluation
- Validation-based threshold selection
- Joblib model export
- FastAPI integration

### Disclaimer

The classification target represents a 30-day subsequent
visit proxy, not a clinically validated unplanned
hospital readmission prediction model.

### Repository copy — research only

Saved outputs are removed. Run sequentially in Google Colab with authorized private V2 CSVs; do not commit datasets, fitted models, or private exports. The notebook is specific to the original V2 cohort and contains fixed cohort and evaluation checks. A different dataset or environment may fail these checks. No clinical deployment or operational alerts are approved. FastAPI integration is a separate task.


In [ ]:
import sys
import os
import platform
import importlib.metadata as metadata

print("TRAINING ENVIRONMENT")
print("=" * 50)
print("Python version:", sys.version.split()[0])
print("Operating system:", platform.system())
print("CPU cores:", os.cpu_count())

print("\nREQUIRED LIBRARIES")
print("=" * 50)

libraries = [
    "pandas",
    "numpy",
    "scikit-learn",
    "xgboost",
    "joblib",
    "matplotlib",
    "seaborn"
]

for library in libraries:
    try:
        version = metadata.version(library)
        print(f"{library:20} {version}")
    except metadata.PackageNotFoundError:
        print(f"{library:20} NOT INSTALLED")

In [ ]:
# CELL 2: Upload corrected V2 datasets

from google.colab import files
from pathlib import Path
import os

os.chdir("/content")

print("UPLOAD CORRECTED V2 DATASETS")
print("=" * 60)

uploaded = files.upload()

print("\nUPLOADED FILES")
print("=" * 60)

for filename in uploaded.keys():
    path = Path("/content") / filename

    print(f"\nFilename: {filename}")
    print(f"Location: {path}")
    print(f"Size: {path.stat().st_size / 1024**2:.2f} MB")

del uploaded

print("\nUpload process completed.")

In [ ]:
from pathlib import Path

LOS_PATH = Path(
    "/content/St_Hubert_LOS_Model_Dataset_v2.csv"
)

PROXY_PATH = Path(
    "/content/St_Hubert_30Day_Subsequent_Visit_Proxy_Dataset_v2.csv"
)

print("CORRECTED DATASET AVAILABILITY")
print("=" * 60)

for path in [LOS_PATH, PROXY_PATH]:

    print(f"\nFile: {path.name}")

    if path.exists():
        print("Status: AVAILABLE")
        print(
            f"Size: "
            f"{path.stat().st_size / (1024**2):.2f} MB"
        )
    else:
        print("Status: NOT FOUND")

assert LOS_PATH.exists()
assert PROXY_PATH.exists()

print("\nBoth corrected datasets are available.")
print("Dataset version: V2")

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/St.-Hubert-Medical-Center")

REPO_URL = (
    "https://github.com/Itarasawrus-afk/"
    "St.-Hubert-Medical-Center.git"
)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    print("Repository already exists.")

print("\nRepository location:", REPO_DIR)

In [ ]:
MODEL_DIR = REPO_DIR / "predictive model"

assert MODEL_DIR.is_dir(), "Predictive model folder not found."

print("EXISTING PREDICTIVE MODEL FILES")
print("=" * 60)

for path in sorted(MODEL_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(MODEL_DIR))

In [ ]:
script_names = [
    "XGBoost",
    "linear regression",
    "random forest"
]

print("EXISTING MODEL SCRIPT INSPECTION")
print("=" * 80)

for name in script_names:

    script = MODEL_DIR / name

    if script.is_file():

        print("\n" + "=" * 80)
        print("SCRIPT:", script.name)
        print("File size:", script.stat().st_size, "bytes")
        print("=" * 80)

        content = script.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print(content)

    else:
        print(f"\nFile not found: {name}")

print("\n" + "=" * 80)
print("Script inspection completed.")

In [ ]:
# CELL 7: Corrected V2 dataset loading

import pandas as pd
import numpy as np
from pathlib import Path

# Define corrected dataset paths
LOS_PATH = Path(
    "/content/St_Hubert_LOS_Model_Dataset_v2.csv"
)

PROXY_PATH = Path(
    "/content/St_Hubert_30Day_Subsequent_Visit_Proxy_Dataset_v2.csv"
)

# Verify file availability
assert LOS_PATH.is_file(), "LOS dataset not found."
assert PROXY_PATH.is_file(), "Proxy dataset not found."

# Load corrected datasets
los_df = pd.read_csv(
    LOS_PATH,
    dtype={"PATIENT_KEY": "string"}
)

proxy_df = pd.read_csv(
    PROXY_PATH,
    dtype={"PATIENT_KEY": "string"}
)

# Display dataset information
print("CORRECTED DATASET LOADING")
print("=" * 60)

for name, df in [
    ("LOS Dataset V2", los_df),
    ("Subsequent Visit Dataset V2", proxy_df)
]:
    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Unique patients: {df['PATIENT_KEY'].nunique():,}")
    print(
        f"Memory (MB): "
        f"{df.memory_usage(deep=True).sum() / 1024**2:.2f}"
    )

# Verify corrected dataset dimensions
assert los_df.shape == (770960, 9)
assert proxy_df.shape == (728679, 11)

# Verify unique patient counts
assert los_df["PATIENT_KEY"].nunique() == 312359
assert proxy_df["PATIENT_KEY"].nunique() == 307470

# Verify reconstructed classification target
assert (
    proxy_df["SUBSEQUENT_VISIT_WITHIN_30D_PROXY"].sum()
    == 84897
)

print("\nBoth corrected datasets loaded successfully.")
print("Dataset version verification: PASSED")

In [ ]:
def inspect_schema(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    schema = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Non-Null Count": df.notna().sum().values,
        "Unique Values": df.nunique().values
    })

    display(schema)

inspect_schema(los_df, "LOS DATASET SCHEMA")

inspect_schema(
    proxy_df,
    "SUBSEQUENT VISIT DATASET SCHEMA"
)

In [ ]:
FEATURES = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

NUM_COLS = [
    "AGE_AT_VISIT",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

CAT_COLS = [
    "GENDER",
    "VISIT_TYPE"
]

EXCLUDED_PROXY_FEATURES = [
    "PATIENT_KEY",
    "VISIT_STATUS_AT_EXIT",
    "LOS_DAYS",
    PROXY_TARGET
]

def verify_columns(df, target, name):

    required = FEATURES + [GROUP_COL, target]

    missing = [
        col for col in required
        if col not in df.columns
    ]

    print("\n", name)
    print("Required columns:", len(required))
    print("Missing columns:", missing)

    assert not missing, (
        f"{name} has missing required columns."
    )

    print("Column validation: PASSED")

verify_columns(los_df, LOS_TARGET, "LOS DATASET")
verify_columns(proxy_df, PROXY_TARGET, "PROXY DATASET")

assert GROUP_COL not in FEATURES
assert LOS_TARGET not in FEATURES

assert not set(FEATURES).intersection(
    EXCLUDED_PROXY_FEATURES
)

print("\nAPPROVED PREDICTORS")

for i, feature in enumerate(FEATURES, start=1):
    print(f"{i}. {feature}")

print("\nExplicit leakage exclusion check: PASSED")

In [ ]:
def missing_value_report(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    report = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing Percentage": (
            df.isna().mean() * 100
        )
    })

    report = report.sort_values(
        "Missing Count",
        ascending=False
    )

    display(report)

    print(
        "Total missing cells:",
        f"{df.isna().sum().sum():,}"
    )

missing_value_report(los_df, "LOS DATASET")

missing_value_report(
    proxy_df,
    "SUBSEQUENT VISIT DATASET"
)

In [ ]:
# CELL 11: Duplicate and patient identifier checks

def check_duplicates_and_patients(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    duplicates = df.duplicated().sum()

    missing_keys = (
        df[GROUP_COL].isna()
        | df[GROUP_COL].str.strip().eq("").fillna(False)
    ).sum()

    visits_per_patient = (
        df.groupby(GROUP_COL).size()
    )

    print("Total rows:", f"{len(df):,}")
    print("Exact duplicate rows:", f"{duplicates:,}")

    print(
        "Duplicate percentage:",
        f"{duplicates / len(df) * 100:.4f}%"
    )

    print(
        "Unique patients:",
        f"{df[GROUP_COL].nunique():,}"
    )

    print(
        "Missing or blank patient keys:",
        f"{missing_keys:,}"
    )

    print(
        "Patients with multiple visits:",
        f"{(visits_per_patient > 1).sum():,}"
    )

    print(
        "Average visits per patient:",
        round(visits_per_patient.mean(), 4)
    )

    print(
        "Maximum visits per patient:",
        visits_per_patient.max()
    )

check_duplicates_and_patients(
    los_df,
    "LOS DATASET"
)

check_duplicates_and_patients(
    proxy_df,
    "SUBSEQUENT VISIT DATASET"
)

In [ ]:
# CELL 12: Feature quality checks

def check_feature_quality(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    numeric = df[NUM_COLS].apply(
        pd.to_numeric,
        errors="coerce"
    )

    print("\nNUMERICAL FEATURE STATISTICS")
    display(numeric.describe().T)

    print("\nINVALID NUMERIC VALUES")

    for col in NUM_COLS:

        invalid = (
            df[col].notna()
            & numeric[col].isna()
        ).sum()

        infinite = np.isinf(
            numeric[col].to_numpy(dtype=float)
        ).sum()

        fractional = (
            numeric[col].notna()
            & np.isfinite(numeric[col])
            & numeric[col].mod(1).ne(0)
        ).sum()

        print(
            f"{col}: "
            f"Invalid={invalid}, "
            f"Infinite={infinite}, "
            f"Fractional={fractional}"
        )

    checks = {
        "Age below 0": (
            numeric["AGE_AT_VISIT"] < 0
        ),
        "Age above 120 (review)": (
            numeric["AGE_AT_VISIT"] > 120
        ),
        "Invalid admission month": (
            ~numeric["ADMISSION_MONTH"].between(1, 12)
            & numeric["ADMISSION_MONTH"].notna()
        ),
        "Weekday outside 0-7": (
            ~numeric["ADMISSION_WEEKDAY"].between(0, 7)
            & numeric["ADMISSION_WEEKDAY"].notna()
        ),
        "Invalid admission hour": (
            ~numeric["ADMISSION_HOUR"].between(0, 23)
            & numeric["ADMISSION_HOUR"].notna()
        ),
        "Negative prior visit count": (
            numeric["PRIOR_VISIT_COUNT"] < 0
        )
    }

    print("\nFEATURE RANGE CHECKS")

    for label, condition in checks.items():
        print(f"{label}: {condition.sum():,}")

    print("\nOBSERVED WEEKDAY VALUES")
    print(sorted(
        numeric["ADMISSION_WEEKDAY"].dropna().unique()
    ))

    print("\nCATEGORICAL FEATURE DISTRIBUTIONS")

    for col in CAT_COLS:
        print(f"\n{col}")
        display(df[col].value_counts(dropna=False))

check_feature_quality(los_df, "LOS DATASET")

check_feature_quality(
    proxy_df,
    "SUBSEQUENT VISIT DATASET"
)

In [ ]:
# CELL 13: Target variable validation

los_target = pd.to_numeric(
    los_df[LOS_TARGET],
    errors="coerce"
)

proxy_target = pd.to_numeric(
    proxy_df[PROXY_TARGET],
    errors="coerce"
)

print("LOS TARGET VALIDATION")
print("=" * 60)

print("Missing or non-numeric:", los_target.isna().sum())
print("Infinite values:", np.isinf(los_target).sum())
print("Negative LOS:", (los_target < 0).sum())
print("Zero LOS:", (los_target == 0).sum())
print("LOS above 30 days:", (los_target > 30).sum())
print("LOS above 90 days:", (los_target > 90).sum())
print("LOS above 180 days:", (los_target > 180).sum())

print("\nLOS DESCRIPTIVE STATISTICS")

display(los_target.describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
))

print("\nSUBSEQUENT VISIT TARGET VALIDATION")
print("=" * 60)

print(
    "Missing or non-numeric:",
    proxy_target.isna().sum()
)

print(
    "Infinite values:",
    np.isinf(proxy_target).sum()
)

invalid_binary = (
    proxy_target.notna()
    & ~proxy_target.isin([0, 1])
).sum()

print("Invalid binary values:", invalid_binary)

class_report = pd.DataFrame({
    "Count": proxy_target.value_counts(
        dropna=False
    ).sort_index(),
    "Percentage": proxy_target.value_counts(
        normalize=True,
        dropna=False
    ).sort_index() * 100
})

display(class_report)

negative_count = (proxy_target == 0).sum()
positive_count = (proxy_target == 1).sum()

if positive_count > 0:
    print(
        "Class imbalance ratio:",
        f"{negative_count / positive_count:.2f}:1"
    )

los_valid = (
    np.isfinite(los_target).all()
    and (los_target >= 0).all()
)

proxy_valid = (
    proxy_target.isin([0, 1]).all()
    and proxy_target.nunique() == 2
)

print("\nTARGET VALIDATION SUMMARY")
print("LOS target:", "PASS" if los_valid else "FAIL")
print("Proxy target:", "PASS" if proxy_valid else "FAIL")

In [ ]:
# CELL 14: Target distribution visualizations

sns.set_theme(style="whitegrid")

# Visualization 1: LOS distribution

valid_los = los_target[
    np.isfinite(los_target)
]

plt.figure(figsize=(12, 5))

sns.histplot(
    valid_los,
    bins=50,
    color="#245C46"
)

plt.axvline(
    valid_los.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {valid_los.mean():.2f} days"
)

plt.axvline(
    valid_los.median(),
    color="orange",
    linestyle="--",
    label=f"Median: {valid_los.median():.2f} days"
)

plt.title("Distribution of Recorded Length of Stay")
plt.xlabel("Length of Stay (Days)")
plt.ylabel("Number of Visits")
plt.legend()
plt.tight_layout()
plt.show()


# Visualization 2: LOS boxplot

plt.figure(figsize=(12, 3))

sns.boxplot(
    x=valid_los,
    color="#245C46"
)

plt.title("Length of Stay — Boxplot")
plt.xlabel("Length of Stay (Days)")
plt.tight_layout()
plt.show()


# Visualization 3: Classification target distribution

class_counts = proxy_target.value_counts().reindex(
    [0, 1],
    fill_value=0
)

plt.figure(figsize=(8, 5))

bars = plt.bar(
    ["No Subsequent Visit", "Subsequent Visit"],
    class_counts.values,
    color=["#94A3B8", "#245C46"]
)

plt.bar_label(
    bars,
    labels=[
        f"{count:,}\n({count / len(proxy_df):.2%})"
        for count in class_counts.values
    ],
    padding=5
)

plt.ylim(0, max(class_counts.values) * 1.2)

plt.title("30-Day Subsequent Visit Target Distribution")
plt.ylabel("Number of Visits")
plt.tight_layout()
plt.show()

In [ ]:
# CELL 15: Final feature configuration

FEATURES = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

NUM_COLS = [
    "AGE_AT_VISIT",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

CAT_COLS = [
    "GENDER",
    "VISIT_TYPE"
]

LOS_TARGET = "TARGET_LOS_DAYS"

PROXY_TARGET = (
    "SUBSEQUENT_VISIT_WITHIN_30D_PROXY"
)

GROUP_COL = "PATIENT_KEY"

FORBIDDEN_FEATURES = {
    GROUP_COL,
    LOS_TARGET,
    PROXY_TARGET,
    "VISIT_STATUS_AT_EXIT",
    "LOS_DAYS"
}

assert not set(FEATURES) & FORBIDDEN_FEATURES

assert set(NUM_COLS + CAT_COLS) == set(FEATURES)

los_model_df = los_df[
    [GROUP_COL] + FEATURES + [LOS_TARGET]
].copy()

proxy_model_df = proxy_df[
    [GROUP_COL] + FEATURES + [PROXY_TARGET]
].copy()

print("FINAL FEATURE CONFIGURATION")
print("=" * 60)

print("Predictive features:", len(FEATURES))
print("Numerical features:", len(NUM_COLS))
print("Categorical features:", len(CAT_COLS))

print("\nLOS modeling dataset:", los_model_df.shape)

print(
    "Proxy modeling dataset:",
    proxy_model_df.shape
)

print("\nExcluded classification columns:")

for col in ["VISIT_STATUS_AT_EXIT", "LOS_DAYS"]:
    assert col not in proxy_model_df.columns
    print(f"- {col}")

print("\nFeature configuration: PASSED")

In [ ]:
# CELL 16: Patient-grouped splitting function

from sklearn.model_selection import GroupShuffleSplit

SEED = 42

def grouped_train_val_test_split(
    df,
    group_col="PATIENT_KEY",
    val_size=0.15,
    test_size=0.15,
    seed=42
):

    assert df[group_col].notna().all()

    # First split: reserve the test patients

    gss_test = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=seed
    )

    train_val_idx, test_idx = next(
        gss_test.split(
            df,
            groups=df[group_col]
        )
    )

    train_val_df = df.iloc[
        train_val_idx
    ].reset_index(drop=True)

    test_df = df.iloc[
        test_idx
    ].reset_index(drop=True)

    # Second split: training and validation

    relative_val_size = (
        val_size / (1.0 - test_size)
    )

    gss_val = GroupShuffleSplit(
        n_splits=1,
        test_size=relative_val_size,
        random_state=seed + 1
    )

    train_idx, val_idx = next(
        gss_val.split(
            train_val_df,
            groups=train_val_df[group_col]
        )
    )

    train_df = train_val_df.iloc[
        train_idx
    ].reset_index(drop=True)

    val_df = train_val_df.iloc[
        val_idx
    ].reset_index(drop=True)

    # Verify patient separation

    train_patients = set(train_df[group_col])
    val_patients = set(val_df[group_col])
    test_patients = set(test_df[group_col])

    assert train_patients.isdisjoint(val_patients)

    assert train_patients.isdisjoint(test_patients)

    assert val_patients.isdisjoint(test_patients)

    # Verify that no rows were lost

    assert (
        len(train_df)
        + len(val_df)
        + len(test_df)
        == len(df)
    )

    return train_df, val_df, test_df

print("Patient-grouped splitting function ready.")

In [ ]:
# CELL 17: Execute patient-grouped splitting

los_train, los_val, los_test = (
    grouped_train_val_test_split(
        los_model_df,
        seed=SEED
    )
)

proxy_train, proxy_val, proxy_test = (
    grouped_train_val_test_split(
        proxy_model_df,
        seed=SEED
    )
)

def split_summary(train, val, test, name):

    total_rows = len(train) + len(val) + len(test)

    total_patients = sum(
        df[GROUP_COL].nunique()
        for df in [train, val, test]
    )

    report = []

    for label, df in [
        ("Training", train),
        ("Validation", val),
        ("Testing", test)
    ]:

        report.append({
            "Dataset": label,
            "Rows": len(df),
            "Row Percentage": (
                len(df) / total_rows * 100
            ),
            "Unique Patients": (
                df[GROUP_COL].nunique()
            ),
            "Patient Percentage": (
                df[GROUP_COL].nunique()
                / total_patients * 100
            )
        })

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    display(pd.DataFrame(report))

    print("Total rows:", f"{total_rows:,}")
    print("Total patients:", f"{total_patients:,}")

split_summary(
    los_train,
    los_val,
    los_test,
    "LOS REGRESSION SPLIT"
)

split_summary(
    proxy_train,
    proxy_val,
    proxy_test,
    "SUBSEQUENT VISIT CLASSIFICATION SPLIT"
)

In [ ]:
# CELL 18: Verify patient separation

def verify_patient_separation(
    train,
    val,
    test,
    name
):

    train_keys = set(train[GROUP_COL])
    val_keys = set(val[GROUP_COL])
    test_keys = set(test[GROUP_COL])

    overlaps = {
        "Training vs Validation": len(
            train_keys & val_keys
        ),
        "Training vs Testing": len(
            train_keys & test_keys
        ),
        "Validation vs Testing": len(
            val_keys & test_keys
        )
    }

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for label, count in overlaps.items():
        print(f"{label}: {count}")

    assert all(
        count == 0
        for count in overlaps.values()
    )

    print("\nPatient leakage check: PASSED")

verify_patient_separation(
    los_train,
    los_val,
    los_test,
    "LOS REGRESSION"
)

verify_patient_separation(
    proxy_train,
    proxy_val,
    proxy_test,
    "SUBSEQUENT VISIT CLASSIFICATION"
)

In [ ]:
# CELL 19: Target distributions after splitting

los_report = []

for name, df in [
    ("Training", los_train),
    ("Validation", los_val)
]:

    target = df[LOS_TARGET]

    los_report.append({
        "Dataset": name,
        "Rows": len(df),
        "Mean LOS": target.mean(),
        "Median LOS": target.median(),
        "Std LOS": target.std(),
        "Min LOS": target.min(),
        "Max LOS": target.max()
    })

print("LOS TARGET DISTRIBUTION")
display(pd.DataFrame(los_report))


proxy_report = []

for name, df in [
    ("Training", proxy_train),
    ("Validation", proxy_val)
]:

    target = df[PROXY_TARGET]

    positive = (target == 1).sum()
    negative = (target == 0).sum()

    assert target.nunique() == 2

    proxy_report.append({
        "Dataset": name,
        "Rows": len(df),
        "Negative Cases": negative,
        "Positive Cases": positive,
        "Positive Percentage": (
            positive / len(df) * 100
        ),
        "Imbalance Ratio": negative / positive
    })

print("\nCLASSIFICATION TARGET DISTRIBUTION")
display(pd.DataFrame(proxy_report))

print("\nSplit distribution checks completed.")

print(
    "Test target distributions are reserved "
    "for final evaluation."
)

In [ ]:
# CELL 20: Prepare features and targets

# ==========================================
# MODEL A: LOS REGRESSION
# ==========================================

X_los_train = los_train[FEATURES].copy()
X_los_val = los_val[FEATURES].copy()
X_los_test = los_test[FEATURES].copy()

y_los_train = los_train[LOS_TARGET].to_numpy(
    dtype=np.float64
)

y_los_val = los_val[LOS_TARGET].to_numpy(
    dtype=np.float64
)

y_los_test = los_test[LOS_TARGET].to_numpy(
    dtype=np.float64
)


# ==========================================
# MODEL B: SUBSEQUENT VISIT CLASSIFICATION
# ==========================================

X_proxy_train = proxy_train[FEATURES].copy()
X_proxy_val = proxy_val[FEATURES].copy()
X_proxy_test = proxy_test[FEATURES].copy()

y_proxy_train = proxy_train[PROXY_TARGET].to_numpy(
    dtype=np.int64
)

y_proxy_val = proxy_val[PROXY_TARGET].to_numpy(
    dtype=np.int64
)

y_proxy_test = proxy_test[PROXY_TARGET].to_numpy(
    dtype=np.int64
)


# ==========================================
# VERIFY DATASET DIMENSIONS
# ==========================================

datasets = [
    ("LOS Training", X_los_train, y_los_train),
    ("LOS Validation", X_los_val, y_los_val),
    ("LOS Testing", X_los_test, y_los_test),
    ("Proxy Training", X_proxy_train, y_proxy_train),
    ("Proxy Validation", X_proxy_val, y_proxy_val),
    ("Proxy Testing", X_proxy_test, y_proxy_test)
]

report = []

for name, X, y in datasets:

    assert len(X) == len(y)
    assert list(X.columns) == FEATURES
    assert not set(X.columns) & FORBIDDEN_FEATURES
    assert np.isfinite(y).all()

    report.append({
        "Dataset": name,
        "X Shape": X.shape,
        "y Shape": y.shape,
        "Target Type": str(y.dtype)
    })

display(pd.DataFrame(report))

print("\nFeature and target preparation: PASSED")
print("All forbidden predictors are excluded.")

In [ ]:
# CELL 21: Build preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

def build_preprocessor(
    num_cols=NUM_COLS,
    cat_cols=CAT_COLS
):

    numerical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "ohe",
            OneHotEncoder(handle_unknown="ignore")
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numerical_pipeline,
            num_cols
        ),
        (
            "cat",
            categorical_pipeline,
            cat_cols
        )
    ])

    return preprocessor


print("PREPROCESSING CONFIGURATION")
print("=" * 60)

print("\nNumerical features:")
for col in NUM_COLS:
    print("-", col)

print("\nNumerical transformations:")
print("- Median imputation")
print("- StandardScaler")

print("\nCategorical features:")
for col in CAT_COLS:
    print("-", col)

print("\nCategorical transformations:")
print("- Most frequent imputation")
print("- One-hot encoding")
print("- Ignore unknown categories")

print("\nPreprocessing function ready.")

In [ ]:
# CELL 22: Verify preprocessing

from scipy import sparse

los_preprocessor_check = build_preprocessor()
proxy_preprocessor_check = build_preprocessor()

# Fit exclusively on training data

X_los_train_check = (
    los_preprocessor_check.fit_transform(X_los_train)
)

X_los_val_check = (
    los_preprocessor_check.transform(X_los_val)
)

X_proxy_train_check = (
    proxy_preprocessor_check.fit_transform(X_proxy_train)
)

X_proxy_val_check = (
    proxy_preprocessor_check.transform(X_proxy_val)
)


def verify_transformed_data(X, name):

    values = X.data if sparse.issparse(X) else X

    missing = np.isnan(values).sum()
    infinite = np.isinf(values).sum()

    print(f"\n{name}")
    print("Transformed shape:", X.shape)
    print("Missing values:", missing)
    print("Infinite values:", infinite)

    assert missing == 0
    assert infinite == 0


verify_transformed_data(
    X_los_train_check,
    "LOS Training"
)

verify_transformed_data(
    X_los_val_check,
    "LOS Validation"
)

verify_transformed_data(
    X_proxy_train_check,
    "Proxy Training"
)

verify_transformed_data(
    X_proxy_val_check,
    "Proxy Validation"
)

print("\nPreprocessing verification: PASSED")

In [ ]:
# CELL 23: Inspect transformed feature names

los_feature_names = (
    los_preprocessor_check.get_feature_names_out()
)

proxy_feature_names = (
    proxy_preprocessor_check.get_feature_names_out()
)

print("LOS TRANSFORMED FEATURES")
print("=" * 60)

for i, feature in enumerate(
    los_feature_names,
    start=1
):
    print(f"{i}. {feature}")

print("\nTotal features:", len(los_feature_names))


print("\nPROXY TRANSFORMED FEATURES")
print("=" * 60)

for i, feature in enumerate(
    proxy_feature_names,
    start=1
):
    print(f"{i}. {feature}")

print("\nTotal features:", len(proxy_feature_names))

assert list(los_feature_names) == list(
    proxy_feature_names
)

print("\nFeature transformation check: PASSED")

In [ ]:
# CELL 24: Inspect preprocessing parameters

def inspect_preprocessor(preprocessor, name):

    num_pipeline = (
        preprocessor.named_transformers_["num"]
    )

    cat_pipeline = (
        preprocessor.named_transformers_["cat"]
    )

    imputer = num_pipeline.named_steps["imputer"]
    scaler = num_pipeline.named_steps["scaler"]

    numerical_report = pd.DataFrame({
        "Feature": NUM_COLS,
        "Imputation Median": imputer.statistics_,
        "Scaling Mean": scaler.mean_,
        "Scaling Std": scaler.scale_
    })

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("\nNumerical preprocessing parameters")
    display(numerical_report)

    print("\nCategorical imputation values")

    cat_imputer = cat_pipeline.named_steps["imputer"]

    for col, value in zip(
        CAT_COLS,
        cat_imputer.statistics_
    ):
        print(f"{col}: {value}")

    print("\nLearned categorical values")

    encoder = cat_pipeline.named_steps["ohe"]

    for col, categories in zip(
        CAT_COLS,
        encoder.categories_
    ):
        print(f"{col}: {list(categories)}")


inspect_preprocessor(
    los_preprocessor_check,
    "LOS PREPROCESSING"
)

inspect_preprocessor(
    proxy_preprocessor_check,
    "PROXY PREPROCESSING"
)

print("\nPreprocessing parameter inspection completed.")

In [ ]:
# CELL 25: Release temporary preprocessing arrays

import gc

del X_los_train_check
del X_los_val_check
del X_proxy_train_check
del X_proxy_val_check

del los_preprocessor_check
del proxy_preprocessor_check

gc.collect()

print("Temporary preprocessing arrays released.")
print("Original feature datasets retained.")
print("Preprocessing function retained.")
print("\nMemory cleanup completed.")

In [ ]:
# CELL 26: Statistical baseline evaluation

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# ==========================================
# LOS REGRESSION BASELINES
# ==========================================

train_mean = y_los_train.mean()
train_median = np.median(y_los_train)

mean_predictions = np.full(
    len(y_los_val),
    train_mean
)

median_predictions = np.full(
    len(y_los_val),
    train_median
)

baseline_los_results = []

for name, predictions in [
    ("Training Mean", mean_predictions),
    ("Training Median", median_predictions)
]:

    baseline_los_results.append({
        "Baseline": name,
        "Validation MAE": mean_absolute_error(
            y_los_val, predictions
        ),
        "Validation RMSE": root_mean_squared_error(
            y_los_val, predictions
        ),
        "Validation R2": r2_score(
            y_los_val, predictions
        )
    })

baseline_los_df = pd.DataFrame(
    baseline_los_results
)

print("LOS REGRESSION BASELINES")
print("=" * 60)

print(f"Training mean: {train_mean:.4f} days")
print(f"Training median: {train_median:.4f} days")

display(baseline_los_df)


# ==========================================
# CLASSIFICATION BASELINE
# ==========================================

train_prevalence = y_proxy_train.mean()

baseline_probabilities = np.full(
    len(y_proxy_val),
    train_prevalence
)

baseline_proxy_df = pd.DataFrame([{
    "Baseline": "Constant Training Prevalence",
    "Validation ROC-AUC": roc_auc_score(
        y_proxy_val,
        baseline_probabilities
    ),
    "Validation PR-AUC (AP)": average_precision_score(
        y_proxy_val,
        baseline_probabilities
    ),
    "Validation Brier Score": brier_score_loss(
        y_proxy_val,
        baseline_probabilities
    )
}])

print("\nCLASSIFICATION BASELINE")
print("=" * 60)

print(
    f"Training prevalence: "
    f"{train_prevalence:.4%}"
)

display(baseline_proxy_df)

print("\nBaseline evaluation completed.")
print("Test sets have not been evaluated.")

In [ ]:
# CELL 27: Classification eligibility audit

print("CLASSIFICATION ELIGIBILITY AUDIT")
print("=" * 65)

status_report = (
    proxy_df.groupby("VISIT_STATUS_AT_EXIT")
    .agg(
        Records=(PROXY_TARGET, "size"),
        Positive_Cases=(PROXY_TARGET, "sum"),
        Positive_Rate=(PROXY_TARGET, "mean"),
        Mean_LOS=("LOS_DAYS", "mean")
    )
)

status_report["Positive_Rate"] *= 100

display(status_report)

print("\nAdmitted records in proxy dataset:")

admitted_count = (
    proxy_df["VISIT_STATUS_AT_EXIT"]
    .str.strip()
    .str.lower()
    .eq("admitted")
    .sum()
)

print(f"{admitted_count:,}")

print("\nREFERENCE SOURCE AUDIT")
print("Source visit records: 917,331")
print("Source admitted records: 146,371")
print("Reconstructed eligible records: 728,679")
print("Reconstructed positive cases: 84,897")

print("\nPREPARED DATASET COMPARISON")

print("Eligible record difference:",
      len(proxy_df) - 728679)

print("Positive case difference:",
      int(proxy_df[PROXY_TARGET].sum()) - 84897)

print("\nEligibility audit completed.")

In [ ]:
# CELL 28: Patient-key consistency audit

los_keys = set(los_df[GROUP_COL])
proxy_keys = set(proxy_df[GROUP_COL])

shared_keys = los_keys & proxy_keys
proxy_only_keys = proxy_keys - los_keys
los_only_keys = los_keys - proxy_keys

print("PATIENT-KEY CONSISTENCY AUDIT")
print("=" * 65)

print("LOS unique patients:",
      f"{len(los_keys):,}")

print("Proxy unique patients:",
      f"{len(proxy_keys):,}")

print("Shared patient keys:",
      f"{len(shared_keys):,}")

print("LOS-only patient keys:",
      f"{len(los_only_keys):,}")

print("Proxy-only patient keys:",
      f"{len(proxy_only_keys):,}")

coverage = len(shared_keys) / len(proxy_keys)

print(f"\nProxy patient-key coverage: {coverage:.4%}")

if len(proxy_only_keys) == 0:
    print("Patient-key subset check: PASSED")
else:
    print("Patient-key subset check: NEEDS REVIEW")

del los_keys, proxy_keys
del shared_keys, proxy_only_keys, los_only_keys

gc.collect()

print("\nPatient-key audit completed.")

In [ ]:
# CELL 29: Linear Regression - LOS Prediction

import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

print("LINEAR REGRESSION - LOS PREDICTION")
print("=" * 70)

# --------------------------------------------------
# 1. Verify training and validation inputs
# --------------------------------------------------

assert len(X_los_train) == 539559
assert len(X_los_val) == 115592

assert len(X_los_train) == len(y_los_train)
assert len(X_los_val) == len(y_los_val)

assert list(X_los_train.columns) == list(FEATURES)
assert list(X_los_val.columns) == list(FEATURES)

assert "PATIENT_KEY" not in X_los_train.columns
assert "TARGET_LOS_DAYS" not in X_los_train.columns

print("Training records:", f"{len(X_los_train):,}")
print("Validation records:", f"{len(X_los_val):,}")
print("Original features:", X_los_train.shape[1])

# --------------------------------------------------
# 2. Build the complete model pipeline
# --------------------------------------------------

lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        ("model", LinearRegression())
    ]
)

print("\nPipeline created successfully.")

# --------------------------------------------------
# 3. Train the model
# --------------------------------------------------

print("\nTraining Linear Regression...")

start_time = time.perf_counter()

lr_pipeline.fit(
    X_los_train,
    y_los_train
)

lr_training_time = time.perf_counter() - start_time

print("Training completed.")
print(f"Training time: {lr_training_time:.4f} seconds")

# --------------------------------------------------
# 4. Generate validation predictions
# --------------------------------------------------

start_time = time.perf_counter()

lr_val_pred = lr_pipeline.predict(X_los_val)

lr_prediction_time = time.perf_counter() - start_time

assert len(lr_val_pred) == len(y_los_val)
assert np.isfinite(lr_val_pred).all()

# --------------------------------------------------
# 5. Calculate validation metrics
# --------------------------------------------------

lr_mae = mean_absolute_error(
    y_los_val, lr_val_pred
)

lr_rmse = root_mean_squared_error(
    y_los_val, lr_val_pred
)

lr_r2 = r2_score(
    y_los_val, lr_val_pred
)

# Recalculate baselines from current V2 training data
mean_baseline_pred = np.full(
    len(y_los_val),
    np.mean(y_los_train)
)

median_baseline_pred = np.full(
    len(y_los_val),
    np.median(y_los_train)
)

mean_baseline_mae = mean_absolute_error(
    y_los_val, mean_baseline_pred
)

median_baseline_mae = mean_absolute_error(
    y_los_val, median_baseline_pred
)

mean_baseline_rmse = root_mean_squared_error(
    y_los_val, mean_baseline_pred
)

median_baseline_rmse = root_mean_squared_error(
    y_los_val, median_baseline_pred
)

mae_improvement = (
    (median_baseline_mae - lr_mae)
    / median_baseline_mae
) * 100

rmse_improvement = (
    (mean_baseline_rmse - lr_rmse)
    / mean_baseline_rmse
) * 100

# --------------------------------------------------
# 6. Store model evaluation results
# --------------------------------------------------

lr_results = {
    "Model": "Linear Regression",
    "Validation MAE": lr_mae,
    "Validation RMSE": lr_rmse,
    "Validation R2": lr_r2,
    "Training Time (s)": lr_training_time,
    "Prediction Time (s)": lr_prediction_time
}

# Initialize the regression model registries
los_models = {
    "Linear Regression": lr_pipeline
}

los_validation_predictions = {
    "Linear Regression": lr_val_pred
}

los_model_results = [lr_results]

# --------------------------------------------------
# 7. Display evaluation results
# --------------------------------------------------

print("\nLINEAR REGRESSION VALIDATION RESULTS")
print("=" * 70)

display(
    pd.DataFrame([lr_results]).round(6)
)

comparison_df = pd.DataFrame([
    {
        "Model": "Training Mean Baseline",
        "Validation MAE": mean_baseline_mae,
        "Validation RMSE": mean_baseline_rmse,
        "Validation R2": r2_score(
            y_los_val, mean_baseline_pred
        )
    },
    {
        "Model": "Training Median Baseline",
        "Validation MAE": median_baseline_mae,
        "Validation RMSE": median_baseline_rmse,
        "Validation R2": r2_score(
            y_los_val, median_baseline_pred
        )
    },
    {
        "Model": "Linear Regression",
        "Validation MAE": lr_mae,
        "Validation RMSE": lr_rmse,
        "Validation R2": lr_r2
    }
])

print("\nBASELINE COMPARISON")
print("=" * 70)

display(comparison_df.round(6))

print(
    f"\nMAE improvement over median baseline: "
    f"{mae_improvement:.6f}%"
)

print(
    f"RMSE improvement over mean baseline: "
    f"{rmse_improvement:.6f}%"
)

# --------------------------------------------------
# 8. Prediction sanity checks
# --------------------------------------------------

print("\nPREDICTION SANITY CHECK")
print("=" * 70)

print(f"Minimum prediction: {lr_val_pred.min():.4f}")
print(f"Maximum prediction: {lr_val_pred.max():.4f}")
print(f"Mean prediction: {lr_val_pred.mean():.4f}")
print(f"Prediction std: {lr_val_pred.std():.4f}")

print(
    "Negative predictions:",
    f"{np.sum(lr_val_pred < 0):,}"
)

print(
    "Non-finite predictions:",
    f"{np.sum(~np.isfinite(lr_val_pred)):,}"
)

print(
    f"\nValidation prediction time: "
    f"{lr_prediction_time:.4f} seconds"
)

print("\nLinear Regression training completed.")
print("Fitted pipeline retained in memory.")
print("Test dataset has NOT been evaluated.")

In [ ]:
# CELL 30: Linear Regression Diagnostics
# and Coefficient Analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("LINEAR REGRESSION DIAGNOSTICS")
print("=" * 70)

# --------------------------------------------------
# 1. Verify fitted model and validation predictions
# --------------------------------------------------

assert "Linear Regression" in los_models
assert len(lr_val_pred) == len(y_los_val)
assert np.isfinite(lr_val_pred).all()

actual = np.asarray(y_los_val)
predicted = np.asarray(lr_val_pred)

# Positive residual = underprediction
# Negative residual = overprediction

lr_residuals = actual - predicted
lr_absolute_errors = np.abs(lr_residuals)

# --------------------------------------------------
# 2. Residual summary
# --------------------------------------------------

residual_summary = pd.DataFrame({
    "Statistic": [
        "Mean Residual",
        "Median Residual",
        "Residual Standard Deviation",
        "Minimum Residual",
        "Maximum Residual",
        "Mean Absolute Error",
        "Median Absolute Error"
    ],
    "Value (Days)": [
        np.mean(lr_residuals),
        np.median(lr_residuals),
        np.std(lr_residuals),
        np.min(lr_residuals),
        np.max(lr_residuals),
        np.mean(lr_absolute_errors),
        np.median(lr_absolute_errors)
    ]
})

print("\nRESIDUAL SUMMARY")
print("=" * 70)

display(residual_summary.round(4))

# --------------------------------------------------
# 3. Prediction accuracy within error tolerances
# --------------------------------------------------

tolerances = [1, 3, 7, 14, 30, 60]

tolerance_results = []

for tolerance in tolerances:

    count = np.sum(
        lr_absolute_errors <= tolerance
    )

    tolerance_results.append({
        "Error Tolerance (Days)": tolerance,
        "Predictions Within Tolerance": count,
        "Percentage": count / len(actual) * 100
    })

lr_tolerance_df = pd.DataFrame(
    tolerance_results
)

print("\nPREDICTION ERROR TOLERANCE")
print("=" * 70)

display(lr_tolerance_df.round(4))

# --------------------------------------------------
# 4. Overprediction and underprediction
# --------------------------------------------------

prediction_direction_df = pd.DataFrame([
    {
        "Prediction Direction": "Underprediction",
        "Count": np.sum(lr_residuals > 0)
    },
    {
        "Prediction Direction": "Overprediction",
        "Count": np.sum(lr_residuals < 0)
    },
    {
        "Prediction Direction": "Exact Prediction",
        "Count": np.sum(lr_residuals == 0)
    }
])

prediction_direction_df["Percentage"] = (
    prediction_direction_df["Count"]
    / len(actual)
    * 100
)

print("\nPREDICTION DIRECTION")
print("=" * 70)

display(prediction_direction_df.round(4))

# --------------------------------------------------
# 5. Error analysis by actual LOS range
# --------------------------------------------------

lr_diagnostics_df = pd.DataFrame({
    "Actual_LOS": actual,
    "Predicted_LOS": predicted,
    "Residual": lr_residuals,
    "Absolute_Error": lr_absolute_errors
})

lr_diagnostics_df["LOS_Range"] = pd.cut(
    lr_diagnostics_df["Actual_LOS"],
    bins=[0, 7, 30, 60, 90, 180, np.inf],
    labels=[
        "0-7 days",
        ">7-30 days",
        ">30-60 days",
        ">60-90 days",
        ">90-180 days",
        ">180 days"
    ],
    include_lowest=True
)

lr_range_analysis = (
    lr_diagnostics_df
    .groupby("LOS_Range", observed=True)
    .agg(
        Records=("Actual_LOS", "size"),
        Mean_Actual_LOS=("Actual_LOS", "mean"),
        Mean_Predicted_LOS=("Predicted_LOS", "mean"),
        MAE=("Absolute_Error", "mean"),
        Mean_Residual=("Residual", "mean")
    )
    .reset_index()
)

print("\nERROR ANALYSIS BY LOS RANGE")
print("=" * 70)

display(lr_range_analysis.round(4))

# --------------------------------------------------
# 6. Extract fitted feature coefficients
# --------------------------------------------------

lr_preprocessor = (
    lr_pipeline.named_steps["preprocessor"]
)

lr_estimator = (
    lr_pipeline.named_steps["model"]
)

lr_feature_names = (
    lr_preprocessor.get_feature_names_out()
)

lr_coefficients = np.asarray(
    lr_estimator.coef_
).ravel()

assert len(lr_feature_names) == len(lr_coefficients)

lr_coefficient_df = pd.DataFrame({
    "Feature": lr_feature_names,
    "Coefficient": lr_coefficients
})

lr_coefficient_df["Absolute_Coefficient"] = (
    lr_coefficient_df["Coefficient"].abs()
)

lr_coefficient_df = (
    lr_coefficient_df
    .sort_values(
        "Absolute_Coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nLINEAR REGRESSION COEFFICIENTS")
print("=" * 70)

print(
    f"Model intercept: "
    f"{lr_estimator.intercept_:.6f}"
)

display(lr_coefficient_df.round(6))

print(
    "\nCoefficient caution: numerical features "
    "are standardized and categorical features "
    "are one-hot encoded."
)

print(
    "Full one-hot encoding with an intercept "
    "creates redundant categorical columns."
)

print(
    "Individual categorical coefficients "
    "must not be interpreted independently "
    "as causal effects or feature importance."
)

# --------------------------------------------------
# 7. Actual versus predicted scatterplot
# --------------------------------------------------

rng = np.random.default_rng(42)

sample_indices = rng.choice(
    len(actual),
    size=min(10000, len(actual)),
    replace=False
)

plot_min = min(actual.min(), predicted.min())
plot_max = max(actual.max(), predicted.max())

plt.figure(figsize=(9, 6))

plt.scatter(
    actual[sample_indices],
    predicted[sample_indices],
    alpha=0.15,
    s=8,
    label="Validation predictions"
)

plt.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    linestyle="--",
    label="Perfect prediction"
)

plt.xlabel("Actual LOS (Days)")
plt.ylabel("Predicted LOS (Days)")
plt.title("Linear Regression: Actual vs Predicted LOS")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 8. Residual distribution
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.hist(
    lr_residuals,
    bins=60,
    edgecolor="white"
)

plt.axvline(
    0,
    linestyle="--",
    label="Zero residual"
)

plt.xlabel("Residual: Actual - Predicted (Days)")
plt.ylabel("Frequency")
plt.title("Linear Regression: Residual Distribution")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 9. Actual versus predicted LOS distribution
# --------------------------------------------------

common_bins = np.linspace(
    plot_min,
    plot_max,
    61
)

plt.figure(figsize=(9, 5))

plt.hist(
    actual,
    bins=common_bins,
    alpha=0.5,
    density=True,
    label="Actual LOS"
)

plt.hist(
    predicted,
    bins=common_bins,
    alpha=0.5,
    density=True,
    label="Predicted LOS"
)

plt.xlabel("LOS (Days)")
plt.ylabel("Density")
plt.title("Actual vs Predicted LOS Distribution")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 10. Residuals versus predicted LOS
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.scatter(
    predicted[sample_indices],
    lr_residuals[sample_indices],
    alpha=0.15,
    s=8
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Predicted LOS (Days)")
plt.ylabel("Residual (Days)")
plt.title("Linear Regression: Residuals vs Predictions")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 11. Completion summary
# --------------------------------------------------

print("\nDIAGNOSTICS COMPLETION SUMMARY")
print("=" * 70)

print("Residual analysis: COMPLETED")
print("Error tolerance analysis: COMPLETED")
print("LOS range analysis: COMPLETED")
print("Coefficient extraction: COMPLETED")
print("Diagnostic visualization: COMPLETED")

print("\nModel has not been retrained.")
print("Test dataset has NOT been evaluated.")
print("Linear Regression diagnostics completed.")

In [ ]:
# CELL 31: Random Forest Regressor - LOS Prediction

import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

print("RANDOM FOREST REGRESSOR - LOS PREDICTION")
print("=" * 70)

# --------------------------------------------------
# 1. Verify training and validation inputs
# --------------------------------------------------

assert len(X_los_train) == 539559
assert len(X_los_val) == 115592

assert len(X_los_train) == len(y_los_train)
assert len(X_los_val) == len(y_los_val)

assert list(X_los_train.columns) == list(FEATURES)
assert list(X_los_val.columns) == list(FEATURES)

assert "PATIENT_KEY" not in X_los_train.columns
assert "TARGET_LOS_DAYS" not in X_los_train.columns

assert "Linear Regression" in los_models

print("Training records:", f"{len(X_los_train):,}")
print("Validation records:", f"{len(X_los_val):,}")
print("Original features:", X_los_train.shape[1])

# --------------------------------------------------
# 2. Define original Random Forest configuration
# --------------------------------------------------

rf_regressor_params = {
    "n_estimators": 100,
    "max_depth": 12,
    "min_samples_leaf": 50,
    "max_samples": 0.5,
    "bootstrap": True,
    "criterion": "squared_error",
    "random_state": 42,
    "n_jobs": -1
}

print("\nRANDOM FOREST CONFIGURATION")
print("=" * 70)

for parameter, value in rf_regressor_params.items():
    print(f"{parameter}: {value}")

# --------------------------------------------------
# 3. Build complete preprocessing/model pipeline
# --------------------------------------------------

rf_los_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "model",
            RandomForestRegressor(
                **rf_regressor_params
            )
        )
    ]
)

print("\nPipeline created successfully.")

# --------------------------------------------------
# 4. Train Random Forest
# --------------------------------------------------

print("\nTraining Random Forest Regressor...")
print("This may take several minutes.", flush=True)

start_time = time.perf_counter()

rf_los_pipeline.fit(
    X_los_train,
    y_los_train
)

rf_los_training_time = (
    time.perf_counter() - start_time
)

print("\nTraining completed.")

print(
    f"Training time: "
    f"{rf_los_training_time:.4f} seconds"
)

# --------------------------------------------------
# 5. Generate validation predictions
# --------------------------------------------------

start_time = time.perf_counter()

rf_los_val_pred = rf_los_pipeline.predict(
    X_los_val
)

rf_los_prediction_time = (
    time.perf_counter() - start_time
)

assert len(rf_los_val_pred) == len(y_los_val)
assert np.isfinite(rf_los_val_pred).all()

# --------------------------------------------------
# 6. Calculate validation metrics
# --------------------------------------------------

rf_los_mae = mean_absolute_error(
    y_los_val,
    rf_los_val_pred
)

rf_los_rmse = root_mean_squared_error(
    y_los_val,
    rf_los_val_pred
)

rf_los_r2 = r2_score(
    y_los_val,
    rf_los_val_pred
)

rf_los_results = {
    "Model": "Random Forest Regressor",
    "Validation MAE": rf_los_mae,
    "Validation RMSE": rf_los_rmse,
    "Validation R2": rf_los_r2,
    "Training Time (s)": rf_los_training_time,
    "Prediction Time (s)": rf_los_prediction_time
}

print("\nRANDOM FOREST VALIDATION RESULTS")
print("=" * 70)

display(
    pd.DataFrame([rf_los_results]).round(6)
)

# --------------------------------------------------
# 7. Calculate improvements over baselines
# --------------------------------------------------

rf_mae_baseline_improvement = (
    (median_baseline_mae - rf_los_mae)
    / median_baseline_mae
) * 100

rf_rmse_baseline_improvement = (
    (mean_baseline_rmse - rf_los_rmse)
    / mean_baseline_rmse
) * 100

rf_mae_lr_improvement = (
    (lr_mae - rf_los_mae)
    / lr_mae
) * 100

rf_rmse_lr_improvement = (
    (lr_rmse - rf_los_rmse)
    / lr_rmse
) * 100

print("\nPERFORMANCE IMPROVEMENTS")
print("=" * 70)

print(
    "MAE improvement over median baseline: "
    f"{rf_mae_baseline_improvement:.6f}%"
)

print(
    "RMSE improvement over mean baseline: "
    f"{rf_rmse_baseline_improvement:.6f}%"
)

print(
    "MAE improvement over Linear Regression: "
    f"{rf_mae_lr_improvement:.6f}%"
)

print(
    "RMSE improvement over Linear Regression: "
    f"{rf_rmse_lr_improvement:.6f}%"
)

# --------------------------------------------------
# 8. Register fitted model and results
# --------------------------------------------------

los_models["Random Forest Regressor"] = (
    rf_los_pipeline
)

los_validation_predictions[
    "Random Forest Regressor"
] = rf_los_val_pred

# Prevent duplicate results if this cell is rerun
los_model_results = [
    result
    for result in los_model_results
    if result["Model"] != "Random Forest Regressor"
]

los_model_results.append(rf_los_results)

# --------------------------------------------------
# 9. Updated regression model comparison
# --------------------------------------------------

los_leaderboard = (
    pd.DataFrame(los_model_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

print("\nREGRESSION MODEL LEADERBOARD")
print("=" * 70)

display(los_leaderboard.round(6))

# --------------------------------------------------
# 10. Prediction sanity checks
# --------------------------------------------------

print("\nPREDICTION SANITY CHECK")
print("=" * 70)

print(
    f"Minimum prediction: "
    f"{rf_los_val_pred.min():.4f}"
)

print(
    f"Maximum prediction: "
    f"{rf_los_val_pred.max():.4f}"
)

print(
    f"Mean prediction: "
    f"{rf_los_val_pred.mean():.4f}"
)

print(
    f"Prediction std: "
    f"{rf_los_val_pred.std():.4f}"
)

print(
    "Negative predictions:",
    f"{np.sum(rf_los_val_pred < 0):,}"
)

print(
    "Non-finite predictions:",
    f"{np.sum(~np.isfinite(rf_los_val_pred)):,}"
)

# --------------------------------------------------
# 11. Verify trained forest structure
# --------------------------------------------------

rf_los_estimator = (
    rf_los_pipeline.named_steps["model"]
)

tree_depths = [
    tree.tree_.max_depth
    for tree in rf_los_estimator.estimators_
]

tree_node_counts = [
    tree.tree_.node_count
    for tree in rf_los_estimator.estimators_
]

print("\nTRAINED FOREST STRUCTURE")
print("=" * 70)

print(
    "Number of trained trees:",
    len(rf_los_estimator.estimators_)
)

print(
    f"Average tree depth: "
    f"{np.mean(tree_depths):.2f}"
)

print(
    "Maximum tree depth:",
    max(tree_depths)
)

print(
    f"Average nodes per tree: "
    f"{np.mean(tree_node_counts):.2f}"
)

assert len(rf_los_estimator.estimators_) == 100
assert max(tree_depths) <= 12

# --------------------------------------------------
# 12. Completion summary
# --------------------------------------------------

print("\nTRAINING COMPLETION SUMMARY")
print("=" * 70)

print("Random Forest training: COMPLETED")
print("Validation evaluation: COMPLETED")
print("Model registry: UPDATED")
print("Prediction sanity checks: PASSED")
print("Forest structure checks: PASSED")

print("\nFitted pipeline retained in memory.")
print("Test dataset has NOT been evaluated.")

In [ ]:
# CELL 32: Random Forest Diagnostics
# and Feature Importance Analysis

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

print("RANDOM FOREST REGRESSOR DIAGNOSTICS")
print("=" * 70)

# --------------------------------------------------
# 1. Verify fitted model and validation predictions
# --------------------------------------------------

assert "Random Forest Regressor" in los_models
assert len(rf_los_val_pred) == len(y_los_val)
assert np.isfinite(rf_los_val_pred).all()

actual = np.asarray(y_los_val)
predicted = np.asarray(rf_los_val_pred)

rf_los_residuals = actual - predicted
rf_los_absolute_errors = np.abs(rf_los_residuals)

# Positive residual = underprediction
# Negative residual = overprediction

# --------------------------------------------------
# 2. Residual summary
# --------------------------------------------------

rf_residual_summary = pd.DataFrame({
    "Statistic": [
        "Mean Residual",
        "Median Residual",
        "Residual Standard Deviation",
        "Minimum Residual",
        "Maximum Residual",
        "Mean Absolute Error",
        "Median Absolute Error"
    ],
    "Value (Days)": [
        np.mean(rf_los_residuals),
        np.median(rf_los_residuals),
        np.std(rf_los_residuals),
        np.min(rf_los_residuals),
        np.max(rf_los_residuals),
        np.mean(rf_los_absolute_errors),
        np.median(rf_los_absolute_errors)
    ]
})

print("\nRESIDUAL SUMMARY")
print("=" * 70)

display(rf_residual_summary.round(4))

# --------------------------------------------------
# 3. Prediction error tolerance comparison
# --------------------------------------------------

tolerances = [1, 3, 7, 14, 30, 60]

rf_tolerance_results = []

for tolerance in tolerances:

    rf_count = np.sum(
        rf_los_absolute_errors <= tolerance
    )

    lr_count = np.sum(
        lr_absolute_errors <= tolerance
    )

    rf_percentage = rf_count / len(actual) * 100
    lr_percentage = lr_count / len(actual) * 100

    rf_tolerance_results.append({
        "Error Tolerance (Days)": tolerance,
        "LR Within Tolerance": lr_count,
        "RF Within Tolerance": rf_count,
        "LR Percentage": lr_percentage,
        "RF Percentage": rf_percentage,
        "RF Improvement (pp)": (
            rf_percentage - lr_percentage
        )
    })

rf_tolerance_df = pd.DataFrame(
    rf_tolerance_results
)

print("\nPREDICTION ERROR TOLERANCE COMPARISON")
print("=" * 70)

display(rf_tolerance_df.round(4))

# --------------------------------------------------
# 4. Prediction direction
# --------------------------------------------------

rf_prediction_direction_df = pd.DataFrame([
    {
        "Prediction Direction": "Underprediction",
        "Count": np.sum(rf_los_residuals > 0)
    },
    {
        "Prediction Direction": "Overprediction",
        "Count": np.sum(rf_los_residuals < 0)
    },
    {
        "Prediction Direction": "Exact Prediction",
        "Count": np.sum(rf_los_residuals == 0)
    }
])

rf_prediction_direction_df["Percentage"] = (
    rf_prediction_direction_df["Count"]
    / len(actual)
    * 100
)

print("\nPREDICTION DIRECTION")
print("=" * 70)

display(rf_prediction_direction_df.round(4))

# --------------------------------------------------
# 5. Error analysis by actual LOS range
# --------------------------------------------------

rf_diagnostics_df = pd.DataFrame({
    "Actual_LOS": actual,
    "Predicted_LOS": predicted,
    "Residual": rf_los_residuals,
    "Absolute_Error": rf_los_absolute_errors
})

rf_diagnostics_df["LOS_Range"] = pd.cut(
    rf_diagnostics_df["Actual_LOS"],
    bins=[0, 7, 30, 60, 90, 180, np.inf],
    labels=[
        "0-7 days",
        ">7-30 days",
        ">30-60 days",
        ">60-90 days",
        ">90-180 days",
        ">180 days"
    ],
    include_lowest=True
)

rf_range_analysis = (
    rf_diagnostics_df
    .groupby("LOS_Range", observed=True)
    .agg(
        Records=("Actual_LOS", "size"),
        Mean_Actual_LOS=("Actual_LOS", "mean"),
        Mean_Predicted_LOS=("Predicted_LOS", "mean"),
        MAE=("Absolute_Error", "mean"),
        Mean_Residual=("Residual", "mean")
    )
    .reset_index()
)

print("\nRANDOM FOREST ERROR ANALYSIS BY LOS RANGE")
print("=" * 70)

display(rf_range_analysis.round(4))

# Compare with Linear Regression

los_range_comparison = (
    lr_range_analysis[
        ["LOS_Range", "Records", "MAE"]
    ]
    .rename(columns={"MAE": "LR_MAE"})
    .merge(
        rf_range_analysis[
            ["LOS_Range", "MAE"]
        ].rename(columns={"MAE": "RF_MAE"}),
        on="LOS_Range",
        validate="one_to_one"
    )
)

los_range_comparison["RF_MAE_Improvement"] = (
    los_range_comparison["LR_MAE"]
    - los_range_comparison["RF_MAE"]
)

print("\nLOS RANGE MODEL COMPARISON")
print("=" * 70)

display(los_range_comparison.round(4))

# --------------------------------------------------
# 6. Extract impurity-based feature importance
# --------------------------------------------------

rf_preprocessor = (
    rf_los_pipeline.named_steps["preprocessor"]
)

rf_estimator = (
    rf_los_pipeline.named_steps["model"]
)

rf_feature_names = (
    rf_preprocessor.get_feature_names_out()
)

rf_importances = (
    rf_estimator.feature_importances_
)

assert len(rf_feature_names) == len(rf_importances)

assert np.isclose(
    rf_importances.sum(),
    1.0
)

rf_feature_importance_df = pd.DataFrame({
    "Transformed_Feature": rf_feature_names,
    "MDI_Importance": rf_importances
})

rf_feature_importance_df["Importance_Percentage"] = (
    rf_feature_importance_df["MDI_Importance"]
    * 100
)

rf_feature_importance_df = (
    rf_feature_importance_df
    .sort_values(
        "MDI_Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTRANSFORMED FEATURE IMPORTANCE (MDI)")
print("=" * 70)

display(
    rf_feature_importance_df.round(6)
)

# --------------------------------------------------
# 7. Aggregate MDI by original feature
# --------------------------------------------------

def get_original_feature(feature_name):

    clean_name = feature_name.split("__", 1)[-1]

    for feature in sorted(
        FEATURES,
        key=len,
        reverse=True
    ):

        if (
            clean_name == feature
            or clean_name.startswith(feature + "_")
        ):
            return feature

    raise ValueError(
        f"Unmapped transformed feature: {feature_name}"
    )

rf_feature_importance_df["Original_Feature"] = (
    rf_feature_importance_df["Transformed_Feature"]
    .apply(get_original_feature)
)

rf_grouped_importance_df = (
    rf_feature_importance_df
    .groupby("Original_Feature", as_index=False)
    ["MDI_Importance"]
    .sum()
    .sort_values(
        "MDI_Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

rf_grouped_importance_df["Importance_Percentage"] = (
    rf_grouped_importance_df["MDI_Importance"]
    * 100
)

assert set(
    rf_grouped_importance_df["Original_Feature"]
) == set(FEATURES)

print("\nGROUPED ORIGINAL FEATURE IMPORTANCE (MDI)")
print("=" * 70)

display(
    rf_grouped_importance_df.round(6)
)

# --------------------------------------------------
# 8. Validation permutation importance
# --------------------------------------------------

# Use a reproducible validation sample to limit
# computational cost in Google Colab.

permutation_sample_size = min(
    30000,
    len(X_los_val)
)

rng = np.random.default_rng(42)

permutation_indices = rng.choice(
    len(X_los_val),
    size=permutation_sample_size,
    replace=False
)

X_rf_permutation = (
    X_los_val.iloc[permutation_indices]
    .copy()
)

y_rf_permutation = (
    np.asarray(y_los_val)[permutation_indices]
)

rf_permutation_baseline_mae = mean_absolute_error(
    y_rf_permutation,
    rf_los_pipeline.predict(X_rf_permutation)
)

print("\nVALIDATION PERMUTATION IMPORTANCE")
print("=" * 70)

print(
    "Validation sample size:",
    f"{permutation_sample_size:,}"
)

print(
    "Sample baseline MAE:",
    f"{rf_permutation_baseline_mae:.6f}"
)

print("Permutation repeats: 5")
print("Scoring metric: Negative MAE")

print(
    "\nCalculating permutation importance...",
    flush=True
)

start_time = time.perf_counter()

rf_permutation_result = permutation_importance(
    estimator=rf_los_pipeline,
    X=X_rf_permutation,
    y=y_rf_permutation,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=1
)

rf_permutation_time = (
    time.perf_counter() - start_time
)

rf_permutation_df = pd.DataFrame({
    "Feature": X_rf_permutation.columns,
    "Mean_MAE_Increase": (
        rf_permutation_result.importances_mean
    ),
    "Importance_Std": (
        rf_permutation_result.importances_std
    )
})

rf_permutation_df = (
    rf_permutation_df
    .sort_values(
        "Mean_MAE_Increase",
        ascending=False
    )
    .reset_index(drop=True)
)

display(rf_permutation_df.round(6))

print(
    f"\nPermutation importance time: "
    f"{rf_permutation_time:.4f} seconds"
)

print(
    "\nPositive importance means that shuffling "
    "the feature increased validation MAE."
)

print(
    "Near-zero importance means limited measured "
    "reliance on that feature."
)

print(
    "Negative importance means that shuffling "
    "the feature improved MAE on this sample."
)

# --------------------------------------------------
# 9. Prior visit count diagnostic
# --------------------------------------------------

rf_prior_visit_analysis = pd.DataFrame({
    "PRIOR_VISIT_COUNT": (
        X_los_val["PRIOR_VISIT_COUNT"].to_numpy()
    ),
    "Actual_LOS": actual,
    "LR_Predicted_LOS": lr_val_pred,
    "RF_Predicted_LOS": predicted,
    "RF_Absolute_Error": rf_los_absolute_errors
})

rf_prior_visit_summary = (
    rf_prior_visit_analysis
    .groupby("PRIOR_VISIT_COUNT")
    .agg(
        Records=("Actual_LOS", "size"),
        Mean_Actual_LOS=("Actual_LOS", "mean"),
        Mean_LR_Prediction=("LR_Predicted_LOS", "mean"),
        Mean_RF_Prediction=("RF_Predicted_LOS", "mean"),
        RF_MAE=("RF_Absolute_Error", "mean")
    )
    .reset_index()
)

print("\nPRIOR VISIT COUNT DIAGNOSTIC")
print("=" * 70)

display(
    rf_prior_visit_summary.round(4)
)

# --------------------------------------------------
# 10. Actual versus predicted scatterplot
# --------------------------------------------------

rng = np.random.default_rng(42)

sample_indices = rng.choice(
    len(actual),
    size=min(10000, len(actual)),
    replace=False
)

plot_min = min(actual.min(), predicted.min())
plot_max = max(actual.max(), predicted.max())

plt.figure(figsize=(9, 6))

plt.scatter(
    actual[sample_indices],
    predicted[sample_indices],
    alpha=0.15,
    s=8,
    label="Validation predictions"
)

plt.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    linestyle="--",
    label="Perfect prediction"
)

plt.xlabel("Actual LOS (Days)")
plt.ylabel("Predicted LOS (Days)")
plt.title("Random Forest: Actual vs Predicted LOS")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 11. Residual distribution
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.hist(
    rf_los_residuals,
    bins=60,
    edgecolor="white"
)

plt.axvline(
    0,
    linestyle="--",
    label="Zero residual"
)

plt.xlabel("Residual: Actual - Predicted (Days)")
plt.ylabel("Frequency")
plt.title("Random Forest: Residual Distribution")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 12. Prediction distribution comparison
# --------------------------------------------------

common_bins = np.linspace(
    min(actual.min(), predicted.min(), lr_val_pred.min()),
    max(actual.max(), predicted.max(), lr_val_pred.max()),
    61
)

plt.figure(figsize=(10, 5))

plt.hist(
    actual,
    bins=common_bins,
    density=True,
    alpha=0.4,
    label="Actual LOS"
)

plt.hist(
    lr_val_pred,
    bins=common_bins,
    density=True,
    alpha=0.4,
    label="Linear Regression"
)

plt.hist(
    predicted,
    bins=common_bins,
    density=True,
    alpha=0.4,
    label="Random Forest"
)

plt.xlabel("LOS (Days)")
plt.ylabel("Density")
plt.title("Actual and Predicted LOS Distributions")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 13. Grouped MDI feature importance plot
# --------------------------------------------------

mdi_plot_df = (
    rf_grouped_importance_df
    .sort_values("Importance_Percentage")
)

plt.figure(figsize=(9, 5))

plt.barh(
    mdi_plot_df["Original_Feature"],
    mdi_plot_df["Importance_Percentage"]
)

plt.xlabel("Impurity-Based Importance (%)")
plt.title("Random Forest: Original Feature Importance")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 14. Permutation importance plot
# --------------------------------------------------

permutation_plot_df = (
    rf_permutation_df
    .sort_values("Mean_MAE_Increase")
)

plt.figure(figsize=(9, 5))

plt.barh(
    permutation_plot_df["Feature"],
    permutation_plot_df["Mean_MAE_Increase"],
    xerr=permutation_plot_df["Importance_Std"],
    capsize=3
)

plt.axvline(0, linestyle="--")

plt.xlabel("Increase in MAE (Days)")
plt.title("Random Forest: Validation Permutation Importance")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 15. Prior visit count relationship plot
# --------------------------------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    rf_prior_visit_summary["PRIOR_VISIT_COUNT"],
    rf_prior_visit_summary["Mean_Actual_LOS"],
    marker="o",
    label="Mean Actual LOS"
)

plt.plot(
    rf_prior_visit_summary["PRIOR_VISIT_COUNT"],
    rf_prior_visit_summary["Mean_LR_Prediction"],
    marker="o",
    label="Linear Regression"
)

plt.plot(
    rf_prior_visit_summary["PRIOR_VISIT_COUNT"],
    rf_prior_visit_summary["Mean_RF_Prediction"],
    marker="o",
    label="Random Forest"
)

plt.xlabel("Prior Visit Count")
plt.ylabel("Mean LOS (Days)")
plt.title("LOS Relationship with Prior Visit Count")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 16. Completion summary
# --------------------------------------------------

print("\nDIAGNOSTICS COMPLETION SUMMARY")
print("=" * 70)

print("Residual analysis: COMPLETED")
print("Error tolerance comparison: COMPLETED")
print("LOS range comparison: COMPLETED")
print("Transformed feature importance: COMPLETED")
print("Grouped feature importance: COMPLETED")
print("Validation permutation importance: COMPLETED")
print("Prior visit count diagnostic: COMPLETED")
print("Diagnostic visualization: COMPLETED")

print("\nModel has not been retrained.")
print("Test dataset has NOT been evaluated.")
print("Random Forest diagnostics completed.")

In [ ]:
# CELL 33: XGBoost Regressor - LOS Prediction

import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

print("XGBOOST REGRESSOR - LOS PREDICTION")
print("=" * 70)

# --------------------------------------------------
# 1. Verify training and validation inputs
# --------------------------------------------------

assert len(X_los_train) == 539559
assert len(X_los_val) == 115592

assert len(X_los_train) == len(y_los_train)
assert len(X_los_val) == len(y_los_val)

assert list(X_los_train.columns) == list(FEATURES)
assert list(X_los_val.columns) == list(FEATURES)

assert "PATIENT_KEY" not in X_los_train.columns
assert "TARGET_LOS_DAYS" not in X_los_train.columns

assert "Linear Regression" in los_models
assert "Random Forest Regressor" in los_models

print("XGBoost version:", xgb.__version__)

print(
    "Training records:",
    f"{len(X_los_train):,}"
)

print(
    "Validation records:",
    f"{len(X_los_val):,}"
)

print(
    "Original features:",
    X_los_train.shape[1]
)

# --------------------------------------------------
# 2. Define XGBoost configuration
# --------------------------------------------------

xgb_regressor_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 50,
    "tree_method": "hist",
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0
}

print("\nXGBOOST CONFIGURATION")
print("=" * 70)

for parameter, value in xgb_regressor_params.items():
    print(f"{parameter}: {value}")

# --------------------------------------------------
# 3. Build preprocessing/model pipeline
# --------------------------------------------------

xgb_los_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "model",
            xgb.XGBRegressor(
                **xgb_regressor_params
            )
        )
    ]
)

print("\nPipeline created successfully.")

# --------------------------------------------------
# 4. Train XGBoost Regressor
# --------------------------------------------------

print("\nTraining XGBoost Regressor...")
print("Please wait for training to complete.", flush=True)

start_time = time.perf_counter()

xgb_los_pipeline.fit(
    X_los_train,
    y_los_train
)

xgb_los_training_time = (
    time.perf_counter() - start_time
)

print("\nTraining completed.")

print(
    f"Training time: "
    f"{xgb_los_training_time:.4f} seconds"
)

# --------------------------------------------------
# 5. Generate validation predictions
# --------------------------------------------------

start_time = time.perf_counter()

xgb_los_val_pred = xgb_los_pipeline.predict(
    X_los_val
)

xgb_los_prediction_time = (
    time.perf_counter() - start_time
)

assert len(xgb_los_val_pred) == len(y_los_val)
assert np.isfinite(xgb_los_val_pred).all()

# --------------------------------------------------
# 6. Calculate validation metrics
# --------------------------------------------------

xgb_los_mae = mean_absolute_error(
    y_los_val,
    xgb_los_val_pred
)

xgb_los_rmse = root_mean_squared_error(
    y_los_val,
    xgb_los_val_pred
)

xgb_los_r2 = r2_score(
    y_los_val,
    xgb_los_val_pred
)

xgb_los_results = {
    "Model": "XGBoost Regressor",
    "Validation MAE": xgb_los_mae,
    "Validation RMSE": xgb_los_rmse,
    "Validation R2": xgb_los_r2,
    "Training Time (s)": xgb_los_training_time,
    "Prediction Time (s)": xgb_los_prediction_time
}

print("\nXGBOOST VALIDATION RESULTS")
print("=" * 70)

display(
    pd.DataFrame([xgb_los_results]).round(6)
)

# --------------------------------------------------
# 7. Calculate performance improvements
# --------------------------------------------------

comparison_models = [
    (
        "Median Baseline",
        median_baseline_mae,
        None
    ),
    (
        "Mean Baseline",
        None,
        mean_baseline_rmse
    ),
    (
        "Linear Regression",
        lr_mae,
        lr_rmse
    ),
    (
        "Random Forest Regressor",
        rf_los_mae,
        rf_los_rmse
    )
]

xgb_improvement_results = []

for model_name, reference_mae, reference_rmse in comparison_models:

    mae_improvement = (
        (reference_mae - xgb_los_mae)
        / reference_mae * 100
        if reference_mae is not None
        else np.nan
    )

    rmse_improvement = (
        (reference_rmse - xgb_los_rmse)
        / reference_rmse * 100
        if reference_rmse is not None
        else np.nan
    )

    xgb_improvement_results.append({
        "Reference Model": model_name,
        "MAE Improvement (%)": mae_improvement,
        "RMSE Improvement (%)": rmse_improvement
    })

xgb_improvement_df = pd.DataFrame(
    xgb_improvement_results
)

print("\nPERFORMANCE IMPROVEMENTS")
print("=" * 70)

display(xgb_improvement_df.round(6))

print(
    "Positive improvement indicates lower error."
)

print(
    "Negative improvement indicates higher error."
)

# --------------------------------------------------
# 8. Register fitted model and results
# --------------------------------------------------

los_models["XGBoost Regressor"] = (
    xgb_los_pipeline
)

los_validation_predictions[
    "XGBoost Regressor"
] = xgb_los_val_pred

# Prevent duplicate results if rerun

los_model_results = [
    result
    for result in los_model_results
    if result["Model"] != "XGBoost Regressor"
]

los_model_results.append(xgb_los_results)

# --------------------------------------------------
# 9. Updated regression model leaderboard
# --------------------------------------------------

los_leaderboard = (
    pd.DataFrame(los_model_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

assert len(los_leaderboard) == 3

print("\nREGRESSION MODEL LEADERBOARD")
print("=" * 70)

display(los_leaderboard.round(6))

# --------------------------------------------------
# 10. Prediction sanity checks
# --------------------------------------------------

print("\nPREDICTION SANITY CHECK")
print("=" * 70)

print(
    f"Minimum prediction: "
    f"{xgb_los_val_pred.min():.4f}"
)

print(
    f"Maximum prediction: "
    f"{xgb_los_val_pred.max():.4f}"
)

print(
    f"Mean prediction: "
    f"{xgb_los_val_pred.mean():.4f}"
)

print(
    f"Prediction std: "
    f"{xgb_los_val_pred.std():.4f}"
)

xgb_negative_predictions = np.sum(
    xgb_los_val_pred < 0
)

xgb_nonfinite_predictions = np.sum(
    ~np.isfinite(xgb_los_val_pred)
)

print(
    "Negative predictions:",
    f"{xgb_negative_predictions:,}"
)

print(
    "Non-finite predictions:",
    f"{xgb_nonfinite_predictions:,}"
)

if xgb_negative_predictions > 0:

    print(
        "WARNING: Negative LOS predictions detected."
    )

    print(
        "Predictions have not been clipped."
    )

# --------------------------------------------------
# 11. Verify trained booster structure
# --------------------------------------------------

xgb_los_estimator = (
    xgb_los_pipeline.named_steps["model"]
)

xgb_los_booster = (
    xgb_los_estimator.get_booster()
)

xgb_boosting_rounds = (
    xgb_los_booster.num_boosted_rounds()
)

xgb_tree_count = len(
    xgb_los_booster.get_dump()
)

print("\nTRAINED BOOSTER STRUCTURE")
print("=" * 70)

print(
    "Boosting rounds:",
    xgb_boosting_rounds
)

print(
    "Number of trees:",
    xgb_tree_count
)

print(
    "Transformed input features:",
    xgb_los_booster.num_features()
)

assert xgb_boosting_rounds == 300
assert xgb_tree_count == 300
assert xgb_los_booster.num_features() == 10

# --------------------------------------------------
# 12. Preliminary proposal target assessment
# --------------------------------------------------

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 70)

print("Proposed LOS MAE: below 1 day")

print(
    f"Actual validation MAE: "
    f"{xgb_los_mae:.4f} days"
)

print(
    "Target status:",
    "ACHIEVED"
    if xgb_los_mae < 1
    else "NOT ACHIEVED"
)

# --------------------------------------------------
# 13. Completion summary
# --------------------------------------------------

print("\nTRAINING COMPLETION SUMMARY")
print("=" * 70)

print("XGBoost training: COMPLETED")
print("Validation evaluation: COMPLETED")
print("Model registry: UPDATED")
print("Prediction finiteness check: PASSED")
print("Booster structure checks: PASSED")

print(
    "Negative prediction check:",
    "PASSED"
    if xgb_negative_predictions == 0
    else "WARNING"
)

print(
    "Registered LOS models:",
    len(los_models)
)

print("\nFitted pipeline retained in memory.")
print("Test dataset has NOT been evaluated.")
print("All three LOS regression models trained.")


In [ ]:
# CELL 34: XGBoost Regressor Diagnostics
# and Feature Importance Analysis

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

print("XGBOOST REGRESSOR DIAGNOSTICS")
print("=" * 70)

# --------------------------------------------------
# 1. Verify fitted models and validation predictions
# --------------------------------------------------

assert "XGBoost Regressor" in los_models
assert "Random Forest Regressor" in los_models
assert "Linear Regression" in los_models

actual = np.asarray(y_los_val)
predicted = np.asarray(xgb_los_val_pred)

assert len(predicted) == len(actual)
assert np.isfinite(predicted).all()

xgb_los_residuals = actual - predicted
xgb_los_absolute_errors = np.abs(xgb_los_residuals)

# Positive residual = underprediction
# Negative residual = overprediction

# --------------------------------------------------
# 2. Residual summary
# --------------------------------------------------

xgb_residual_summary = pd.DataFrame({
    "Statistic": [
        "Mean Residual",
        "Median Residual",
        "Residual Standard Deviation",
        "Minimum Residual",
        "Maximum Residual",
        "Mean Absolute Error",
        "Median Absolute Error"
    ],
    "Value (Days)": [
        np.mean(xgb_los_residuals),
        np.median(xgb_los_residuals),
        np.std(xgb_los_residuals),
        np.min(xgb_los_residuals),
        np.max(xgb_los_residuals),
        np.mean(xgb_los_absolute_errors),
        np.median(xgb_los_absolute_errors)
    ]
})

print("\nRESIDUAL SUMMARY")
print("=" * 70)

display(xgb_residual_summary.round(4))

# --------------------------------------------------
# 3. Error tolerance comparison
# --------------------------------------------------

tolerances = [1, 3, 7, 14, 30, 60]

xgb_tolerance_results = []

for tolerance in tolerances:

    lr_count = np.sum(
        lr_absolute_errors <= tolerance
    )

    rf_count = np.sum(
        rf_los_absolute_errors <= tolerance
    )

    xgb_count = np.sum(
        xgb_los_absolute_errors <= tolerance
    )

    lr_pct = lr_count / len(actual) * 100
    rf_pct = rf_count / len(actual) * 100
    xgb_pct = xgb_count / len(actual) * 100

    xgb_tolerance_results.append({
        "Error Tolerance (Days)": tolerance,
        "LR Within Tolerance": lr_count,
        "RF Within Tolerance": rf_count,
        "XGB Within Tolerance": xgb_count,
        "LR Percentage": lr_pct,
        "RF Percentage": rf_pct,
        "XGB Percentage": xgb_pct,
        "XGB vs RF (pp)": xgb_pct - rf_pct
    })

xgb_tolerance_df = pd.DataFrame(
    xgb_tolerance_results
)

print("\nPREDICTION ERROR TOLERANCE COMPARISON")
print("=" * 70)

display(xgb_tolerance_df.round(4))

# --------------------------------------------------
# 4. Prediction direction
# --------------------------------------------------

xgb_prediction_direction_df = pd.DataFrame({
    "Prediction Direction": [
        "Underprediction",
        "Overprediction",
        "Exact Prediction"
    ],
    "Count": [
        np.sum(xgb_los_residuals > 0),
        np.sum(xgb_los_residuals < 0),
        np.sum(xgb_los_residuals == 0)
    ]
})

xgb_prediction_direction_df["Percentage"] = (
    xgb_prediction_direction_df["Count"]
    / len(actual)
    * 100
)

print("\nPREDICTION DIRECTION")
print("=" * 70)

display(
    xgb_prediction_direction_df.round(4)
)

# --------------------------------------------------
# 5. Error analysis by actual LOS range
# --------------------------------------------------

xgb_diagnostics_df = pd.DataFrame({
    "Actual_LOS": actual,
    "Predicted_LOS": predicted,
    "Residual": xgb_los_residuals,
    "Absolute_Error": xgb_los_absolute_errors
})

xgb_diagnostics_df["LOS_Range"] = pd.cut(
    xgb_diagnostics_df["Actual_LOS"],
    bins=[0, 7, 30, 60, 90, 180, np.inf],
    labels=[
        "0-7 days",
        ">7-30 days",
        ">30-60 days",
        ">60-90 days",
        ">90-180 days",
        ">180 days"
    ],
    include_lowest=True
)

xgb_range_analysis = (
    xgb_diagnostics_df
    .groupby("LOS_Range", observed=True)
    .agg(
        Records=("Actual_LOS", "size"),
        Mean_Actual_LOS=("Actual_LOS", "mean"),
        Mean_Predicted_LOS=("Predicted_LOS", "mean"),
        MAE=("Absolute_Error", "mean"),
        Mean_Residual=("Residual", "mean")
    )
    .reset_index()
)

print("\nXGBOOST ERROR ANALYSIS BY LOS RANGE")
print("=" * 70)

display(xgb_range_analysis.round(4))

# Compare all three regression models

all_los_range_comparison = (
    lr_range_analysis[
        ["LOS_Range", "Records", "MAE"]
    ]
    .rename(columns={"MAE": "LR_MAE"})
    .merge(
        rf_range_analysis[
            ["LOS_Range", "MAE"]
        ].rename(columns={"MAE": "RF_MAE"}),
        on="LOS_Range",
        validate="one_to_one"
    )
    .merge(
        xgb_range_analysis[
            ["LOS_Range", "MAE"]
        ].rename(columns={"MAE": "XGB_MAE"}),
        on="LOS_Range",
        validate="one_to_one"
    )
)

all_los_range_comparison["XGB_vs_RF_Improvement"] = (
    all_los_range_comparison["RF_MAE"]
    - all_los_range_comparison["XGB_MAE"]
)

all_los_range_comparison["Best_Model"] = (
    all_los_range_comparison[
        ["LR_MAE", "RF_MAE", "XGB_MAE"]
    ]
    .idxmin(axis=1)
    .map({
        "LR_MAE": "Linear Regression",
        "RF_MAE": "Random Forest",
        "XGB_MAE": "XGBoost"
    })
)

print("\nALL REGRESSION MODELS - LOS RANGE COMPARISON")
print("=" * 70)

display(all_los_range_comparison.round(4))

# --------------------------------------------------
# 6. Extract normalized gain-based importance
# --------------------------------------------------

xgb_preprocessor = (
    xgb_los_pipeline.named_steps["preprocessor"]
)

xgb_estimator = (
    xgb_los_pipeline.named_steps["model"]
)

xgb_feature_names = (
    xgb_preprocessor.get_feature_names_out()
)

# Explicitly retrieve average gain for each feature.
# Features unused in splits receive zero gain.

xgb_gain_scores = (
    xgb_estimator.get_booster()
    .get_score(importance_type="gain")
)

xgb_raw_gains = np.array([
    xgb_gain_scores.get(f"f{i}", 0.0)
    for i in range(len(xgb_feature_names))
])

assert xgb_raw_gains.sum() > 0

xgb_gain_importances = (
    xgb_raw_gains / xgb_raw_gains.sum()
)

assert len(xgb_feature_names) == len(
    xgb_gain_importances
)

assert np.isclose(
    xgb_gain_importances.sum(),
    1.0
)

xgb_feature_importance_df = pd.DataFrame({
    "Transformed_Feature": xgb_feature_names,
    "Average_Gain": xgb_raw_gains,
    "Normalized_Gain": xgb_gain_importances
})

xgb_feature_importance_df["Importance_Percentage"] = (
    xgb_feature_importance_df["Normalized_Gain"]
    * 100
)

xgb_feature_importance_df = (
    xgb_feature_importance_df
    .sort_values(
        "Normalized_Gain",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTRANSFORMED FEATURE IMPORTANCE (GAIN)")
print("=" * 70)

display(
    xgb_feature_importance_df.round(6)
)

# --------------------------------------------------
# 7. Aggregate gain importance by original feature
# --------------------------------------------------

# Reuse the feature-mapping function from Cell 32.

xgb_feature_importance_df["Original_Feature"] = (
    xgb_feature_importance_df["Transformed_Feature"]
    .apply(get_original_feature)
)

xgb_grouped_importance_df = (
    xgb_feature_importance_df
    .groupby("Original_Feature", as_index=False)
    ["Normalized_Gain"]
    .sum()
    .sort_values(
        "Normalized_Gain",
        ascending=False
    )
    .reset_index(drop=True)
)

xgb_grouped_importance_df["Importance_Percentage"] = (
    xgb_grouped_importance_df["Normalized_Gain"]
    * 100
)

assert set(
    xgb_grouped_importance_df["Original_Feature"]
) == set(FEATURES)

print("\nGROUPED ORIGINAL FEATURE IMPORTANCE (GAIN)")
print("=" * 70)

display(
    xgb_grouped_importance_df.round(6)
)

# --------------------------------------------------
# 8. Validation permutation importance
# --------------------------------------------------

# Reuse exactly the same validation sample
# from the Random Forest diagnostics.

X_xgb_permutation = X_rf_permutation.copy()
y_xgb_permutation = y_rf_permutation.copy()

assert len(X_xgb_permutation) == 30000
assert len(y_xgb_permutation) == 30000

xgb_permutation_baseline_mae = (
    mean_absolute_error(
        y_xgb_permutation,
        xgb_los_pipeline.predict(
            X_xgb_permutation
        )
    )
)

print("\nVALIDATION PERMUTATION IMPORTANCE")
print("=" * 70)

print(
    "Validation sample size:",
    f"{len(X_xgb_permutation):,}"
)

print(
    "Sample baseline MAE:",
    f"{xgb_permutation_baseline_mae:.6f}"
)

print("Permutation repeats: 5")
print("Scoring metric: Negative MAE")
print("Same validation sample as Random Forest.")

print(
    "\nCalculating permutation importance...",
    flush=True
)

start_time = time.perf_counter()

xgb_permutation_result = permutation_importance(
    estimator=xgb_los_pipeline,
    X=X_xgb_permutation,
    y=y_xgb_permutation,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=1
)

xgb_permutation_time = (
    time.perf_counter() - start_time
)

xgb_permutation_df = pd.DataFrame({
    "Feature": X_xgb_permutation.columns,
    "Mean_MAE_Increase": (
        xgb_permutation_result.importances_mean
    ),
    "Importance_Std": (
        xgb_permutation_result.importances_std
    )
})

xgb_permutation_df = (
    xgb_permutation_df
    .sort_values(
        "Mean_MAE_Increase",
        ascending=False
    )
    .reset_index(drop=True)
)

display(xgb_permutation_df.round(6))

print(
    f"\nPermutation importance time: "
    f"{xgb_permutation_time:.4f} seconds"
)

# --------------------------------------------------
# 9. Compare RF and XGB permutation importance
# --------------------------------------------------

tree_permutation_comparison = (
    rf_permutation_df.rename(columns={
        "Mean_MAE_Increase": "RF_MAE_Increase",
        "Importance_Std": "RF_Std"
    })
    .merge(
        xgb_permutation_df.rename(columns={
            "Mean_MAE_Increase": "XGB_MAE_Increase",
            "Importance_Std": "XGB_Std"
        }),
        on="Feature",
        validate="one_to_one"
    )
)

tree_permutation_comparison["Average_MAE_Increase"] = (
    (
        tree_permutation_comparison["RF_MAE_Increase"]
        + tree_permutation_comparison["XGB_MAE_Increase"]
    ) / 2
)

tree_permutation_comparison = (
    tree_permutation_comparison
    .sort_values(
        "Average_MAE_Increase",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nRANDOM FOREST VS XGBOOST PERMUTATION IMPORTANCE")
print("=" * 70)

display(
    tree_permutation_comparison.round(6)
)

# --------------------------------------------------
# 10. Prior visit count diagnostic
# --------------------------------------------------

xgb_prior_visit_analysis = pd.DataFrame({
    "PRIOR_VISIT_COUNT": (
        X_los_val["PRIOR_VISIT_COUNT"].to_numpy()
    ),
    "Actual_LOS": actual,
    "LR_Predicted_LOS": lr_val_pred,
    "RF_Predicted_LOS": rf_los_val_pred,
    "XGB_Predicted_LOS": predicted,
    "XGB_Absolute_Error": xgb_los_absolute_errors
})

xgb_prior_visit_summary = (
    xgb_prior_visit_analysis
    .groupby("PRIOR_VISIT_COUNT")
    .agg(
        Records=("Actual_LOS", "size"),
        Mean_Actual_LOS=("Actual_LOS", "mean"),
        Mean_LR_Prediction=("LR_Predicted_LOS", "mean"),
        Mean_RF_Prediction=("RF_Predicted_LOS", "mean"),
        Mean_XGB_Prediction=("XGB_Predicted_LOS", "mean"),
        XGB_MAE=("XGB_Absolute_Error", "mean")
    )
    .reset_index()
)

print("\nPRIOR VISIT COUNT DIAGNOSTIC")
print("=" * 70)

display(
    xgb_prior_visit_summary.round(4)
)

# --------------------------------------------------
# 11. Admission month diagnostic
# --------------------------------------------------

# Investigate the calendar-related signal observed
# in the Random Forest diagnostics.

los_month_analysis = pd.DataFrame({
    "ADMISSION_MONTH": (
        X_los_val["ADMISSION_MONTH"].to_numpy()
    ),
    "Actual_LOS": actual,
    "LR_Predicted_LOS": lr_val_pred,
    "RF_Predicted_LOS": rf_los_val_pred,
    "XGB_Predicted_LOS": predicted,
    "Prior_Visit_Count": (
        X_los_val["PRIOR_VISIT_COUNT"].to_numpy()
    ),
    "LR_Absolute_Error": lr_absolute_errors,
    "RF_Absolute_Error": rf_los_absolute_errors,
    "XGB_Absolute_Error": xgb_los_absolute_errors
})

los_month_summary = (
    los_month_analysis
    .groupby("ADMISSION_MONTH")
    .agg(
        Records=("Actual_LOS", "size"),
        Mean_Actual_LOS=("Actual_LOS", "mean"),
        Std_Actual_LOS=("Actual_LOS", "std"),
        Max_Actual_LOS=("Actual_LOS", "max"),
        Mean_Prior_Visits=("Prior_Visit_Count", "mean"),
        Mean_LR_Prediction=("LR_Predicted_LOS", "mean"),
        Mean_RF_Prediction=("RF_Predicted_LOS", "mean"),
        Mean_XGB_Prediction=("XGB_Predicted_LOS", "mean"),
        LR_MAE=("LR_Absolute_Error", "mean"),
        RF_MAE=("RF_Absolute_Error", "mean"),
        XGB_MAE=("XGB_Absolute_Error", "mean")
    )
    .reset_index()
)

print("\nADMISSION MONTH DIAGNOSTIC")
print("=" * 70)

display(los_month_summary.round(4))

print(
    "\nAdmission month alone does not identify "
    "the admission year or prove observation-window bias."
)

# --------------------------------------------------
# 12. Actual versus predicted scatterplot
# --------------------------------------------------

rng = np.random.default_rng(42)

sample_indices = rng.choice(
    len(actual),
    size=min(10000, len(actual)),
    replace=False
)

plot_min = min(actual.min(), predicted.min())
plot_max = max(actual.max(), predicted.max())

plt.figure(figsize=(9, 6))

plt.scatter(
    actual[sample_indices],
    predicted[sample_indices],
    alpha=0.15,
    s=8,
    label="Validation predictions"
)

plt.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    linestyle="--",
    label="Perfect prediction"
)

plt.xlabel("Actual LOS (Days)")
plt.ylabel("Predicted LOS (Days)")
plt.title("XGBoost: Actual vs Predicted LOS")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 13. Residual distribution
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.hist(
    xgb_los_residuals,
    bins=60,
    edgecolor="white"
)

plt.axvline(
    0,
    linestyle="--",
    label="Zero residual"
)

plt.xlabel("Residual: Actual - Predicted (Days)")
plt.ylabel("Frequency")
plt.title("XGBoost: Residual Distribution")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 14. All-model prediction distributions
# --------------------------------------------------

common_bins = np.linspace(
    min(
        actual.min(),
        lr_val_pred.min(),
        rf_los_val_pred.min(),
        predicted.min()
    ),
    max(
        actual.max(),
        lr_val_pred.max(),
        rf_los_val_pred.max(),
        predicted.max()
    ),
    61
)

plt.figure(figsize=(10, 5))

for values, label in [
    (actual, "Actual LOS"),
    (lr_val_pred, "Linear Regression"),
    (rf_los_val_pred, "Random Forest"),
    (predicted, "XGBoost")
]:

    plt.hist(
        values,
        bins=common_bins,
        density=True,
        alpha=0.35,
        label=label
    )

plt.xlabel("LOS (Days)")
plt.ylabel("Density")
plt.title("Actual and Predicted LOS: All Models")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 15. Grouped gain importance plot
# --------------------------------------------------

gain_plot_df = (
    xgb_grouped_importance_df
    .sort_values("Importance_Percentage")
)

plt.figure(figsize=(9, 5))

plt.barh(
    gain_plot_df["Original_Feature"],
    gain_plot_df["Importance_Percentage"]
)

plt.xlabel("Normalized Gain Importance (%)")
plt.title("XGBoost: Original Feature Importance")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 16. RF versus XGB permutation importance
# --------------------------------------------------

comparison_plot_df = (
    tree_permutation_comparison
    .sort_values("Average_MAE_Increase")
)

positions = np.arange(
    len(comparison_plot_df)
)

bar_height = 0.35

plt.figure(figsize=(10, 6))

plt.barh(
    positions - bar_height / 2,
    comparison_plot_df["RF_MAE_Increase"],
    height=bar_height,
    xerr=comparison_plot_df["RF_Std"],
    capsize=3,
    label="Random Forest"
)

plt.barh(
    positions + bar_height / 2,
    comparison_plot_df["XGB_MAE_Increase"],
    height=bar_height,
    xerr=comparison_plot_df["XGB_Std"],
    capsize=3,
    label="XGBoost"
)

plt.yticks(
    positions,
    comparison_plot_df["Feature"]
)

plt.axvline(0, linestyle="--")

plt.xlabel("Increase in MAE (Days)")
plt.title("Validation Permutation Importance Comparison")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 17. Prior visit count relationship
# --------------------------------------------------

plt.figure(figsize=(10, 5))

for column, label in [
    ("Mean_Actual_LOS", "Mean Actual LOS"),
    ("Mean_LR_Prediction", "Linear Regression"),
    ("Mean_RF_Prediction", "Random Forest"),
    ("Mean_XGB_Prediction", "XGBoost")
]:

    plt.plot(
        xgb_prior_visit_summary["PRIOR_VISIT_COUNT"],
        xgb_prior_visit_summary[column],
        marker="o",
        label=label
    )

plt.xlabel("Prior Visit Count")
plt.ylabel("Mean LOS (Days)")
plt.title("LOS Relationship with Prior Visit Count")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 18. Admission month relationship
# --------------------------------------------------

plt.figure(figsize=(10, 5))

for column, label in [
    ("Mean_Actual_LOS", "Mean Actual LOS"),
    ("Mean_LR_Prediction", "Linear Regression"),
    ("Mean_RF_Prediction", "Random Forest"),
    ("Mean_XGB_Prediction", "XGBoost")
]:

    plt.plot(
        los_month_summary["ADMISSION_MONTH"],
        los_month_summary[column],
        marker="o",
        label=label
    )

plt.xticks(
    range(1, 13),
    [
        "Jan", "Feb", "Mar", "Apr",
        "May", "Jun", "Jul", "Aug",
        "Sep", "Oct", "Nov", "Dec"
    ]
)

plt.xlabel("Admission Month")
plt.ylabel("Mean LOS (Days)")
plt.title("LOS Relationship with Admission Month")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 19. Completion summary
# --------------------------------------------------

print("\nDIAGNOSTICS COMPLETION SUMMARY")
print("=" * 70)

print("Residual analysis: COMPLETED")
print("Error tolerance comparison: COMPLETED")
print("LOS range comparison: COMPLETED")
print("Gain-based feature importance: COMPLETED")
print("Grouped feature importance: COMPLETED")
print("Validation permutation importance: COMPLETED")
print("Tree model importance comparison: COMPLETED")
print("Prior visit count diagnostic: COMPLETED")
print("Admission month diagnostic: COMPLETED")
print("Diagnostic visualization: COMPLETED")

print("\nNo models have been retrained.")
print("Test dataset has NOT been evaluated.")
print("XGBoost diagnostics completed.")

In [ ]:
# CELL 35: Comprehensive LOS Regression Model Comparison

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

print("COMPREHENSIVE LOS REGRESSION MODEL COMPARISON")
print("=" * 75)

# --------------------------------------------------
# 1. Verify model registry and validation inputs
# --------------------------------------------------

expected_models = [
    "Linear Regression",
    "Random Forest Regressor",
    "XGBoost Regressor"
]

assert set(los_models.keys()) == set(expected_models)

assert set(
    los_validation_predictions.keys()
) == set(expected_models)

actual = np.asarray(
    y_los_val,
    dtype=np.float64
)

assert len(actual) == 115592
assert np.isfinite(actual).all()

for model_name in expected_models:

    predictions = np.asarray(
        los_validation_predictions[model_name]
    )

    assert len(predictions) == len(actual)
    assert np.isfinite(predictions).all()

print("Registered regression models:", len(los_models))
print("Validation records:", f"{len(actual):,}")
print("Model registry verification: PASSED")

# --------------------------------------------------
# 2. Construct baseline predictions
# --------------------------------------------------

baseline_mean_predictions = np.full(
    len(actual),
    np.mean(y_los_train),
    dtype=np.float64
)

baseline_median_predictions = np.full(
    len(actual),
    np.median(y_los_train),
    dtype=np.float64
)

all_regression_predictions = {
    "Training Mean Baseline": baseline_mean_predictions,
    "Training Median Baseline": baseline_median_predictions
}

for model_name in expected_models:

    all_regression_predictions[model_name] = (
        np.asarray(
            los_validation_predictions[model_name],
            dtype=np.float64
        )
    )

# --------------------------------------------------
# 3. Unified validation metrics
# --------------------------------------------------

regression_comparison_results = []

for model_name, predictions in (
    all_regression_predictions.items()
):

    residuals = actual - predictions
    absolute_errors = np.abs(residuals)

    regression_comparison_results.append({
        "Model": model_name,
        "Validation MAE": mean_absolute_error(
            actual, predictions
        ),
        "Validation RMSE": root_mean_squared_error(
            actual, predictions
        ),
        "Validation R2": r2_score(
            actual, predictions
        ),
        "Median Absolute Error": np.median(
            absolute_errors
        ),
        "Mean Residual": np.mean(residuals),
        "Residual Std": np.std(residuals),
        "Prediction Mean": np.mean(predictions),
        "Prediction Std": np.std(predictions),
        "Negative Predictions": np.sum(
            predictions < 0
        ),
        "Nonfinite Predictions": np.sum(
            ~np.isfinite(predictions)
        )
    })

regression_comparison_df = (
    pd.DataFrame(regression_comparison_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

print("\nUNIFIED REGRESSION VALIDATION RESULTS")
print("=" * 75)

display(regression_comparison_df.round(6))

# --------------------------------------------------
# 4. Calculate baseline improvements
# --------------------------------------------------

reference_mae = mean_absolute_error(
    actual,
    baseline_median_predictions
)

reference_rmse = root_mean_squared_error(
    actual,
    baseline_mean_predictions
)

regression_comparison_df[
    "MAE Improvement (%)"
] = (
    (
        reference_mae
        - regression_comparison_df["Validation MAE"]
    )
    / reference_mae
    * 100
)

regression_comparison_df[
    "RMSE Improvement (%)"
] = (
    (
        reference_rmse
        - regression_comparison_df["Validation RMSE"]
    )
    / reference_rmse
    * 100
)

print("\nBASELINE IMPROVEMENT COMPARISON")
print("=" * 75)

display(
    regression_comparison_df[
        [
            "Model",
            "Validation MAE",
            "MAE Improvement (%)",
            "Validation RMSE",
            "RMSE Improvement (%)",
            "Validation R2"
        ]
    ].round(6)
)

# --------------------------------------------------
# 5. Prediction tolerance comparison
# --------------------------------------------------

comparison_tolerances = [
    1, 3, 7, 14, 30, 60
]

regression_tolerance_results = []

for model_name, predictions in (
    all_regression_predictions.items()
):

    absolute_errors = np.abs(
        actual - predictions
    )

    result = {"Model": model_name}

    for tolerance in comparison_tolerances:

        result[f"Within {tolerance} Day(s) (%)"] = (
            np.mean(absolute_errors <= tolerance)
            * 100
        )

    regression_tolerance_results.append(result)

regression_tolerance_df = pd.DataFrame(
    regression_tolerance_results
)

print("\nCOMPREHENSIVE ERROR TOLERANCE COMPARISON")
print("=" * 75)

display(regression_tolerance_df.round(4))

# --------------------------------------------------
# 6. Computational performance comparison
# --------------------------------------------------

regression_efficiency_df = (
    pd.DataFrame(los_model_results)[
        [
            "Model",
            "Training Time (s)",
            "Prediction Time (s)"
        ]
    ]
    .copy()
)

regression_efficiency_df[
    "Validation Records per Second"
] = (
    len(actual)
    / regression_efficiency_df["Prediction Time (s)"]
)

regression_efficiency_df[
    "Batch Time per Record (ms)"
] = (
    regression_efficiency_df["Prediction Time (s)"]
    / len(actual)
    * 1000
)

regression_efficiency_df = (
    regression_efficiency_df
    .sort_values("Training Time (s)")
    .reset_index(drop=True)
)

print("\nCOMPUTATIONAL PERFORMANCE COMPARISON")
print("=" * 75)

display(regression_efficiency_df.round(6))

print(
    "\nTiming results describe the observed Colab "
    "training and batch prediction environment."
)

print(
    "Batch time per record is not equivalent "
    "to individual FastAPI request latency."
)

# --------------------------------------------------
# 7. Random Forest versus XGBoost
# --------------------------------------------------

rf_predictions = np.asarray(
    los_validation_predictions[
        "Random Forest Regressor"
    ],
    dtype=np.float64
)

xgb_predictions = np.asarray(
    los_validation_predictions[
        "XGBoost Regressor"
    ],
    dtype=np.float64
)

rf_errors = np.abs(
    actual - rf_predictions
)

xgb_errors = np.abs(
    actual - xgb_predictions
)

# Positive difference favors Random Forest.

paired_error_difference = (
    xgb_errors - rf_errors
)

observed_mae_difference = (
    paired_error_difference.mean()
)

print("\nRANDOM FOREST VS XGBOOST")
print("=" * 75)

print(
    "RF validation MAE:",
    f"{rf_errors.mean():.6f}"
)

print(
    "XGB validation MAE:",
    f"{xgb_errors.mean():.6f}"
)

print(
    "Observed MAE difference (XGB - RF):",
    f"{observed_mae_difference:.6f} days"
)

print(
    "Equivalent difference:",
    f"{observed_mae_difference * 1440:.4f} minutes"
)

paired_comparison_df = pd.DataFrame({
    "Comparison": [
        "RF has lower absolute error",
        "XGB has lower absolute error",
        "Equal absolute error"
    ],
    "Records": [
        np.sum(rf_errors < xgb_errors),
        np.sum(xgb_errors < rf_errors),
        np.sum(rf_errors == xgb_errors)
    ]
})

paired_comparison_df["Percentage"] = (
    paired_comparison_df["Records"]
    / len(actual)
    * 100
)

display(paired_comparison_df.round(4))

# --------------------------------------------------
# 8. Verify patient alignment
# --------------------------------------------------

assert len(los_val) == len(actual)

assert los_val["PATIENT_KEY"].notna().all()

assert np.array_equal(
    los_val["TARGET_LOS_DAYS"].to_numpy(
        dtype=np.float64
    ),
    actual
), "Validation target alignment failed."

assert X_los_val.reset_index(drop=True).equals(
    los_val[list(FEATURES)].reset_index(drop=True)
), "Validation feature alignment failed."

validation_patient_keys = (
    los_val["PATIENT_KEY"].to_numpy()
)

assert len(validation_patient_keys) == len(actual)

print("\nPATIENT ALIGNMENT VERIFICATION")
print("=" * 75)

print("Validation feature alignment: PASSED")
print("Validation target alignment: PASSED")

print(
    "Unique validation patients:",
    f"{pd.Series(validation_patient_keys).nunique():,}"
)

# --------------------------------------------------
# 9. Patient-grouped paired bootstrap
# --------------------------------------------------

# Aggregate paired differences by patient.
# Resampling patients preserves within-patient
# clustering and paired model comparisons.
#
# The resulting statistic remains visit-weighted.

patient_bootstrap_data = pd.DataFrame({
    "PATIENT_KEY": validation_patient_keys,
    "Error_Difference": paired_error_difference
})

patient_bootstrap_summary = (
    patient_bootstrap_data
    .groupby("PATIENT_KEY", sort=False)
    .agg(
        Error_Difference_Sum=(
            "Error_Difference", "sum"
        ),
        Visit_Count=(
            "Error_Difference", "size"
        )
    )
    .reset_index(drop=True)
)

patient_error_sums = (
    patient_bootstrap_summary[
        "Error_Difference_Sum"
    ].to_numpy(dtype=np.float64)
)

patient_visit_counts = (
    patient_bootstrap_summary[
        "Visit_Count"
    ].to_numpy(dtype=np.int64)
)

n_validation_patients = len(
    patient_bootstrap_summary
)

assert n_validation_patients == 46854

assert patient_visit_counts.sum() == len(actual)

assert np.isclose(
    patient_error_sums.sum()
    / patient_visit_counts.sum(),
    observed_mae_difference
)

n_bootstrap_repeats = 2000

bootstrap_rng = np.random.default_rng(42)

bootstrap_mae_differences = np.empty(
    n_bootstrap_repeats,
    dtype=np.float64
)

print("\nPATIENT-GROUPED PAIRED BOOTSTRAP")
print("=" * 75)

print(
    "Unique patients:",
    f"{n_validation_patients:,}"
)

print("Bootstrap repetitions:", n_bootstrap_repeats)
print("Random seed: 42")
print("Confidence interval: 95%")
print("Statistic: XGB MAE - RF MAE")

print("\nRunning bootstrap...", flush=True)

bootstrap_start_time = time.perf_counter()

for iteration in range(n_bootstrap_repeats):

    sampled_patients = bootstrap_rng.integers(
        low=0,
        high=n_validation_patients,
        size=n_validation_patients
    )

    bootstrap_mae_differences[iteration] = (
        patient_error_sums[
            sampled_patients
        ].sum()
        /
        patient_visit_counts[
            sampled_patients
        ].sum()
    )

bootstrap_elapsed = (
    time.perf_counter() - bootstrap_start_time
)

assert np.isfinite(
    bootstrap_mae_differences
).all()

bootstrap_ci_lower, bootstrap_ci_upper = (
    np.percentile(
        bootstrap_mae_differences,
        [2.5, 97.5]
    )
)

bootstrap_comparison_df = pd.DataFrame([{
    "Comparison": "XGBoost MAE - Random Forest MAE",
    "Observed Difference": observed_mae_difference,
    "Bootstrap Mean": bootstrap_mae_differences.mean(),
    "Bootstrap Std": bootstrap_mae_differences.std(),
    "95% CI Lower": bootstrap_ci_lower,
    "95% CI Upper": bootstrap_ci_upper,
    "Bootstrap Repeats": n_bootstrap_repeats,
    "Validation Patients": n_validation_patients
}])

display(bootstrap_comparison_df.round(6))

print(
    "Bootstrap execution time:",
    f"{bootstrap_elapsed:.4f} seconds"
)

if bootstrap_ci_lower > 0:

    bootstrap_interpretation = (
        "The interval favors Random Forest."
    )

elif bootstrap_ci_upper < 0:

    bootstrap_interpretation = (
        "The interval favors XGBoost."
    )

else:

    bootstrap_interpretation = (
        "The interval includes zero. "
        "The observed MAE difference is not clearly "
        "distinguishable from zero under this "
        "bootstrap analysis."
    )

print("\nBootstrap interpretation:")
print(bootstrap_interpretation)

print(
    "\nThis interval reflects validation-patient "
    "sampling uncertainty conditional on the "
    "already fitted models."
)

print(
    "It does not include model retraining "
    "uncertainty or future temporal drift."
)

# --------------------------------------------------
# 10. Proposal target assessment
# --------------------------------------------------

los_proposal_target_mae = 1.0

los_proposal_assessment_df = (
    regression_comparison_df[
        regression_comparison_df["Model"].isin(
            expected_models
        )
    ][
        ["Model", "Validation MAE"]
    ]
    .copy()
)

los_proposal_assessment_df[
    "Proposal MAE Boundary"
] = los_proposal_target_mae

los_proposal_assessment_df[
    "MAE Above Boundary (Days)"
] = (
    los_proposal_assessment_df["Validation MAE"]
    - los_proposal_target_mae
)

los_proposal_assessment_df[
    "MAE / Proposal Boundary"
] = (
    los_proposal_assessment_df["Validation MAE"]
    / los_proposal_target_mae
)

los_proposal_assessment_df["Target Status"] = (
    np.where(
        los_proposal_assessment_df["Validation MAE"]
        < los_proposal_target_mae,
        "ACHIEVED",
        "NOT ACHIEVED"
    )
)

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 75)

display(
    los_proposal_assessment_df.round(6)
)

# --------------------------------------------------
# 11. Provisional model ranking
# --------------------------------------------------

final_regression_validation_ranking = (
    regression_comparison_df[
        regression_comparison_df["Model"].isin(
            expected_models
        )
    ][
        [
            "Model",
            "Validation MAE",
            "Validation RMSE",
            "Validation R2",
            "MAE Improvement (%)",
            "RMSE Improvement (%)"
        ]
    ]
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

final_regression_validation_ranking.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(final_regression_validation_ranking) + 1
    )
)

provisional_los_leader = (
    final_regression_validation_ranking
    .iloc[0]["Model"]
)

print("\nFINAL VALIDATION RANKING")
print("=" * 75)

display(
    final_regression_validation_ranking.round(6)
)

print(
    "Provisional MAE-based leader:",
    provisional_los_leader
)

print(
    "Final model selection is reserved for Cell 36."
)

# --------------------------------------------------
# 12. Visualization: MAE comparison
# --------------------------------------------------

plot_df = (
    regression_comparison_df
    .sort_values("Validation MAE", ascending=False)
)

plt.figure(figsize=(10, 5))

bars = plt.barh(
    plot_df["Model"],
    plot_df["Validation MAE"]
)

plt.bar_label(
    bars,
    fmt="%.4f",
    padding=4
)

plt.axvline(
    los_proposal_target_mae,
    linestyle="--",
    label="Proposal boundary: 1 day"
)

plt.xlim(
    0,
    plot_df["Validation MAE"].max() * 1.15
)

plt.xlabel("Validation MAE (Days)")
plt.title("LOS Regression: Validation MAE Comparison")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 13. Visualization: RMSE comparison
# --------------------------------------------------

plot_df = (
    regression_comparison_df
    .sort_values("Validation RMSE", ascending=False)
)

plt.figure(figsize=(10, 5))

bars = plt.barh(
    plot_df["Model"],
    plot_df["Validation RMSE"]
)

plt.bar_label(
    bars,
    fmt="%.4f",
    padding=4
)

plt.xlim(
    0,
    plot_df["Validation RMSE"].max() * 1.15
)

plt.xlabel("Validation RMSE (Days)")
plt.title("LOS Regression: Validation RMSE Comparison")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 14. Visualization: R-squared comparison
# --------------------------------------------------

plot_df = (
    regression_comparison_df
    .sort_values("Validation R2")
)

plt.figure(figsize=(10, 5))

plt.barh(
    plot_df["Model"],
    plot_df["Validation R2"]
)

plt.axvline(0, linestyle="--")

plt.xlabel("Validation R-squared")
plt.title("LOS Regression: Explained Variance Comparison")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 15. Visualization: Error tolerance curves
# --------------------------------------------------

plt.figure(figsize=(10, 6))

for model_name, predictions in (
    all_regression_predictions.items()
):

    errors = np.abs(actual - predictions)

    tolerance_percentages = [
        np.mean(errors <= tolerance) * 100
        for tolerance in comparison_tolerances
    ]

    plt.plot(
        comparison_tolerances,
        tolerance_percentages,
        marker="o",
        label=model_name
    )

plt.xlabel("Absolute Error Tolerance (Days)")
plt.ylabel("Predictions Within Tolerance (%)")
plt.title("LOS Regression: Prediction Error Tolerance")
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 16. Visualization: Training efficiency
# --------------------------------------------------

plt.figure(figsize=(9, 5))

bars = plt.barh(
    regression_efficiency_df["Model"],
    regression_efficiency_df["Training Time (s)"]
)

plt.bar_label(
    bars,
    fmt="%.4f s",
    padding=4
)

plt.xlim(
    0,
    regression_efficiency_df[
        "Training Time (s)"
    ].max() * 1.2
)

plt.xlabel("Training Time (Seconds)")
plt.title("LOS Regression: Training Time Comparison")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 17. Visualization: Paired bootstrap distribution
# --------------------------------------------------

plt.figure(figsize=(10, 5))

plt.hist(
    bootstrap_mae_differences,
    bins=50,
    edgecolor="white"
)

plt.axvline(
    0,
    linestyle="--",
    label="No MAE difference"
)

plt.axvline(
    observed_mae_difference,
    label="Observed MAE difference"
)

plt.axvline(
    bootstrap_ci_lower,
    linestyle=":",
    label="95% CI bounds"
)

plt.axvline(
    bootstrap_ci_upper,
    linestyle=":"
)

plt.xlabel("XGBoost MAE - Random Forest MAE (Days)")
plt.ylabel("Bootstrap Frequency")
plt.title("Patient-Grouped Paired Bootstrap Comparison")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 18. Visualization: Accuracy versus training time
# --------------------------------------------------

accuracy_efficiency_df = (
    final_regression_validation_ranking
    .merge(
        regression_efficiency_df,
        on="Model",
        validate="one_to_one"
    )
)

plt.figure(figsize=(9, 5))

plt.scatter(
    accuracy_efficiency_df["Training Time (s)"],
    accuracy_efficiency_df["Validation MAE"],
    s=100
)

for _, row in accuracy_efficiency_df.iterrows():

    plt.annotate(
        row["Model"],
        (
            row["Training Time (s)"],
            row["Validation MAE"]
        ),
        xytext=(5, 8),
        textcoords="offset points"
    )

plt.xlabel("Training Time (Seconds)")
plt.ylabel("Validation MAE (Days)")
plt.title("LOS Regression: Accuracy-Efficiency Trade-off")
plt.margins(x=0.25, y=0.25)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 19. Completion summary
# --------------------------------------------------

print("\nCOMPREHENSIVE COMPARISON COMPLETION SUMMARY")
print("=" * 75)

print("Model registry verification: PASSED")
print("Unified validation comparison: COMPLETED")
print("Baseline improvement comparison: COMPLETED")
print("Error tolerance comparison: COMPLETED")
print("Computational comparison: COMPLETED")
print("Patient alignment verification: PASSED")
print("Patient-grouped paired bootstrap: COMPLETED")
print("Proposal target assessment: COMPLETED")
print("Validation ranking: COMPLETED")
print("Comparative visualization: COMPLETED")

print(
    "\nProvisional MAE-based leader:",
    provisional_los_leader
)

print(
    "Models meeting proposal MAE target:",
    (
        los_proposal_assessment_df["Target Status"]
        == "ACHIEVED"
    ).sum()
)

print("\nNo models have been retrained.")
print("No final model selection has been made.")
print("Test dataset has NOT been evaluated.")
print("Comprehensive regression comparison completed.")


In [ ]:
# CELL 36: Regression Model Selection and Readiness Assessment

import numpy as np
import pandas as pd

print("LOS REGRESSION MODEL SELECTION")
print("=" * 75)

# --------------------------------------------------
# 1. Verify required comparison results
# --------------------------------------------------

required_models = [
    "Linear Regression",
    "Random Forest Regressor",
    "XGBoost Regressor"
]

assert set(los_models.keys()) == set(required_models)

assert len(final_regression_validation_ranking) == 3

assert len(bootstrap_comparison_df) == 1

assert len(los_proposal_assessment_df) == 3

assert np.isfinite(
    final_regression_validation_ranking[
        [
            "Validation MAE",
            "Validation RMSE",
            "Validation R2"
        ]
    ].to_numpy()
).all()

print("Model registry verification: PASSED")
print("Validation ranking verification: PASSED")
print("Bootstrap results verification: PASSED")
print("Proposal assessment verification: PASSED")

# --------------------------------------------------
# 2. Formalize model selection policy
# --------------------------------------------------

los_selection_policy = {
    "primary_metric": "Validation MAE",
    "secondary_metrics": [
        "Validation RMSE",
        "Validation R2"
    ],
    "selection_dataset": "Validation",
    "selection_direction": "Minimize MAE",
    "bootstrap_method": "Patient-grouped paired bootstrap",
    "bootstrap_repetitions": int(n_bootstrap_repeats),
    "bootstrap_confidence_level": 0.95,
    "test_set_used_for_selection": False,
    "deployment_approval": False
}

print("\nMODEL SELECTION POLICY")
print("=" * 75)

for key, value in los_selection_policy.items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 3. Select primary LOS model
# --------------------------------------------------

los_selection_ranking = (
    final_regression_validation_ranking
    .sort_values(
        [
            "Validation MAE",
            "Validation RMSE",
            "Validation R2"
        ],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

selected_los_model_name = str(
    los_selection_ranking.iloc[0]["Model"]
)

selected_los_pipeline = los_models[
    selected_los_model_name
]

alternative_los_model_name = str(
    los_selection_ranking.iloc[1]["Model"]
)

alternative_los_pipeline = los_models[
    alternative_los_model_name
]

benchmark_los_model_name = "Linear Regression"

benchmark_los_pipeline = los_models[
    benchmark_los_model_name
]

selected_los_metrics = (
    los_selection_ranking.iloc[0].to_dict()
)

print("\nSELECTED PRIMARY LOS MODEL")
print("=" * 75)

print("Selected model:", selected_los_model_name)

print(
    "Validation MAE:",
    f"{selected_los_metrics['Validation MAE']:.6f}"
)

print(
    "Validation RMSE:",
    f"{selected_los_metrics['Validation RMSE']:.6f}"
)

print(
    "Validation R2:",
    f"{selected_los_metrics['Validation R2']:.6f}"
)

print(
    "MAE improvement over median baseline:",
    f"{selected_los_metrics['MAE Improvement (%)']:.6f}%"
)

print("Alternative model:", alternative_los_model_name)
print("Benchmark model:", benchmark_los_model_name)

# --------------------------------------------------
# 4. Bootstrap-based uncertainty assessment
# --------------------------------------------------

los_bootstrap_includes_zero = bool(
    bootstrap_ci_lower <= 0 <= bootstrap_ci_upper
)

los_bootstrap_summary = {
    "comparison": "XGBoost MAE - Random Forest MAE",
    "observed_difference_days": float(
        observed_mae_difference
    ),
    "observed_difference_minutes": float(
        observed_mae_difference * 1440
    ),
    "confidence_interval_lower": float(
        bootstrap_ci_lower
    ),
    "confidence_interval_upper": float(
        bootstrap_ci_upper
    ),
    "confidence_interval_includes_zero": (
        los_bootstrap_includes_zero
    ),
    "bootstrap_repetitions": int(
        n_bootstrap_repeats
    ),
    "validation_patients": int(
        n_validation_patients
    )
}

print("\nBOOTSTRAP UNCERTAINTY ASSESSMENT")
print("=" * 75)

for key, value in los_bootstrap_summary.items():
    print(f"{key}: {value}")

if los_bootstrap_includes_zero:

    los_statistical_conclusion = (
        "The paired bootstrap confidence interval "
        "includes zero. A clear MAE advantage "
        "between Random Forest and XGBoost "
        "has not been established."
    )

elif bootstrap_ci_lower > 0:

    los_statistical_conclusion = (
        "The paired bootstrap confidence interval "
        "favors Random Forest."
    )

else:

    los_statistical_conclusion = (
        "The paired bootstrap confidence interval "
        "favors XGBoost."
    )

print("\nStatistical conclusion:")
print(los_statistical_conclusion)

# --------------------------------------------------
# 5. Computational efficiency assessment
# --------------------------------------------------

efficiency_lookup = (
    regression_efficiency_df
    .set_index("Model")
)

rf_training_time = float(
    efficiency_lookup.loc[
        "Random Forest Regressor",
        "Training Time (s)"
    ]
)

xgb_training_time = float(
    efficiency_lookup.loc[
        "XGBoost Regressor",
        "Training Time (s)"
    ]
)

rf_prediction_time = float(
    efficiency_lookup.loc[
        "Random Forest Regressor",
        "Prediction Time (s)"
    ]
)

xgb_prediction_time = float(
    efficiency_lookup.loc[
        "XGBoost Regressor",
        "Prediction Time (s)"
    ]
)

xgb_training_time_reduction = (
    (rf_training_time - xgb_training_time)
    / rf_training_time * 100
)

xgb_prediction_time_reduction = (
    (rf_prediction_time - xgb_prediction_time)
    / rf_prediction_time * 100
)

print("\nCOMPUTATIONAL TRADE-OFF ASSESSMENT")
print("=" * 75)

print(
    "XGBoost training time reduction:",
    f"{xgb_training_time_reduction:.4f}%"
)

print(
    "XGBoost batch prediction time reduction:",
    f"{xgb_prediction_time_reduction:.4f}%"
)

print(
    "Production API latency and memory "
    "consumption have not been measured."
)

# --------------------------------------------------
# 6. Assign model roles
# --------------------------------------------------

los_model_roles = {
    selected_los_model_name: "PRIMARY_EXPERIMENTAL",
    alternative_los_model_name: "ALTERNATIVE",
    benchmark_los_model_name: "BENCHMARK"
}

los_model_role_df = (
    los_selection_ranking.copy()
)

los_model_role_df["Assigned Role"] = (
    los_model_role_df["Model"]
    .map(los_model_roles)
)

los_model_role_df["Clinical Deployment"] = (
    "NOT APPROVED"
)

print("\nFINAL MODEL ROLE ASSIGNMENT")
print("=" * 75)

display(
    los_model_role_df[
        [
            "Rank",
            "Model",
            "Assigned Role",
            "Validation MAE",
            "Validation RMSE",
            "Validation R2",
            "Clinical Deployment"
        ]
    ].round(6)
)

# --------------------------------------------------
# 7. Proposal target and readiness decision
# --------------------------------------------------

los_selected_mae = float(
    selected_los_metrics["Validation MAE"]
)

los_target_achieved = bool(
    los_selected_mae < los_proposal_target_mae
)

los_clinical_deployment_approved = False

los_test_evaluation_completed = False

los_readiness_status = (
    "EXPERIMENTAL_ONLY"
)

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 75)

print(
    "Required MAE:",
    f"Below {los_proposal_target_mae:.1f} day"
)

print(
    "Selected model MAE:",
    f"{los_selected_mae:.6f} days"
)

print(
    "Target achieved:",
    "YES" if los_target_achieved else "NO"
)

print(
    "MAE above proposal boundary:",
    f"{los_selected_mae - los_proposal_target_mae:.6f} days"
)

print("Model readiness:", los_readiness_status)

print(
    "Clinical deployment approved:",
    los_clinical_deployment_approved
)

# --------------------------------------------------
# 8. Selected model error tolerance
# --------------------------------------------------

selected_los_validation_predictions = np.asarray(
    los_validation_predictions[
        selected_los_model_name
    ],
    dtype=np.float64
)

assert len(
    selected_los_validation_predictions
) == len(y_los_val)

selected_los_absolute_errors = np.abs(
    np.asarray(y_los_val, dtype=np.float64)
    - selected_los_validation_predictions
)

selected_los_tolerance_df = pd.DataFrame([
    {
        "Tolerance (Days)": tolerance,
        "Records Within Tolerance": int(
            np.sum(
                selected_los_absolute_errors <= tolerance
            )
        ),
        "Percentage Within Tolerance": float(
            np.mean(
                selected_los_absolute_errors <= tolerance
            ) * 100
        )
    }
    for tolerance in [1, 3, 7, 14, 30, 60]
])

print("\nSELECTED MODEL ERROR TOLERANCE")
print("=" * 75)

display(
    selected_los_tolerance_df.round(6)
)

# --------------------------------------------------
# 9. Document readiness checklist
# --------------------------------------------------

los_readiness_df = pd.DataFrame([
    ("Corrected V2 dataset", "PASSED"),
    ("Patient-grouped splitting", "PASSED"),
    ("Patient overlap verification", "PASSED"),
    ("Training-only preprocessing", "PASSED"),
    ("Model training", "COMPLETED"),
    ("Validation evaluation", "COMPLETED"),
    ("Comparative diagnostics", "COMPLETED"),
    ("Patient-grouped bootstrap", "COMPLETED"),
    (
        "Proposal MAE target",
        "PASSED" if los_target_achieved else "FAILED"
    ),
    ("Clinical target validity", "NOT ESTABLISHED"),
    ("Temporal generalization", "NOT ESTABLISHED"),
    ("Held-out test evaluation", "PENDING"),
    ("Serialized model verification", "PENDING"),
    ("FastAPI integration testing", "PENDING"),
    ("Academic prototype", "CONDITIONAL"),
    ("Clinical deployment", "NOT APPROVED")
], columns=["Readiness Criterion", "Status"])

print("\nMODEL READINESS CHECKLIST")
print("=" * 75)

display(los_readiness_df)

# --------------------------------------------------
# 10. Document known limitations
# --------------------------------------------------

los_model_limitations = [
    (
        "The selected model does not achieve the "
        "proposal MAE target of below one day."
    ),
    (
        "Validation R2 is approximately 0.03, "
        "indicating limited explained variation."
    ),
    (
        "Predictions concentrate near the "
        "population mean LOS."
    ),
    (
        "Short visits tend to be overpredicted "
        "and long visits underpredicted."
    ),
    (
        "Admission month and prior visit count "
        "dominate validation permutation importance."
    ),
    (
        "The observed admission-month pattern "
        "may reflect observation-window selection, "
        "source-data characteristics or seasonality."
    ),
    (
        "The clinical interpretation of recorded "
        "visit duration as inpatient LOS "
        "has not been independently established."
    ),
    (
        "The original historical dataset-generation "
        "procedure could not be recovered."
    ),
    (
        "Excluding ongoing visits may introduce "
        "completed-visit selection bias."
    ),
    (
        "Patient-grouped random splitting does not "
        "establish future temporal generalization."
    ),
    (
        "Bootstrap uncertainty is conditional on "
        "the fitted models and validation population."
    ),
    (
        "Clinical variables such as diagnosis severity, "
        "comorbidities and procedures are not included "
        "in the current feature set."
    ),
    (
        "The model is not approved for clinical "
        "decision-making or operational bed allocation."
    )
]

print("\nDOCUMENTED MODEL LIMITATIONS")
print("=" * 75)

for number, limitation in enumerate(
    los_model_limitations,
    start=1
):
    print(f"{number}. {limitation}")

# --------------------------------------------------
# 11. Verify feature contract
# --------------------------------------------------

los_expected_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

assert list(FEATURES) == los_expected_features

assert list(X_los_train.columns) == los_expected_features

assert list(X_los_val.columns) == los_expected_features

assert list(
    selected_los_pipeline.feature_names_in_
) == los_expected_features

assert "PATIENT_KEY" not in los_expected_features

assert "TARGET_LOS_DAYS" not in los_expected_features

assert "VISIT_STATUS_AT_EXIT" not in los_expected_features

assert "LOS_DAYS" not in los_expected_features

print("\nSELECTED MODEL FEATURE CONTRACT")
print("=" * 75)

for number, feature in enumerate(
    los_expected_features,
    start=1
):
    print(f"{number}. {feature}")

print("Feature contract verification: PASSED")

# --------------------------------------------------
# 12. Create structured model selection metadata
# --------------------------------------------------

los_selection_metadata = {
    "project": "St. Hubert Medical Center BI",
    "task": "Length of Stay Prediction",
    "task_type": "regression",
    "dataset_version": "V2",
    "target": "TARGET_LOS_DAYS",
    "prediction_unit": "days",
    "selected_model": selected_los_model_name,
    "alternative_model": alternative_los_model_name,
    "benchmark_model": benchmark_los_model_name,
    "model_roles": los_model_roles,
    "features": list(los_expected_features),
    "numerical_features": list(NUM_COLS),
    "categorical_features": list(CAT_COLS),
    "preprocessing": {
        "numerical_imputation": "median",
        "numerical_scaling": "StandardScaler",
        "categorical_imputation": "most_frequent",
        "categorical_encoding": "OneHotEncoder",
        "unknown_categories": "ignore",
        "fit_dataset": "training"
    },
    "data_split": {
        "method": "patient-grouped random split",
        "group_column": "PATIENT_KEY",
        "training_rows": int(len(X_los_train)),
        "validation_rows": int(len(X_los_val)),
        "test_rows": int(len(X_los_test))
    },
    "validation_metrics": {
        "mae": float(
            selected_los_metrics["Validation MAE"]
        ),
        "rmse": float(
            selected_los_metrics["Validation RMSE"]
        ),
        "r2": float(
            selected_los_metrics["Validation R2"]
        ),
        "mae_improvement_percent": float(
            selected_los_metrics["MAE Improvement (%)"]
        ),
        "rmse_improvement_percent": float(
            selected_los_metrics["RMSE Improvement (%)"]
        )
    },
    "selection_policy": los_selection_policy,
    "bootstrap_comparison": los_bootstrap_summary,
    "statistical_conclusion": los_statistical_conclusion,
    "proposal_mae_boundary": float(
        los_proposal_target_mae
    ),
    "proposal_target_achieved": los_target_achieved,
    "readiness_status": los_readiness_status,
    "clinical_deployment_approved": False,
    "test_evaluation_completed": False,
    "limitations": los_model_limitations
}

# Verify metadata can be serialized to JSON.
# No file is written at this stage.

import json

json.dumps(
    los_selection_metadata,
    allow_nan=False
)

print("\nMODEL SELECTION METADATA")
print("=" * 75)

print("Metadata created successfully.")
print("JSON serialization verification: PASSED")

print(
    "Selected model:",
    los_selection_metadata["selected_model"]
)

print(
    "Dataset version:",
    los_selection_metadata["dataset_version"]
)

print(
    "Number of input features:",
    len(los_selection_metadata["features"])
)

print(
    "Documented limitations:",
    len(los_selection_metadata["limitations"])
)

# --------------------------------------------------
# 13. Formal selection rationale
# --------------------------------------------------

los_selection_rationale = (
    f"{selected_los_model_name} was selected as "
    "the primary experimental LOS regression model "
    "because it achieved the lowest validation MAE "
    "under the predefined ranking criterion. "
    f"Its validation MAE was {los_selected_mae:.4f} "
    "days, with only modest improvement over the "
    "training median baseline. "
    f"{los_statistical_conclusion} "
    "XGBoost remains an alternative due to its "
    "comparable accuracy and lower observed "
    "computational cost. Linear Regression is "
    "retained as a benchmark. None of the three "
    "models achieved the proposal accuracy target. "
    "The selected model is restricted to "
    "experimental academic use and is not "
    "approved for clinical decision-making."
)

los_selection_metadata[
    "selection_rationale"
] = los_selection_rationale

print("\nFORMAL MODEL SELECTION RATIONALE")
print("=" * 75)

print(los_selection_rationale)

# --------------------------------------------------
# 14. Completion summary
# --------------------------------------------------

los_model_selection_completed = True

print("\nREGRESSION SELECTION COMPLETION SUMMARY")
print("=" * 75)

print("Model selection policy: DOCUMENTED")
print("Primary LOS model: SELECTED")
print("Alternative LOS model: ASSIGNED")
print("Benchmark LOS model: ASSIGNED")
print("Bootstrap uncertainty: DOCUMENTED")
print("Computational trade-offs: DOCUMENTED")
print("Proposal target assessment: COMPLETED")
print("Readiness assessment: COMPLETED")
print("Feature contract verification: PASSED")
print("Model limitations: DOCUMENTED")
print("Selection metadata: CREATED")

print("\nSelected model:", selected_los_model_name)

print("Model readiness:", los_readiness_status)

print(
    "Clinical deployment approved:",
    los_clinical_deployment_approved
)

print("\nAll three fitted pipelines retained in memory.")
print("No models have been retrained.")
print("No model files have been exported.")
print("Test dataset has NOT been evaluated.")

print("\nLOS regression validation phase completed.")
print("Ready for classification training preparation.")


In [ ]:
# CELL 37: 30-Day Subsequent Visit Classification
# Training Preparation

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

from sklearn.calibration import calibration_curve

import xgboost as xgb
from xgboost import XGBClassifier

print("SUBSEQUENT VISIT CLASSIFICATION PREPARATION")
print("=" * 75)

# --------------------------------------------------
# 1. Verify regression phase completion
# --------------------------------------------------

assert los_model_selection_completed is True

assert len(los_models) == 3

assert selected_los_model_name == (
    "Random Forest Regressor"
)

assert los_test_evaluation_completed is False

print("Regression phase completion: VERIFIED")
print("Existing regression pipelines:", len(los_models))
print("XGBoost version:", xgb.__version__)

# --------------------------------------------------
# 2. Define classification task contract
# --------------------------------------------------

classification_target = (
    "SUBSEQUENT_VISIT_WITHIN_30D_PROXY"
)

classification_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

classification_excluded_features = [
    "PATIENT_KEY",
    "VISIT_STATUS_AT_EXIT",
    "LOS_DAYS",
    "TARGET_LOS_DAYS",
    classification_target
]

assert list(FEATURES) == classification_features

assert classification_target in proxy_df.columns

assert len(proxy_df) == 728679

assert proxy_df["PATIENT_KEY"].nunique() == 307470

assert int(proxy_df[classification_target].sum()) == 84897

print("\nCLASSIFICATION TASK CONTRACT")
print("=" * 75)

print("Dataset version: V2")
print("Task: Binary classification")
print("Target:", classification_target)
print("Prediction reference point: Admission")
print("Negative class: 0")
print("Positive class: 1")
print("Original features:", len(classification_features))

print(
    "Clinical interpretation: Subsequent visit proxy, "
    "not validated unplanned readmission."
)

print("Corrected dataset verification: PASSED")

# --------------------------------------------------
# 3. Verify existing classification splits
# --------------------------------------------------

classification_split_df = pd.DataFrame([
    {
        "Dataset": "Training",
        "Records": len(proxy_train),
        "Unique Patients": (
            proxy_train["PATIENT_KEY"].nunique()
        )
    },
    {
        "Dataset": "Validation",
        "Records": len(proxy_val),
        "Unique Patients": (
            proxy_val["PATIENT_KEY"].nunique()
        )
    },
    {
        "Dataset": "Testing",
        "Records": len(proxy_test),
        "Unique Patients": (
            proxy_test["PATIENT_KEY"].nunique()
        )
    }
])

classification_split_df["Record Percentage"] = (
    classification_split_df["Records"]
    / len(proxy_df) * 100
)

assert len(proxy_train) == 509976
assert len(proxy_val) == 109458
assert len(proxy_test) == 109245

assert (
    classification_split_df["Records"].sum()
    == len(proxy_df)
)

print("\nCLASSIFICATION DATA SPLITS")
print("=" * 75)

display(classification_split_df.round(4))

# --------------------------------------------------
# 4. Verify patient-grouped separation
# --------------------------------------------------

classification_train_patients = set(
    proxy_train["PATIENT_KEY"]
)

classification_val_patients = set(
    proxy_val["PATIENT_KEY"]
)

classification_test_patients = set(
    proxy_test["PATIENT_KEY"]
)

classification_overlap_df = pd.DataFrame([
    {
        "Comparison": "Training vs Validation",
        "Shared Patients": len(
            classification_train_patients
            & classification_val_patients
        )
    },
    {
        "Comparison": "Training vs Testing",
        "Shared Patients": len(
            classification_train_patients
            & classification_test_patients
        )
    },
    {
        "Comparison": "Validation vs Testing",
        "Shared Patients": len(
            classification_val_patients
            & classification_test_patients
        )
    }
])

assert (
    classification_overlap_df["Shared Patients"]
    == 0
).all()

assert (
    classification_train_patients
    | classification_val_patients
    | classification_test_patients
) == set(proxy_df["PATIENT_KEY"])

print("\nPATIENT LEAKAGE VERIFICATION")
print("=" * 75)

display(classification_overlap_df)

print("Patient overlap verification: PASSED")

# --------------------------------------------------
# 5. Verify feature matrices
# --------------------------------------------------

classification_feature_sets = {
    "Training": X_proxy_train,
    "Validation": X_proxy_val,
    "Testing": X_proxy_test
}

for split_name, feature_matrix in (
    classification_feature_sets.items()
):

    assert list(
        feature_matrix.columns
    ) == classification_features

    assert not any(
        column in feature_matrix.columns
        for column in classification_excluded_features
    )

    assert feature_matrix.shape[1] == 7

    print(
        f"{split_name} feature matrix:",
        feature_matrix.shape
    )

print("\nFEATURE CONTRACT")
print("=" * 75)

classification_feature_contract_df = pd.DataFrame([
    {
        "Feature": feature,
        "Type": (
            "Numerical"
            if feature in NUM_COLS
            else "Categorical"
        ),
        "Included": True
    }
    for feature in classification_features
])

display(classification_feature_contract_df)

print("Excluded features:")

for feature in classification_excluded_features:
    print("-", feature)

print("Feature leakage verification: PASSED")

# --------------------------------------------------
# 6. Verify training and validation alignment
# --------------------------------------------------

assert X_proxy_train.reset_index(
    drop=True
).equals(
    proxy_train[
        classification_features
    ].reset_index(drop=True)
)

assert X_proxy_val.reset_index(
    drop=True
).equals(
    proxy_val[
        classification_features
    ].reset_index(drop=True)
)

classification_y_train = np.asarray(y_proxy_train)

classification_y_val = np.asarray(y_proxy_val)

assert classification_y_train.ndim == 1
assert classification_y_val.ndim == 1

assert len(classification_y_train) == len(X_proxy_train)
assert len(classification_y_val) == len(X_proxy_val)

assert np.array_equal(
    classification_y_train,
    proxy_train[classification_target].to_numpy()
)

assert np.array_equal(
    classification_y_val,
    proxy_val[classification_target].to_numpy()
)

assert np.isfinite(classification_y_train).all()
assert np.isfinite(classification_y_val).all()

assert set(np.unique(classification_y_train)) == {0, 1}
assert set(np.unique(classification_y_val)) == {0, 1}

print("\nTARGET AND FEATURE ALIGNMENT")
print("=" * 75)

print("Training alignment: PASSED")
print("Validation alignment: PASSED")
print("Binary target verification: PASSED")

# --------------------------------------------------
# 7. Class imbalance assessment
# --------------------------------------------------

classification_distribution_results = []

for split_name, target_values in [
    ("Training", classification_y_train),
    ("Validation", classification_y_val)
]:

    negative_count = int(
        np.sum(target_values == 0)
    )

    positive_count = int(
        np.sum(target_values == 1)
    )

    classification_distribution_results.append({
        "Dataset": split_name,
        "Records": len(target_values),
        "Negative Cases": negative_count,
        "Positive Cases": positive_count,
        "Negative Percentage": (
            negative_count / len(target_values) * 100
        ),
        "Positive Percentage": (
            positive_count / len(target_values) * 100
        ),
        "Imbalance Ratio": (
            negative_count / positive_count
        )
    })

classification_distribution_df = pd.DataFrame(
    classification_distribution_results
)

classification_train_negatives = int(
    np.sum(classification_y_train == 0)
)

classification_train_positives = int(
    np.sum(classification_y_train == 1)
)

classification_train_prevalence = float(
    classification_y_train.mean()
)

classification_val_prevalence = float(
    classification_y_val.mean()
)

classification_imbalance_ratio = (
    classification_train_negatives
    / classification_train_positives
)

assert classification_train_negatives == 450620
assert classification_train_positives == 59356

print("\nCLASS IMBALANCE ASSESSMENT")
print("=" * 75)

display(classification_distribution_df.round(6))

print(
    "Training prevalence:",
    f"{classification_train_prevalence:.6%}"
)

print(
    "Validation prevalence:",
    f"{classification_val_prevalence:.6%}"
)

print(
    "Training imbalance ratio:",
    f"{classification_imbalance_ratio:.6f}:1"
)

print(
    "Test target distribution remains reserved "
    "for final evaluation."
)

# --------------------------------------------------
# 8. Document imbalance handling policy
# --------------------------------------------------

classification_imbalance_policy = {
    "training_distribution": "Original",
    "smote": False,
    "random_oversampling": False,
    "random_undersampling": False,
    "random_forest_class_weight": None,
    "xgboost_scale_pos_weight": 1.0,
    "threshold_selection_dataset": "Validation",
    "probability_calibration_assessment": True
}

print("\nCLASS IMBALANCE HANDLING POLICY")
print("=" * 75)

for key, value in (
    classification_imbalance_policy.items()
):
    print(f"{key}: {value}")

# --------------------------------------------------
# 9. Reconfirm probability baselines
# --------------------------------------------------

classification_baseline_probabilities = np.full(
    len(classification_y_val),
    classification_train_prevalence,
    dtype=np.float64
)

classification_baseline_metrics = {
    "Model": "Constant Training Prevalence",
    "Validation ROC-AUC": float(
        roc_auc_score(
            classification_y_val,
            classification_baseline_probabilities
        )
    ),
    "Validation PR-AUC (AP)": float(
        average_precision_score(
            classification_y_val,
            classification_baseline_probabilities
        )
    ),
    "Validation Brier Score": float(
        brier_score_loss(
            classification_y_val,
            classification_baseline_probabilities
        )
    )
}

classification_baseline_df = pd.DataFrame([
    classification_baseline_metrics
])

assert np.isclose(
    classification_baseline_metrics[
        "Validation ROC-AUC"
    ],
    0.5
)

assert np.isclose(
    classification_baseline_metrics[
        "Validation PR-AUC (AP)"
    ],
    classification_val_prevalence
)

print("\nPROBABILITY BASELINE VERIFICATION")
print("=" * 75)

display(classification_baseline_df.round(6))

# --------------------------------------------------
# 10. Majority-class baseline
# --------------------------------------------------

classification_majority_predictions = np.zeros(
    len(classification_y_val),
    dtype=np.int64
)

classification_majority_baseline_df = pd.DataFrame([
    {
        "Model": "Always Predict Negative",
        "Accuracy": accuracy_score(
            classification_y_val,
            classification_majority_predictions
        ),
        "Precision": precision_score(
            classification_y_val,
            classification_majority_predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            classification_y_val,
            classification_majority_predictions,
            zero_division=0
        ),
        "F1": f1_score(
            classification_y_val,
            classification_majority_predictions,
            zero_division=0
        )
    }
])

print("\nMAJORITY-CLASS BASELINE")
print("=" * 75)

display(
    classification_majority_baseline_df.round(6)
)

majority_confusion_matrix = confusion_matrix(
    classification_y_val,
    classification_majority_predictions,
    labels=[0, 1]
)

display(pd.DataFrame(
    majority_confusion_matrix,
    index=["Actual Negative", "Actual Positive"],
    columns=[
        "Predicted Negative",
        "Predicted Positive"
    ]
))

# --------------------------------------------------
# 11. Random Forest Classifier configuration
# --------------------------------------------------

rf_classifier_params = {
    "n_estimators": 100,
    "max_depth": 12,
    "min_samples_leaf": 50,
    "max_samples": 0.5,
    "bootstrap": True,
    "criterion": "gini",
    "class_weight": None,
    "random_state": 42,
    "n_jobs": -1
}

print("\nRANDOM FOREST CLASSIFIER CONFIGURATION")
print("=" * 75)

for key, value in rf_classifier_params.items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 12. XGBoost Classifier configuration
# --------------------------------------------------

xgb_classifier_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 50,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "scale_pos_weight": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0
}

print("\nXGBOOST CLASSIFIER CONFIGURATION")
print("=" * 75)

for key, value in xgb_classifier_params.items():
    print(f"{key}: {value}")

# --------------------------------------------------
# 13. Create unfitted classifier pipelines
# --------------------------------------------------

rf_classifier_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    (
        "model",
        RandomForestClassifier(
            **rf_classifier_params
        )
    )
])

xgb_classifier_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    (
        "model",
        XGBClassifier(
            **xgb_classifier_params
        )
    )
])

assert (
    rf_classifier_pipeline.named_steps[
        "preprocessor"
    ]
    is not
    xgb_classifier_pipeline.named_steps[
        "preprocessor"
    ]
)

assert not hasattr(
    rf_classifier_pipeline.named_steps["model"],
    "classes_"
)

assert not hasattr(
    xgb_classifier_pipeline.named_steps["model"],
    "classes_"
)

print("\nCLASSIFICATION PIPELINE PREPARATION")
print("=" * 75)

print("Random Forest pipeline: CREATED")
print("XGBoost pipeline: CREATED")
print("Independent preprocessors: VERIFIED")
print("Classifier training status: UNFITTED")

# --------------------------------------------------
# 14. Define evaluation and selection policy
# --------------------------------------------------

classification_evaluation_policy = {
    "primary_selection_metric": "Validation PR-AUC (AP)",
    "primary_selection_direction": "Maximize",
    "secondary_metrics": [
        "Validation ROC-AUC",
        "Validation Brier Score"
    ],
    "secondary_metric_directions": [
        "Maximize",
        "Minimize"
    ],
    "threshold_dependent_metrics": [
        "Precision",
        "Recall",
        "F1",
        "Specificity",
        "Accuracy",
        "Confusion Matrix"
    ],
    "positive_class": 1,
    "default_threshold": 0.50,
    "proposal_threshold": 0.65,
    "threshold_optimization_metric": "F1",
    "threshold_selection_dataset": "Validation",
    "threshold_search_method": (
        "Distinct validation probability thresholds"
    ),
    "test_set_used_for_selection": False
}

classification_proposal_targets = {
    "roc_auc": 0.80,
    "pr_auc_ap": 0.75,
    "comparison": "Strictly greater than",
    "proposed_risk_threshold": 0.65
}

print("\nCLASSIFICATION EVALUATION POLICY")
print("=" * 75)

for key, value in (
    classification_evaluation_policy.items()
):
    print(f"{key}: {value}")

print("\nPROPOSAL TARGETS")
print("=" * 75)

for key, value in (
    classification_proposal_targets.items()
):
    print(f"{key}: {value}")

# --------------------------------------------------
# 15. Initialize classification registries
# --------------------------------------------------

# Prevent accidental overwriting of trained classifiers
# if this preparation cell is rerun later.

assert not globals().get(
    "classification_models", {}
), "Trained classifiers already exist."

classification_models = {}

classification_model_results = []

classification_validation_probabilities = {}

classification_validation_predictions = {}

classification_threshold_results = {}

classification_confusion_matrices = {}

classification_training_prepared = True

classification_model_selection_completed = False

classification_test_evaluation_completed = False

print("\nCLASSIFICATION MODEL REGISTRY")
print("=" * 75)

print("Trained classifiers:", len(classification_models))
print("Model results registry: INITIALIZED")
print("Validation probability registry: INITIALIZED")
print("Threshold results registry: INITIALIZED")
print("Confusion matrix registry: INITIALIZED")

# --------------------------------------------------
# 16. Create classification preparation metadata
# --------------------------------------------------

classification_preparation_metadata = {
    "project": "St. Hubert Medical Center BI",
    "task": "30-Day Subsequent Visit Prediction",
    "task_type": "binary_classification",
    "dataset_version": "V2",
    "target": classification_target,
    "clinical_readmission_validated": False,
    "prediction_reference_point": "Admission",
    "features": classification_features,
    "excluded_features": classification_excluded_features,
    "numerical_features": list(NUM_COLS),
    "categorical_features": list(CAT_COLS),
    "positive_class": 1,
    "training_rows": int(len(X_proxy_train)),
    "validation_rows": int(len(X_proxy_val)),
    "test_rows": int(len(X_proxy_test)),
    "training_prevalence": (
        classification_train_prevalence
    ),
    "validation_prevalence": (
        classification_val_prevalence
    ),
    "training_imbalance_ratio": float(
        classification_imbalance_ratio
    ),
    "imbalance_policy": (
        classification_imbalance_policy
    ),
    "evaluation_policy": (
        classification_evaluation_policy
    ),
    "proposal_targets": (
        classification_proposal_targets
    ),
    "baseline_metrics": (
        classification_baseline_metrics
    ),
    "classifier_configurations": {
        "Random Forest Classifier": (
            rf_classifier_params
        ),
        "XGBoost Classifier": (
            xgb_classifier_params
        )
    },
    "clinical_deployment_approved": False,
    "test_evaluation_completed": False
}

json.dumps(
    classification_preparation_metadata,
    allow_nan=False
)

print("\nCLASSIFICATION PREPARATION METADATA")
print("=" * 75)

print("Metadata creation: COMPLETED")
print("JSON serialization verification: PASSED")

# --------------------------------------------------
# 17. Visualization: Class distribution
# --------------------------------------------------

plot_df = classification_distribution_df.set_index(
    "Dataset"
)

ax = plot_df[
    [
        "Negative Percentage",
        "Positive Percentage"
    ]
].plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        label_type="center"
    )

plt.ylabel("Class Percentage")
plt.xlabel("Dataset")
plt.title("Classification: Training and Validation Distribution")
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.legend(["Negative", "Positive"])
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 18. Visualization: Class counts
# --------------------------------------------------

ax = plot_df[
    ["Negative Cases", "Positive Cases"]
].plot(
    kind="bar",
    figsize=(9, 5)
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        padding=3
    )

plt.ylabel("Number of Records")
plt.xlabel("Dataset")
plt.title("Classification: Class Imbalance")
plt.xticks(rotation=0)
plt.margins(y=0.15)
plt.legend(["Negative", "Positive"])
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 19. Completion summary
# --------------------------------------------------

print("\nCLASSIFICATION PREPARATION SUMMARY")
print("=" * 75)

print("Regression phase verification: PASSED")
print("Corrected V2 dataset verification: PASSED")
print("Existing split verification: PASSED")
print("Patient leakage verification: PASSED")
print("Feature contract verification: PASSED")
print("Target alignment verification: PASSED")
print("Class imbalance assessment: COMPLETED")
print("Probability baseline verification: PASSED")
print("Majority-class baseline: COMPLETED")
print("Random Forest configuration: PREPARED")
print("XGBoost configuration: PREPARED")
print("Independent classifier pipelines: CREATED")
print("Evaluation policy: DOCUMENTED")
print("Classification registries: INITIALIZED")
print("Preparation metadata: CREATED")
print("Diagnostic visualization: COMPLETED")

print("\nRegression models retained:", len(los_models))

print(
    "Classification models trained:",
    len(classification_models)
)

print(
    "Classification training prepared:",
    classification_training_prepared
)

print("\nNo classifiers have been trained.")
print("No regression models have been retrained.")
print("Test dataset has NOT been evaluated.")
print("No model files have been exported.")

print(
    "\nReady for Cell 38: "
    "Random Forest Classifier Training."
)


In [ ]:
# CELL 38: Random Forest Classifier Training
# 30-Day Subsequent Visit Prediction

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import ConfusionMatrixDisplay

print("RANDOM FOREST CLASSIFIER - SUBSEQUENT VISIT PREDICTION")
print("=" * 75)

# --------------------------------------------------
# 1. Verify training preparation
# --------------------------------------------------

assert classification_training_prepared is True

assert classification_model_selection_completed is False

assert classification_test_evaluation_completed is False

assert len(los_models) == 3

rf_classifier_name = "Random Forest Classifier"

assert rf_classifier_name not in classification_models, (
    "Random Forest Classifier has already been registered."
)

assert list(X_proxy_train.columns) == classification_features

assert list(X_proxy_val.columns) == classification_features

assert len(X_proxy_train) == len(classification_y_train)

assert len(X_proxy_val) == len(classification_y_val)

assert set(np.unique(classification_y_train)) == {0, 1}

print("Classification preparation: VERIFIED")
print("Training records:", f"{len(X_proxy_train):,}")
print("Validation records:", f"{len(X_proxy_val):,}")
print("Original features:", len(classification_features))
print("Regression pipelines retained:", len(los_models))

# --------------------------------------------------
# 2. Verify classifier configuration
# --------------------------------------------------

print("\nRANDOM FOREST CLASSIFIER CONFIGURATION")
print("=" * 75)

for key, value in rf_classifier_params.items():
    print(f"{key}: {value}")

assert isinstance(
    rf_classifier_pipeline.named_steps["model"],
    RandomForestClassifier
)

assert not hasattr(
    rf_classifier_pipeline.named_steps["model"],
    "classes_"
), "The classifier has already been fitted."

assert (
    rf_classifier_pipeline.named_steps["model"]
    .get_params()["class_weight"] is None
)

print("\nExisting pipeline verification: PASSED")
print("Original class distribution: PRESERVED")

# --------------------------------------------------
# 3. Train Random Forest Classifier
# --------------------------------------------------

print("\nTraining Random Forest Classifier...")
print("This may take several minutes.")

rf_classifier_training_start = time.perf_counter()

rf_classifier_pipeline.fit(
    X_proxy_train,
    classification_y_train
)

rf_classifier_training_time = (
    time.perf_counter()
    - rf_classifier_training_start
)

check_is_fitted(
    rf_classifier_pipeline.named_steps["preprocessor"]
)

check_is_fitted(
    rf_classifier_pipeline.named_steps["model"]
)

print("Training completed.")

print(
    "Training time:",
    f"{rf_classifier_training_time:.4f} seconds"
)

# --------------------------------------------------
# 4. Verify fitted classifier classes
# --------------------------------------------------

rf_classifier_estimator = (
    rf_classifier_pipeline.named_steps["model"]
)

rf_classifier_classes = (
    rf_classifier_estimator.classes_
)

assert np.array_equal(
    rf_classifier_classes,
    np.array([0, 1])
)

rf_positive_class_index = int(
    np.flatnonzero(
        rf_classifier_classes == 1
    )[0]
)

print("\nFITTED CLASS VERIFICATION")
print("=" * 75)

print("Classifier classes:", rf_classifier_classes)

print(
    "Positive probability column:",
    rf_positive_class_index
)

print("Positive class verification: PASSED")

# --------------------------------------------------
# 5. Generate validation probabilities
# --------------------------------------------------

print("\nGenerating validation probabilities...")

rf_classifier_prediction_start = time.perf_counter()

rf_validation_probability_matrix = (
    rf_classifier_pipeline.predict_proba(
        X_proxy_val
    )
)

rf_classifier_prediction_time = (
    time.perf_counter()
    - rf_classifier_prediction_start
)

rf_classifier_val_probabilities = (
    rf_validation_probability_matrix[
        :, rf_positive_class_index
    ]
)

assert rf_validation_probability_matrix.shape == (
    len(X_proxy_val), 2
)

assert np.isfinite(
    rf_validation_probability_matrix
).all()

assert (
    rf_validation_probability_matrix >= 0
).all()

assert (
    rf_validation_probability_matrix <= 1
).all()

assert np.allclose(
    rf_validation_probability_matrix.sum(axis=1),
    1.0
)

assert len(
    rf_classifier_val_probabilities
) == len(classification_y_val)

print("Validation probability generation: COMPLETED")

print(
    "Prediction time:",
    f"{rf_classifier_prediction_time:.4f} seconds"
)

# --------------------------------------------------
# 6. Threshold-independent evaluation
# --------------------------------------------------

rf_classifier_val_roc_auc = float(
    roc_auc_score(
        classification_y_val,
        rf_classifier_val_probabilities
    )
)

rf_classifier_val_ap = float(
    average_precision_score(
        classification_y_val,
        rf_classifier_val_probabilities
    )
)

rf_classifier_val_brier = float(
    brier_score_loss(
        classification_y_val,
        rf_classifier_val_probabilities
    )
)

rf_classifier_metrics = {
    "Model": rf_classifier_name,
    "Validation ROC-AUC": (
        rf_classifier_val_roc_auc
    ),
    "Validation PR-AUC (AP)": (
        rf_classifier_val_ap
    ),
    "Validation Brier Score": (
        rf_classifier_val_brier
    ),
    "Training Time (s)": (
        float(rf_classifier_training_time)
    ),
    "Prediction Time (s)": (
        float(rf_classifier_prediction_time)
    )
}

rf_classifier_results_df = pd.DataFrame([
    rf_classifier_metrics
])

print("\nRANDOM FOREST CLASSIFIER VALIDATION RESULTS")
print("=" * 75)

display(
    rf_classifier_results_df.round(6)
)

# --------------------------------------------------
# 7. Compare against probability baseline
# --------------------------------------------------

rf_classifier_baseline_comparison_df = pd.DataFrame([
    classification_baseline_metrics,
    {
        "Model": rf_classifier_name,
        "Validation ROC-AUC": (
            rf_classifier_val_roc_auc
        ),
        "Validation PR-AUC (AP)": (
            rf_classifier_val_ap
        ),
        "Validation Brier Score": (
            rf_classifier_val_brier
        )
    }
])

baseline_roc_auc = (
    classification_baseline_metrics[
        "Validation ROC-AUC"
    ]
)

baseline_ap = (
    classification_baseline_metrics[
        "Validation PR-AUC (AP)"
    ]
)

baseline_brier = (
    classification_baseline_metrics[
        "Validation Brier Score"
    ]
)

rf_roc_auc_improvement = (
    rf_classifier_val_roc_auc
    - baseline_roc_auc
)

rf_ap_improvement = (
    rf_classifier_val_ap
    - baseline_ap
)

rf_ap_lift = (
    rf_classifier_val_ap
    / baseline_ap
)

rf_brier_skill_score = (
    1
    - rf_classifier_val_brier
    / baseline_brier
)

print("\nPROBABILITY BASELINE COMPARISON")
print("=" * 75)

display(
    rf_classifier_baseline_comparison_df.round(6)
)

print(
    "Absolute ROC-AUC improvement:",
    f"{rf_roc_auc_improvement:.6f}"
)

print(
    "Absolute AP improvement:",
    f"{rf_ap_improvement:.6f}"
)

print(
    "AP lift over baseline:",
    f"{rf_ap_lift:.6f}x"
)

print(
    "Brier skill score:",
    f"{rf_brier_skill_score:.6f}"
)

print(
    "Relative Brier error reduction:",
    f"{rf_brier_skill_score * 100:.4f}%"
)

# --------------------------------------------------
# 8. Define reusable threshold evaluation function
# --------------------------------------------------

def evaluate_classification_threshold(
    y_true,
    probabilities,
    threshold
):

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    assert y_true.ndim == 1
    assert probabilities.ndim == 1
    assert len(y_true) == len(probabilities)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) &
            (probabilities <= 1)).all()
    assert 0 <= threshold <= 1

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    cm = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    results = {
        "Threshold": float(threshold),
        "Accuracy": float(
            accuracy_score(
                y_true, predictions
            )
        ),
        "Precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "Recall": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "F1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "Specificity": float(specificity),
        "Predicted Positives": int(tp + fp),
        "Predicted Positive Rate": float(
            predictions.mean()
        ),
        "True Negatives": int(tn),
        "False Positives": int(fp),
        "False Negatives": int(fn),
        "True Positives": int(tp)
    }

    return results, predictions, cm

# --------------------------------------------------
# 9. Evaluate default and proposal thresholds
# --------------------------------------------------

rf_threshold_configurations = {
    "Default": 0.50,
    "Proposal": 0.65
}

rf_classifier_threshold_rows = []

rf_classifier_threshold_predictions = {}

rf_classifier_confusion_matrices = {}

for threshold_name, threshold_value in (
    rf_threshold_configurations.items()
):

    metrics, predictions, cm = (
        evaluate_classification_threshold(
            classification_y_val,
            rf_classifier_val_probabilities,
            threshold_value
        )
    )

    metrics["Threshold Type"] = threshold_name

    rf_classifier_threshold_rows.append(metrics)

    rf_classifier_threshold_predictions[
        threshold_name
    ] = predictions

    rf_classifier_confusion_matrices[
        threshold_name
    ] = cm

rf_classifier_threshold_df = pd.DataFrame(
    rf_classifier_threshold_rows
)

threshold_display_columns = [
    "Threshold Type",
    "Threshold",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
    "Predicted Positives",
    "Predicted Positive Rate"
]

print("\nTHRESHOLD-DEPENDENT VALIDATION RESULTS")
print("=" * 75)

display(
    rf_classifier_threshold_df[
        threshold_display_columns
    ].round(6)
)

print(
    "These thresholds are evaluated for comparison only."
)

print(
    "Final threshold optimization is reserved "
    "for Cell 42."
)

# --------------------------------------------------
# 10. Display confusion matrices
# --------------------------------------------------

for threshold_name, cm in (
    rf_classifier_confusion_matrices.items()
):

    threshold_value = (
        rf_threshold_configurations[
            threshold_name
        ]
    )

    print(
        f"\nCONFUSION MATRIX - "
        f"{threshold_name.upper()} "
        f"THRESHOLD ({threshold_value:.2f})"
    )

    print("=" * 75)

    display(pd.DataFrame(
        cm,
        index=[
            "Actual Negative",
            "Actual Positive"
        ],
        columns=[
            "Predicted Negative",
            "Predicted Positive"
        ]
    ))

    assert cm.sum() == len(
        classification_y_val
    )

    assert cm[0].sum() == 96613
    assert cm[1].sum() == 12845

# --------------------------------------------------
# 11. Classification reports
# --------------------------------------------------

for threshold_name, predictions in (
    rf_classifier_threshold_predictions.items()
):

    print(
        f"\nCLASSIFICATION REPORT - "
        f"{threshold_name.upper()} THRESHOLD"
    )

    print("=" * 75)

    report = classification_report(
        classification_y_val,
        predictions,
        labels=[0, 1],
        target_names=[
            "Negative (0)",
            "Positive (1)"
        ],
        zero_division=0,
        output_dict=True
    )

    display(
        pd.DataFrame(report).transpose().round(6)
    )

# --------------------------------------------------
# 12. Probability sanity checks
# --------------------------------------------------

rf_probability_summary_df = pd.DataFrame([
    {
        "Statistic": "Minimum probability",
        "Value": float(
            rf_classifier_val_probabilities.min()
        )
    },
    {
        "Statistic": "Maximum probability",
        "Value": float(
            rf_classifier_val_probabilities.max()
        )
    },
    {
        "Statistic": "Mean probability",
        "Value": float(
            rf_classifier_val_probabilities.mean()
        )
    },
    {
        "Statistic": "Median probability",
        "Value": float(
            np.median(
                rf_classifier_val_probabilities
            )
        )
    },
    {
        "Statistic": "Probability standard deviation",
        "Value": float(
            rf_classifier_val_probabilities.std()
        )
    },
    {
        "Statistic": "Actual validation prevalence",
        "Value": classification_val_prevalence
    },
    {
        "Statistic": "Mean probability minus prevalence",
        "Value": float(
            rf_classifier_val_probabilities.mean()
            - classification_val_prevalence
        )
    }
])

print("\nPROBABILITY SANITY CHECK")
print("=" * 75)

display(
    rf_probability_summary_df.round(6)
)

print(
    "Non-finite probabilities:",
    int(np.sum(
        ~np.isfinite(
            rf_classifier_val_probabilities
        )
    ))
)

print(
    "Probabilities outside [0, 1]:",
    int(np.sum(
        (rf_classifier_val_probabilities < 0)
        | (rf_classifier_val_probabilities > 1)
    ))
)

print("\nPROBABILITY QUANTILES")
print("=" * 75)

rf_probability_quantiles_df = pd.DataFrame({
    "Quantile": [
        "0%", "1%", "5%", "25%",
        "50%", "75%", "95%", "99%", "100%"
    ],
    "Probability": np.quantile(
        rf_classifier_val_probabilities,
        [
            0.00, 0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99, 1.00
        ]
    )
})

display(
    rf_probability_quantiles_df.round(6)
)

# --------------------------------------------------
# 13. Verify trained forest structure
# --------------------------------------------------

rf_classifier_tree_depths = np.array([
    tree.tree_.max_depth
    for tree in rf_classifier_estimator.estimators_
])

rf_classifier_tree_nodes = np.array([
    tree.tree_.node_count
    for tree in rf_classifier_estimator.estimators_
])

rf_classifier_transformed_features = (
    rf_classifier_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

assert len(
    rf_classifier_estimator.estimators_
) == rf_classifier_params["n_estimators"]

assert rf_classifier_tree_depths.max() <= (
    rf_classifier_params["max_depth"]
)

assert len(
    rf_classifier_transformed_features
) == 10

assert rf_classifier_estimator.n_features_in_ == 10

print("\nTRAINED FOREST STRUCTURE")
print("=" * 75)

print(
    "Number of trained trees:",
    len(rf_classifier_estimator.estimators_)
)

print(
    "Average tree depth:",
    f"{rf_classifier_tree_depths.mean():.2f}"
)

print(
    "Maximum tree depth:",
    rf_classifier_tree_depths.max()
)

print(
    "Average nodes per tree:",
    f"{rf_classifier_tree_nodes.mean():.2f}"
)

print(
    "Transformed input features:",
    len(rf_classifier_transformed_features)
)

print("Forest structure verification: PASSED")

# --------------------------------------------------
# 14. Proposal target assessment
# --------------------------------------------------

rf_classifier_proposal_df = pd.DataFrame([
    {
        "Metric": "ROC-AUC",
        "Validation Result": (
            rf_classifier_val_roc_auc
        ),
        "Proposal Boundary": 0.80,
        "Target Status": (
            "ACHIEVED"
            if rf_classifier_val_roc_auc > 0.80
            else "NOT ACHIEVED"
        )
    },
    {
        "Metric": "PR-AUC (AP)",
        "Validation Result": (
            rf_classifier_val_ap
        ),
        "Proposal Boundary": 0.75,
        "Target Status": (
            "ACHIEVED"
            if rf_classifier_val_ap > 0.75
            else "NOT ACHIEVED"
        )
    }
])

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 75)

display(
    rf_classifier_proposal_df.round(6)
)

# --------------------------------------------------
# 15. Update classification model registries
# --------------------------------------------------

classification_models[
    rf_classifier_name
] = rf_classifier_pipeline

classification_model_results.append(
    rf_classifier_metrics
)

classification_validation_probabilities[
    rf_classifier_name
] = rf_classifier_val_probabilities

classification_validation_predictions[
    rf_classifier_name
] = rf_classifier_threshold_predictions

classification_threshold_results[
    rf_classifier_name
] = rf_classifier_threshold_df

classification_confusion_matrices[
    rf_classifier_name
] = rf_classifier_confusion_matrices

rf_classifier_training_completed = True

assert len(classification_models) == 1

assert len(classification_model_results) == 1

assert len(los_models) == 3

print("\nCLASSIFICATION MODEL REGISTRY")
print("=" * 75)

print(
    "Registered classifiers:",
    len(classification_models)
)

display(
    pd.DataFrame(
        classification_model_results
    ).round(6)
)

# --------------------------------------------------
# 16. Visualization: Confusion matrices
# --------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5)
)

for ax, threshold_name in zip(
    axes,
    ["Default", "Proposal"]
):

    cm = rf_classifier_confusion_matrices[
        threshold_name
    ]

    threshold_value = (
        rf_threshold_configurations[
            threshold_name
        ]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "Negative",
            "Positive"
        ]
    ).plot(
        ax=ax,
        cmap="Blues",
        values_format=",d",
        colorbar=False
    )

    ax.set_title(
        f"{threshold_name} Threshold "
        f"({threshold_value:.2f})"
    )

fig.suptitle(
    "Random Forest Classifier: "
    "Validation Confusion Matrices"
)

plt.tight_layout()
plt.show()

# --------------------------------------------------
# 17. Visualization: ROC curve
# --------------------------------------------------

rf_fpr, rf_tpr, rf_roc_thresholds = roc_curve(
    classification_y_val,
    rf_classifier_val_probabilities
)

plt.figure(figsize=(8, 6))

plt.plot(
    rf_fpr,
    rf_tpr,
    label=(
        "Random Forest "
        f"(AUC = {rf_classifier_val_roc_auc:.4f})"
    )
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="No-discrimination baseline"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest Classifier: Validation ROC Curve")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 18. Visualization: Precision-recall curve
# --------------------------------------------------

rf_pr_precision, rf_pr_recall, rf_pr_thresholds = (
    precision_recall_curve(
        classification_y_val,
        rf_classifier_val_probabilities
    )
)

plt.figure(figsize=(8, 6))

plt.plot(
    rf_pr_recall,
    rf_pr_precision,
    label=(
        "Random Forest "
        f"(AP = {rf_classifier_val_ap:.4f})"
    )
)

plt.axhline(
    classification_val_prevalence,
    linestyle="--",
    label=(
        "Prevalence baseline "
        f"({classification_val_prevalence:.4f})"
    )
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    "Random Forest Classifier: "
    "Validation Precision-Recall Curve"
)
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 19. Training completion summary
# --------------------------------------------------

print("\nTRAINING COMPLETION SUMMARY")
print("=" * 75)

print("Random Forest Classifier training: COMPLETED")
print("Validation probability generation: COMPLETED")
print("Probability evaluation: COMPLETED")
print("Baseline comparison: COMPLETED")
print("Default threshold evaluation: COMPLETED")
print("Proposal threshold evaluation: COMPLETED")
print("Confusion matrices: COMPLETED")
print("Classification reports: COMPLETED")
print("Probability sanity checks: PASSED")
print("Forest structure checks: PASSED")
print("Proposal target assessment: COMPLETED")
print("Classification model registry: UPDATED")
print("Validation visualization: COMPLETED")

print("\nRegression models retained:", len(los_models))

print(
    "Classification models trained:",
    len(classification_models)
)

print(
    "Total trained predictive models:",
    len(los_models) + len(classification_models)
)

print("\nFitted classifier pipeline retained in memory.")
print("No regression models have been retrained.")
print("XGBoost Classifier has NOT been trained.")
print("No classification threshold has been optimized.")
print("No final classification model has been selected.")
print("Test datasets have NOT been evaluated.")
print("No model files have been exported.")

print(
    "\nReady for Cell 39: "
    "Random Forest Classifier Diagnostics."
)


In [ ]:
# CELL 39: Random Forest Classifier Diagnostics
# Validation-based diagnostics without retraining

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

print("RANDOM FOREST CLASSIFIER DIAGNOSTICS")
print("=" * 75)

# --------------------------------------------------
# 1. Verify existing fitted model
# --------------------------------------------------

assert rf_classifier_training_completed is True

assert classification_model_selection_completed is False

assert classification_test_evaluation_completed is False

assert len(classification_models) == 1
assert len(los_models) == 3

assert classification_models[
    rf_classifier_name
] is rf_classifier_pipeline

rf_diag_y_val = np.asarray(
    classification_y_val
)

rf_diag_y_train = np.asarray(
    classification_y_train
)

rf_diag_prob = np.asarray(
    rf_classifier_val_probabilities
)

assert len(rf_diag_prob) == len(rf_diag_y_val)

assert np.isfinite(rf_diag_prob).all()

assert (
    (rf_diag_prob >= 0)
    & (rf_diag_prob <= 1)
).all()

assert set(np.unique(rf_diag_y_val)) == {0, 1}

print("Fitted classifier verification: PASSED")
print("Validation records:", f"{len(rf_diag_y_val):,}")
print("Regression pipelines retained:", len(los_models))

# --------------------------------------------------
# 2. Training versus validation performance
# --------------------------------------------------

print("\nTRAINING VS VALIDATION PERFORMANCE")
print("=" * 75)

print("Generating in-sample training probabilities...")

rf_diag_train_start = time.perf_counter()

rf_diag_train_prob = (
    rf_classifier_pipeline.predict_proba(
        X_proxy_train
    )[:, rf_positive_class_index]
)

rf_diag_train_prediction_time = (
    time.perf_counter() - rf_diag_train_start
)

assert len(rf_diag_train_prob) == len(
    rf_diag_y_train
)

assert np.isfinite(rf_diag_train_prob).all()

def summarize_probability_metrics(
    y_true,
    probabilities
):
    return {
        "ROC-AUC": float(
            roc_auc_score(y_true, probabilities)
        ),
        "PR-AUC (AP)": float(
            average_precision_score(
                y_true, probabilities
            )
        ),
        "Brier Score": float(
            brier_score_loss(
                y_true, probabilities
            )
        ),
        "Log Loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1]
            )
        ),
        "Mean Probability": float(
            np.mean(probabilities)
        ),
        "Probability Std": float(
            np.std(probabilities)
        ),
        "Actual Prevalence": float(
            np.mean(y_true)
        )
    }

rf_diag_train_metrics = (
    summarize_probability_metrics(
        rf_diag_y_train,
        rf_diag_train_prob
    )
)

rf_diag_val_metrics = (
    summarize_probability_metrics(
        rf_diag_y_val,
        rf_diag_prob
    )
)

rf_train_val_comparison_df = pd.DataFrame([
    {
        "Dataset": "Training (In-sample)",
        **rf_diag_train_metrics
    },
    {
        "Dataset": "Validation",
        **rf_diag_val_metrics
    }
])

display(
    rf_train_val_comparison_df.round(6)
)

rf_roc_generalization_gap = (
    rf_diag_train_metrics["ROC-AUC"]
    - rf_diag_val_metrics["ROC-AUC"]
)

rf_ap_generalization_gap = (
    rf_diag_train_metrics["PR-AUC (AP)"]
    - rf_diag_val_metrics["PR-AUC (AP)"]
)

print(
    "ROC-AUC training-validation gap:",
    f"{rf_roc_generalization_gap:.6f}"
)

print(
    "AP training-validation gap:",
    f"{rf_ap_generalization_gap:.6f}"
)

print(
    "Training probability generation time:",
    f"{rf_diag_train_prediction_time:.4f} seconds"
)

print(
    "\nTraining metrics are in-sample estimates "
    "and are not independent performance estimates."
)

# --------------------------------------------------
# 3. Probability separation by actual class
# --------------------------------------------------

rf_probability_separation_rows = []

for actual_class in [0, 1]:

    class_probabilities = rf_diag_prob[
        rf_diag_y_val == actual_class
    ]

    rf_probability_separation_rows.append({
        "Actual Class": actual_class,
        "Records": int(len(class_probabilities)),
        "Mean Probability": float(
            class_probabilities.mean()
        ),
        "Median Probability": float(
            np.median(class_probabilities)
        ),
        "Probability Std": float(
            class_probabilities.std()
        ),
        "Minimum": float(
            class_probabilities.min()
        ),
        "5th Percentile": float(
            np.quantile(class_probabilities, 0.05)
        ),
        "95th Percentile": float(
            np.quantile(class_probabilities, 0.95)
        ),
        "Maximum": float(
            class_probabilities.max()
        )
    })

rf_probability_separation_df = pd.DataFrame(
    rf_probability_separation_rows
)

rf_negative_mean_probability = float(
    rf_diag_prob[rf_diag_y_val == 0].mean()
)

rf_positive_mean_probability = float(
    rf_diag_prob[rf_diag_y_val == 1].mean()
)

rf_probability_separation_gap = (
    rf_positive_mean_probability
    - rf_negative_mean_probability
)

print("\nPROBABILITY SEPARATION BY ACTUAL CLASS")
print("=" * 75)

display(
    rf_probability_separation_df.round(6)
)

print(
    "Positive minus negative mean probability:",
    f"{rf_probability_separation_gap:.8f}"
)

print(
    "Difference in percentage points:",
    f"{rf_probability_separation_gap * 100:.6f}"
)

# --------------------------------------------------
# 4. Calibration analysis
# --------------------------------------------------

print("\nCALIBRATION ANALYSIS")
print("=" * 75)

rf_calibration_df = pd.DataFrame({
    "Actual": rf_diag_y_val,
    "Probability": rf_diag_prob
})

# Quantile-based bins provide approximately
# equal-sized groups while preserving score ties.

rf_calibration_df["Calibration_Bin"] = pd.qcut(
    rf_calibration_df["Probability"],
    q=10,
    duplicates="drop"
)

rf_calibration_summary_df = (
    rf_calibration_df
    .groupby(
        "Calibration_Bin",
        observed=True
    )
    .agg(
        Records=("Actual", "size"),
        Positive_Cases=("Actual", "sum"),
        Mean_Predicted_Probability=(
            "Probability", "mean"
        ),
        Observed_Positive_Rate=(
            "Actual", "mean"
        ),
        Minimum_Probability=(
            "Probability", "min"
        ),
        Maximum_Probability=(
            "Probability", "max"
        )
    )
    .reset_index()
)

rf_calibration_summary_df[
    "Calibration_Gap"
] = (
    rf_calibration_summary_df[
        "Mean_Predicted_Probability"
    ]
    - rf_calibration_summary_df[
        "Observed_Positive_Rate"
    ]
)

rf_calibration_summary_df[
    "Absolute_Calibration_Gap"
] = rf_calibration_summary_df[
    "Calibration_Gap"
].abs()

assert rf_calibration_summary_df[
    "Records"
].sum() == len(rf_diag_y_val)

rf_expected_calibration_error = float(
    np.sum(
        rf_calibration_summary_df["Records"]
        * rf_calibration_summary_df[
            "Absolute_Calibration_Gap"
        ]
    ) / len(rf_diag_y_val)
)

rf_calibration_in_large_gap = float(
    rf_diag_prob.mean()
    - rf_diag_y_val.mean()
)

display(
    rf_calibration_summary_df.round(6)
)

print(
    "Expected Calibration Error (ECE):",
    f"{rf_expected_calibration_error:.6f}"
)

print(
    "Mean predicted minus observed probability:",
    f"{rf_calibration_in_large_gap:.6f}"
)

print(
    "Number of calibration bins:",
    len(rf_calibration_summary_df)
)

print(
    "\nECE is descriptive and depends on "
    "the selected binning procedure."
)

print(
    "Agreement with overall prevalence does not "
    "establish individual-level calibration."
)

# --------------------------------------------------
# 5. Brier score and log loss comparison
# --------------------------------------------------

rf_diag_baseline_prob = np.full(
    len(rf_diag_y_val),
    float(classification_train_prevalence),
    dtype=np.float64
)

rf_diag_baseline_metrics = (
    summarize_probability_metrics(
        rf_diag_y_val,
        rf_diag_baseline_prob
    )
)

rf_diag_brier_skill = (
    1
    - rf_diag_val_metrics["Brier Score"]
    / rf_diag_baseline_metrics["Brier Score"]
)

rf_diag_log_loss_skill = (
    1
    - rf_diag_val_metrics["Log Loss"]
    / rf_diag_baseline_metrics["Log Loss"]
)

rf_probability_quality_df = pd.DataFrame([
    {
        "Model": "Constant Training Prevalence",
        "Brier Score": (
            rf_diag_baseline_metrics["Brier Score"]
        ),
        "Log Loss": (
            rf_diag_baseline_metrics["Log Loss"]
        ),
        "Brier Skill Score": 0.0,
        "Relative Log Loss Reduction": 0.0
    },
    {
        "Model": rf_classifier_name,
        "Brier Score": (
            rf_diag_val_metrics["Brier Score"]
        ),
        "Log Loss": (
            rf_diag_val_metrics["Log Loss"]
        ),
        "Brier Skill Score": rf_diag_brier_skill,
        "Relative Log Loss Reduction": (
            rf_diag_log_loss_skill
        )
    }
])

print("\nPROBABILITY QUALITY COMPARISON")
print("=" * 75)

display(
    rf_probability_quality_df.round(6)
)

# --------------------------------------------------
# 6. Risk decile and lift analysis
# --------------------------------------------------

rf_risk_decile_df = (
    rf_calibration_summary_df[
        [
            "Records",
            "Positive_Cases",
            "Mean_Predicted_Probability",
            "Observed_Positive_Rate",
            "Minimum_Probability",
            "Maximum_Probability"
        ]
    ]
    .sort_values(
        "Mean_Predicted_Probability",
        ascending=False
    )
    .reset_index(drop=True)
)

rf_risk_decile_df.insert(
    0,
    "Risk_Group",
    np.arange(
        1, len(rf_risk_decile_df) + 1
    )
)

rf_diag_prevalence = float(
    rf_diag_y_val.mean()
)

rf_risk_decile_df["Lift"] = (
    rf_risk_decile_df[
        "Observed_Positive_Rate"
    ] / rf_diag_prevalence
)

rf_risk_decile_df[
    "Cumulative_Records"
] = rf_risk_decile_df[
    "Records"
].cumsum()

rf_risk_decile_df[
    "Cumulative_Positive_Cases"
] = rf_risk_decile_df[
    "Positive_Cases"
].cumsum()

rf_risk_decile_df[
    "Cumulative_Population_Percentage"
] = (
    rf_risk_decile_df["Cumulative_Records"]
    / len(rf_diag_y_val)
    * 100
)

rf_risk_decile_df[
    "Cumulative_Positive_Capture_Percentage"
] = (
    rf_risk_decile_df[
        "Cumulative_Positive_Cases"
    ]
    / rf_diag_y_val.sum()
    * 100
)

rf_top_risk_group = (
    rf_risk_decile_df.iloc[0]
)

rf_bottom_risk_group = (
    rf_risk_decile_df.iloc[-1]
)

print("\nRISK DECILE AND LIFT ANALYSIS")
print("=" * 75)

display(
    rf_risk_decile_df.round(6)
)

print(
    "Highest-score group observed positive rate:",
    f"{rf_top_risk_group['Observed_Positive_Rate']:.6%}"
)

print(
    "Highest-score group lift:",
    f"{rf_top_risk_group['Lift']:.6f}x"
)

print(
    "Highest-score group positive capture:",
    f"{rf_top_risk_group['Cumulative_Positive_Capture_Percentage']:.4f}%"
)

print(
    "Lowest-score group observed positive rate:",
    f"{rf_bottom_risk_group['Observed_Positive_Rate']:.6%}"
)

print(
    "\nRisk Group 1 contains the highest "
    "predicted probabilities."
)

print(
    "Quantile groups may have unequal sizes "
    "when probabilities contain ties."
)

# --------------------------------------------------
# 7. Transformed feature importance (MDI)
# --------------------------------------------------

rf_classifier_feature_names = (
    rf_classifier_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

rf_classifier_mdi_values = (
    rf_classifier_estimator.feature_importances_
)

assert len(rf_classifier_feature_names) == len(
    rf_classifier_mdi_values
)

rf_classifier_mdi_df = pd.DataFrame({
    "Transformed_Feature": (
        rf_classifier_feature_names
    ),
    "MDI_Importance": (
        rf_classifier_mdi_values
    )
})

rf_classifier_mdi_df[
    "Importance_Percentage"
] = (
    rf_classifier_mdi_df["MDI_Importance"]
    * 100
)

rf_classifier_mdi_df = (
    rf_classifier_mdi_df
    .sort_values(
        "MDI_Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTRANSFORMED FEATURE IMPORTANCE (MDI)")
print("=" * 75)

display(
    rf_classifier_mdi_df.round(6)
)

# --------------------------------------------------
# 8. Group importance by original feature
# --------------------------------------------------

def rf_classifier_original_feature(
    transformed_name
):

    feature_name = transformed_name.split(
        "__", 1
    )[-1]

    for original_feature in sorted(
        classification_features,
        key=len,
        reverse=True
    ):

        if (
            feature_name == original_feature
            or feature_name.startswith(
                original_feature + "_"
            )
        ):
            return original_feature

    raise ValueError(
        f"Unrecognized feature: {transformed_name}"
    )

rf_classifier_mdi_df["Original_Feature"] = (
    rf_classifier_mdi_df[
        "Transformed_Feature"
    ].apply(
        rf_classifier_original_feature
    )
)

rf_classifier_grouped_mdi_df = (
    rf_classifier_mdi_df
    .groupby(
        "Original_Feature",
        as_index=False
    )["MDI_Importance"]
    .sum()
    .sort_values(
        "MDI_Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

rf_classifier_grouped_mdi_df[
    "Importance_Percentage"
] = (
    rf_classifier_grouped_mdi_df[
        "MDI_Importance"
    ] * 100
)

assert set(
    rf_classifier_grouped_mdi_df[
        "Original_Feature"
    ]
) == set(classification_features)

assert np.isclose(
    rf_classifier_grouped_mdi_df[
        "MDI_Importance"
    ].sum(),
    1.0
)

print("\nGROUPED ORIGINAL FEATURE IMPORTANCE (MDI)")
print("=" * 75)

display(
    rf_classifier_grouped_mdi_df.round(6)
)

print(
    "MDI importance measures internal tree usage, "
    "not necessarily validation predictive value."
)

# --------------------------------------------------
# 9. Validation permutation importance
# --------------------------------------------------

print("\nVALIDATION PERMUTATION IMPORTANCE")
print("=" * 75)

# Fixed sample retained for the later
# XGBoost diagnostic comparison.

classification_permutation_sample_size = min(
    30000,
    len(X_proxy_val)
)

classification_permutation_rng = (
    np.random.default_rng(42)
)

classification_permutation_indices = np.sort(
    classification_permutation_rng.choice(
        len(X_proxy_val),
        size=classification_permutation_sample_size,
        replace=False
    )
)

X_classification_permutation = (
    X_proxy_val.iloc[
        classification_permutation_indices
    ].copy()
)

y_classification_permutation = (
    rf_diag_y_val[
        classification_permutation_indices
    ].copy()
)

assert len(
    X_classification_permutation
) == len(
    y_classification_permutation
)

assert set(
    np.unique(y_classification_permutation)
) == {0, 1}

rf_classifier_permutation_sample_prob = (
    rf_classifier_pipeline.predict_proba(
        X_classification_permutation
    )[:, rf_positive_class_index]
)

rf_classifier_permutation_baseline_ap = (
    average_precision_score(
        y_classification_permutation,
        rf_classifier_permutation_sample_prob
    )
)

print(
    "Validation sample size:",
    f"{classification_permutation_sample_size:,}"
)

print(
    "Sample positive prevalence:",
    f"{y_classification_permutation.mean():.6f}"
)

print(
    "Sample baseline AP:",
    f"{rf_classifier_permutation_baseline_ap:.6f}"
)

print("Permutation repeats: 5")
print("Scoring metric: Average Precision")
print("Random seed: 42")

print("\nCalculating permutation importance...")

rf_classifier_permutation_start = (
    time.perf_counter()
)

rf_classifier_permutation_result = (
    permutation_importance(
        rf_classifier_pipeline,
        X_classification_permutation,
        y_classification_permutation,
        scoring="average_precision",
        n_repeats=5,
        random_state=42,
        n_jobs=1
    )
)

rf_classifier_permutation_time = (
    time.perf_counter()
    - rf_classifier_permutation_start
)

rf_classifier_permutation_df = pd.DataFrame({
    "Feature": classification_features,
    "Mean_AP_Decrease": (
        rf_classifier_permutation_result
        .importances_mean
    ),
    "Importance_Std": (
        rf_classifier_permutation_result
        .importances_std
    )
})

rf_classifier_permutation_df = (
    rf_classifier_permutation_df
    .sort_values(
        "Mean_AP_Decrease",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    rf_classifier_permutation_df.round(6)
)

print(
    "Permutation importance time:",
    f"{rf_classifier_permutation_time:.4f} seconds"
)

print(
    "\nPositive importance means shuffling "
    "the feature reduced validation AP."
)

print(
    "Near-zero importance indicates limited "
    "measured predictive contribution."
)

print(
    "Negative importance means shuffling "
    "the feature improved AP on this sample."
)

# --------------------------------------------------
# 10. Marginal feature-target relationships
# --------------------------------------------------

rf_classifier_feature_diagnostic_df = (
    X_proxy_val.copy()
)

rf_classifier_feature_diagnostic_df[
    "Actual"
] = rf_diag_y_val

rf_classifier_feature_diagnostic_df[
    "Predicted_Probability"
] = rf_diag_prob

# Age groups are descriptive only.
# Missing ages are retained as a separate group.

rf_classifier_feature_diagnostic_df[
    "AGE_GROUP"
] = (
    pd.cut(
        rf_classifier_feature_diagnostic_df[
            "AGE_AT_VISIT"
        ],
        bins=[-np.inf, 18, 35, 50, 65, np.inf],
        labels=[
            "0-17",
            "18-34",
            "35-49",
            "50-64",
            "65+"
        ],
        right=False
    )
    .astype("object")
    .fillna("Missing")
)

def build_classifier_feature_summary(
    feature
):

    result = (
        rf_classifier_feature_diagnostic_df
        .groupby(
            feature,
            observed=True,
            dropna=False
        )
        .agg(
            Records=("Actual", "size"),
            Positive_Cases=("Actual", "sum"),
            Actual_Positive_Rate=(
                "Actual", "mean"
            ),
            Mean_Predicted_Probability=(
                "Predicted_Probability", "mean"
            )
        )
        .reset_index()
    )

    result["Observed_Lift"] = (
        result["Actual_Positive_Rate"]
        / rf_diag_prevalence
    )

    result["Prediction_Gap"] = (
        result["Mean_Predicted_Probability"]
        - result["Actual_Positive_Rate"]
    )

    return result

rf_classifier_feature_summaries = {}

for feature in [
    "PRIOR_VISIT_COUNT",
    "ADMISSION_MONTH",
    "AGE_GROUP",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR"
]:

    feature_summary = (
        build_classifier_feature_summary(
            feature
        )
    )

    rf_classifier_feature_summaries[
        feature
    ] = feature_summary

    print(
        f"\nFEATURE-TARGET DIAGNOSTIC: {feature}"
    )

    print("=" * 75)

    display(
        feature_summary.round(6)
    )

print(
    "\nMarginal associations are descriptive "
    "and do not establish causal relationships."
)

print(
    "Groups with very small record counts "
    "must be interpreted cautiously."
)

# --------------------------------------------------
# 11. Threshold sensitivity analysis
# --------------------------------------------------

# These are fixed illustrative thresholds.
# No optimal threshold is selected here.

rf_diagnostic_thresholds = [
    0.05,
    0.08,
    0.10,
    0.11,
    float(classification_train_prevalence),
    0.12,
    0.13,
    0.14,
    0.15,
    0.17,
    0.20,
    0.50,
    0.65
]

rf_threshold_sensitivity_rows = []

for threshold in sorted(
    set(rf_diagnostic_thresholds)
):

    metrics, _, _ = (
        evaluate_classification_threshold(
            rf_diag_y_val,
            rf_diag_prob,
            threshold
        )
    )

    rf_threshold_sensitivity_rows.append(
        metrics
    )

rf_threshold_sensitivity_df = pd.DataFrame(
    rf_threshold_sensitivity_rows
)

print("\nTHRESHOLD SENSITIVITY ANALYSIS")
print("=" * 75)

display(
    rf_threshold_sensitivity_df[
        [
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Specificity",
            "Predicted Positives",
            "Predicted Positive Rate"
        ]
    ].round(6)
)

# Always-positive reference classifier.

rf_always_positive_precision = (
    rf_diag_prevalence
)

rf_always_positive_recall = 1.0

rf_always_positive_f1 = (
    2 * rf_diag_prevalence
    / (1 + rf_diag_prevalence)
)

print("\nALWAYS-POSITIVE REFERENCE")
print("=" * 75)

print(
    "Precision:",
    f"{rf_always_positive_precision:.6f}"
)

print(
    "Recall:",
    f"{rf_always_positive_recall:.6f}"
)

print(
    "F1:",
    f"{rf_always_positive_f1:.6f}"
)

print(
    "\nNo threshold has been selected."
)

print(
    "Formal threshold optimization remains "
    "reserved for Cell 42."
)

# --------------------------------------------------
# 12. Visualization: probability distributions
# --------------------------------------------------

plt.figure(figsize=(10, 6))

rf_probability_bins = np.linspace(
    rf_diag_prob.min(),
    rf_diag_prob.max(),
    50
)

for actual_class, class_label in [
    (0, "Actual Negative"),
    (1, "Actual Positive")
]:

    plt.hist(
        rf_diag_prob[
            rf_diag_y_val == actual_class
        ],
        bins=rf_probability_bins,
        density=True,
        alpha=0.45,
        label=class_label
    )

plt.axvline(
    rf_diag_prevalence,
    linestyle="--",
    label="Validation prevalence"
)

plt.xlabel("Predicted Positive Probability")
plt.ylabel("Density")
plt.title(
    "Random Forest: Probability Separation "
    "by Actual Class"
)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 13. Visualization: calibration curve
# --------------------------------------------------

plt.figure(figsize=(8, 7))

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.plot(
    rf_calibration_summary_df[
        "Mean_Predicted_Probability"
    ],
    rf_calibration_summary_df[
        "Observed_Positive_Rate"
    ],
    marker="o",
    label=(
        "Random Forest "
        f"(ECE = {rf_expected_calibration_error:.4f})"
    )
)

plt.scatter(
    classification_train_prevalence,
    rf_diag_prevalence,
    marker="x",
    s=100,
    label="Constant prevalence baseline"
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Positive Rate")
plt.title(
    "Random Forest: Validation Calibration Curve"
)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Zoomed calibration view.

rf_calibration_zoom_min = max(
    0,
    min(
        rf_calibration_summary_df[
            "Mean_Predicted_Probability"
        ].min(),
        rf_calibration_summary_df[
            "Observed_Positive_Rate"
        ].min(),
        classification_train_prevalence,
        rf_diag_prevalence
    ) - 0.01
)

rf_calibration_zoom_max = min(
    1,
    max(
        rf_calibration_summary_df[
            "Mean_Predicted_Probability"
        ].max(),
        rf_calibration_summary_df[
            "Observed_Positive_Rate"
        ].max(),
        classification_train_prevalence,
        rf_diag_prevalence
    ) + 0.01
)

plt.figure(figsize=(8, 7))

plt.plot(
    [
        rf_calibration_zoom_min,
        rf_calibration_zoom_max
    ],
    [
        rf_calibration_zoom_min,
        rf_calibration_zoom_max
    ],
    linestyle="--",
    label="Perfect calibration"
)

plt.plot(
    rf_calibration_summary_df[
        "Mean_Predicted_Probability"
    ],
    rf_calibration_summary_df[
        "Observed_Positive_Rate"
    ],
    marker="o",
    label="Random Forest"
)

plt.scatter(
    classification_train_prevalence,
    rf_diag_prevalence,
    marker="x",
    s=100,
    label="Constant prevalence baseline"
)

plt.xlim(
    rf_calibration_zoom_min,
    rf_calibration_zoom_max
)

plt.ylim(
    rf_calibration_zoom_min,
    rf_calibration_zoom_max
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Positive Rate")
plt.title(
    "Random Forest: Zoomed Calibration Curve"
)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 14. Visualization: risk group positive rates
# --------------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    rf_risk_decile_df["Risk_Group"],
    rf_risk_decile_df[
        "Observed_Positive_Rate"
    ] * 100,
    alpha=0.7,
    label="Observed positive rate"
)

plt.plot(
    rf_risk_decile_df["Risk_Group"],
    rf_risk_decile_df[
        "Mean_Predicted_Probability"
    ] * 100,
    marker="o",
    label="Mean predicted probability"
)

plt.axhline(
    rf_diag_prevalence * 100,
    linestyle="--",
    label="Validation prevalence"
)

plt.xticks(
    rf_risk_decile_df["Risk_Group"]
)

plt.xlabel("Risk Group (1 = Highest Score)")
plt.ylabel("Positive Rate (%)")
plt.title(
    "Random Forest: Risk Group Analysis"
)
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 15. Visualization: cumulative gains
# --------------------------------------------------

plt.figure(figsize=(8, 6))

plt.plot(
    np.r_[
        0,
        rf_risk_decile_df[
            "Cumulative_Population_Percentage"
        ].to_numpy()
    ],
    np.r_[
        0,
        rf_risk_decile_df[
            "Cumulative_Positive_Capture_Percentage"
        ].to_numpy()
    ],
    marker="o",
    label="Random Forest"
)

plt.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    label="Random selection"
)

plt.xlabel("Population Selected (%)")
plt.ylabel("Positive Cases Captured (%)")
plt.title(
    "Random Forest: Cumulative Gains Curve"
)
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 16. Visualization: grouped MDI importance
# --------------------------------------------------

rf_mdi_plot_df = (
    rf_classifier_grouped_mdi_df
    .sort_values("Importance_Percentage")
)

plt.figure(figsize=(10, 6))

plt.barh(
    rf_mdi_plot_df["Original_Feature"],
    rf_mdi_plot_df["Importance_Percentage"]
)

plt.xlabel("Impurity-Based Importance (%)")
plt.title(
    "Random Forest Classifier: "
    "Original Feature Importance"
)
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 17. Visualization: permutation importance
# --------------------------------------------------

rf_classifier_permutation_plot_df = (
    rf_classifier_permutation_df
    .sort_values("Mean_AP_Decrease")
)

plt.figure(figsize=(10, 6))

plt.barh(
    rf_classifier_permutation_plot_df[
        "Feature"
    ],
    rf_classifier_permutation_plot_df[
        "Mean_AP_Decrease"
    ],
    xerr=rf_classifier_permutation_plot_df[
        "Importance_Std"
    ],
    capsize=3
)

plt.axvline(
    0,
    linestyle="--"
)

plt.xlabel("Decrease in Average Precision")
plt.title(
    "Random Forest Classifier: "
    "Validation Permutation Importance"
)
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 18. Visualization: threshold sensitivity
# --------------------------------------------------

plt.figure(figsize=(10, 6))

for metric in [
    "Precision",
    "Recall",
    "F1",
    "Specificity"
]:

    plt.plot(
        rf_threshold_sensitivity_df["Threshold"],
        rf_threshold_sensitivity_df[metric],
        marker="o",
        label=metric
    )

plt.axhline(
    rf_always_positive_f1,
    linestyle=":",
    label="Always-positive F1 reference"
)

plt.axvline(
    0.50,
    linestyle="--",
    label="Default threshold"
)

plt.axvline(
    0.65,
    linestyle="--",
    label="Proposal threshold"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Metric Value")
plt.title(
    "Random Forest: Threshold Sensitivity"
)
plt.xlim(0, 0.70)
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 19. Visualization: prior visit count relationship
# --------------------------------------------------

rf_prior_visit_classification_df = (
    rf_classifier_feature_summaries[
        "PRIOR_VISIT_COUNT"
    ]
)

plt.figure(figsize=(10, 6))

plt.plot(
    rf_prior_visit_classification_df[
        "PRIOR_VISIT_COUNT"
    ],
    rf_prior_visit_classification_df[
        "Actual_Positive_Rate"
    ] * 100,
    marker="o",
    label="Observed positive rate"
)

plt.plot(
    rf_prior_visit_classification_df[
        "PRIOR_VISIT_COUNT"
    ],
    rf_prior_visit_classification_df[
        "Mean_Predicted_Probability"
    ] * 100,
    marker="o",
    label="Mean predicted probability"
)

plt.axhline(
    rf_diag_prevalence * 100,
    linestyle="--",
    label="Validation prevalence"
)

plt.xlabel("Prior Visit Count")
plt.ylabel("Positive Rate (%)")
plt.title(
    "Subsequent Visit Relationship "
    "with Prior Visit Count"
)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 20. Visualization: admission month relationship
# --------------------------------------------------

rf_month_classification_df = (
    rf_classifier_feature_summaries[
        "ADMISSION_MONTH"
    ]
)

plt.figure(figsize=(10, 6))

plt.plot(
    rf_month_classification_df[
        "ADMISSION_MONTH"
    ],
    rf_month_classification_df[
        "Actual_Positive_Rate"
    ] * 100,
    marker="o",
    label="Observed positive rate"
)

plt.plot(
    rf_month_classification_df[
        "ADMISSION_MONTH"
    ],
    rf_month_classification_df[
        "Mean_Predicted_Probability"
    ] * 100,
    marker="o",
    label="Mean predicted probability"
)

plt.axhline(
    rf_diag_prevalence * 100,
    linestyle="--",
    label="Validation prevalence"
)

plt.xticks(
    range(1, 13),
    [
        "Jan", "Feb", "Mar", "Apr",
        "May", "Jun", "Jul", "Aug",
        "Sep", "Oct", "Nov", "Dec"
    ]
)

plt.xlabel("Admission Month")
plt.ylabel("Positive Rate (%)")
plt.title(
    "Subsequent Visit Relationship "
    "with Admission Month"
)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 21. Diagnostic summary
# --------------------------------------------------

rf_classifier_diagnostic_summary = {
    "model": rf_classifier_name,
    "dataset_version": "V2",
    "training_roc_auc": float(
        rf_diag_train_metrics["ROC-AUC"]
    ),
    "validation_roc_auc": float(
        rf_diag_val_metrics["ROC-AUC"]
    ),
    "training_ap": float(
        rf_diag_train_metrics["PR-AUC (AP)"]
    ),
    "validation_ap": float(
        rf_diag_val_metrics["PR-AUC (AP)"]
    ),
    "roc_auc_generalization_gap": float(
        rf_roc_generalization_gap
    ),
    "ap_generalization_gap": float(
        rf_ap_generalization_gap
    ),
    "validation_brier_score": float(
        rf_diag_val_metrics["Brier Score"]
    ),
    "validation_log_loss": float(
        rf_diag_val_metrics["Log Loss"]
    ),
    "brier_skill_score": float(
        rf_diag_brier_skill
    ),
    "expected_calibration_error": float(
        rf_expected_calibration_error
    ),
    "calibration_bins": int(
        len(rf_calibration_summary_df)
    ),
    "mean_probability_gap": float(
        rf_calibration_in_large_gap
    ),
    "positive_negative_probability_gap": float(
        rf_probability_separation_gap
    ),
    "highest_risk_group_lift": float(
        rf_top_risk_group["Lift"]
    ),
    "highest_risk_group_positive_capture_pct": float(
        rf_top_risk_group[
            "Cumulative_Positive_Capture_Percentage"
        ]
    ),
    "always_positive_reference_f1": float(
        rf_always_positive_f1
    ),
    "permutation_sample_size": int(
        classification_permutation_sample_size
    ),
    "permutation_scoring": "average_precision",
    "permutation_repeats": 5,
    "permutation_random_state": 42,
    "threshold_optimized": False,
    "model_retrained": False,
    "test_set_evaluated": False
}

import json

json.dumps(
    rf_classifier_diagnostic_summary,
    allow_nan=False
)

rf_classifier_diagnostics_completed = True

print("\nKEY DIAGNOSTIC RESULTS")
print("=" * 75)

display(
    pd.DataFrame([
        {
            "Diagnostic": "Training ROC-AUC",
            "Value": rf_diag_train_metrics[
                "ROC-AUC"
            ]
        },
        {
            "Diagnostic": "Validation ROC-AUC",
            "Value": rf_diag_val_metrics[
                "ROC-AUC"
            ]
        },
        {
            "Diagnostic": "Training AP",
            "Value": rf_diag_train_metrics[
                "PR-AUC (AP)"
            ]
        },
        {
            "Diagnostic": "Validation AP",
            "Value": rf_diag_val_metrics[
                "PR-AUC (AP)"
            ]
        },
        {
            "Diagnostic": "Expected Calibration Error",
            "Value": rf_expected_calibration_error
        },
        {
            "Diagnostic": "Brier Skill Score",
            "Value": rf_diag_brier_skill
        },
        {
            "Diagnostic": "Highest Risk Group Lift",
            "Value": rf_top_risk_group["Lift"]
        },
        {
            "Diagnostic": "Positive-Negative Probability Gap",
            "Value": rf_probability_separation_gap
        }
    ]).round(6)
)

# --------------------------------------------------
# 22. Completion verification
# --------------------------------------------------

assert len(los_models) == 3

assert len(classification_models) == 1

assert classification_model_selection_completed is False

assert classification_test_evaluation_completed is False

assert rf_classifier_diagnostics_completed is True

print("\nDIAGNOSTICS COMPLETION SUMMARY")
print("=" * 75)

print("Training-validation comparison: COMPLETED")
print("Probability separation: COMPLETED")
print("Calibration analysis: COMPLETED")
print("Brier and log loss comparison: COMPLETED")
print("Risk decile analysis: COMPLETED")
print("Cumulative gains analysis: COMPLETED")
print("Transformed feature importance: COMPLETED")
print("Grouped feature importance: COMPLETED")
print("Validation permutation importance: COMPLETED")
print("Marginal feature-target diagnostics: COMPLETED")
print("Threshold sensitivity analysis: COMPLETED")
print("Diagnostic visualization: COMPLETED")
print("Diagnostic metadata: CREATED")
print("JSON serialization verification: PASSED")

print("\nRegression models retained:", len(los_models))

print(
    "Classification models retained:",
    len(classification_models)
)

print("\nNo models have been retrained.")
print("No probability calibration has been fitted.")
print("No classification threshold has been optimized.")
print("No final classification model has been selected.")
print("XGBoost Classifier has NOT been trained.")
print("Test datasets have NOT been evaluated.")
print("No model files have been exported.")

print(
    "\nReady for Cell 40: "
    "XGBoost Classifier Training."
)


In [ ]:
# CELL 40: XGBoost Classifier Training
# St. Hubert Medical Center
# Subsequent Visit Within 30 Days Proxy
# Corrected V2 Dataset

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
    classification_report,
    ConfusionMatrixDisplay
)

print("XGBOOST CLASSIFIER - SUBSEQUENT VISIT PREDICTION")
print("=" * 75)

# --------------------------------------------------
# 1. Verify previous stages
# --------------------------------------------------

assert rf_classifier_training_completed is True
assert rf_classifier_diagnostics_completed is True

assert classification_model_selection_completed is False
assert classification_test_evaluation_completed is False

assert len(los_models) == 3
assert len(classification_models) == 1

assert classification_models[
    rf_classifier_name
] is rf_classifier_pipeline

check_is_fitted(rf_classifier_pipeline)

xgb_classifier_name = "XGBoost Classifier"

assert xgb_classifier_name not in classification_models

xgb_classifier_y_train = np.asarray(
    classification_y_train
)

xgb_classifier_y_val = np.asarray(
    classification_y_val
)

assert len(X_proxy_train) == len(
    xgb_classifier_y_train
)

assert len(X_proxy_val) == len(
    xgb_classifier_y_val
)

assert set(
    np.unique(xgb_classifier_y_train)
) == {0, 1}

assert set(
    np.unique(xgb_classifier_y_val)
) == {0, 1}

assert list(
    X_proxy_train.columns
) == list(classification_features)

assert list(
    X_proxy_val.columns
) == list(classification_features)

print("Classification preparation: VERIFIED")
print("Random Forest diagnostics: VERIFIED")

print(
    "Training records:",
    f"{len(X_proxy_train):,}"
)

print(
    "Validation records:",
    f"{len(X_proxy_val):,}"
)

print(
    "Original features:",
    len(classification_features)
)

print(
    "Regression pipelines retained:",
    len(los_models)
)

# --------------------------------------------------
# 2. Verify XGBoost configuration
# --------------------------------------------------

xgb_classifier_expected_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 50,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "scale_pos_weight": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0
}

# Locate the existing unfitted XGBoost pipeline
# prepared in Cell 37.
# Do not replace the original pipeline.

xgb_pipeline_candidates = {}

for variable_name, variable_value in list(
    globals().items()
):

    if isinstance(variable_value, Pipeline):

        final_estimator = variable_value.steps[-1][1]

        if isinstance(final_estimator, XGBClassifier):

            xgb_pipeline_candidates[
                id(variable_value)
            ] = (
                variable_name,
                variable_value
            )

assert len(xgb_pipeline_candidates) == 1, (
    "Expected exactly one prepared XGBoost "
    "classifier pipeline. Inspect Cell 37."
)

(
    xgb_pipeline_original_variable,
    xgb_classifier_pipeline
) = next(
    iter(xgb_pipeline_candidates.values())
)

xgb_classifier_estimator = (
    xgb_classifier_pipeline.steps[-1][1]
)

xgb_classifier_actual_params = (
    xgb_classifier_estimator.get_params()
)

print("\nXGBOOST CLASSIFIER CONFIGURATION")
print("=" * 75)

for parameter, expected_value in (
    xgb_classifier_expected_params.items()
):

    actual_value = (
        xgb_classifier_actual_params.get(parameter)
    )

    assert actual_value == expected_value, (
        f"Unexpected parameter: {parameter}. "
        f"Expected {expected_value}, "
        f"found {actual_value}."
    )

    print(f"{parameter}: {actual_value}")

assert (
    xgb_classifier_actual_params.get(
        "early_stopping_rounds"
    ) is None
)

assert not hasattr(
    xgb_classifier_estimator,
    "n_features_in_"
), "XGBoost classifier is already fitted."

assert (
    xgb_classifier_pipeline.named_steps[
        "preprocessor"
    ]
    is not
    rf_classifier_pipeline.named_steps[
        "preprocessor"
    ]
)

print(
    "\nExisting pipeline:",
    xgb_pipeline_original_variable
)

print("Existing pipeline verification: PASSED")
print("Independent preprocessor: VERIFIED")
print("Original class distribution: PRESERVED")

# --------------------------------------------------
# 3. Train XGBoost classifier
# --------------------------------------------------

print("\nTraining XGBoost Classifier...")
print("This may take several minutes.")

xgb_classifier_training_start = (
    time.perf_counter()
)

xgb_classifier_pipeline.fit(
    X_proxy_train,
    xgb_classifier_y_train
)

xgb_classifier_training_time = (
    time.perf_counter()
    - xgb_classifier_training_start
)

check_is_fitted(xgb_classifier_pipeline)

print("Training completed.")

print(
    "Training time:",
    f"{xgb_classifier_training_time:.4f} seconds"
)

# --------------------------------------------------
# 4. Verify fitted class order
# --------------------------------------------------

xgb_classifier_classes = np.asarray(
    xgb_classifier_estimator.classes_
)

assert set(
    xgb_classifier_classes.tolist()
) == {0, 1}

xgb_positive_class_index = int(
    np.flatnonzero(
        xgb_classifier_classes == 1
    )[0]
)

print("\nFITTED CLASS VERIFICATION")
print("=" * 75)

print(
    "Classifier classes:",
    xgb_classifier_classes
)

print(
    "Positive probability column:",
    xgb_positive_class_index
)

print("Positive class verification: PASSED")

# --------------------------------------------------
# 5. Generate validation probabilities
# --------------------------------------------------

print("\nGenerating validation probabilities...")

xgb_classifier_prediction_start = (
    time.perf_counter()
)

xgb_classifier_val_probabilities = (
    xgb_classifier_pipeline.predict_proba(
        X_proxy_val
    )[:, xgb_positive_class_index]
)

xgb_classifier_prediction_time = (
    time.perf_counter()
    - xgb_classifier_prediction_start
)

assert len(
    xgb_classifier_val_probabilities
) == len(xgb_classifier_y_val)

assert np.isfinite(
    xgb_classifier_val_probabilities
).all()

assert (
    (xgb_classifier_val_probabilities >= 0)
    &
    (xgb_classifier_val_probabilities <= 1)
).all()

print("Validation probability generation: COMPLETED")

print(
    "Prediction time:",
    f"{xgb_classifier_prediction_time:.4f} seconds"
)

# --------------------------------------------------
# 6. Probability-based evaluation
# --------------------------------------------------

xgb_classifier_val_roc_auc = float(
    roc_auc_score(
        xgb_classifier_y_val,
        xgb_classifier_val_probabilities
    )
)

xgb_classifier_val_ap = float(
    average_precision_score(
        xgb_classifier_y_val,
        xgb_classifier_val_probabilities
    )
)

xgb_classifier_val_brier = float(
    brier_score_loss(
        xgb_classifier_y_val,
        xgb_classifier_val_probabilities
    )
)

xgb_classifier_result = {
    "Model": xgb_classifier_name,
    "Validation ROC-AUC": (
        xgb_classifier_val_roc_auc
    ),
    "Validation PR-AUC (AP)": (
        xgb_classifier_val_ap
    ),
    "Validation Brier Score": (
        xgb_classifier_val_brier
    ),
    "Training Time (s)": float(
        xgb_classifier_training_time
    ),
    "Prediction Time (s)": float(
        xgb_classifier_prediction_time
    )
}

xgb_classifier_results_df = pd.DataFrame([
    xgb_classifier_result
])

print("\nXGBOOST CLASSIFIER VALIDATION RESULTS")
print("=" * 75)

display(
    xgb_classifier_results_df.round(6)
)

# --------------------------------------------------
# 7. Compare against probability baseline
# --------------------------------------------------

xgb_classifier_baseline_probabilities = (
    np.full(
        len(xgb_classifier_y_val),
        float(classification_train_prevalence),
        dtype=np.float64
    )
)

xgb_classifier_baseline_roc = float(
    roc_auc_score(
        xgb_classifier_y_val,
        xgb_classifier_baseline_probabilities
    )
)

xgb_classifier_baseline_ap = float(
    average_precision_score(
        xgb_classifier_y_val,
        xgb_classifier_baseline_probabilities
    )
)

xgb_classifier_baseline_brier = float(
    brier_score_loss(
        xgb_classifier_y_val,
        xgb_classifier_baseline_probabilities
    )
)

xgb_classifier_roc_improvement = (
    xgb_classifier_val_roc_auc
    - xgb_classifier_baseline_roc
)

xgb_classifier_ap_improvement = (
    xgb_classifier_val_ap
    - xgb_classifier_baseline_ap
)

xgb_classifier_ap_lift = (
    xgb_classifier_val_ap
    / xgb_classifier_baseline_ap
)

xgb_classifier_brier_skill = (
    1
    - xgb_classifier_val_brier
    / xgb_classifier_baseline_brier
)

xgb_classifier_baseline_comparison_df = (
    pd.DataFrame([
        {
            "Model": "Constant Training Prevalence",
            "Validation ROC-AUC": (
                xgb_classifier_baseline_roc
            ),
            "Validation PR-AUC (AP)": (
                xgb_classifier_baseline_ap
            ),
            "Validation Brier Score": (
                xgb_classifier_baseline_brier
            )
        },
        {
            "Model": xgb_classifier_name,
            "Validation ROC-AUC": (
                xgb_classifier_val_roc_auc
            ),
            "Validation PR-AUC (AP)": (
                xgb_classifier_val_ap
            ),
            "Validation Brier Score": (
                xgb_classifier_val_brier
            )
        }
    ])
)

print("\nPROBABILITY BASELINE COMPARISON")
print("=" * 75)

display(
    xgb_classifier_baseline_comparison_df.round(6)
)

print(
    "Absolute ROC-AUC improvement:",
    f"{xgb_classifier_roc_improvement:.6f}"
)

print(
    "Absolute AP improvement:",
    f"{xgb_classifier_ap_improvement:.6f}"
)

print(
    "AP lift over baseline:",
    f"{xgb_classifier_ap_lift:.6f}x"
)

print(
    "Brier skill score:",
    f"{xgb_classifier_brier_skill:.6f}"
)

print(
    "Relative Brier error reduction:",
    f"{xgb_classifier_brier_skill * 100:.4f}%"
)

# --------------------------------------------------
# 8. Default and proposal threshold evaluation
# --------------------------------------------------

xgb_classifier_threshold_rows = []

xgb_classifier_threshold_metrics = {}

xgb_classifier_confusion_matrices = {}

xgb_classifier_threshold_predictions = {}

for threshold_type, threshold in [
    ("Default", 0.50),
    ("Proposal", 0.65)
]:

    metrics, _, _ = (
        evaluate_classification_threshold(
            xgb_classifier_y_val,
            xgb_classifier_val_probabilities,
            threshold
        )
    )

    predictions = (
        xgb_classifier_val_probabilities >= threshold
    ).astype(np.int64)

    from sklearn.metrics import confusion_matrix

    matrix = confusion_matrix(
        xgb_classifier_y_val,
        predictions,
        labels=[0, 1]
    )

    assert matrix.sum() == len(
        xgb_classifier_y_val
    )

    xgb_classifier_threshold_rows.append({
        "Threshold Type": threshold_type,
        **metrics
    })

    xgb_classifier_threshold_metrics[
        threshold_type
    ] = metrics

    xgb_classifier_confusion_matrices[
        threshold_type
    ] = matrix

    xgb_classifier_threshold_predictions[
        threshold_type
    ] = predictions

xgb_classifier_threshold_results_df = (
    pd.DataFrame(
        xgb_classifier_threshold_rows
    )
)

print("\nTHRESHOLD-DEPENDENT VALIDATION RESULTS")
print("=" * 75)

display(
    xgb_classifier_threshold_results_df[
        [
            "Threshold Type",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Specificity",
            "Predicted Positives",
            "Predicted Positive Rate"
        ]
    ].round(6)
)

print(
    "These thresholds are evaluated "
    "for comparison only."
)

print(
    "Final threshold optimization "
    "is reserved for Cell 42."
)

# --------------------------------------------------
# 9. Confusion matrices and classification reports
# --------------------------------------------------

for threshold_type, threshold in [
    ("Default", 0.50),
    ("Proposal", 0.65)
]:

    print(
        f"\nCONFUSION MATRIX - "
        f"{threshold_type.upper()} "
        f"THRESHOLD ({threshold:.2f})"
    )

    print("=" * 75)

    display(
        pd.DataFrame(
            xgb_classifier_confusion_matrices[
                threshold_type
            ],
            index=[
                "Actual Negative",
                "Actual Positive"
            ],
            columns=[
                "Predicted Negative",
                "Predicted Positive"
            ]
        )
    )

    print(
        f"\nCLASSIFICATION REPORT - "
        f"{threshold_type.upper()} THRESHOLD"
    )

    print("=" * 75)

    report = classification_report(
        xgb_classifier_y_val,
        xgb_classifier_threshold_predictions[
            threshold_type
        ],
        labels=[0, 1],
        target_names=[
            "Negative (0)",
            "Positive (1)"
        ],
        zero_division=0,
        output_dict=True
    )

    display(
        pd.DataFrame(report).T.round(6)
    )

# --------------------------------------------------
# 10. Probability sanity checks
# --------------------------------------------------

xgb_classifier_probability_summary = {
    "Minimum probability": float(
        xgb_classifier_val_probabilities.min()
    ),
    "Maximum probability": float(
        xgb_classifier_val_probabilities.max()
    ),
    "Mean probability": float(
        xgb_classifier_val_probabilities.mean()
    ),
    "Median probability": float(
        np.median(
            xgb_classifier_val_probabilities
        )
    ),
    "Probability standard deviation": float(
        xgb_classifier_val_probabilities.std()
    ),
    "Actual validation prevalence": float(
        xgb_classifier_y_val.mean()
    ),
    "Mean probability minus prevalence": float(
        xgb_classifier_val_probabilities.mean()
        - xgb_classifier_y_val.mean()
    )
}

xgb_classifier_probability_summary_df = (
    pd.DataFrame(
        list(
            xgb_classifier_probability_summary.items()
        ),
        columns=["Statistic", "Value"]
    )
)

print("\nPROBABILITY SANITY CHECK")
print("=" * 75)

display(
    xgb_classifier_probability_summary_df.round(6)
)

print(
    "Non-finite probabilities:",
    int(
        (~np.isfinite(
            xgb_classifier_val_probabilities
        )).sum()
    )
)

print(
    "Probabilities outside [0, 1]:",
    int(
        (
            (xgb_classifier_val_probabilities < 0)
            |
            (xgb_classifier_val_probabilities > 1)
        ).sum()
    )
)

xgb_classifier_quantiles = [
    0, 0.01, 0.05, 0.25,
    0.50, 0.75, 0.95, 0.99, 1
]

xgb_classifier_probability_quantiles_df = (
    pd.DataFrame({
        "Quantile": [
            f"{q * 100:.0f}%"
            for q in xgb_classifier_quantiles
        ],
        "Probability": np.quantile(
            xgb_classifier_val_probabilities,
            xgb_classifier_quantiles
        )
    })
)

print("\nPROBABILITY QUANTILES")
print("=" * 75)

display(
    xgb_classifier_probability_quantiles_df.round(6)
)

# --------------------------------------------------
# 11. Verify trained boosting structure
# --------------------------------------------------

xgb_classifier_booster = (
    xgb_classifier_estimator.get_booster()
)

xgb_classifier_boosting_rounds = int(
    xgb_classifier_booster.num_boosted_rounds()
)

xgb_classifier_transformed_features = int(
    xgb_classifier_booster.num_features()
)

xgb_classifier_feature_names = (
    xgb_classifier_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

assert xgb_classifier_boosting_rounds == 300

assert xgb_classifier_transformed_features == len(
    xgb_classifier_feature_names
)

assert xgb_classifier_transformed_features == 10

xgb_classifier_tree_df = (
    xgb_classifier_booster.trees_to_dataframe()
)

xgb_classifier_tree_count = int(
    xgb_classifier_tree_df["Tree"].nunique()
)

assert xgb_classifier_tree_count == 300

print("\nTRAINED BOOSTING STRUCTURE")
print("=" * 75)

print(
    "Boosting rounds:",
    xgb_classifier_boosting_rounds
)

print(
    "Number of trees:",
    xgb_classifier_tree_count
)

print(
    "Configured maximum depth:",
    xgb_classifier_actual_params["max_depth"]
)

print(
    "Total tree nodes:",
    len(xgb_classifier_tree_df)
)

print(
    "Average nodes per tree:",
    f"{len(xgb_classifier_tree_df) / xgb_classifier_tree_count:.2f}"
)

print(
    "Transformed input features:",
    xgb_classifier_transformed_features
)

print("Boosting structure verification: PASSED")

# Release temporary tree structure table.

del xgb_classifier_tree_df

# --------------------------------------------------
# 12. Proposal target assessment
# --------------------------------------------------

xgb_classifier_roc_target_achieved = bool(
    xgb_classifier_val_roc_auc > 0.80
)

xgb_classifier_ap_target_achieved = bool(
    xgb_classifier_val_ap > 0.75
)

xgb_classifier_proposal_assessment_df = (
    pd.DataFrame([
        {
            "Metric": "ROC-AUC",
            "Validation Result": (
                xgb_classifier_val_roc_auc
            ),
            "Proposal Boundary": 0.80,
            "Target Status": (
                "ACHIEVED"
                if xgb_classifier_roc_target_achieved
                else "NOT ACHIEVED"
            )
        },
        {
            "Metric": "PR-AUC (AP)",
            "Validation Result": (
                xgb_classifier_val_ap
            ),
            "Proposal Boundary": 0.75,
            "Target Status": (
                "ACHIEVED"
                if xgb_classifier_ap_target_achieved
                else "NOT ACHIEVED"
            )
        }
    ])
)

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 75)

display(
    xgb_classifier_proposal_assessment_df.round(6)
)

# --------------------------------------------------
# 13. Register fitted classifier
# --------------------------------------------------

classification_models[
    xgb_classifier_name
] = xgb_classifier_pipeline

assert len(classification_models) == 2

assert classification_models[
    rf_classifier_name
] is rf_classifier_pipeline

assert classification_models[
    xgb_classifier_name
] is xgb_classifier_pipeline

# Preserve explicit XGBoost result objects for
# later diagnostics and comparison.
#
# Construct a comparison table using the existing
# Random Forest diagnostic metrics and XGBoost
# validation metrics.

classification_cell40_comparison_df = (
    pd.DataFrame([
        {
            "Model": "Constant Training Prevalence",
            "Validation ROC-AUC": (
                xgb_classifier_baseline_roc
            ),
            "Validation PR-AUC (AP)": (
                xgb_classifier_baseline_ap
            ),
            "Validation Brier Score": (
                xgb_classifier_baseline_brier
            )
        },
        {
            "Model": rf_classifier_name,
            "Validation ROC-AUC": float(
                rf_diag_val_metrics["ROC-AUC"]
            ),
            "Validation PR-AUC (AP)": float(
                rf_diag_val_metrics["PR-AUC (AP)"]
            ),
            "Validation Brier Score": float(
                rf_diag_val_metrics["Brier Score"]
            )
        },
        {
            "Model": xgb_classifier_name,
            "Validation ROC-AUC": (
                xgb_classifier_val_roc_auc
            ),
            "Validation PR-AUC (AP)": (
                xgb_classifier_val_ap
            ),
            "Validation Brier Score": (
                xgb_classifier_val_brier
            )
        }
    ])
)

print("\nCLASSIFICATION MODEL REGISTRY")
print("=" * 75)

print(
    "Registered classifiers:",
    len(classification_models)
)

print(
    "Registered model names:",
    list(classification_models.keys())
)

print("\nPRELIMINARY VALIDATION COMPARISON")
print("=" * 75)

display(
    classification_cell40_comparison_df.round(6)
)

print(
    "Preliminary comparison only. "
    "No final model has been selected."
)

# --------------------------------------------------
# 14. ROC curve visualization
# --------------------------------------------------

xgb_classifier_fpr, xgb_classifier_tpr, _ = (
    roc_curve(
        xgb_classifier_y_val,
        xgb_classifier_val_probabilities
    )
)

plt.figure(figsize=(8, 7))

plt.plot(
    xgb_classifier_fpr,
    xgb_classifier_tpr,
    label=(
        "XGBoost "
        f"(AUC = {xgb_classifier_val_roc_auc:.4f})"
    )
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="No-discrimination baseline"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "XGBoost Classifier: Validation ROC Curve"
)

plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 15. Precision-recall curve visualization
# --------------------------------------------------

(
    xgb_classifier_pr_precision,
    xgb_classifier_pr_recall,
    _
) = precision_recall_curve(
    xgb_classifier_y_val,
    xgb_classifier_val_probabilities
)

plt.figure(figsize=(8, 7))

plt.plot(
    xgb_classifier_pr_recall,
    xgb_classifier_pr_precision,
    label=(
        "XGBoost "
        f"(AP = {xgb_classifier_val_ap:.4f})"
    )
)

plt.axhline(
    xgb_classifier_y_val.mean(),
    linestyle="--",
    label=(
        "Prevalence baseline "
        f"({xgb_classifier_baseline_ap:.4f})"
    )
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    "XGBoost Classifier: "
    "Validation Precision-Recall Curve"
)

plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(loc="upper right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 16. Confusion matrix visualizations
# --------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5)
)

for axis, threshold_type, threshold in [
    (axes[0], "Default", 0.50),
    (axes[1], "Proposal", 0.65)
]:

    matrix_display = ConfusionMatrixDisplay(
        confusion_matrix=(
            xgb_classifier_confusion_matrices[
                threshold_type
            ]
        ),
        display_labels=[
            "Negative",
            "Positive"
        ]
    )

    matrix_display.plot(
        ax=axis,
        cmap="Blues",
        colorbar=False,
        values_format=",d"
    )

    axis.set_title(
        f"{threshold_type} Threshold "
        f"({threshold:.2f})"
    )

fig.suptitle(
    "XGBoost Classifier: "
    "Validation Confusion Matrices"
)

plt.tight_layout()
plt.show()

# --------------------------------------------------
# 17. Predicted probability distribution
# --------------------------------------------------

plt.figure(figsize=(10, 6))

plt.hist(
    xgb_classifier_val_probabilities,
    bins=60,
    alpha=0.75,
    label="XGBoost validation probabilities"
)

plt.axvline(
    classification_train_prevalence,
    linestyle="--",
    label="Training prevalence"
)

plt.axvline(
    xgb_classifier_y_val.mean(),
    linestyle=":",
    label="Validation prevalence"
)

plt.xlabel("Predicted Positive Probability")
plt.ylabel("Records")

plt.title(
    "XGBoost Classifier: "
    "Validation Probability Distribution"
)

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 18. Training metadata
# --------------------------------------------------

xgb_classifier_training_metadata = {
    "model": xgb_classifier_name,
    "dataset_version": "V2",
    "task": "binary_classification",
    "target": (
        "SUBSEQUENT_VISIT_WITHIN_30D_PROXY"
    ),
    "prediction_reference_point": "Admission",
    "clinical_interpretation": (
        "Subsequent visit proxy, "
        "not validated unplanned readmission."
    ),
    "features": list(classification_features),
    "training_records": int(
        len(X_proxy_train)
    ),
    "validation_records": int(
        len(X_proxy_val)
    ),
    "hyperparameters": (
        xgb_classifier_expected_params.copy()
    ),
    "training_time_seconds": float(
        xgb_classifier_training_time
    ),
    "validation_prediction_time_seconds": float(
        xgb_classifier_prediction_time
    ),
    "validation_roc_auc": float(
        xgb_classifier_val_roc_auc
    ),
    "validation_average_precision": float(
        xgb_classifier_val_ap
    ),
    "validation_brier_score": float(
        xgb_classifier_val_brier
    ),
    "ap_lift_over_baseline": float(
        xgb_classifier_ap_lift
    ),
    "brier_skill_score": float(
        xgb_classifier_brier_skill
    ),
    "positive_class": 1,
    "positive_probability_column": int(
        xgb_positive_class_index
    ),
    "boosting_rounds": int(
        xgb_classifier_boosting_rounds
    ),
    "transformed_features": int(
        xgb_classifier_transformed_features
    ),
    "roc_auc_target_achieved": (
        xgb_classifier_roc_target_achieved
    ),
    "ap_target_achieved": (
        xgb_classifier_ap_target_achieved
    ),
    "resampling_used": False,
    "early_stopping_used": False,
    "probability_calibration_fitted": False,
    "threshold_optimized": False,
    "test_set_evaluated": False,
    "clinical_deployment_approved": False
}

json.dumps(
    xgb_classifier_training_metadata,
    allow_nan=False
)

print("\nTRAINING METADATA")
print("=" * 75)

print("Metadata creation: COMPLETED")
print("JSON serialization verification: PASSED")

# --------------------------------------------------
# 19. Completion verification
# --------------------------------------------------

assert len(los_models) == 3
assert len(classification_models) == 2

assert classification_model_selection_completed is False

assert classification_test_evaluation_completed is False

assert classification_models[
    rf_classifier_name
] is rf_classifier_pipeline

assert classification_models[
    xgb_classifier_name
] is xgb_classifier_pipeline

check_is_fitted(
    classification_models[rf_classifier_name]
)

check_is_fitted(
    classification_models[xgb_classifier_name]
)

xgb_classifier_training_completed = True

print("\nTRAINING COMPLETION SUMMARY")
print("=" * 75)

print("XGBoost Classifier training: COMPLETED")
print("Validation probability generation: COMPLETED")
print("Probability evaluation: COMPLETED")
print("Baseline comparison: COMPLETED")
print("Default threshold evaluation: COMPLETED")
print("Proposal threshold evaluation: COMPLETED")
print("Confusion matrices: COMPLETED")
print("Classification reports: COMPLETED")
print("Probability sanity checks: PASSED")
print("Boosting structure checks: PASSED")
print("Proposal target assessment: COMPLETED")
print("Fitted model registry: UPDATED")
print("Preliminary comparison: COMPLETED")
print("Validation visualization: COMPLETED")
print("Training metadata: CREATED")

print(
    "\nRegression models retained:",
    len(los_models)
)

print(
    "Classification models trained:",
    len(classification_models)
)

print(
    "Total trained predictive models:",
    len(los_models)
    + len(classification_models)
)

print("\nAll five fitted pipelines retained in memory.")

print("No regression models have been retrained.")
print("Random Forest Classifier has not been retrained.")
print("No classification threshold has been optimized.")
print("No final classification model has been selected.")
print("Test datasets have NOT been evaluated.")
print("No model files have been exported.")

print(
    "\nReady for Cell 41: "
    "XGBoost Classifier Diagnostics."
)


In [ ]:
# CELL 41: XGBoost Classifier Diagnostics
# St. Hubert Medical Center
# Corrected V2 Dataset
# Training and validation diagnostics only

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.utils.validation import check_is_fitted
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

print("XGBOOST CLASSIFIER DIAGNOSTICS")
print("=" * 75)

# ==================================================
# 1. Verify existing fitted models
# ==================================================

assert xgb_classifier_training_completed is True
assert rf_classifier_diagnostics_completed is True

assert classification_model_selection_completed is False
assert classification_test_evaluation_completed is False

assert len(los_models) == 3
assert len(classification_models) == 2

assert classification_models[
    xgb_classifier_name
] is xgb_classifier_pipeline

check_is_fitted(xgb_classifier_pipeline)

xgb_diag_y_train = np.asarray(
    classification_y_train
).reshape(-1)

xgb_diag_y_val = np.asarray(
    classification_y_val
).reshape(-1)

xgb_diag_val_prob = np.asarray(
    xgb_classifier_val_probabilities,
    dtype=np.float64
).reshape(-1)

assert len(xgb_diag_y_train) == len(X_proxy_train)

assert (
    len(xgb_diag_y_val)
    == len(xgb_diag_val_prob)
    == len(X_proxy_val)
)

assert np.isfinite(xgb_diag_val_prob).all()

assert (
    (xgb_diag_val_prob >= 0)
    & (xgb_diag_val_prob <= 1)
).all()

xgb_diag_prevalence = float(
    xgb_diag_y_val.mean()
)

print("Fitted classifier verification: PASSED")
print("Validation records:", f"{len(xgb_diag_y_val):,}")
print("Regression pipelines retained:", len(los_models))

# ==================================================
# 2. Training versus validation performance
# ==================================================

print("\nTRAINING VS VALIDATION PERFORMANCE")
print("=" * 75)

print("Generating in-sample training probabilities...")

xgb_diag_train_start = time.perf_counter()

xgb_diag_train_prob = (
    xgb_classifier_pipeline.predict_proba(
        X_proxy_train
    )[:, xgb_positive_class_index]
)

xgb_diag_train_prediction_time = (
    time.perf_counter() - xgb_diag_train_start
)

assert np.isfinite(xgb_diag_train_prob).all()

assert (
    (xgb_diag_train_prob >= 0)
    & (xgb_diag_train_prob <= 1)
).all()

def xgb_diag_probability_metrics(y, probabilities):

    y = np.asarray(y)
    probabilities = np.asarray(probabilities)

    return {
        "ROC-AUC": float(
            roc_auc_score(y, probabilities)
        ),
        "PR-AUC (AP)": float(
            average_precision_score(y, probabilities)
        ),
        "Brier Score": float(
            brier_score_loss(y, probabilities)
        ),
        "Log Loss": float(
            log_loss(y, probabilities, labels=[0, 1])
        ),
        "Mean Probability": float(
            probabilities.mean()
        ),
        "Probability Std": float(
            probabilities.std()
        ),
        "Actual Prevalence": float(y.mean())
    }

xgb_diag_train_metrics = (
    xgb_diag_probability_metrics(
        xgb_diag_y_train,
        xgb_diag_train_prob
    )
)

xgb_diag_val_metrics = (
    xgb_diag_probability_metrics(
        xgb_diag_y_val,
        xgb_diag_val_prob
    )
)

xgb_diag_train_val_df = pd.DataFrame([
    {
        "Dataset": "Training (In-sample)",
        **xgb_diag_train_metrics
    },
    {
        "Dataset": "Validation",
        **xgb_diag_val_metrics
    }
])

display(xgb_diag_train_val_df.round(6))

xgb_diag_roc_gap = (
    xgb_diag_train_metrics["ROC-AUC"]
    - xgb_diag_val_metrics["ROC-AUC"]
)

xgb_diag_ap_gap = (
    xgb_diag_train_metrics["PR-AUC (AP)"]
    - xgb_diag_val_metrics["PR-AUC (AP)"]
)

print(
    "ROC-AUC training-validation gap:",
    f"{xgb_diag_roc_gap:.6f}"
)

print(
    "AP training-validation gap:",
    f"{xgb_diag_ap_gap:.6f}"
)

print(
    "Training probability generation time:",
    f"{xgb_diag_train_prediction_time:.4f} seconds"
)

print(
    "\nTraining metrics are in-sample estimates "
    "and are not independent performance estimates."
)

# ==================================================
# 3. Probability separation by actual class
# ==================================================

print("\nPROBABILITY SEPARATION BY ACTUAL CLASS")
print("=" * 75)

xgb_diag_class_rows = []

for actual_class in [0, 1]:

    probabilities = xgb_diag_val_prob[
        xgb_diag_y_val == actual_class
    ]

    xgb_diag_class_rows.append({
        "Actual Class": actual_class,
        "Records": int(len(probabilities)),
        "Mean Probability": float(
            probabilities.mean()
        ),
        "Median Probability": float(
            np.median(probabilities)
        ),
        "Probability Std": float(
            probabilities.std()
        ),
        "Minimum": float(probabilities.min()),
        "5th Percentile": float(
            np.quantile(probabilities, 0.05)
        ),
        "95th Percentile": float(
            np.quantile(probabilities, 0.95)
        ),
        "Maximum": float(probabilities.max())
    })

xgb_diag_class_separation_df = pd.DataFrame(
    xgb_diag_class_rows
)

display(xgb_diag_class_separation_df.round(6))

xgb_diag_positive_negative_gap = float(
    xgb_diag_val_prob[
        xgb_diag_y_val == 1
    ].mean()
    -
    xgb_diag_val_prob[
        xgb_diag_y_val == 0
    ].mean()
)

print(
    "Positive minus negative mean probability:",
    f"{xgb_diag_positive_negative_gap:.8f}"
)

print(
    "Difference in percentage points:",
    f"{xgb_diag_positive_negative_gap * 100:.6f}"
)

# ==================================================
# 4. Calibration analysis
# ==================================================

print("\nCALIBRATION ANALYSIS")
print("=" * 75)

xgb_diag_probability_df = pd.DataFrame({
    "Actual": xgb_diag_y_val,
    "Probability": xgb_diag_val_prob
})

xgb_diag_probability_df["Calibration_Bin"] = (
    pd.qcut(
        xgb_diag_probability_df["Probability"],
        q=10,
        duplicates="drop"
    )
)

xgb_diag_calibration_df = (
    xgb_diag_probability_df
    .groupby("Calibration_Bin", observed=True)
    .agg(
        Records=("Actual", "size"),
        Positive_Cases=("Actual", "sum"),
        Mean_Predicted_Probability=(
            "Probability", "mean"
        ),
        Observed_Positive_Rate=("Actual", "mean"),
        Minimum_Probability=("Probability", "min"),
        Maximum_Probability=("Probability", "max")
    )
    .reset_index()
)

xgb_diag_calibration_df["Calibration_Gap"] = (
    xgb_diag_calibration_df[
        "Mean_Predicted_Probability"
    ]
    -
    xgb_diag_calibration_df[
        "Observed_Positive_Rate"
    ]
)

xgb_diag_calibration_df[
    "Absolute_Calibration_Gap"
] = xgb_diag_calibration_df[
    "Calibration_Gap"
].abs()

assert (
    xgb_diag_calibration_df["Records"].sum()
    == len(xgb_diag_y_val)
)

xgb_diag_ece = float(
    np.average(
        xgb_diag_calibration_df[
            "Absolute_Calibration_Gap"
        ],
        weights=xgb_diag_calibration_df["Records"]
    )
)

xgb_diag_mean_calibration_gap = float(
    xgb_diag_val_prob.mean()
    - xgb_diag_prevalence
)

display(xgb_diag_calibration_df.round(6))

print(
    "Expected Calibration Error (ECE):",
    f"{xgb_diag_ece:.6f}"
)

print(
    "Mean predicted minus observed probability:",
    f"{xgb_diag_mean_calibration_gap:.6f}"
)

print(
    "Number of calibration bins:",
    len(xgb_diag_calibration_df)
)

print(
    "\nECE is descriptive and depends on binning."
)

print(
    "Agreement with overall prevalence does not "
    "establish individual-level calibration."
)

# ==================================================
# 5. Probability quality comparison
# ==================================================

print("\nPROBABILITY QUALITY COMPARISON")
print("=" * 75)

xgb_diag_baseline_prob = np.full(
    len(xgb_diag_y_val),
    float(classification_train_prevalence),
    dtype=np.float64
)

xgb_diag_baseline_metrics = (
    xgb_diag_probability_metrics(
        xgb_diag_y_val,
        xgb_diag_baseline_prob
    )
)

xgb_diag_brier_skill = float(
    1
    - xgb_diag_val_metrics["Brier Score"]
    / xgb_diag_baseline_metrics["Brier Score"]
)

xgb_diag_log_loss_reduction = float(
    1
    - xgb_diag_val_metrics["Log Loss"]
    / xgb_diag_baseline_metrics["Log Loss"]
)

xgb_diag_probability_quality_df = pd.DataFrame([
    {
        "Model": "Constant Training Prevalence",
        "Brier Score": (
            xgb_diag_baseline_metrics["Brier Score"]
        ),
        "Log Loss": (
            xgb_diag_baseline_metrics["Log Loss"]
        ),
        "Brier Skill Score": 0.0,
        "Relative Log Loss Reduction": 0.0
    },
    {
        "Model": xgb_classifier_name,
        "Brier Score": (
            xgb_diag_val_metrics["Brier Score"]
        ),
        "Log Loss": (
            xgb_diag_val_metrics["Log Loss"]
        ),
        "Brier Skill Score": xgb_diag_brier_skill,
        "Relative Log Loss Reduction": (
            xgb_diag_log_loss_reduction
        )
    }
])

display(xgb_diag_probability_quality_df.round(6))

# ==================================================
# 6. Risk decile and cumulative gains analysis
# ==================================================

print("\nRISK DECILE AND LIFT ANALYSIS")
print("=" * 75)

# Reverse calibration-bin order so that
# Risk Group 1 contains the highest probabilities.
# Tied probabilities remain in the same bin.

xgb_diag_risk_df = (
    xgb_diag_calibration_df
    .sort_values(
        "Mean_Predicted_Probability",
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)

xgb_diag_risk_df.insert(
    0,
    "Risk_Group",
    np.arange(1, len(xgb_diag_risk_df) + 1)
)

xgb_diag_risk_df["Lift"] = (
    xgb_diag_risk_df["Observed_Positive_Rate"]
    / xgb_diag_prevalence
)

xgb_diag_risk_df["Cumulative_Records"] = (
    xgb_diag_risk_df["Records"].cumsum()
)

xgb_diag_risk_df["Cumulative_Positive_Cases"] = (
    xgb_diag_risk_df["Positive_Cases"].cumsum()
)

xgb_diag_risk_df[
    "Cumulative_Population_Percentage"
] = (
    xgb_diag_risk_df["Cumulative_Records"]
    / len(xgb_diag_y_val)
    * 100
)

xgb_diag_risk_df[
    "Cumulative_Positive_Capture_Percentage"
] = (
    xgb_diag_risk_df["Cumulative_Positive_Cases"]
    / xgb_diag_y_val.sum()
    * 100
)

xgb_diag_risk_df = xgb_diag_risk_df[
    [
        "Risk_Group",
        "Records",
        "Positive_Cases",
        "Mean_Predicted_Probability",
        "Observed_Positive_Rate",
        "Minimum_Probability",
        "Maximum_Probability",
        "Lift",
        "Cumulative_Records",
        "Cumulative_Positive_Cases",
        "Cumulative_Population_Percentage",
        "Cumulative_Positive_Capture_Percentage"
    ]
]

assert (
    xgb_diag_risk_df["Positive_Cases"].sum()
    == xgb_diag_y_val.sum()
)

display(xgb_diag_risk_df.round(6))

xgb_diag_top_lift = float(
    xgb_diag_risk_df.iloc[0]["Lift"]
)

xgb_diag_top_positive_rate = float(
    xgb_diag_risk_df.iloc[0][
        "Observed_Positive_Rate"
    ]
)

xgb_diag_top_capture = float(
    xgb_diag_risk_df.iloc[0][
        "Cumulative_Positive_Capture_Percentage"
    ]
)

print(
    "Highest-score group observed positive rate:",
    f"{xgb_diag_top_positive_rate * 100:.6f}%"
)

print(
    "Highest-score group lift:",
    f"{xgb_diag_top_lift:.6f}x"
)

print(
    "Highest-score group positive capture:",
    f"{xgb_diag_top_capture:.4f}%"
)

print(
    "Lowest-score group observed positive rate:",
    f"{xgb_diag_risk_df.iloc[-1]['Observed_Positive_Rate'] * 100:.6f}%"
)

print(
    "\nRisk Group 1 contains the highest scores."
)

print(
    "Quantile groups may have unequal sizes "
    "when probabilities contain ties."
)

# ==================================================
# 7. Internal feature importance
# ==================================================

print("\nTRANSFORMED FEATURE IMPORTANCE (GAIN)")
print("=" * 75)

xgb_diag_feature_names = (
    xgb_classifier_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

xgb_diag_booster = (
    xgb_classifier_estimator.get_booster()
)

# Use explicit gain and total_gain definitions.
# Missing entries represent features unused in splits.

xgb_diag_gain_scores = xgb_diag_booster.get_score(
    importance_type="gain"
)

xgb_diag_total_gain_scores = (
    xgb_diag_booster.get_score(
        importance_type="total_gain"
    )
)

xgb_diag_split_scores = (
    xgb_diag_booster.get_score(
        importance_type="weight"
    )
)

assert len(xgb_diag_feature_names) == (
    xgb_diag_booster.num_features()
)

xgb_diag_booster_names = (
    xgb_diag_booster.feature_names
)

if xgb_diag_booster_names is None:

    xgb_diag_booster_names = [
        f"f{i}"
        for i in range(len(xgb_diag_feature_names))
    ]

assert len(xgb_diag_booster_names) == len(
    xgb_diag_feature_names
)

xgb_diag_importance_rows = []

for feature_name, booster_name in zip(
    xgb_diag_feature_names,
    xgb_diag_booster_names
):

    xgb_diag_importance_rows.append({
        "Transformed_Feature": str(feature_name),
        "Mean_Gain": float(
            xgb_diag_gain_scores.get(
                booster_name, 0.0
            )
        ),
        "Total_Gain": float(
            xgb_diag_total_gain_scores.get(
                booster_name, 0.0
            )
        ),
        "Split_Count": int(
            xgb_diag_split_scores.get(
                booster_name, 0
            )
        )
    })

xgb_diag_transformed_importance_df = (
    pd.DataFrame(xgb_diag_importance_rows)
)

xgb_diag_total_gain = float(
    xgb_diag_transformed_importance_df[
        "Total_Gain"
    ].sum()
)

assert xgb_diag_total_gain > 0

xgb_diag_transformed_importance_df[
    "Total_Gain_Percentage"
] = (
    xgb_diag_transformed_importance_df["Total_Gain"]
    / xgb_diag_total_gain
    * 100
)

xgb_diag_transformed_importance_df = (
    xgb_diag_transformed_importance_df
    .sort_values("Total_Gain", ascending=False)
    .reset_index(drop=True)
)

display(
    xgb_diag_transformed_importance_df.round(6)
)

# Map transformed features back to original features.

def xgb_diag_original_feature_name(name):

    clean_name = str(name).split("__", 1)[-1]

    for original_feature in sorted(
        classification_features,
        key=len,
        reverse=True
    ):

        if (
            clean_name == original_feature
            or clean_name.startswith(
                original_feature + "_"
            )
        ):
            return original_feature

    raise ValueError(
        f"Unmapped transformed feature: {name}"
    )

xgb_diag_transformed_importance_df[
    "Original_Feature"
] = (
    xgb_diag_transformed_importance_df[
        "Transformed_Feature"
    ].map(xgb_diag_original_feature_name)
)

xgb_diag_grouped_importance_df = (
    xgb_diag_transformed_importance_df
    .groupby("Original_Feature", as_index=False)
    .agg(
        Total_Gain=("Total_Gain", "sum"),
        Split_Count=("Split_Count", "sum"),
        Total_Gain_Percentage=(
            "Total_Gain_Percentage", "sum"
        )
    )
    .sort_values("Total_Gain", ascending=False)
    .reset_index(drop=True)
)

assert set(
    xgb_diag_grouped_importance_df[
        "Original_Feature"
    ]
) == set(classification_features)

assert np.isclose(
    xgb_diag_grouped_importance_df[
        "Total_Gain_Percentage"
    ].sum(),
    100.0
)

print("\nGROUPED ORIGINAL FEATURE IMPORTANCE")
print("=" * 75)

display(
    xgb_diag_grouped_importance_df.round(6)
)

print(
    "Total gain measures cumulative training "
    "objective improvement from tree splits."
)

print(
    "It does not establish validation predictive "
    "value or clinical importance."
)

# ==================================================
# 8. Validation permutation importance
# ==================================================

print("\nVALIDATION PERMUTATION IMPORTANCE")
print("=" * 75)

xgb_diag_sample_size = min(
    30000,
    len(X_proxy_val)
)

xgb_diag_rng = np.random.default_rng(42)

xgb_diag_sample_positions = (
    xgb_diag_rng.choice(
        len(X_proxy_val),
        size=xgb_diag_sample_size,
        replace=False
    )
)

xgb_diag_sample_X = (
    X_proxy_val
    .iloc[xgb_diag_sample_positions]
    .copy()
)

xgb_diag_sample_y = (
    xgb_diag_y_val[xgb_diag_sample_positions]
)

assert len(xgb_diag_sample_X) == len(
    xgb_diag_sample_y
)

assert set(
    np.unique(xgb_diag_sample_y)
) == {0, 1}

xgb_diag_sample_prob = (
    xgb_classifier_pipeline.predict_proba(
        xgb_diag_sample_X
    )[:, xgb_positive_class_index]
)

xgb_diag_sample_ap = float(
    average_precision_score(
        xgb_diag_sample_y,
        xgb_diag_sample_prob
    )
)

print(
    "Validation sample size:",
    f"{xgb_diag_sample_size:,}"
)

print(
    "Sample positive prevalence:",
    f"{xgb_diag_sample_y.mean():.6f}"
)

print(
    "Sample baseline AP:",
    f"{xgb_diag_sample_ap:.6f}"
)

print("Permutation repeats: 5")
print("Scoring metric: Average Precision")
print("Random seed: 42")

print("\nCalculating permutation importance...")

xgb_diag_permutation_start = time.perf_counter()

# Sequential permutation execution avoids
# nested parallelism with the fitted XGBoost model.

xgb_diag_permutation_result = (
    permutation_importance(
        xgb_classifier_pipeline,
        xgb_diag_sample_X,
        xgb_diag_sample_y,
        scoring="average_precision",
        n_repeats=5,
        random_state=42,
        n_jobs=1
    )
)

xgb_diag_permutation_time = (
    time.perf_counter()
    - xgb_diag_permutation_start
)

xgb_diag_permutation_df = (
    pd.DataFrame({
        "Feature": list(
            xgb_diag_sample_X.columns
        ),
        "Mean_AP_Decrease": (
            xgb_diag_permutation_result.importances_mean
        ),
        "Importance_Std": (
            xgb_diag_permutation_result.importances_std
        )
    })
    .sort_values(
        "Mean_AP_Decrease",
        ascending=False
    )
    .reset_index(drop=True)
)

display(xgb_diag_permutation_df.round(6))

print(
    "Permutation importance time:",
    f"{xgb_diag_permutation_time:.4f} seconds"
)

print(
    "\nPositive importance means shuffling "
    "the feature reduced validation AP."
)

print(
    "Near-zero importance indicates limited "
    "measured predictive contribution."
)

print(
    "Negative importance means shuffling "
    "the feature improved AP on this sample."
)

# ==================================================
# 9. Marginal feature-target diagnostics
# ==================================================

xgb_diag_feature_data = (
    X_proxy_val.reset_index(drop=True).copy()
)

xgb_diag_feature_data["Actual"] = (
    xgb_diag_y_val
)

xgb_diag_feature_data["Probability"] = (
    xgb_diag_val_prob
)

xgb_diag_feature_data["AGE_GROUP"] = (
    pd.cut(
        xgb_diag_feature_data["AGE_AT_VISIT"],
        bins=[
            -np.inf, 18, 35, 50, 65, np.inf
        ],
        labels=[
            "0-17",
            "18-34",
            "35-49",
            "50-64",
            "65+"
        ],
        right=False
    )
    .astype("object")
    .fillna("Missing")
)

def xgb_diag_feature_summary(feature):

    result = (
        xgb_diag_feature_data
        .groupby(
            feature,
            observed=True,
            dropna=False
        )
        .agg(
            Records=("Actual", "size"),
            Positive_Cases=("Actual", "sum"),
            Actual_Positive_Rate=("Actual", "mean"),
            Mean_Predicted_Probability=(
                "Probability", "mean"
            )
        )
        .reset_index()
    )

    result["Observed_Lift"] = (
        result["Actual_Positive_Rate"]
        / xgb_diag_prevalence
    )

    result["Prediction_Gap"] = (
        result["Mean_Predicted_Probability"]
        - result["Actual_Positive_Rate"]
    )

    assert result["Records"].sum() == len(
        xgb_diag_y_val
    )

    return result

xgb_diag_feature_tables = {}

for feature in [
    "PRIOR_VISIT_COUNT",
    "ADMISSION_MONTH",
    "AGE_GROUP",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR"
]:

    print(
        f"\nFEATURE-TARGET DIAGNOSTIC: {feature}"
    )

    print("=" * 75)

    xgb_diag_feature_tables[feature] = (
        xgb_diag_feature_summary(feature)
    )

    display(
        xgb_diag_feature_tables[feature].round(6)
    )

print(
    "\nMarginal associations are descriptive "
    "and do not establish causal relationships."
)

print(
    "Groups with very small record counts "
    "must be interpreted cautiously."
)

# ==================================================
# 10. Threshold sensitivity analysis
# ==================================================

print("\nTHRESHOLD SENSITIVITY ANALYSIS")
print("=" * 75)

xgb_diag_thresholds = sorted(set([
    0.05,
    0.08,
    0.10,
    0.11,
    float(classification_train_prevalence),
    0.12,
    0.13,
    0.14,
    0.15,
    0.17,
    0.20,
    0.50,
    0.65
]))

xgb_diag_threshold_rows = []

for threshold in xgb_diag_thresholds:

    predictions = (
        xgb_diag_val_prob >= threshold
    ).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        xgb_diag_y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    xgb_diag_threshold_rows.append({
        "Threshold": float(threshold),
        "Accuracy": float(
            accuracy_score(
                xgb_diag_y_val,
                predictions
            )
        ),
        "Precision": float(
            precision_score(
                xgb_diag_y_val,
                predictions,
                zero_division=0
            )
        ),
        "Recall": float(
            recall_score(
                xgb_diag_y_val,
                predictions,
                zero_division=0
            )
        ),
        "F1": float(
            f1_score(
                xgb_diag_y_val,
                predictions,
                zero_division=0
            )
        ),
        "Specificity": float(
            tn / (tn + fp)
        ),
        "Predicted Positives": int(
            predictions.sum()
        ),
        "Predicted Positive Rate": float(
            predictions.mean()
        )
    })

xgb_diag_threshold_sensitivity_df = (
    pd.DataFrame(xgb_diag_threshold_rows)
)

display(
    xgb_diag_threshold_sensitivity_df.round(6)
)

xgb_diag_always_positive_precision = (
    xgb_diag_prevalence
)

xgb_diag_always_positive_recall = 1.0

xgb_diag_always_positive_f1 = float(
    2 * xgb_diag_prevalence
    / (1 + xgb_diag_prevalence)
)

print("\nALWAYS-POSITIVE REFERENCE")
print("=" * 75)

print(
    "Precision:",
    f"{xgb_diag_always_positive_precision:.6f}"
)

print("Recall: 1.000000")

print(
    "F1:",
    f"{xgb_diag_always_positive_f1:.6f}"
)

print("\nNo threshold has been selected.")

print(
    "Formal threshold optimization "
    "remains reserved for Cell 42."
)

# ==================================================
# 11. Probability separation visualization
# ==================================================

plt.figure(figsize=(10, 6))

xgb_diag_hist_bins = np.linspace(
    xgb_diag_val_prob.min(),
    xgb_diag_val_prob.max(),
    60
)

for actual_class, label in [
    (0, "Actual Negative"),
    (1, "Actual Positive")
]:

    plt.hist(
        xgb_diag_val_prob[
            xgb_diag_y_val == actual_class
        ],
        bins=xgb_diag_hist_bins,
        density=True,
        alpha=0.45,
        label=label
    )

plt.axvline(
    xgb_diag_prevalence,
    linestyle="--",
    label="Validation prevalence"
)

plt.xlabel("Predicted Positive Probability")
plt.ylabel("Density")
plt.title("XGBoost: Probability Separation by Actual Class")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 12. Calibration visualizations
# ==================================================

for zoomed in [False, True]:

    plt.figure(figsize=(8, 7))

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    plt.plot(
        xgb_diag_calibration_df[
            "Mean_Predicted_Probability"
        ],
        xgb_diag_calibration_df[
            "Observed_Positive_Rate"
        ],
        marker="o",
        label=f"XGBoost (ECE = {xgb_diag_ece:.4f})"
    )

    plt.scatter(
        classification_train_prevalence,
        xgb_diag_prevalence,
        marker="x",
        s=100,
        label="Constant prevalence baseline"
    )

    if zoomed:

        calibration_values = np.concatenate([
            xgb_diag_calibration_df[
                "Mean_Predicted_Probability"
            ].to_numpy(),
            xgb_diag_calibration_df[
                "Observed_Positive_Rate"
            ].to_numpy(),
            [xgb_diag_prevalence]
        ])

        lower = max(
            0,
            calibration_values.min() - 0.01
        )

        upper = min(
            1,
            calibration_values.max() + 0.01
        )

        plt.xlim(lower, upper)
        plt.ylim(lower, upper)

        plt.title(
            "XGBoost: Zoomed Calibration Curve"
        )

    else:

        plt.xlim(0, 1)
        plt.ylim(0, 1)

        plt.title(
            "XGBoost: Validation Calibration Curve"
        )

    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Positive Rate")
    plt.legend(loc="upper left")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

# ==================================================
# 13. Risk group visualization
# ==================================================

plt.figure(figsize=(10, 6))

plt.bar(
    xgb_diag_risk_df["Risk_Group"],
    xgb_diag_risk_df[
        "Observed_Positive_Rate"
    ] * 100,
    alpha=0.65,
    label="Observed positive rate"
)

plt.plot(
    xgb_diag_risk_df["Risk_Group"],
    xgb_diag_risk_df[
        "Mean_Predicted_Probability"
    ] * 100,
    marker="o",
    label="Mean predicted probability"
)

plt.axhline(
    xgb_diag_prevalence * 100,
    linestyle="--",
    label="Validation prevalence"
)

plt.xticks(xgb_diag_risk_df["Risk_Group"])
plt.xlabel("Risk Group (1 = Highest Score)")
plt.ylabel("Positive Rate (%)")
plt.title("XGBoost: Risk Group Analysis")
plt.legend()
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 14. Cumulative gains visualization
# ==================================================

plt.figure(figsize=(8, 6))

plt.plot(
    np.r_[
        0,
        xgb_diag_risk_df[
            "Cumulative_Population_Percentage"
        ].to_numpy()
    ],
    np.r_[
        0,
        xgb_diag_risk_df[
            "Cumulative_Positive_Capture_Percentage"
        ].to_numpy()
    ],
    marker="o",
    label="XGBoost"
)

plt.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    label="Random selection"
)

plt.xlabel("Population Selected (%)")
plt.ylabel("Positive Cases Captured (%)")
plt.title("XGBoost: Cumulative Gains Curve")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 15. Internal feature importance visualization
# ==================================================

xgb_diag_gain_plot_df = (
    xgb_diag_grouped_importance_df
    .sort_values("Total_Gain_Percentage")
)

plt.figure(figsize=(10, 6))

plt.barh(
    xgb_diag_gain_plot_df["Original_Feature"],
    xgb_diag_gain_plot_df["Total_Gain_Percentage"]
)

plt.xlabel("Total Gain Importance (%)")

plt.title(
    "XGBoost Classifier: Original Feature Importance"
)

plt.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 16. Permutation importance visualization
# ==================================================

xgb_diag_permutation_plot_df = (
    xgb_diag_permutation_df
    .sort_values("Mean_AP_Decrease")
)

plt.figure(figsize=(10, 6))

plt.barh(
    xgb_diag_permutation_plot_df["Feature"],
    xgb_diag_permutation_plot_df["Mean_AP_Decrease"],
    xerr=xgb_diag_permutation_plot_df["Importance_Std"],
    capsize=3
)

plt.axvline(0, linestyle="--")

plt.xlabel("Decrease in Average Precision")

plt.title(
    "XGBoost Classifier: Validation Permutation Importance"
)

plt.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 17. Threshold sensitivity visualization
# ==================================================

plt.figure(figsize=(11, 6))

for metric in [
    "Precision",
    "Recall",
    "F1",
    "Specificity"
]:

    plt.plot(
        xgb_diag_threshold_sensitivity_df["Threshold"],
        xgb_diag_threshold_sensitivity_df[metric],
        marker="o",
        label=metric
    )

plt.axhline(
    xgb_diag_always_positive_f1,
    linestyle=":",
    label="Always-positive F1 reference"
)

plt.axvline(
    0.50,
    linestyle="--",
    label="Default threshold"
)

plt.axvline(
    0.65,
    linestyle="--",
    label="Proposal threshold"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Metric Value")
plt.title("XGBoost: Threshold Sensitivity")
plt.ylim(0, 1.03)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 18. Marginal relationship visualizations
# ==================================================

for feature, title_text in [
    (
        "PRIOR_VISIT_COUNT",
        "Subsequent Visit Relationship with Prior Visit Count"
    ),
    (
        "ADMISSION_MONTH",
        "Subsequent Visit Relationship with Admission Month"
    )
]:

    plot_df = xgb_diag_feature_tables[feature]

    plt.figure(figsize=(10, 6))

    plt.plot(
        plot_df[feature],
        plot_df["Actual_Positive_Rate"] * 100,
        marker="o",
        label="Observed positive rate"
    )

    plt.plot(
        plot_df[feature],
        plot_df["Mean_Predicted_Probability"] * 100,
        marker="o",
        label="Mean predicted probability"
    )

    plt.axhline(
        xgb_diag_prevalence * 100,
        linestyle="--",
        label="Validation prevalence"
    )

    if feature == "ADMISSION_MONTH":

        plt.xticks(
            range(1, 13),
            [
                "Jan", "Feb", "Mar", "Apr",
                "May", "Jun", "Jul", "Aug",
                "Sep", "Oct", "Nov", "Dec"
            ]
        )

    else:

        plt.xticks(plot_df[feature])

    plt.ylabel("Positive Rate (%)")
    plt.title(title_text)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

# ==================================================
# 19. Key diagnostic results
# ==================================================

xgb_diag_summary = {
    "Training ROC-AUC": (
        xgb_diag_train_metrics["ROC-AUC"]
    ),
    "Validation ROC-AUC": (
        xgb_diag_val_metrics["ROC-AUC"]
    ),
    "Training AP": (
        xgb_diag_train_metrics["PR-AUC (AP)"]
    ),
    "Validation AP": (
        xgb_diag_val_metrics["PR-AUC (AP)"]
    ),
    "Expected Calibration Error": xgb_diag_ece,
    "Brier Skill Score": xgb_diag_brier_skill,
    "Highest Risk Group Lift": xgb_diag_top_lift,
    "Positive-Negative Probability Gap": (
        xgb_diag_positive_negative_gap
    )
}

xgb_diag_summary_df = pd.DataFrame(
    list(xgb_diag_summary.items()),
    columns=["Diagnostic", "Value"]
)

print("\nKEY DIAGNOSTIC RESULTS")
print("=" * 75)

display(xgb_diag_summary_df.round(6))

# ==================================================
# 20. Diagnostic metadata
# ==================================================

xgb_classifier_diagnostics_metadata = {
    "model": xgb_classifier_name,
    "dataset_version": "V2",
    "target": (
        "SUBSEQUENT_VISIT_WITHIN_30D_PROXY"
    ),
    "prediction_reference_point": "Admission",
    "training_metrics": xgb_diag_train_metrics,
    "validation_metrics": xgb_diag_val_metrics,
    "roc_auc_generalization_gap": float(
        xgb_diag_roc_gap
    ),
    "ap_generalization_gap": float(
        xgb_diag_ap_gap
    ),
    "positive_negative_probability_gap": float(
        xgb_diag_positive_negative_gap
    ),
    "expected_calibration_error": float(
        xgb_diag_ece
    ),
    "calibration_bins": int(
        len(xgb_diag_calibration_df)
    ),
    "mean_calibration_gap": float(
        xgb_diag_mean_calibration_gap
    ),
    "brier_skill_score": float(
        xgb_diag_brier_skill
    ),
    "relative_log_loss_reduction": float(
        xgb_diag_log_loss_reduction
    ),
    "highest_risk_group_lift": float(
        xgb_diag_top_lift
    ),
    "highest_risk_group_positive_rate": float(
        xgb_diag_top_positive_rate
    ),
    "highest_risk_group_capture_percentage": float(
        xgb_diag_top_capture
    ),
    "internal_importance_type": "total_gain",
    "permutation_scoring": "average_precision",
    "permutation_sample_size": int(
        xgb_diag_sample_size
    ),
    "permutation_sample_baseline_ap": float(
        xgb_diag_sample_ap
    ),
    "permutation_repeats": 5,
    "permutation_random_seed": 42,
    "always_positive_f1": float(
        xgb_diag_always_positive_f1
    ),
    "training_metrics_are_in_sample": True,
    "model_retrained": False,
    "probability_calibration_fitted": False,
    "threshold_optimized": False,
    "test_set_evaluated": False,
    "clinical_deployment_approved": False
}

json.dumps(
    xgb_classifier_diagnostics_metadata,
    allow_nan=False
)

# ==================================================
# 21. Completion verification
# ==================================================

assert len(los_models) == 3
assert len(classification_models) == 2

assert classification_model_selection_completed is False

assert classification_test_evaluation_completed is False

assert classification_models[
    xgb_classifier_name
] is xgb_classifier_pipeline

assert classification_models[
    rf_classifier_name
] is rf_classifier_pipeline

check_is_fitted(xgb_classifier_pipeline)
check_is_fitted(rf_classifier_pipeline)

xgb_classifier_diagnostics_completed = True

# Release unnecessary temporary feature copies.
# Retain fitted pipelines, metrics, and diagnostic tables.

del xgb_diag_feature_data
del xgb_diag_sample_X
del xgb_diag_sample_y

print("\nDIAGNOSTICS COMPLETION SUMMARY")
print("=" * 75)

print("Training-validation comparison: COMPLETED")
print("Probability separation: COMPLETED")
print("Calibration analysis: COMPLETED")
print("Brier and log loss comparison: COMPLETED")
print("Risk decile analysis: COMPLETED")
print("Cumulative gains analysis: COMPLETED")
print("Transformed feature importance: COMPLETED")
print("Grouped feature importance: COMPLETED")
print("Validation permutation importance: COMPLETED")
print("Marginal feature-target diagnostics: COMPLETED")
print("Threshold sensitivity analysis: COMPLETED")
print("Diagnostic visualization: COMPLETED")
print("Diagnostic metadata: CREATED")
print("JSON serialization verification: PASSED")

print("\nRegression models retained:", len(los_models))

print(
    "Classification models retained:",
    len(classification_models)
)

print("\nAll five fitted pipelines retained in memory.")
print("No models have been retrained.")
print("No probability calibration has been fitted.")
print("No classification threshold has been optimized.")
print("No final classification model has been selected.")
print("Test datasets have NOT been evaluated.")
print("No model files have been exported.")

print(
    "\nReady for Cell 42: "
    "Classification Threshold Optimization."
)


In [ ]:
# CELL 42: Classification Threshold Optimization
# St. Hubert Medical Center
# Corrected V2 Dataset
# Validation-only threshold optimization

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

print("CLASSIFICATION THRESHOLD OPTIMIZATION")
print("=" * 75)

# ==================================================
# 1. Verify existing fitted models and diagnostics
# ==================================================

assert rf_classifier_diagnostics_completed is True
assert xgb_classifier_diagnostics_completed is True

assert classification_model_selection_completed is False
assert classification_test_evaluation_completed is False

assert len(los_models) == 3
assert len(classification_models) == 2

check_is_fitted(rf_classifier_pipeline)
check_is_fitted(xgb_classifier_pipeline)

assert classification_models[
    rf_classifier_name
] is rf_classifier_pipeline

assert classification_models[
    xgb_classifier_name
] is xgb_classifier_pipeline

threshold_y_val = np.asarray(
    classification_y_val
).reshape(-1)

assert set(np.unique(threshold_y_val)) == {0, 1}

# Generate validation probabilities directly
# from existing fitted classifier pipelines.

threshold_validation_probabilities = {}

for model_name, pipeline in {
    rf_classifier_name: rf_classifier_pipeline,
    xgb_classifier_name: xgb_classifier_pipeline
}.items():

    check_is_fitted(pipeline)

    classes = np.asarray(pipeline.classes_)

    positive_class_index = np.flatnonzero(
        classes == 1
    )

    assert len(positive_class_index) == 1

    probabilities = pipeline.predict_proba(
        X_proxy_val
    )[:, positive_class_index[0]]

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    ).reshape(-1)

    assert len(probabilities) == len(
        classification_y_val
    )

    assert np.isfinite(probabilities).all()

    assert (
        (probabilities >= 0) &
        (probabilities <= 1)
    ).all()

    threshold_validation_probabilities[
        model_name
    ] = probabilities

    print(
        f"{model_name}: "
        f"{len(probabilities):,} validation "
        "probabilities generated."
    )

for model_name, probabilities in (
    threshold_validation_probabilities.items()
):

    assert len(probabilities) == len(threshold_y_val)

    assert np.isfinite(probabilities).all()

    assert (
        (probabilities >= 0)
        & (probabilities <= 1)
    ).all()

threshold_validation_prevalence = float(
    threshold_y_val.mean()
)

threshold_validation_records = int(
    len(threshold_y_val)
)

threshold_validation_positives = int(
    threshold_y_val.sum()
)

threshold_validation_negatives = (
    threshold_validation_records
    - threshold_validation_positives
)

print("Random Forest diagnostics: VERIFIED")
print("XGBoost diagnostics: VERIFIED")
print("Fitted classifier pipelines: VERIFIED")

print(
    "Validation records:",
    f"{threshold_validation_records:,}"
)

print(
    "Validation positives:",
    f"{threshold_validation_positives:,}"
)

print(
    "Validation negatives:",
    f"{threshold_validation_negatives:,}"
)

print(
    "Validation prevalence:",
    f"{threshold_validation_prevalence:.6%}"
)

print("Test dataset accessed: NO")

# ==================================================
# 2. Threshold optimization policy
# ==================================================

print("\nTHRESHOLD OPTIMIZATION POLICY")
print("=" * 75)

classification_threshold_policy = {
    "dataset_version": "V2",
    "optimization_dataset": "Validation",
    "optimization_metric": "Positive-class F1",
    "optimization_direction": "Maximize",
    "candidate_thresholds": (
        "All distinct validation probabilities"
    ),
    "prediction_rule": (
        "Positive if probability >= threshold"
    ),
    "tie_breaking": (
        "Highest threshold among numerical F1 ties"
    ),
    "tie_absolute_tolerance": 1e-12,
    "default_threshold": 0.50,
    "proposal_threshold": 0.65,
    "positive_class": 1,
    "model_retraining": False,
    "probability_calibration_fitted": False,
    "test_set_used_for_optimization": False,
    "clinical_threshold_approved": False
}

for key, value in (
    classification_threshold_policy.items()
):
    print(f"{key}: {value}")

# ==================================================
# 3. Reusable threshold evaluation function
# ==================================================

def evaluate_classification_threshold(
    y_true,
    probabilities,
    threshold
):

    y_true = np.asarray(y_true).reshape(-1)

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    ).reshape(-1)

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    matrix = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    )

    tn, fp, fn, tp = [
        int(value)
        for value in matrix.ravel()
    ]

    specificity = (
        tn / (tn + fp)
        if tn + fp > 0
        else 0.0
    )

    recall = float(
        recall_score(
            y_true,
            predictions,
            zero_division=0
        )
    )

    metrics = {
        "Threshold": float(threshold),
        "Accuracy": float(
            accuracy_score(y_true, predictions)
        ),
        "Precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "Recall": recall,
        "F1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0
            )
        ),
        "Specificity": float(specificity),
        "Balanced Accuracy": float(
            (recall + specificity) / 2
        ),
        "Predicted Positives": int(
            predictions.sum()
        ),
        "Predicted Positive Rate": float(
            predictions.mean()
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    assert tn + fp + fn + tp == len(y_true)

    return metrics, matrix

# ==================================================
# 4. Reference baseline evaluation
# ==================================================

print("\nTHRESHOLD REFERENCE BASELINES")
print("=" * 75)

# Constant zero scores with threshold 0.5:
# every record is classified as negative.

always_negative_metrics, always_negative_matrix = (
    evaluate_classification_threshold(
        threshold_y_val,
        np.zeros(threshold_validation_records),
        0.50
    )
)

# Constant one scores with threshold 0.5:
# every record is classified as positive.

always_positive_metrics, always_positive_matrix = (
    evaluate_classification_threshold(
        threshold_y_val,
        np.ones(threshold_validation_records),
        0.50
    )
)

classification_threshold_baseline_df = (
    pd.DataFrame([
        {
            "Model": "Always Negative",
            **always_negative_metrics
        },
        {
            "Model": "Always Positive",
            **always_positive_metrics
        }
    ])
    .drop(columns=["Threshold"])
)

display(
    classification_threshold_baseline_df.round(6)
)

threshold_reference_f1 = float(
    always_positive_metrics["F1"]
)

assert np.isclose(
    threshold_reference_f1,
    (
        2 * threshold_validation_prevalence
        / (1 + threshold_validation_prevalence)
    )
)

print(
    "Always-positive reference F1:",
    f"{threshold_reference_f1:.9f}"
)

print("\nALWAYS-NEGATIVE CONFUSION MATRIX")

display(pd.DataFrame(
    always_negative_matrix,
    index=["Actual Negative", "Actual Positive"],
    columns=[
        "Predicted Negative",
        "Predicted Positive"
    ]
))

print("\nALWAYS-POSITIVE CONFUSION MATRIX")

display(pd.DataFrame(
    always_positive_matrix,
    index=["Actual Negative", "Actual Positive"],
    columns=[
        "Predicted Negative",
        "Predicted Positive"
    ]
))

# ==================================================
# 5. Efficient distinct-threshold optimization
# ==================================================

def optimize_validation_f1(
    y_true,
    probabilities
):

    start_time = time.perf_counter()

    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            probabilities,
            pos_label=1,
            drop_intermediate=False
        )
    )

    # The final precision and recall pair has
    # no corresponding probability threshold.

    precision = precision[:-1]
    recall = recall[:-1]

    assert (
        len(precision)
        == len(recall)
        == len(thresholds)
    )

    assert len(thresholds) > 0

    assert len(thresholds) == len(
        np.unique(probabilities)
    )

    denominator = precision + recall

    f1_values = np.divide(
        2 * precision * recall,
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0
    )

    maximum_f1 = float(f1_values.max())

    # Resolve numerical ties using the highest
    # threshold, which minimizes predicted positives
    # among the tied candidates.

    tied_positions = np.flatnonzero(
        np.isclose(
            f1_values,
            maximum_f1,
            rtol=0,
            atol=1e-12
        )
    )

    best_position = int(
        tied_positions[-1]
    )

    best_threshold = float(
        thresholds[best_position]
    )

    verified_metrics, verified_matrix = (
        evaluate_classification_threshold(
            y_true,
            probabilities,
            best_threshold
        )
    )

    assert np.isclose(
        verified_metrics["F1"],
        maximum_f1,
        rtol=0,
        atol=1e-12
    )

    assert np.isclose(
        verified_metrics["Precision"],
        precision[best_position]
    )

    assert np.isclose(
        verified_metrics["Recall"],
        recall[best_position]
    )

    # Minimum observed threshold predicts all
    # validation records as positive.

    assert np.isclose(
        f1_values[0],
        threshold_reference_f1
    )

    assert (
        maximum_f1 + 1e-12
        >= threshold_reference_f1
    )

    curve_df = pd.DataFrame({
        "Threshold": thresholds,
        "Precision": precision,
        "Recall": recall,
        "F1": f1_values
    })

    optimization_time = (
        time.perf_counter() - start_time
    )

    return {
        "threshold": best_threshold,
        "maximum_f1": maximum_f1,
        "candidate_count": int(len(thresholds)),
        "tie_count": int(len(tied_positions)),
        "optimization_time_seconds": float(
            optimization_time
        ),
        "metrics": verified_metrics,
        "confusion_matrix": verified_matrix,
        "curve": curve_df
    }

# ==================================================
# 6. Optimize both fitted classifiers
# ==================================================

classification_threshold_optimization_results = {}
classification_optimized_thresholds = {}
classification_threshold_curves = {}

classification_threshold_comparison_rows = []
classification_threshold_summary_rows = []

classification_optimized_confusion_matrices = {}
classification_optimized_reports = {}

for model_name, probabilities in (
    threshold_validation_probabilities.items()
):

    print(f"\n{model_name.upper()}")
    print("=" * 75)

    result = optimize_validation_f1(
        threshold_y_val,
        probabilities
    )

    classification_threshold_optimization_results[
        model_name
    ] = result

    classification_optimized_thresholds[
        model_name
    ] = result["threshold"]

    classification_threshold_curves[
        model_name
    ] = result["curve"]

    optimized_metrics = result["metrics"]

    absolute_f1_improvement = float(
        optimized_metrics["F1"]
        - threshold_reference_f1
    )

    relative_f1_improvement = float(
        absolute_f1_improvement
        / threshold_reference_f1
        * 100
    )

    print(
        "Distinct candidate thresholds:",
        f"{result['candidate_count']:,}"
    )

    print(
        "Numerically tied optimal thresholds:",
        result["tie_count"]
    )

    print(
        "Optimized threshold:",
        f"{result['threshold']:.12f}"
    )

    print(
        "Maximum validation F1:",
        f"{result['maximum_f1']:.9f}"
    )

    print(
        "Optimization time:",
        f"{result['optimization_time_seconds']:.4f} seconds"
    )

    print(
        "Absolute F1 improvement over always-positive:",
        f"{absolute_f1_improvement:.9f}"
    )

    print(
        "Relative F1 improvement over always-positive:",
        f"{relative_f1_improvement:.6f}%"
    )

    # Evaluate default, proposal, and optimized
    # thresholds using the same metric function.

    model_threshold_rows = []

    for threshold_type, threshold in [
        ("Default", 0.50),
        ("Proposal", 0.65),
        ("F1-Optimized", result["threshold"])
    ]:

        metrics, matrix = (
            evaluate_classification_threshold(
                threshold_y_val,
                probabilities,
                threshold
            )
        )

        row = {
            "Model": model_name,
            "Threshold Type": threshold_type,
            **metrics
        }

        model_threshold_rows.append(row)

        classification_threshold_comparison_rows.append(
            row
        )

    model_threshold_df = pd.DataFrame(
        model_threshold_rows
    )

    display(model_threshold_df.round(6))

    classification_optimized_confusion_matrices[
        model_name
    ] = result["confusion_matrix"]

    print("\nOPTIMIZED CONFUSION MATRIX")

    display(pd.DataFrame(
        result["confusion_matrix"],
        index=[
            "Actual Negative",
            "Actual Positive"
        ],
        columns=[
            "Predicted Negative",
            "Predicted Positive"
        ]
    ))

    optimized_predictions = (
        probabilities >= result["threshold"]
    ).astype(np.int64)

    report = classification_report(
        threshold_y_val,
        optimized_predictions,
        labels=[0, 1],
        target_names=[
            "Negative (0)",
            "Positive (1)"
        ],
        output_dict=True,
        zero_division=0
    )

    classification_optimized_reports[
        model_name
    ] = report

    print("\nOPTIMIZED CLASSIFICATION REPORT")

    display(
        pd.DataFrame(report).T.round(6)
    )

    classification_threshold_summary_rows.append({
        "Model": model_name,
        "Optimized Threshold": result["threshold"],
        "Optimized F1": optimized_metrics["F1"],
        "Always-Positive F1": threshold_reference_f1,
        "Absolute F1 Improvement": (
            absolute_f1_improvement
        ),
        "Relative F1 Improvement (%)": (
            relative_f1_improvement
        ),
        "Precision": optimized_metrics["Precision"],
        "Recall": optimized_metrics["Recall"],
        "Specificity": optimized_metrics["Specificity"],
        "Balanced Accuracy": (
            optimized_metrics["Balanced Accuracy"]
        ),
        "Predicted Positives": (
            optimized_metrics["Predicted Positives"]
        ),
        "Predicted Positive Rate": (
            optimized_metrics["Predicted Positive Rate"]
        ),
        "True Positives": optimized_metrics["TP"],
        "False Positives": optimized_metrics["FP"],
        "True Negatives": optimized_metrics["TN"],
        "False Negatives": optimized_metrics["FN"],
        "Candidate Thresholds": (
            result["candidate_count"]
        ),
        "Optimization Time (s)": (
            result["optimization_time_seconds"]
        )
    })

# ==================================================
# 7. Consolidated threshold comparison
# ==================================================

classification_threshold_comparison_df = (
    pd.DataFrame(
        classification_threshold_comparison_rows
    )
)

classification_threshold_optimization_summary_df = (
    pd.DataFrame(
        classification_threshold_summary_rows
    )
)

print("\nCONSOLIDATED THRESHOLD COMPARISON")
print("=" * 75)

display(
    classification_threshold_comparison_df.round(6)
)

print("\nOPTIMIZED THRESHOLD SUMMARY")
print("=" * 75)

display(
    classification_threshold_optimization_summary_df
    .round(9)
)

# ==================================================
# 8. Assess optimized thresholds against baselines
# ==================================================

print("\nTHRESHOLD UTILITY ASSESSMENT")
print("=" * 75)

classification_threshold_utility_rows = []

for model_name, result in (
    classification_threshold_optimization_results.items()
):

    metrics = result["metrics"]

    f1_gain = float(
        metrics["F1"]
        - threshold_reference_f1
    )

    precision_gain = float(
        metrics["Precision"]
        - threshold_validation_prevalence
    )

    balanced_accuracy_gain = float(
        metrics["Balanced Accuracy"] - 0.50
    )

    positive_rate = float(
        metrics["Predicted Positive Rate"]
    )

    classification_threshold_utility_rows.append({
        "Model": model_name,
        "F1 Above Always-Positive": bool(
            f1_gain > 1e-12
        ),
        "Precision Above Prevalence": bool(
            precision_gain > 1e-12
        ),
        "Balanced Accuracy Above Chance": bool(
            balanced_accuracy_gain > 1e-12
        ),
        "Predicted Positive Rate (%)": (
            positive_rate * 100
        ),
        "Records Not Flagged": int(
            threshold_validation_records
            - metrics["Predicted Positives"]
        ),
        "False Positives": metrics["FP"],
        "False Negatives": metrics["FN"],
        "Threshold Role": "EXPERIMENTAL_ONLY",
        "Clinical Approval": False
    })

classification_threshold_utility_df = pd.DataFrame(
    classification_threshold_utility_rows
)

display(
    classification_threshold_utility_df.round(6)
)

print(
    "\nA numerical improvement over a baseline "
    "does not establish statistical significance "
    "or clinical usefulness."
)

print(
    "F1 does not account for true negatives "
    "or clinical intervention costs."
)

print(
    "Threshold optimization cannot improve "
    "ROC-AUC, average precision, or Brier score "
    "when the underlying probabilities are unchanged."
)

# ==================================================
# 9. Full F1 optimization curves
# ==================================================

for model_name, result in (
    classification_threshold_optimization_results.items()
):

    curve = result["curve"]

    plt.figure(figsize=(10, 6))

    plt.plot(
        curve["Threshold"],
        curve["F1"],
        label="Validation F1",
        linewidth=1.5
    )

    plt.axhline(
        threshold_reference_f1,
        linestyle=":",
        label="Always-positive F1"
    )

    plt.axvline(
        result["threshold"],
        linestyle="--",
        label=(
            f"Optimized threshold "
            f"({result['threshold']:.6f})"
        )
    )

    plt.scatter(
        result["threshold"],
        result["metrics"]["F1"],
        s=70,
        zorder=5
    )

    plt.xlabel("Classification Threshold")
    plt.ylabel("Positive-Class F1")

    plt.title(
        f"{model_name}: Full Validation F1 Search"
    )

    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

# ==================================================
# 10. Zoomed F1 optimization curves
# ==================================================

for model_name, result in (
    classification_threshold_optimization_results.items()
):

    curve = result["curve"]

    best_threshold = result["threshold"]

    zoom_lower = max(
        float(curve["Threshold"].min()),
        best_threshold - 0.015
    )

    zoom_upper = min(
        float(curve["Threshold"].max()),
        best_threshold + 0.015
    )

    zoom_curve = curve[
        curve["Threshold"].between(
            zoom_lower,
            zoom_upper
        )
    ]

    plt.figure(figsize=(10, 6))

    plt.plot(
        zoom_curve["Threshold"],
        zoom_curve["F1"],
        label="Validation F1"
    )

    plt.axhline(
        threshold_reference_f1,
        linestyle=":",
        label="Always-positive F1"
    )

    plt.axvline(
        best_threshold,
        linestyle="--",
        label="Optimized threshold"
    )

    plt.scatter(
        best_threshold,
        result["metrics"]["F1"],
        s=70,
        zorder=5
    )

    plt.xlabel("Classification Threshold")
    plt.ylabel("Positive-Class F1")

    plt.title(
        f"{model_name}: Zoomed F1 Optimization"
    )

    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

# ==================================================
# 11. Optimized confusion matrix visualization
# ==================================================

for model_name, matrix in (
    classification_optimized_confusion_matrices.items()
):

    fig, ax = plt.subplots(figsize=(6, 5))

    image = ax.imshow(
        matrix,
        cmap="Blues"
    )

    for row in range(2):
        for column in range(2):

            value = int(matrix[row, column])

            text_color = (
                "white"
                if value > matrix.max() / 2
                else "black"
            )

            ax.text(
                column,
                row,
                f"{value:,}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=13
            )

    ax.set_xticks(
        [0, 1],
        labels=["Negative", "Positive"]
    )

    ax.set_yticks(
        [0, 1],
        labels=["Negative", "Positive"]
    )

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

    ax.set_title(
        f"{model_name}\n"
        f"Optimized Threshold: "
        f"{classification_optimized_thresholds[model_name]:.6f}"
    )

    fig.colorbar(image, ax=ax)

    plt.tight_layout()
    plt.show()

# ==================================================
# 12. Optimized metric comparison visualization
# ==================================================

threshold_metric_names = [
    "Precision",
    "Recall",
    "F1",
    "Specificity",
    "Balanced Accuracy"
]

threshold_plot_models = {
    "Always Negative": always_negative_metrics,
    "Always Positive": always_positive_metrics
}

for model_name, result in (
    classification_threshold_optimization_results.items()
):

    threshold_plot_models[model_name] = (
        result["metrics"]
    )

positions = np.arange(
    len(threshold_metric_names)
)

bar_width = 0.19

plt.figure(figsize=(12, 6))

for index, (model_name, metrics) in enumerate(
    threshold_plot_models.items()
):

    values = [
        metrics[metric]
        for metric in threshold_metric_names
    ]

    offset = (
        index
        - (len(threshold_plot_models) - 1) / 2
    ) * bar_width

    plt.bar(
        positions + offset,
        values,
        width=bar_width,
        label=model_name
    )

plt.xticks(
    positions,
    threshold_metric_names
)

plt.ylabel("Metric Value")

plt.title(
    "Optimized Classification Thresholds "
    "versus Reference Baselines"
)

plt.ylim(0, 1.08)
plt.legend()
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 13. Predicted positive rate comparison
# ==================================================

threshold_rate_labels = list(
    threshold_plot_models.keys()
)

threshold_rate_values = [
    metrics["Predicted Positive Rate"] * 100
    for metrics in threshold_plot_models.values()
]

plt.figure(figsize=(10, 5))

bars = plt.bar(
    threshold_rate_labels,
    threshold_rate_values
)

plt.bar_label(
    bars,
    fmt="%.2f%%",
    padding=3
)

plt.axhline(
    threshold_validation_prevalence * 100,
    linestyle="--",
    label="Actual validation prevalence"
)

plt.ylabel("Records Flagged Positive (%)")

plt.title(
    "Predicted Positive Rate at Optimized Thresholds"
)

plt.xticks(rotation=15, ha="right")
plt.ylim(0, 110)
plt.legend()
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ==================================================
# 14. Threshold optimization metadata
# ==================================================

classification_threshold_optimization_metadata = {
    "dataset_version": "V2",
    "target": (
        "SUBSEQUENT_VISIT_WITHIN_30D_PROXY"
    ),
    "prediction_reference_point": "Admission",
    "policy": classification_threshold_policy,
    "validation_records": (
        threshold_validation_records
    ),
    "validation_positives": (
        threshold_validation_positives
    ),
    "validation_negatives": (
        threshold_validation_negatives
    ),
    "validation_prevalence": (
        threshold_validation_prevalence
    ),
    "always_positive_reference_f1": (
        threshold_reference_f1
    ),
    "baseline_metrics": {
        "always_negative": always_negative_metrics,
        "always_positive": always_positive_metrics
    },
    "models": {},
    "limitations": [
        (
            "Thresholds were optimized and evaluated "
            "on the same validation dataset."
        ),
        (
            "Optimized validation F1 may be "
            "optimistically biased."
        ),
        (
            "F1 optimization does not establish "
            "clinical decision utility."
        ),
        (
            "Threshold changes do not improve "
            "underlying probability discrimination."
        ),
        (
            "Neither classifier has demonstrated "
            "validation discrimination above baseline."
        ),
        (
            "The target is a subsequent visit proxy, "
            "not validated unplanned readmission."
        ),
        (
            "No clinically justified intervention "
            "cost or capacity constraint is available."
        )
    ],
    "threshold_role": "EXPERIMENTAL_ONLY",
    "test_set_evaluated": False,
    "clinical_deployment_approved": False
}

for model_name, result in (
    classification_threshold_optimization_results.items()
):

    classification_threshold_optimization_metadata[
        "models"
    ][model_name] = {
        "optimized_threshold": result["threshold"],
        "candidate_count": result["candidate_count"],
        "tie_count": result["tie_count"],
        "maximum_validation_f1": result["maximum_f1"],
        "validation_metrics": result["metrics"],
        "confusion_matrix": (
            result["confusion_matrix"].tolist()
        ),
        "absolute_f1_improvement_over_always_positive": (
            float(
                result["metrics"]["F1"]
                - threshold_reference_f1
            )
        ),
        "optimization_time_seconds": (
            result["optimization_time_seconds"]
        ),
        "clinical_threshold_approved": False
    }

json.dumps(
    classification_threshold_optimization_metadata,
    allow_nan=False
)

print("\nTHRESHOLD OPTIMIZATION METADATA")
print("=" * 75)

print("Metadata creation: COMPLETED")
print("JSON serialization verification: PASSED")

# ==================================================
# 15. Final completion verification
# ==================================================

assert len(
    classification_optimized_thresholds
) == 2

assert len(
    classification_threshold_optimization_results
) == 2

assert len(
    classification_optimized_confusion_matrices
) == 2

assert len(
    classification_threshold_comparison_df
) == 6

assert len(los_models) == 3
assert len(classification_models) == 2

assert classification_model_selection_completed is False
assert classification_test_evaluation_completed is False

check_is_fitted(rf_classifier_pipeline)
check_is_fitted(xgb_classifier_pipeline)

classification_threshold_optimization_completed = True

print("\nTHRESHOLD OPTIMIZATION COMPLETION SUMMARY")
print("=" * 75)

print("Existing fitted classifiers: VERIFIED")
print("Validation-only optimization: VERIFIED")
print("Always-negative baseline: COMPLETED")
print("Always-positive baseline: COMPLETED")
print("Random Forest threshold optimization: COMPLETED")
print("XGBoost threshold optimization: COMPLETED")
print("Distinct threshold search: COMPLETED")
print("Optimized F1 verification: PASSED")
print("Default threshold comparison: COMPLETED")
print("Proposal threshold comparison: COMPLETED")
print("Optimized confusion matrices: COMPLETED")
print("Optimized classification reports: COMPLETED")
print("Threshold utility assessment: COMPLETED")
print("Diagnostic visualization: COMPLETED")
print("Threshold metadata: CREATED")
print("JSON serialization verification: PASSED")

print("\nEXPERIMENTAL OPTIMIZED THRESHOLDS")

for model_name, threshold in (
    classification_optimized_thresholds.items()
):

    print(f"{model_name}: {threshold:.12f}")

print("\nRegression models retained:", len(los_models))

print(
    "Classification models retained:",
    len(classification_models)
)

print("\nAll five fitted pipelines retained in memory.")
print("No models have been retrained.")
print("No probability calibration has been fitted.")
print("No final classification model has been selected.")
print("No clinical threshold has been approved.")
print("Test datasets have NOT been evaluated.")
print("No model files have been exported.")

print(
    "\nReady for Cell 43: "
    "Comprehensive Classification Model Comparison."
)


In [ ]:

# CELL 43
# COMPREHENSIVE CLASSIFICATION MODEL COMPARISON

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from IPython.display import display
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("COMPREHENSIVE CLASSIFICATION MODEL COMPARISON")
print("=" * 75)

# ============================================================
# 1. VERIFY EXISTING CLASSIFICATION COMPONENTS
# ============================================================

required_variables = [
    "rf_classifier_pipeline",
    "xgb_classifier_pipeline",
    "X_proxy_val",
    "classification_y_val",
    "y_proxy_train",
    "threshold_validation_probabilities",
    "rf_classifier_name",
    "xgb_classifier_name"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing variables: {missing_variables}"
)

check_is_fitted(rf_classifier_pipeline)
check_is_fitted(xgb_classifier_pipeline)

comparison_y_val = np.asarray(
    classification_y_val
).reshape(-1)

comparison_y_train = np.asarray(
    y_proxy_train
).reshape(-1)

assert len(comparison_y_val) == len(X_proxy_val)
assert len(comparison_y_val) == 109458
assert set(np.unique(comparison_y_val)) == {0, 1}

comparison_train_prevalence = float(
    comparison_y_train.mean()
)

comparison_val_prevalence = float(
    comparison_y_val.mean()
)

print("Existing fitted classifiers: VERIFIED")
print("Validation dataset: VERIFIED")
print(f"Validation records: {len(comparison_y_val):,}")
print(f"Training prevalence: {comparison_train_prevalence:.6f}")
print(f"Validation prevalence: {comparison_val_prevalence:.6f}")
print("Test dataset accessed: NO")

# ============================================================
# 2. RETRIEVE VALIDATION PROBABILITIES
# ============================================================

comparison_probabilities = {
    "Constant Training Prevalence": np.full(
        len(comparison_y_val),
        comparison_train_prevalence,
        dtype=np.float64
    ),
    "Random Forest Classifier": np.asarray(
        threshold_validation_probabilities[
            rf_classifier_name
        ],
        dtype=np.float64
    ).reshape(-1),
    "XGBoost Classifier": np.asarray(
        threshold_validation_probabilities[
            xgb_classifier_name
        ],
        dtype=np.float64
    ).reshape(-1)
}

for name, probabilities in comparison_probabilities.items():

    assert len(probabilities) == len(comparison_y_val)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) &
            (probabilities <= 1)).all()

    print(f"{name}: probabilities VERIFIED")


# Recover full-precision thresholds from Cell 42
from sklearn.metrics import precision_recall_curve

comparison_thresholds = {}

print("\nFULL-PRECISION THRESHOLD RECOVERY")
print("=" * 75)

for model_name in [
    "Random Forest Classifier",
    "XGBoost Classifier"
]:

    probabilities = comparison_probabilities[model_name]

    precision, recall, thresholds = precision_recall_curve(
        comparison_y_val,
        probabilities,
        pos_label=1
    )

    precision = precision[:-1]
    recall = recall[:-1]

    denominator = precision + recall

    f1_values = np.divide(
        2 * precision * recall,
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0
    )

    maximum_f1 = np.max(f1_values)

    optimal_indices = np.flatnonzero(
        np.isclose(
            f1_values,
            maximum_f1,
            rtol=0,
            atol=1e-12
        )
    )

    # Same tie-breaking policy as Cell 42:
    # choose the highest threshold among F1 ties.
    exact_threshold = float(
        np.max(thresholds[optimal_indices])
    )

    comparison_thresholds[model_name] = exact_threshold

    print(f"Model: {model_name}")
    print(f"Full-precision threshold: {exact_threshold!r}")
    print(f"Maximum validation F1: {maximum_f1:.12f}")
    print(f"Candidate thresholds: {len(thresholds):,}")
    print("-" * 75)

print("Full-precision threshold recovery: COMPLETED")


# ============================================================
# 3. PROBABILITY-BASED PERFORMANCE COMPARISON
# ============================================================

comparison_probability_rows = []

for name, probabilities in comparison_probabilities.items():

    comparison_probability_rows.append({
        "Model": name,
        "ROC-AUC": float(
            roc_auc_score(
                comparison_y_val,
                probabilities
            )
        ),
        "PR-AUC (AP)": float(
            average_precision_score(
                comparison_y_val,
                probabilities
            )
        ),
        "Brier Score": float(
            brier_score_loss(
                comparison_y_val,
                probabilities
            )
        ),
        "Log Loss": float(
            log_loss(
                comparison_y_val,
                probabilities,
                labels=[0, 1]
            )
        ),
        "Mean Probability": float(
            probabilities.mean()
        ),
        "Probability Std": float(
            probabilities.std()
        )
    })

classification_probability_comparison_df = pd.DataFrame(
    comparison_probability_rows
)

comparison_baseline = (
    classification_probability_comparison_df.iloc[0]
)

classification_probability_comparison_df[
    "ROC-AUC Improvement"
] = (
    classification_probability_comparison_df["ROC-AUC"]
    - comparison_baseline["ROC-AUC"]
)

classification_probability_comparison_df[
    "AP Improvement"
] = (
    classification_probability_comparison_df["PR-AUC (AP)"]
    - comparison_baseline["PR-AUC (AP)"]
)

classification_probability_comparison_df[
    "AP Lift"
] = (
    classification_probability_comparison_df["PR-AUC (AP)"]
    / comparison_baseline["PR-AUC (AP)"]
)

classification_probability_comparison_df[
    "Brier Skill Score"
] = (
    1 -
    classification_probability_comparison_df["Brier Score"]
    / comparison_baseline["Brier Score"]
)

classification_probability_comparison_df[
    "Relative Log Loss Reduction"
] = (
    1 -
    classification_probability_comparison_df["Log Loss"]
    / comparison_baseline["Log Loss"]
)

print("\nPROBABILITY PERFORMANCE COMPARISON")
print("=" * 75)

display(
    classification_probability_comparison_df.round(6)
)

# ============================================================
# 4. THRESHOLD-DEPENDENT PERFORMANCE
# ============================================================

def comparison_binary_metrics(
    model_name,
    threshold_type,
    threshold,
    predictions
):

    predictions = np.asarray(
        predictions
    ).reshape(-1)

    tn, fp, fn, tp = confusion_matrix(
        comparison_y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Model": model_name,
        "Threshold Type": threshold_type,
        "Threshold": threshold,
        "Accuracy": float(
            accuracy_score(
                comparison_y_val,
                predictions
            )
        ),
        "Precision": float(
            precision_score(
                comparison_y_val,
                predictions,
                zero_division=0
            )
        ),
        "Recall": float(
            recall_score(
                comparison_y_val,
                predictions,
                zero_division=0
            )
        ),
        "F1": float(
            f1_score(
                comparison_y_val,
                predictions,
                zero_division=0
            )
        ),
        "Specificity": float(tn / (tn + fp)),
        "Balanced Accuracy": float(
            balanced_accuracy_score(
                comparison_y_val,
                predictions
            )
        ),
        "Predicted Positives": int(predictions.sum()),
        "Predicted Positive Rate": float(
            predictions.mean()
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


comparison_threshold_rows = []

comparison_threshold_rows.append(
    comparison_binary_metrics(
        "Always Negative",
        "Reference",
        None,
        np.zeros_like(comparison_y_val)
    )
)

comparison_threshold_rows.append(
    comparison_binary_metrics(
        "Always Positive",
        "Reference",
        None,
        np.ones_like(comparison_y_val)
    )
)

for name, optimized_threshold in comparison_thresholds.items():

    probabilities = comparison_probabilities[name]

    for threshold_type, threshold in [
        ("Default", 0.50),
        ("Proposal", 0.65),
        ("F1-Optimized", optimized_threshold)
    ]:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        comparison_threshold_rows.append(
            comparison_binary_metrics(
                name,
                threshold_type,
                threshold,
                predictions
            )
        )

classification_threshold_comparison_df = pd.DataFrame(
    comparison_threshold_rows
)

print("\nTHRESHOLD PERFORMANCE COMPARISON")
print("=" * 75)

display(
    classification_threshold_comparison_df.round(6)
)

# ============================================================
# 5. VERIFY CELL 42 THRESHOLD REPRODUCIBILITY
# ============================================================

comparison_optimized_df = (
    classification_threshold_comparison_df[
        classification_threshold_comparison_df[
            "Threshold Type"
        ] == "F1-Optimized"
    ].copy()
)

comparison_expected_confusion = {
    "Random Forest Classifier": [268, 96345, 24, 12821],
    "XGBoost Classifier": [270, 96343, 27, 12818]
}

for _, row in comparison_optimized_df.iterrows():

    actual_counts = [
        int(row["TN"]),
        int(row["FP"]),
        int(row["FN"]),
        int(row["TP"])
    ]

    expected_counts = comparison_expected_confusion[
        row["Model"]
    ]

    assert actual_counts == expected_counts, (
        f"Threshold reproduction mismatch: {row['Model']}. "
        "Check the full-precision Cell 42 threshold."
    )

    print(
        f"{row['Model']}: "
        "Cell 42 confusion matrix reproduced."
    )

comparison_always_positive_f1 = float(
    classification_threshold_comparison_df.loc[
        classification_threshold_comparison_df[
            "Model"
        ] == "Always Positive",
        "F1"
    ].iloc[0]
)

comparison_optimized_df[
    "F1 Improvement"
] = (
    comparison_optimized_df["F1"]
    - comparison_always_positive_f1
)

comparison_optimized_df[
    "Relative F1 Improvement (%)"
] = (
    comparison_optimized_df["F1 Improvement"]
    / comparison_always_positive_f1
    * 100
)

comparison_optimized_df[
    "False Positives per True Positive"
] = (
    comparison_optimized_df["FP"]
    / comparison_optimized_df["TP"].replace(0, np.nan)
)

print("\nOPTIMIZED THRESHOLD UTILITY")
print("=" * 75)

display(comparison_optimized_df.round(6))

# ============================================================
# 6. RISK RANKING AND TOP-DECILE LIFT
# ============================================================

comparison_risk_rows = []

for name in comparison_thresholds:

    probabilities = comparison_probabilities[name]

    risk_data = pd.DataFrame({
        "Probability": probabilities,
        "Actual": comparison_y_val
    })

    risk_data["Risk_Group"] = (
        10 - pd.qcut(
            risk_data["Probability"],
            q=10,
            labels=False,
            duplicates="raise"
        )
    )

    top_group = risk_data[
        risk_data["Risk_Group"] == 1
    ]

    bottom_group = risk_data[
        risk_data["Risk_Group"] == 10
    ]

    top_rate = float(top_group["Actual"].mean())

    comparison_risk_rows.append({
        "Model": name,
        "Highest Risk Group Records": int(len(top_group)),
        "Highest Risk Group Positives": int(
            top_group["Actual"].sum()
        ),
        "Highest Risk Group Positive Rate": top_rate,
        "Highest Risk Group Lift": (
            top_rate / comparison_val_prevalence
        ),
        "Positive Capture Rate": float(
            top_group["Actual"].sum()
            / comparison_y_val.sum()
        ),
        "Lowest Risk Group Positive Rate": float(
            bottom_group["Actual"].mean()
        )
    })

classification_risk_comparison_df = pd.DataFrame(
    comparison_risk_rows
)

print("\nRISK RANKING COMPARISON")
print("=" * 75)

display(classification_risk_comparison_df.round(6))

# ============================================================
# 7. PROPOSAL TARGET ASSESSMENT
# ============================================================

comparison_target_rows = []

for name in comparison_thresholds:

    row = classification_probability_comparison_df.loc[
        classification_probability_comparison_df[
            "Model"
        ] == name
    ].iloc[0]

    for metric, target in [
        ("ROC-AUC", 0.80),
        ("PR-AUC (AP)", 0.75)
    ]:

        result = float(row[metric])

        comparison_target_rows.append({
            "Model": name,
            "Metric": metric,
            "Validation Result": result,
            "Proposal Boundary": target,
            "Required Direction": ">",
            "Gap to Boundary": target - result,
            "Target Status": (
                "ACHIEVED"
                if result > target
                else "NOT ACHIEVED"
            )
        })

classification_target_assessment_df = pd.DataFrame(
    comparison_target_rows
)

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 75)

display(classification_target_assessment_df.round(6))

# ============================================================
# 8. CONSOLIDATED MODEL COMPARISON
# ============================================================

classification_comprehensive_comparison_df = (
    classification_probability_comparison_df[
        classification_probability_comparison_df[
            "Model"
        ].isin(comparison_thresholds)
    ].merge(
        comparison_optimized_df[[
            "Model",
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "Specificity",
            "Balanced Accuracy",
            "Predicted Positive Rate",
            "F1 Improvement",
            "False Positives per True Positive"
        ]],
        on="Model",
        validate="one_to_one"
    ).merge(
        classification_risk_comparison_df[[
            "Model",
            "Highest Risk Group Lift",
            "Positive Capture Rate"
        ]],
        on="Model",
        validate="one_to_one"
    )
)

print("\nCOMPREHENSIVE CLASSIFIER COMPARISON")
print("=" * 75)

display(
    classification_comprehensive_comparison_df.round(6)
)

# ============================================================
# 9. METRIC-SPECIFIC MODEL RANKING
# ============================================================

comparison_ranking_rules = {
    "ROC-AUC": "maximize",
    "PR-AUC (AP)": "maximize",
    "Brier Score": "minimize",
    "Log Loss": "minimize",
    "F1": "maximize",
    "Balanced Accuracy": "maximize",
    "Highest Risk Group Lift": "maximize"
}

comparison_ranking_rows = []

for metric, direction in comparison_ranking_rules.items():

    ranked = (
        classification_comprehensive_comparison_df
        .sort_values(
            metric,
            ascending=(direction == "minimize")
        )
        .reset_index(drop=True)
    )

    comparison_ranking_rows.append({
        "Metric": metric,
        "Direction": direction,
        "Numerically Leading Model": ranked.loc[0, "Model"],
        "Leading Value": float(ranked.loc[0, metric]),
        "Other Model Value": float(ranked.loc[1, metric]),
        "Absolute Difference": abs(
            float(ranked.loc[0, metric])
            - float(ranked.loc[1, metric])
        )
    })

classification_metric_ranking_df = pd.DataFrame(
    comparison_ranking_rows
)

print("\nMETRIC-SPECIFIC MODEL RANKING")
print("=" * 75)

display(classification_metric_ranking_df.round(8))

print(
    "Numerical rankings do not establish statistical "
    "significance or operational superiority."
)

# ============================================================
# 10. VISUALIZATION
# ============================================================

comparison_plot_names = [
    "Baseline",
    "Random Forest",
    "XGBoost"
]

for metric, target in [
    ("ROC-AUC", 0.80),
    ("PR-AUC (AP)", 0.75),
    ("Brier Score", None),
    ("Log Loss", None)
]:

    values = (
        classification_probability_comparison_df[
            metric
        ].to_numpy()
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))

    bars = ax.bar(
        comparison_plot_names,
        values,
        color=["gray", "steelblue", "darkorange"]
    )

    ax.bar_label(
        bars,
        labels=[f"{v:.6f}" for v in values],
        padding=4
    )

    ax.axhline(
        values[0],
        linestyle="--",
        color="gray",
        label="Constant prevalence baseline"
    )

    if target is not None:

        ax.axhline(
            target,
            linestyle=":",
            color="red",
            label=f"Proposal boundary ({target:.2f})"
        )

        ax.set_ylim(0, 1)

    else:

        ax.set_ylim(0, max(values) * 1.25)

    ax.set_title(
        f"Classification Validation Comparison: {metric}"
    )

    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# Operational alert burden

fig, ax = plt.subplots(figsize=(9, 5))

plot_models = comparison_optimized_df["Model"].tolist()

plot_tp = comparison_optimized_df["TP"].to_numpy()
plot_fp = comparison_optimized_df["FP"].to_numpy()

ax.bar(
    plot_models,
    plot_tp,
    label="True positives",
    color="seagreen"
)

ax.bar(
    plot_models,
    plot_fp,
    bottom=plot_tp,
    label="False positives",
    color="indianred"
)

ax.axhline(
    comparison_y_val.sum(),
    linestyle="--",
    color="black",
    label="Actual positive records"
)

ax.set_title(
    "Experimental Thresholds: Validation Alert Burden"
)

ax.set_ylabel("Flagged visit records")
ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()
plt.close(fig)

# ============================================================
# 11. DEPLOYMENT READINESS ASSESSMENT
# ============================================================

comparison_readiness_rows = []

for _, row in (
    classification_comprehensive_comparison_df.iterrows()
):

    comparison_readiness_rows.append({
        "Model": row["Model"],
        "ROC-AUC Above Baseline": bool(
            row["ROC-AUC"] > comparison_baseline["ROC-AUC"]
        ),
        "AP Above Baseline": bool(
            row["PR-AUC (AP)"]
            > comparison_baseline["PR-AUC (AP)"]
        ),
        "Positive Brier Skill": bool(
            row["Brier Skill Score"] > 0
        ),
        "Highest Decile Lift Above 1": bool(
            row["Highest Risk Group Lift"] > 1
        ),
        "ROC-AUC Target Achieved": bool(
            row["ROC-AUC"] > 0.80
        ),
        "AP Target Achieved": bool(
            row["PR-AUC (AP)"] > 0.75
        ),
        "Experimental Threshold Available": True,
        "Clinical Threshold Approved": False,
        "Deployment Status": "NOT RECOMMENDED",
        "Model Role": "RESEARCH / EXPERIMENTAL"
    })

classification_readiness_df = pd.DataFrame(
    comparison_readiness_rows
)

print("\nDEPLOYMENT READINESS ASSESSMENT")
print("=" * 75)

display(classification_readiness_df)

# ============================================================
# 12. COMPARISON METADATA
# ============================================================

def comparison_json_records(dataframe):

    return json.loads(
        dataframe.to_json(
            orient="records",
            double_precision=15
        )
    )


classification_comparison_metadata = {
    "dataset_version": "V2",
    "task": "30-Day Subsequent Visit Proxy Classification",
    "evaluation_dataset": "Validation",
    "validation_records": int(len(comparison_y_val)),
    "validation_positive_cases": int(
        comparison_y_val.sum()
    ),
    "validation_prevalence": comparison_val_prevalence,
    "training_prevalence": comparison_train_prevalence,
    "probability_comparison": comparison_json_records(
        classification_probability_comparison_df
    ),
    "threshold_comparison": comparison_json_records(
        classification_threshold_comparison_df
    ),
    "risk_comparison": comparison_json_records(
        classification_risk_comparison_df
    ),
    "proposal_assessment": comparison_json_records(
        classification_target_assessment_df
    ),
    "comprehensive_comparison": comparison_json_records(
        classification_comprehensive_comparison_df
    ),
    "metric_rankings": comparison_json_records(
        classification_metric_ranking_df
    ),
    "deployment_readiness": comparison_json_records(
        classification_readiness_df
    ),
    "thresholds": comparison_thresholds,
    "threshold_role": "EXPERIMENTAL_ONLY",
    "final_model_selected": False,
    "clinical_threshold_approved": False,
    "clinical_deployment_recommended": False,
    "test_dataset_accessed": False,
    "model_retraining_performed": False,
    "model_files_exported": False,
    "limitations": [
        "Neither classifier exceeds the constant "
        "prevalence baseline in ROC-AUC or average precision.",
        "Both classifiers have negative Brier skill scores.",
        "Optimized thresholds flag nearly all validation records.",
        "Proposal discrimination targets were not achieved.",
        "Numerical model differences do not establish "
        "statistical significance.",
        "The target is a subsequent visit proxy, not "
        "a validated unplanned readmission outcome.",
        "Clinical deployment is not supported by "
        "the current validation evidence."
    ]
}

json.dumps(
    classification_comparison_metadata,
    allow_nan=False
)

print("\nCOMPARISON METADATA")
print("=" * 75)
print("Metadata creation: COMPLETED")
print("JSON serialization verification: PASSED")

# ============================================================
# 13. COMPLETION SUMMARY
# ============================================================

print("\nCLASSIFICATION COMPARISON COMPLETION SUMMARY")
print("=" * 75)

print("Existing fitted classifiers: VERIFIED")
print("Validation probability comparison: COMPLETED")
print("Baseline comparison: COMPLETED")
print("Threshold comparison: COMPLETED")
print("Cell 42 threshold reproduction: PASSED")
print("Risk ranking comparison: COMPLETED")
print("Proposal target assessment: COMPLETED")
print("Comprehensive comparison: COMPLETED")
print("Metric-specific ranking: COMPLETED")
print("Deployment readiness assessment: COMPLETED")
print("Comparison visualization: COMPLETED")
print("Comparison metadata: CREATED")

print("\nCLASSIFICATION CONCLUSION")
print("=" * 75)

print(
    "Neither classifier demonstrates meaningful "
    "validation improvement over the reference baseline."
)

print(
    "Both classifiers are retained as experimental "
    "predictive model candidates."
)

print(
    "Neither classifier is recommended for "
    "clinical risk alert deployment."
)

print("\nNo models have been retrained.")
print("No thresholds have been reoptimized.")
print("No final classification model has been selected.")
print("No clinical threshold has been approved.")
print("Test datasets have NOT been evaluated.")
print("No model files have been exported.")

print("\nReady for Cell 44: Final Model Selection.")


In [ ]:

# ============================================================
# CELL 44A — FINAL MODEL SELECTION READINESS CHECK
# ============================================================

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted

print("FINAL MODEL SELECTION READINESS CHECK")
print("=" * 75)

# ------------------------------------------------------------
# 1. Discover existing fitted pipelines
# ------------------------------------------------------------

pipeline_inventory = []

for variable_name, obj in list(globals().items()):

    if variable_name.startswith("_"):
        continue

    if isinstance(obj, Pipeline):

        estimator = obj.steps[-1][1]

        try:
            check_is_fitted(obj)
            fitted_status = True
        except Exception:
            fitted_status = False

        pipeline_inventory.append({
            "Variable": variable_name,
            "Estimator": type(estimator).__name__,
            "Fitted": fitted_status
        })

pipeline_inventory_df = pd.DataFrame(pipeline_inventory)

print("\nAVAILABLE PIPELINES")
print("=" * 75)

display(pipeline_inventory_df)

# ------------------------------------------------------------
# 2. Discover validation comparison tables
# ------------------------------------------------------------

comparison_inventory = []

for variable_name, obj in list(globals().items()):

    if variable_name.startswith("_"):
        continue

    if isinstance(obj, pd.DataFrame):

        columns = [str(col) for col in obj.columns]

        has_mae = any(
            "mae" in col.lower()
            for col in columns
        )

        has_ap = any(
            "pr-auc" in col.lower()
            or "average precision" in col.lower()
            or col.lower() == "ap"
            for col in columns
        )

        has_model = any(
            "model" in col.lower()
            for col in columns
        )

        if has_model and (has_mae or has_ap):

            comparison_inventory.append({
                "Variable": variable_name,
                "Rows": len(obj),
                "Columns": columns
            })

comparison_inventory_df = pd.DataFrame(
    comparison_inventory
)

print("\nAVAILABLE MODEL COMPARISON TABLES")
print("=" * 75)

display(comparison_inventory_df)

# ------------------------------------------------------------
# 3. Discover threshold-related variables
# ------------------------------------------------------------

print("\nTHRESHOLD VARIABLE INVENTORY")
print("=" * 75)

for variable_name, obj in list(globals().items()):

    if variable_name.startswith("_"):
        continue

    if "threshold" in variable_name.lower():

        if isinstance(obj, dict):

            print(
                f"{variable_name}: dictionary, "
                f"keys={list(obj.keys())[:10]}"
            )

        elif isinstance(obj, pd.DataFrame):

            print(
                f"{variable_name}: DataFrame, "
                f"shape={obj.shape}"
            )

        elif isinstance(obj, (int, float)):

            print(f"{variable_name}: {obj!r}")

# ------------------------------------------------------------
# 4. Verify required fitted estimator types
# ------------------------------------------------------------

required_estimators = {
    "LinearRegression",
    "RandomForestRegressor",
    "XGBRegressor",
    "RandomForestClassifier",
    "XGBClassifier"
}

available_fitted_estimators = {
    item["Estimator"]
    for item in pipeline_inventory
    if item["Fitted"]
}

print("\nREQUIRED MODEL VERIFICATION")
print("=" * 75)

for estimator_name in sorted(required_estimators):

    status = (
        "PASSED"
        if estimator_name in available_fitted_estimators
        else "MISSING"
    )

    print(f"{estimator_name}: {status}")

missing_estimators = (
    required_estimators - available_fitted_estimators
)

assert not missing_estimators, (
    f"Missing fitted estimators: {missing_estimators}"
)

# ------------------------------------------------------------
# 5. Completion summary
# ------------------------------------------------------------

print("\nREADINESS CHECK COMPLETION SUMMARY")
print("=" * 75)

print("Required fitted model types: VERIFIED")
print("Pipeline variable inventory: COMPLETED")
print("Comparison table inventory: COMPLETED")
print("Threshold variable inventory: COMPLETED")
print("Model selection: NOT YET PERFORMED")
print("Model retraining: NONE")
print("Threshold reoptimization: NONE")
print("Test dataset accessed: NO")
print("Model files exported: NO")

print("\nReady for Cell 44B: Final Model Selection.")


In [ ]:

# ============================================================
# CELL 44B — FINAL MODEL SELECTION
# St. Hubert Medical Center BI Project
# Dataset Version: V2
# ============================================================

import numpy as np
import pandas as pd
import json

from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    f1_score
)

print("FINAL MODEL SELECTION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify prerequisites
# ------------------------------------------------------------

required_variables = [
    "lr_pipeline",
    "rf_los_pipeline",
    "xgb_los_pipeline",
    "rf_classifier_pipeline",
    "xgb_classifier_pipeline",
    "final_regression_validation_ranking",
    "regression_comparison_df",
    "classification_comprehensive_comparison_df",
    "classification_probability_comparison_df",
    "comparison_thresholds",
    "classification_threshold_optimization_results",
    "threshold_validation_probabilities",
    "X_los_val",
    "y_los_val",
    "X_proxy_val",
    "classification_y_val"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing required variables: {missing_variables}"
)

print("Required variables: VERIFIED")

# ------------------------------------------------------------
# 2. Establish canonical fitted model registries
# ------------------------------------------------------------

final_regression_registry = {
    "Linear Regression": lr_pipeline,
    "Random Forest Regressor": rf_los_pipeline,
    "XGBoost Regressor": xgb_los_pipeline
}

final_classification_registry = {
    "Random Forest Classifier": rf_classifier_pipeline,
    "XGBoost Classifier": xgb_classifier_pipeline
}

final_model_registry = {
    **final_regression_registry,
    **final_classification_registry
}

assert len(final_model_registry) == 5

for model_name, fitted_pipeline in final_model_registry.items():

    check_is_fitted(fitted_pipeline)

    print(f"{model_name}: FITTED")

print("\nCanonical model registry: VERIFIED")
print(f"Regression models: {len(final_regression_registry)}")
print(f"Classification models: {len(final_classification_registry)}")
print(f"Total fitted models: {len(final_model_registry)}")

# ------------------------------------------------------------
# 3. Freeze model selection policy
# ------------------------------------------------------------

final_model_selection_policy = {
    "dataset_version": "V2",
    "selection_dataset": "Validation",
    "regression_primary_metric": "MAE",
    "regression_primary_direction": "minimize",
    "regression_secondary_metrics": [
        "RMSE",
        "R2"
    ],
    "classification_primary_metric": "Average Precision",
    "classification_primary_direction": "maximize",
    "classification_secondary_metrics": [
        "ROC-AUC",
        "Brier Score",
        "Log Loss"
    ],
    "classification_threshold_source": "Cell 42 / Cell 43",
    "classification_threshold_role": "EXPERIMENTAL_ONLY",
    "classification_prediction_rule": "probability >= threshold",
    "regression_proposal_mae_boundary": 1.0,
    "classification_proposal_roc_auc_boundary": 0.80,
    "classification_proposal_ap_boundary": 0.75,
    "deployment_approval_separate_from_selection": True,
    "test_set_used_for_selection": False,
    "model_retraining": False,
    "threshold_reoptimization": False,
    "probability_calibration_fitted": False
}

print("\nMODEL SELECTION POLICY")
print("=" * 75)

for key, value in final_model_selection_policy.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# 4. Verify validation datasets
# ------------------------------------------------------------

los_selection_y = np.asarray(
    y_los_val,
    dtype=np.float64
).reshape(-1)

classification_selection_y = np.asarray(
    classification_y_val,
    dtype=np.int64
).reshape(-1)

assert len(X_los_val) == len(los_selection_y)

assert len(X_proxy_val) == len(
    classification_selection_y
)

assert np.isfinite(los_selection_y).all()

assert set(
    np.unique(classification_selection_y)
) == {0, 1}

assert len(classification_selection_y) == 109458
assert classification_selection_y.sum() == 12845

print("\nVALIDATION DATASET VERIFICATION")
print("=" * 75)

print(f"LOS validation records: {len(los_selection_y):,}")

print(
    "Classification validation records: "
    f"{len(classification_selection_y):,}"
)

print(
    "Classification validation prevalence: "
    f"{classification_selection_y.mean():.6%}"
)

print("Validation datasets: VERIFIED")

# ------------------------------------------------------------
# 5. Recover full-precision regression validation metrics
# ------------------------------------------------------------
# Recompute predictions from existing fitted pipelines.
# No fitting or retraining is performed.
# This avoids selecting models using rounded display values.

print("\nREGRESSION VALIDATION METRIC VERIFICATION")
print("=" * 75)

final_regression_validation_records = []

for model_name, fitted_pipeline in (
    final_regression_registry.items()
):

    predictions = np.asarray(
        fitted_pipeline.predict(X_los_val),
        dtype=np.float64
    ).reshape(-1)

    assert len(predictions) == len(los_selection_y)
    assert np.isfinite(predictions).all()

    final_regression_validation_records.append({
        "Model": model_name,
        "Validation MAE": float(
            mean_absolute_error(
                los_selection_y,
                predictions
            )
        ),
        "Validation RMSE": float(
            root_mean_squared_error(
                los_selection_y,
                predictions
            )
        ),
        "Validation R2": float(
            r2_score(
                los_selection_y,
                predictions
            )
        )
    })

    print(f"{model_name}: validation metrics recovered")

final_regression_selection_df = pd.DataFrame(
    final_regression_validation_records
)

# Primary: lowest MAE
# Secondary: lowest RMSE
# Tertiary: highest R2
# Final deterministic tie-break: model name

final_regression_selection_df = (
    final_regression_selection_df
    .sort_values(
        by=[
            "Validation MAE",
            "Validation RMSE",
            "Validation R2",
            "Model"
        ],
        ascending=[
            True,
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

final_regression_selection_df.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(final_regression_selection_df) + 1
    )
)

print("\nFINAL REGRESSION VALIDATION RANKING")
print("=" * 75)

display(
    final_regression_selection_df.style.format({
        "Validation MAE": "{:.9f}",
        "Validation RMSE": "{:.9f}",
        "Validation R2": "{:.9f}"
    })
)

# ------------------------------------------------------------
# 6. Select primary regression model
# ------------------------------------------------------------

final_selected_los_row = (
    final_regression_selection_df.iloc[0]
)

final_selected_los_name = str(
    final_selected_los_row["Model"]
)

final_selected_los_pipeline = (
    final_regression_registry[
        final_selected_los_name
    ]
)

final_selected_los_mae = float(
    final_selected_los_row["Validation MAE"]
)

final_selected_los_rmse = float(
    final_selected_los_row["Validation RMSE"]
)

final_selected_los_r2 = float(
    final_selected_los_row["Validation R2"]
)

final_los_proposal_target_achieved = bool(
    final_selected_los_mae < 1.0
)

# Verify consistency with previous LOS selection.

previous_los_selection_consistent = (
    final_selected_los_pipeline
    is selected_los_pipeline
)

assert previous_los_selection_consistent, (
    "Final LOS selection differs from the previous "
    "selected_los_pipeline. Review earlier regression "
    "selection and prediction postprocessing before proceeding."
)

print("\nPRIMARY LOS MODEL SELECTION")
print("=" * 75)

print(f"Selected model: {final_selected_los_name}")
print(f"Validation MAE: {final_selected_los_mae:.9f}")
print(f"Validation RMSE: {final_selected_los_rmse:.9f}")
print(f"Validation R2: {final_selected_los_r2:.9f}")

print(
    "Proposal MAE target achieved: "
    f"{final_los_proposal_target_achieved}"
)

print(
    "Previous LOS selection consistency: PASSED"
)

# ------------------------------------------------------------
# 7. Recover full-precision classification thresholds
# ------------------------------------------------------------

print("\nEXPERIMENTAL THRESHOLD VERIFICATION")
print("=" * 75)

final_classification_thresholds = {}

for model_name in final_classification_registry:

    assert model_name in comparison_thresholds

    recovered_threshold = float(
        comparison_thresholds[model_name]
    )

    assert np.isfinite(recovered_threshold)
    assert 0.0 <= recovered_threshold <= 1.0

    final_classification_thresholds[
        model_name
    ] = recovered_threshold

    print(
        f"{model_name}: "
        f"{recovered_threshold!r}"
    )

print("Full-precision thresholds: VERIFIED")

# ------------------------------------------------------------
# 8. Recover full-precision classification metrics
# ------------------------------------------------------------

print("\nCLASSIFICATION VALIDATION METRIC VERIFICATION")
print("=" * 75)

final_classification_validation_records = []

# Expected confusion matrices from Cells 42 and 43.
# Order: TN, FP, FN, TP.

expected_validation_confusion_counts = {
    "Random Forest Classifier": (
        268, 96345, 24, 12821
    ),
    "XGBoost Classifier": (
        270, 96343, 27, 12818
    )
}

for model_name, fitted_pipeline in (
    final_classification_registry.items()
):

    classes = np.asarray(
        fitted_pipeline.classes_
    )

    positive_columns = np.flatnonzero(
        classes == 1
    )

    assert len(positive_columns) == 1

    positive_column = int(
        positive_columns[0]
    )

    # Reuse the original Cell 42 validation probabilities
    # to preserve exact threshold reproducibility.

    probabilities = np.asarray(
        threshold_validation_probabilities[model_name],
        dtype=np.float64
    ).reshape(-1).copy()

    assert len(probabilities) == len(
        classification_selection_y
    )

    assert np.isfinite(probabilities).all()
    assert np.all(probabilities >= 0)
    assert np.all(probabilities <= 1)

    experimental_threshold = (
        final_classification_thresholds[
            model_name
        ]
    )

    predictions = (
        probabilities >= experimental_threshold
    ).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        classification_selection_y,
        predictions,
        labels=[0, 1]
    ).ravel()

    actual_counts = tuple(
        int(value)
        for value in [tn, fp, fn, tp]
    )

    assert actual_counts == (
        expected_validation_confusion_counts[
            model_name
        ]
    ), (
        f"Threshold reproduction mismatch: {model_name}"
    )

    final_classification_validation_records.append({
        "Model": model_name,
        "ROC-AUC": float(
            roc_auc_score(
                classification_selection_y,
                probabilities
            )
        ),
        "PR-AUC (AP)": float(
            average_precision_score(
                classification_selection_y,
                probabilities
            )
        ),
        "Brier Score": float(
            brier_score_loss(
                classification_selection_y,
                probabilities
            )
        ),
        "Log Loss": float(
            log_loss(
                classification_selection_y,
                probabilities,
                labels=[0, 1]
            )
        ),
        "Experimental Threshold": experimental_threshold,
        "Optimized F1": float(
            f1_score(
                classification_selection_y,
                predictions,
                zero_division=0
            )
        ),
        "Predicted Positive Rate": float(
            predictions.mean()
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

    print(
        f"{model_name}: metrics recovered; "
        "confusion matrix reproduced"
    )

# ------------------------------------------------------------
# 9. Rank classification models
# ------------------------------------------------------------

final_classification_selection_df = pd.DataFrame(
    final_classification_validation_records
)

final_classification_selection_df = (
    final_classification_selection_df
    .sort_values(
        by=[
            "PR-AUC (AP)",
            "ROC-AUC",
            "Brier Score",
            "Log Loss",
            "Model"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

final_classification_selection_df.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(final_classification_selection_df) + 1
    )
)

print("\nFINAL CLASSIFICATION VALIDATION RANKING")
print("=" * 75)

display(
    final_classification_selection_df.style.format({
        "ROC-AUC": "{:.9f}",
        "PR-AUC (AP)": "{:.9f}",
        "Brier Score": "{:.9f}",
        "Log Loss": "{:.9f}",
        "Experimental Threshold": "{:.17g}",
        "Optimized F1": "{:.9f}",
        "Predicted Positive Rate": "{:.4%}"
    })
)

# ------------------------------------------------------------
# 10. Select primary experimental classifier
# ------------------------------------------------------------

final_selected_classifier_row = (
    final_classification_selection_df.iloc[0]
)

final_selected_classifier_name = str(
    final_selected_classifier_row["Model"]
)

final_selected_classifier_pipeline = (
    final_classification_registry[
        final_selected_classifier_name
    ]
)

final_selected_classifier_threshold = (
    final_classification_thresholds[
        final_selected_classifier_name
    ]
)

final_selected_classifier_ap = float(
    final_selected_classifier_row["PR-AUC (AP)"]
)

final_selected_classifier_roc_auc = float(
    final_selected_classifier_row["ROC-AUC"]
)

final_selected_classifier_brier = float(
    final_selected_classifier_row["Brier Score"]
)

final_selected_classifier_log_loss = float(
    final_selected_classifier_row["Log Loss"]
)

final_classifier_roc_target_achieved = bool(
    final_selected_classifier_roc_auc > 0.80
)

final_classifier_ap_target_achieved = bool(
    final_selected_classifier_ap > 0.75
)

assert (
    final_selected_classifier_name
    == "XGBoost Classifier"
), "Unexpected classification selection."

print("\nPRIMARY EXPERIMENTAL CLASSIFIER")
print("=" * 75)

print(
    f"Selected model: {final_selected_classifier_name}"
)

print(
    f"Validation AP: {final_selected_classifier_ap:.9f}"
)

print(
    f"Validation ROC-AUC: "
    f"{final_selected_classifier_roc_auc:.9f}"
)

print(
    f"Validation Brier Score: "
    f"{final_selected_classifier_brier:.9f}"
)

print(
    "Experimental threshold: "
    f"{final_selected_classifier_threshold!r}"
)

print("Clinical threshold approved: FALSE")
print("Clinical deployment recommended: FALSE")

# ------------------------------------------------------------
# 11. Assign final model roles
# ------------------------------------------------------------

final_model_role_records = []

for _, row in final_regression_selection_df.iterrows():

    model_name = str(row["Model"])
    rank = int(row["Rank"])

    if rank == 1:
        model_role = "PRIMARY_EXPERIMENTAL_CANDIDATE"
    elif rank == 2:
        model_role = "SECONDARY_EXPERIMENTAL_CANDIDATE"
    else:
        model_role = "COMPARATIVE_MODEL"

    final_model_role_records.append({
        "Task": "LOS Regression",
        "Model": model_name,
        "Validation Rank": rank,
        "Selection Metric": "MAE",
        "Selection Value": float(row["Validation MAE"]),
        "Model Role": model_role,
        "Proposal Target Achieved": bool(
            row["Validation MAE"] < 1.0
        ),
        "Experimental Threshold": None,
        "Clinical Deployment Approved": False,
        "Deployment Status": (
            "PENDING_FINAL_EVALUATION"
        ),
        "Retain for Test Evaluation": True,
        "Retain for Export": True
    })

for _, row in (
    final_classification_selection_df.iterrows()
):

    model_name = str(row["Model"])
    rank = int(row["Rank"])

    model_role = (
        "PRIMARY_EXPERIMENTAL_CANDIDATE"
        if rank == 1
        else "SECONDARY_EXPERIMENTAL_CANDIDATE"
    )

    final_model_role_records.append({
        "Task": "30-Day Subsequent Visit Proxy",
        "Model": model_name,
        "Validation Rank": rank,
        "Selection Metric": "Average Precision",
        "Selection Value": float(row["PR-AUC (AP)"]),
        "Model Role": model_role,
        "Proposal Target Achieved": bool(
            row["ROC-AUC"] > 0.80
            and row["PR-AUC (AP)"] > 0.75
        ),
        "Experimental Threshold": (
            final_classification_thresholds[
                model_name
            ]
        ),
        "Clinical Deployment Approved": False,
        "Deployment Status": "NOT_RECOMMENDED",
        "Retain for Test Evaluation": True,
        "Retain for Export": True
    })

final_model_role_df = pd.DataFrame(
    final_model_role_records
)

print("\nFINAL MODEL ROLE ASSIGNMENTS")
print("=" * 75)

display(final_model_role_df)

# ------------------------------------------------------------
# 12. Establish selected model registry
# ------------------------------------------------------------

final_selected_model_registry = {
    "LOS_REGRESSION": {
        "model_name": final_selected_los_name,
        "pipeline": final_selected_los_pipeline,
        "selection_metric": "MAE",
        "selection_value": final_selected_los_mae,
        "deployment_approved": False
    },
    "SUBSEQUENT_VISIT_CLASSIFICATION": {
        "model_name": final_selected_classifier_name,
        "pipeline": final_selected_classifier_pipeline,
        "selection_metric": "Average Precision",
        "selection_value": final_selected_classifier_ap,
        "experimental_threshold": (
            final_selected_classifier_threshold
        ),
        "clinical_threshold_approved": False,
        "deployment_approved": False
    }
}

# ------------------------------------------------------------
# 13. Create final selection metadata
# ------------------------------------------------------------

# Convert missing DataFrame values to JSON-compatible nulls.

final_model_roles_serializable = (
    final_model_role_df
    .astype(object)
    .where(
        pd.notna(final_model_role_df),
        None
    )
    .to_dict(orient="records")
)

final_model_selection_metadata = {
    "project": "St. Hubert Medical Center BI Project",
    "dataset_version": "V2",
    "selection_stage": "PRE_TEST_FINAL_SELECTION",
    "selection_policy": final_model_selection_policy,

    "regression": {
        "target": "TARGET_LOS_DAYS",
        "selected_model": final_selected_los_name,
        "selection_metric": "MAE",
        "validation_mae": final_selected_los_mae,
        "validation_rmse": final_selected_los_rmse,
        "validation_r2": final_selected_los_r2,
        "proposal_target_achieved": (
            final_los_proposal_target_achieved
        ),
        "previous_selection_consistent": (
            previous_los_selection_consistent
        ),
        "clinical_deployment_approved": False,
        "deployment_status": "PENDING_FINAL_EVALUATION",
        "validation_ranking": (
            final_regression_selection_df
            .to_dict(orient="records")
        )
    },

    "classification": {
        "target": (
            "SUBSEQUENT_VISIT_WITHIN_30D_PROXY"
        ),
        "selected_model": (
            final_selected_classifier_name
        ),
        "selection_metric": "Average Precision",
        "validation_ap": final_selected_classifier_ap,
        "validation_roc_auc": (
            final_selected_classifier_roc_auc
        ),
        "validation_brier_score": (
            final_selected_classifier_brier
        ),
        "validation_log_loss": (
            final_selected_classifier_log_loss
        ),
        "experimental_threshold": (
            final_selected_classifier_threshold
        ),
        "all_experimental_thresholds": (
            final_classification_thresholds
        ),
        "roc_auc_target_achieved": (
            final_classifier_roc_target_achieved
        ),
        "ap_target_achieved": (
            final_classifier_ap_target_achieved
        ),
        "clinical_threshold_approved": False,
        "clinical_deployment_approved": False,
        "deployment_status": "NOT_RECOMMENDED",
        "model_role": "PRIMARY_EXPERIMENTAL_CANDIDATE",
        "validation_ranking": (
            final_classification_selection_df
            .to_dict(orient="records")
        ),
        "target_limitation": (
            "Subsequent visit proxy, not clinically "
            "validated unplanned readmission."
        ),
        "selection_limitation": (
            "Highest validation AP among trained "
            "classifiers does not establish meaningful "
            "baseline improvement, statistical superiority, "
            "or clinical usefulness."
        )
    },

    "model_roles": final_model_roles_serializable,

    "model_inventory": {
        "regression_models": 3,
        "classification_models": 2,
        "total_models": 5,
        "all_models_retained": True
    },

    "test_evaluation_policy": {
        "evaluate_all_five_models": True,
        "selection_frozen_before_test": True,
        "thresholds_frozen_before_test": True,
        "test_results_used_for_reselection": False,
        "test_results_used_for_threshold_optimization": False
    },

    "test_datasets_evaluated": False,
    "models_exported": False
}

# Strict JSON serialization verification.
# NaN and Infinity are not permitted.

final_model_selection_json = json.dumps(
    final_model_selection_metadata,
    indent=4,
    allow_nan=False
)

print("\nFINAL SELECTION METADATA")
print("=" * 75)

print("Metadata creation: COMPLETED")
print("JSON serialization verification: PASSED")

# ------------------------------------------------------------
# 14. Final integrity checks
# ------------------------------------------------------------

assert len(final_model_registry) == 5

assert len(final_regression_selection_df) == 3
assert len(final_classification_selection_df) == 2
assert len(final_model_role_df) == 5

assert len(final_selected_model_registry) == 2

assert final_selected_los_pipeline is (
    final_regression_registry[
        final_selected_los_name
    ]
)

assert final_selected_classifier_pipeline is (
    final_classification_registry[
        final_selected_classifier_name
    ]
)

assert final_selected_classifier_threshold == (
    comparison_thresholds[
        final_selected_classifier_name
    ]
)

assert not final_model_role_df[
    "Clinical Deployment Approved"
].any()

assert final_model_role_df[
    "Retain for Test Evaluation"
].all()

assert final_model_role_df[
    "Retain for Export"
].all()

# Mark selection as frozen only after all checks pass.

final_model_selection_completed = True
final_model_selection_frozen = True

# ------------------------------------------------------------
# 15. Completion summary
# ------------------------------------------------------------

print("\nFINAL MODEL SELECTION COMPLETION SUMMARY")
print("=" * 75)

print("Existing fitted pipelines: VERIFIED")
print("Canonical model registry: CREATED")
print("Validation-only selection policy: VERIFIED")
print("Regression metric recovery: COMPLETED")
print("Regression ranking: COMPLETED")
print("Previous regression selection consistency: PASSED")
print("Classification metric recovery: COMPLETED")
print("Classification ranking: COMPLETED")
print("Full-precision threshold recovery: COMPLETED")
print("Classification confusion matrix reproduction: PASSED")
print("Primary regression model selection: COMPLETED")
print("Primary experimental classifier selection: COMPLETED")
print("Final model role assignment: COMPLETED")
print("Final selection metadata: CREATED")
print("JSON serialization verification: PASSED")
print("Final integrity checks: PASSED")

print("\nFROZEN MODEL SELECTIONS")
print("=" * 75)

print(f"Primary LOS model: {final_selected_los_name}")

print(
    "Primary experimental classifier: "
    f"{final_selected_classifier_name}"
)

print(
    "Frozen experimental threshold: "
    f"{final_selected_classifier_threshold!r}"
)

print("\nRegression models retained: 3")
print("Classification models retained: 2")
print("Total fitted pipelines retained: 5")

print("\nModel selection frozen: TRUE")
print("Experimental thresholds frozen: TRUE")
print("Clinical deployment approved: FALSE")

print("\nNo models have been retrained.")
print("No thresholds have been reoptimized.")
print("No probability calibration has been fitted.")
print("Test datasets have NOT been evaluated.")
print("No model files have been exported.")

print(
    "\nReady for Cell 45: "
    "Held-Out Regression Test Evaluation."
)


In [ ]:

# ============================================================
# CELL 45 — HELD-OUT REGRESSION TEST EVALUATION
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)
from sklearn.utils.validation import check_is_fitted
from IPython.display import display

print("HELD-OUT REGRESSION TEST EVALUATION")
print("=" * 75)

# ============================================================
# 1. REQUIRED VARIABLE VERIFICATION
# ============================================================

required_variables = [
    "lr_pipeline",
    "rf_los_pipeline",
    "xgb_los_pipeline",
    "selected_los_pipeline",
    "X_los_test",
    "y_los_test",
    "y_los_train",
    "X_los_val",
    "y_los_val",
    "final_regression_validation_ranking",
    "rf_classifier_pipeline",
    "xgb_classifier_pipeline"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing required variables: {missing_variables}"
)

print("Required variables: VERIFIED")

# ============================================================
# 2. FROZEN REGRESSION MODEL REGISTRY
# ============================================================

regression_test_models = {
    "Linear Regression": lr_pipeline,
    "Random Forest Regressor": rf_los_pipeline,
    "XGBoost Regressor": xgb_los_pipeline
}

frozen_primary_los_model = "Random Forest Regressor"

assert selected_los_pipeline is rf_los_pipeline, (
    "Frozen primary LOS pipeline mismatch."
)

for model_name, fitted_pipeline in regression_test_models.items():
    check_is_fitted(fitted_pipeline)
    print(f"{model_name}: FITTED")

check_is_fitted(rf_classifier_pipeline)
check_is_fitted(xgb_classifier_pipeline)

print("Frozen primary LOS model:", frozen_primary_los_model)
print("Classification pipelines retained: 2")

# ============================================================
# 3. HELD-OUT TEST DATA VERIFICATION
# ============================================================

print("\nHELD-OUT TEST DATA VERIFICATION")
print("=" * 75)

regression_test_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

assert list(X_los_test.columns) == regression_test_features
assert list(X_los_val.columns) == regression_test_features

regression_test_y = np.asarray(
    y_los_test,
    dtype=np.float64
).reshape(-1)

regression_training_y = np.asarray(
    y_los_train,
    dtype=np.float64
).reshape(-1)

assert len(X_los_test) == len(regression_test_y)
assert len(regression_test_y) > 0
assert np.isfinite(regression_test_y).all()
assert np.isfinite(regression_training_y).all()
assert (regression_test_y >= 0).all()

if isinstance(y_los_test, pd.Series):
    assert X_los_test.index.equals(y_los_test.index), (
        "Test feature-target index mismatch."
    )

print(f"Test records: {len(regression_test_y):,}")
print(f"Original features: {X_los_test.shape[1]}")
print(f"Missing target values: {np.isnan(regression_test_y).sum()}")
print("Test dataset verification: PASSED")

print("\nTEST TARGET DISTRIBUTION")

regression_test_target_summary = pd.DataFrame({
    "Statistic": [
        "Records",
        "Mean",
        "Median",
        "Standard deviation",
        "Minimum",
        "Maximum"
    ],
    "Value": [
        len(regression_test_y),
        np.mean(regression_test_y),
        np.median(regression_test_y),
        np.std(regression_test_y),
        np.min(regression_test_y),
        np.max(regression_test_y)
    ]
})

display(regression_test_target_summary)

# ============================================================
# 4. REGRESSION METRIC FUNCTION
# ============================================================

def calculate_regression_test_metrics(actual, predicted):

    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)

    assert actual.shape == predicted.shape
    assert np.isfinite(predicted).all()

    errors = predicted - actual

    return {
        "Test MAE": float(
            mean_absolute_error(actual, predicted)
        ),
        "Test RMSE": float(
            root_mean_squared_error(actual, predicted)
        ),
        "Test R2": float(
            r2_score(actual, predicted)
        ),
        "Mean Error": float(np.mean(errors)),
        "Median Absolute Error": float(
            np.median(np.abs(errors))
        ),
        "Negative Predictions": int(
            np.sum(predicted < 0)
        ),
        "Within 1 Day (%)": float(
            np.mean(np.abs(errors) <= 1) * 100
        ),
        "Within 7 Days (%)": float(
            np.mean(np.abs(errors) <= 7) * 100
        )
    }

# ============================================================
# 5. TRAINING-DERIVED REFERENCE BASELINES
# ============================================================

print("\nTRAINING-DERIVED REFERENCE BASELINES")
print("=" * 75)

regression_test_train_mean = float(
    np.mean(regression_training_y)
)

regression_test_train_median = float(
    np.median(regression_training_y)
)

regression_test_predictions = {
    "Constant Training Mean": np.full(
        len(regression_test_y),
        regression_test_train_mean
    ),
    "Constant Training Median": np.full(
        len(regression_test_y),
        regression_test_train_median
    )
}

regression_test_baseline_rows = []

for baseline_name, predictions in regression_test_predictions.items():

    metrics = calculate_regression_test_metrics(
        regression_test_y,
        predictions
    )

    regression_test_baseline_rows.append({
        "Model": baseline_name,
        **metrics
    })

regression_test_baseline_df = pd.DataFrame(
    regression_test_baseline_rows
)

display(regression_test_baseline_df.round(6))

print("Training mean:", regression_test_train_mean)
print("Training median:", regression_test_train_median)
print("Baselines derived exclusively from training targets.")

# ============================================================
# 6. HELD-OUT MODEL PREDICTIONS
# ============================================================

print("\nHELD-OUT MODEL PREDICTIONS")
print("=" * 75)

regression_test_rows = []

for model_name, fitted_pipeline in regression_test_models.items():

    print(f"\nEvaluating {model_name}...")

    prediction_start = time.perf_counter()

    predictions = np.asarray(
        fitted_pipeline.predict(X_los_test),
        dtype=np.float64
    ).reshape(-1)

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    assert len(predictions) == len(regression_test_y)
    assert np.isfinite(predictions).all()

    regression_test_predictions[model_name] = predictions

    metrics = calculate_regression_test_metrics(
        regression_test_y,
        predictions
    )

    regression_test_rows.append({
        "Model": model_name,
        **metrics,
        "Prediction Time (s)": float(prediction_time)
    })

    print(f"Test MAE: {metrics['Test MAE']:.6f}")
    print(f"Test RMSE: {metrics['Test RMSE']:.6f}")
    print(f"Test R2: {metrics['Test R2']:.6f}")
    print(f"Prediction time: {prediction_time:.4f}s")

regression_test_results_df = pd.DataFrame(
    regression_test_rows
)

print("\nHELD-OUT REGRESSION TEST RESULTS")
print("=" * 75)

display(regression_test_results_df.round(6))

# ============================================================
# 7. BASELINE IMPROVEMENT COMPARISON
# ============================================================

print("\nBASELINE IMPROVEMENT COMPARISON")
print("=" * 75)

mean_baseline = regression_test_baseline_df.loc[
    regression_test_baseline_df["Model"]
    == "Constant Training Mean"
].iloc[0]

median_baseline = regression_test_baseline_df.loc[
    regression_test_baseline_df["Model"]
    == "Constant Training Median"
].iloc[0]

regression_test_comparison_df = (
    regression_test_results_df.copy()
)

regression_test_comparison_df[
    "MAE Improvement vs Median (%)"
] = (
    (
        median_baseline["Test MAE"]
        - regression_test_comparison_df["Test MAE"]
    )
    / median_baseline["Test MAE"]
    * 100
)

regression_test_comparison_df[
    "RMSE Improvement vs Mean (%)"
] = (
    (
        mean_baseline["Test RMSE"]
        - regression_test_comparison_df["Test RMSE"]
    )
    / mean_baseline["Test RMSE"]
    * 100
)

regression_test_comparison_df[
    "R2 Improvement vs Mean"
] = (
    regression_test_comparison_df["Test R2"]
    - mean_baseline["Test R2"]
)

display(regression_test_comparison_df.round(6))

# ============================================================
# 8. VALIDATION-TO-TEST COMPARISON
# ============================================================

print("\nVALIDATION-TO-TEST PERFORMANCE COMPARISON")
print("=" * 75)

validation_reference = (
    final_regression_validation_ranking[
        [
            "Model",
            "Validation MAE",
            "Validation RMSE",
            "Validation R2"
        ]
    ].copy()
)

regression_validation_test_df = (
    validation_reference.merge(
        regression_test_results_df[
            ["Model", "Test MAE", "Test RMSE", "Test R2"]
        ],
        on="Model",
        how="inner",
        validate="one_to_one"
    )
)

assert len(regression_validation_test_df) == 3

regression_validation_test_df["MAE Change"] = (
    regression_validation_test_df["Test MAE"]
    - regression_validation_test_df["Validation MAE"]
)

regression_validation_test_df["RMSE Change"] = (
    regression_validation_test_df["Test RMSE"]
    - regression_validation_test_df["Validation RMSE"]
)

regression_validation_test_df["R2 Change"] = (
    regression_validation_test_df["Test R2"]
    - regression_validation_test_df["Validation R2"]
)

display(regression_validation_test_df.round(6))

print("Positive MAE/RMSE change indicates higher test error.")
print("Negative R2 change indicates lower test R2.")
print("Test results will not be used to revise model selection.")

# ============================================================
# 9. RESIDUAL DIAGNOSTICS
# ============================================================

print("\nTEST RESIDUAL DIAGNOSTICS")
print("=" * 75)

regression_test_residuals = {}
regression_test_residual_rows = []

for model_name in regression_test_models:

    predictions = regression_test_predictions[model_name]

    residuals = regression_test_y - predictions

    regression_test_residuals[model_name] = residuals

    regression_test_residual_rows.append({
        "Model": model_name,
        "Mean Residual": float(np.mean(residuals)),
        "Median Residual": float(np.median(residuals)),
        "Residual Std": float(np.std(residuals)),
        "5th Percentile": float(np.quantile(residuals, 0.05)),
        "95th Percentile": float(np.quantile(residuals, 0.95)),
        "Underprediction Rate (%)": float(
            np.mean(residuals > 0) * 100
        ),
        "Overprediction Rate (%)": float(
            np.mean(residuals < 0) * 100
        )
    })

regression_test_residual_df = pd.DataFrame(
    regression_test_residual_rows
)

display(regression_test_residual_df.round(6))

print("Residual definition: Actual LOS minus predicted LOS.")
print("Positive residuals indicate underprediction.")

# ============================================================
# 10. LOS RANGE PERFORMANCE
# ============================================================

print("\nHELD-OUT LOS RANGE PERFORMANCE")
print("=" * 75)

los_test_range_bins = [
    -np.inf, 1, 7, 30, 90, 180, np.inf
]

los_test_range_labels = [
    "0–1 day",
    ">1–7 days",
    ">7–30 days",
    ">30–90 days",
    ">90–180 days",
    ">180 days"
]

los_test_ranges = pd.cut(
    regression_test_y,
    bins=los_test_range_bins,
    labels=los_test_range_labels,
    include_lowest=True
)

regression_test_range_rows = []

for range_label in los_test_range_labels:

    mask = np.asarray(los_test_ranges == range_label)
    records = int(mask.sum())

    for model_name in regression_test_models:

        if records == 0:
            continue

        actual = regression_test_y[mask]

        predicted = (
            regression_test_predictions[model_name][mask]
        )

        regression_test_range_rows.append({
            "LOS Range": range_label,
            "Model": model_name,
            "Records": records,
            "Mean Actual LOS": float(np.mean(actual)),
            "Mean Predicted LOS": float(np.mean(predicted)),
            "MAE": float(
                mean_absolute_error(actual, predicted)
            ),
            "RMSE": float(
                root_mean_squared_error(actual, predicted)
            ),
            "Mean Error": float(
                np.mean(predicted - actual)
            )
        })

regression_test_range_df = pd.DataFrame(
    regression_test_range_rows
)

display(regression_test_range_df.round(4))

# ============================================================
# 11. PROPOSAL TARGET ASSESSMENT
# ============================================================

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 75)

regression_test_proposal_df = (
    regression_test_results_df[
        ["Model", "Test MAE"]
    ].copy()
)

regression_test_proposal_df["Proposal MAE Boundary"] = 1.0

regression_test_proposal_df["Required Direction"] = "<"

regression_test_proposal_df["Gap to Boundary"] = (
    regression_test_proposal_df["Test MAE"] - 1.0
)

regression_test_proposal_df["Target Status"] = np.where(
    regression_test_proposal_df["Test MAE"] < 1.0,
    "ACHIEVED",
    "NOT ACHIEVED"
)

display(regression_test_proposal_df.round(6))

# ============================================================
# 12. FROZEN PRIMARY MODEL TEST ASSESSMENT
# ============================================================

print("\nFROZEN PRIMARY LOS MODEL TEST ASSESSMENT")
print("=" * 75)

primary_los_test_result = (
    regression_test_comparison_df.loc[
        regression_test_comparison_df["Model"]
        == frozen_primary_los_model
    ].iloc[0]
)

print("Selected model:", frozen_primary_los_model)
print("Selection source: Validation")
print(f"Test MAE: {primary_los_test_result['Test MAE']:.9f}")
print(f"Test RMSE: {primary_los_test_result['Test RMSE']:.9f}")
print(f"Test R2: {primary_los_test_result['Test R2']:.9f}")

print(
    "MAE improvement over training median:",
    f"{primary_los_test_result['MAE Improvement vs Median (%)']:.6f}%"
)

print(
    "Proposal target achieved:",
    bool(primary_los_test_result["Test MAE"] < 1.0)
)

print("Model selection changed: FALSE")
print("Clinical deployment approved: FALSE")

# ============================================================
# 13. TEST PERFORMANCE VISUALIZATIONS
# ============================================================

regression_test_plot_df = pd.concat(
    [
        regression_test_baseline_df,
        regression_test_results_df
    ],
    ignore_index=True
)

short_model_names = {
    "Constant Training Mean": "Mean baseline",
    "Constant Training Median": "Median baseline",
    "Linear Regression": "Linear Regression",
    "Random Forest Regressor": "Random Forest",
    "XGBoost Regressor": "XGBoost"
}

plot_labels = (
    regression_test_plot_df["Model"]
    .map(short_model_names)
)

for metric in ["Test MAE", "Test RMSE", "Test R2"]:

    fig, ax = plt.subplots(figsize=(10, 5))

    bars = ax.bar(
        plot_labels,
        regression_test_plot_df[metric]
    )

    ax.bar_label(bars, fmt="%.4f", padding=3)

    if metric == "Test MAE":
        ax.axhline(
            1.0,
            linestyle="--",
            label="Proposal MAE boundary (1 day)"
        )
        ax.legend()

    if metric == "Test R2":
        ax.axhline(0, linestyle="--")

    ax.set_title(f"Held-Out Regression Comparison: {metric}")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.2)
    ax.margins(y=0.15)

    plt.tight_layout()
    plt.show()

# ============================================================
# 14. VALIDATION VS TEST MAE
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

positions = np.arange(len(regression_validation_test_df))
width = 0.35

ax.bar(
    positions - width / 2,
    regression_validation_test_df["Validation MAE"],
    width,
    label="Validation"
)

ax.bar(
    positions + width / 2,
    regression_validation_test_df["Test MAE"],
    width,
    label="Held-out test"
)

ax.set_xticks(positions)

ax.set_xticklabels(
    regression_validation_test_df["Model"]
    .map(short_model_names)
)

ax.set_ylabel("MAE (days)")
ax.set_title("Regression Generalization: Validation vs Test MAE")
ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# ============================================================
# 15. ACTUAL VS PREDICTED LOS
# ============================================================

rng = np.random.default_rng(42)

sample_indices = rng.choice(
    len(regression_test_y),
    size=min(10000, len(regression_test_y)),
    replace=False
)

for model_name in regression_test_models:

    predictions = regression_test_predictions[model_name]

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(
        regression_test_y[sample_indices],
        predictions[sample_indices],
        alpha=0.15,
        s=8
    )

    lower = min(
        regression_test_y.min(),
        predictions.min()
    )

    upper = max(
        regression_test_y.max(),
        predictions.max()
    )

    ax.plot(
        [lower, upper],
        [lower, upper],
        linestyle="--",
        label="Perfect prediction"
    )

    ax.set_xlabel("Actual LOS (days)")
    ax.set_ylabel("Predicted LOS (days)")
    ax.set_title(f"{model_name}: Held-Out Actual vs Predicted")
    ax.legend()
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

# ============================================================
# 16. RESIDUAL DISTRIBUTIONS
# ============================================================

for model_name in regression_test_models:

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.hist(
        regression_test_residuals[model_name],
        bins=60,
        alpha=0.8
    )

    ax.axvline(
        0,
        linestyle="--",
        label="Zero residual"
    )

    ax.set_xlabel("Actual minus predicted LOS (days)")
    ax.set_ylabel("Records")
    ax.set_title(f"{model_name}: Held-Out Residual Distribution")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)

    plt.tight_layout()
    plt.show()

# ============================================================
# 17. LOS RANGE MAE VISUALIZATION
# ============================================================

range_plot_df = regression_test_range_df.pivot(
    index="LOS Range",
    columns="Model",
    values="MAE"
).reindex(los_test_range_labels)

ax = range_plot_df.plot(
    kind="bar",
    figsize=(12, 6)
)

ax.set_title("Held-Out Regression MAE by Actual LOS Range")
ax.set_ylabel("MAE (days)")
ax.set_xlabel("Actual LOS range")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# ============================================================
# 18. TEST EVALUATION METADATA
# ============================================================

print("\nREGRESSION TEST EVALUATION METADATA")
print("=" * 75)

regression_test_evaluation_metadata = {
    "dataset_version": "V2",
    "task": "LOS Regression",
    "target": "TARGET_LOS_DAYS",
    "evaluation_dataset": "Held-out test",
    "test_records": int(len(regression_test_y)),
    "original_features": regression_test_features,
    "frozen_primary_model": frozen_primary_los_model,
    "selection_dataset": "Validation",
    "selection_metric": "MAE",
    "proposal_mae_boundary": 1.0,
    "proposal_required_direction": "<",
    "baseline_source": "Training targets only",
    "prediction_clipping_applied": False,
    "model_retraining": False,
    "model_selection_changed": False,
    "test_used_for_model_selection": False,
    "clinical_deployment_approved": False,
    "baseline_results": json.loads(
        regression_test_baseline_df.to_json(
            orient="records",
            double_precision=15
        )
    ),
    "model_results": json.loads(
        regression_test_comparison_df.to_json(
            orient="records",
            double_precision=15
        )
    ),
    "validation_test_comparison": json.loads(
        regression_validation_test_df.to_json(
            orient="records",
            double_precision=15
        )
    ),
    "residual_diagnostics": json.loads(
        regression_test_residual_df.to_json(
            orient="records",
            double_precision=15
        )
    ),
    "los_range_performance": json.loads(
        regression_test_range_df.to_json(
            orient="records",
            double_precision=15
        )
    ),
    "proposal_assessment": json.loads(
        regression_test_proposal_df.to_json(
            orient="records",
            double_precision=15
        )
    )
}

json.dumps(
    regression_test_evaluation_metadata,
    allow_nan=False
)

print("Metadata creation: COMPLETED")
print("JSON serialization verification: PASSED")

# ============================================================
# 19. FINAL INTEGRITY CHECKS
# ============================================================

assert selected_los_pipeline is rf_los_pipeline
assert len(regression_test_models) == 3
assert len(regression_test_results_df) == 3
assert len(regression_test_baseline_df) == 2

for fitted_pipeline in regression_test_models.values():
    check_is_fitted(fitted_pipeline)

check_is_fitted(rf_classifier_pipeline)
check_is_fitted(xgb_classifier_pipeline)

regression_test_evaluation_completed = True

print("\nREGRESSION TEST EVALUATION COMPLETION SUMMARY")
print("=" * 75)

print("Existing fitted regression pipelines: VERIFIED")
print("Frozen primary LOS selection: VERIFIED")
print("Held-out regression dataset: VERIFIED")
print("Training-derived baselines: COMPLETED")
print("Linear Regression test evaluation: COMPLETED")
print("Random Forest Regressor test evaluation: COMPLETED")
print("XGBoost Regressor test evaluation: COMPLETED")
print("Baseline improvement comparison: COMPLETED")
print("Validation-to-test comparison: COMPLETED")
print("Residual diagnostics: COMPLETED")
print("LOS range analysis: COMPLETED")
print("Proposal target assessment: COMPLETED")
print("Test evaluation visualizations: COMPLETED")
print("Evaluation metadata: CREATED")
print("JSON serialization verification: PASSED")
print("Final integrity checks: PASSED")

print("\nPrimary LOS model:", frozen_primary_los_model)
print("Regression models retained: 3")
print("Classification models retained: 2")
print("Total fitted pipelines retained: 5")

print("\nRegression test dataset evaluated: YES")
print("Classification test dataset evaluated: NO")
print("Model selection changed: NO")
print("Models retrained: NO")
print("Thresholds reoptimized: NO")
print("Clinical deployment approved: NO")
print("Model files exported: NO")

print(
    "\nReady for Cell 46: "
    "Held-Out Classification Test Evaluation."
)


In [ ]:

# ============================================================
# CELL 46 — HELD-OUT CLASSIFICATION TEST EVALUATION
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay
)

from sklearn.calibration import calibration_curve
from sklearn.utils.validation import check_is_fitted
from IPython.display import display

print("HELD-OUT CLASSIFICATION TEST EVALUATION")
print("=" * 75)

# ============================================================
# 1. REQUIRED VARIABLE VERIFICATION
# ============================================================

required_variables = [
    "lr_pipeline",
    "rf_los_pipeline",
    "xgb_los_pipeline",
    "selected_los_pipeline",
    "rf_classifier_pipeline",
    "xgb_classifier_pipeline",
    "X_proxy_test",
    "y_proxy_test",
    "y_proxy_train",
    "comparison_thresholds",
    "classification_probability_comparison_df",
    "regression_test_evaluation_completed"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing required variables: {missing_variables}"
)

assert regression_test_evaluation_completed

print("Required variables: VERIFIED")
print("Cell 45 completion: VERIFIED")

# ============================================================
# 2. FROZEN MODEL REGISTRY
# ============================================================

classification_test_models = {
    "Random Forest Classifier": rf_classifier_pipeline,
    "XGBoost Classifier": xgb_classifier_pipeline
}

frozen_primary_classifier = "XGBoost Classifier"

classification_test_frozen_thresholds = {
    "Random Forest Classifier": 0.09556332971472799,
    "XGBoost Classifier": 0.08537115901708603
}

for model_name, fitted_pipeline in classification_test_models.items():

    check_is_fitted(fitted_pipeline)

    assert np.array_equal(
        fitted_pipeline.classes_,
        np.array([0, 1])
    ), f"Unexpected class ordering: {model_name}"

    print(f"{model_name}: FITTED")

    print(
        "Frozen experimental threshold:",
        repr(classification_test_frozen_thresholds[model_name])
    )

assert selected_los_pipeline is rf_los_pipeline

print("\nPrimary experimental classifier:", frozen_primary_classifier)
print("Clinical threshold approved: FALSE")

# ============================================================
# 3. HELD-OUT TEST DATA VERIFICATION
# ============================================================

print("\nHELD-OUT CLASSIFICATION TEST DATA VERIFICATION")
print("=" * 75)

classification_test_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

assert list(X_proxy_test.columns) == classification_test_features

classification_test_y_raw = np.asarray(
    y_proxy_test
).reshape(-1)

classification_training_y_raw = np.asarray(
    y_proxy_train
).reshape(-1)

assert np.isin(
    classification_test_y_raw, [0, 1]
).all()

assert np.isin(
    classification_training_y_raw, [0, 1]
).all()

classification_test_y = (
    classification_test_y_raw.astype(np.int64)
)

classification_training_y = (
    classification_training_y_raw.astype(np.int64)
)

assert len(X_proxy_test) == len(classification_test_y)
assert len(classification_test_y) > 0

assert np.unique(classification_test_y).size == 2

if isinstance(y_proxy_test, pd.Series):

    assert X_proxy_test.index.equals(y_proxy_test.index), (
        "Test feature-target index mismatch."
    )

classification_test_records = len(classification_test_y)

classification_test_positives = int(
    classification_test_y.sum()
)

classification_test_negatives = (
    classification_test_records
    - classification_test_positives
)

classification_test_prevalence = float(
    classification_test_y.mean()
)

classification_test_train_prevalence = float(
    classification_training_y.mean()
)

print(f"Test records: {classification_test_records:,}")
print(f"Test positives: {classification_test_positives:,}")
print(f"Test negatives: {classification_test_negatives:,}")

print(
    f"Test prevalence: "
    f"{classification_test_prevalence:.6%}"
)

print(
    f"Training prevalence: "
    f"{classification_test_train_prevalence:.6%}"
)

print("Original features:", X_proxy_test.shape[1])
print("Test dataset verification: PASSED")

# ============================================================
# 4. EVALUATION METRIC FUNCTIONS
# ============================================================

def calculate_classification_probability_metrics(
    actual,
    probabilities
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    ).reshape(-1)

    assert len(actual) == len(probabilities)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    return {
        "ROC-AUC": float(
            roc_auc_score(actual, probabilities)
        ),
        "PR-AUC (AP)": float(
            average_precision_score(actual, probabilities)
        ),
        "Brier Score": float(
            brier_score_loss(actual, probabilities)
        ),
        "Log Loss": float(
            log_loss(actual, probabilities, labels=[0, 1])
        ),
        "Mean Probability": float(
            probabilities.mean()
        ),
        "Probability Std": float(
            probabilities.std()
        )
    }


def calculate_classification_decision_metrics(
    actual,
    predicted
):

    predicted = np.asarray(
        predicted,
        dtype=np.int64
    ).reshape(-1)

    assert len(actual) == len(predicted)

    tn, fp, fn, tp = confusion_matrix(
        actual,
        predicted,
        labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": float(
            accuracy_score(actual, predicted)
        ),
        "Precision": float(
            precision_score(
                actual, predicted, zero_division=0
            )
        ),
        "Recall": float(
            recall_score(
                actual, predicted, zero_division=0
            )
        ),
        "F1": float(
            f1_score(
                actual, predicted, zero_division=0
            )
        ),
        "Specificity": float(
            tn / (tn + fp)
        ),
        "Balanced Accuracy": float(
            balanced_accuracy_score(actual, predicted)
        ),
        "Predicted Positives": int(
            predicted.sum()
        ),
        "Predicted Positive Rate": float(
            predicted.mean()
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

# ============================================================
# 5. TRAINING-DERIVED PROBABILITY BASELINE
# ============================================================

print("\nTRAINING-DERIVED PROBABILITY BASELINE")
print("=" * 75)

classification_test_probabilities = {
    "Constant Training Prevalence": np.full(
        classification_test_records,
        classification_test_train_prevalence,
        dtype=np.float64
    )
}

classification_test_probability_rows = [{
    "Model": "Constant Training Prevalence",
    **calculate_classification_probability_metrics(
        classification_test_y,
        classification_test_probabilities[
            "Constant Training Prevalence"
        ]
    )
}]

display(
    pd.DataFrame(
        classification_test_probability_rows
    ).round(6)
)

# ============================================================
# 6. HELD-OUT CLASSIFIER PROBABILITIES
# ============================================================

print("\nHELD-OUT CLASSIFIER PROBABILITIES")
print("=" * 75)

classification_test_prediction_times = {}

for model_name, fitted_pipeline in classification_test_models.items():

    print(f"\nEvaluating {model_name}...")

    prediction_start = time.perf_counter()

    probabilities = np.asarray(
        fitted_pipeline.predict_proba(X_proxy_test)[:, 1],
        dtype=np.float64
    ).reshape(-1)

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    classification_test_probabilities[model_name] = probabilities

    classification_test_prediction_times[model_name] = float(
        prediction_time
    )

    metrics = calculate_classification_probability_metrics(
        classification_test_y,
        probabilities
    )

    classification_test_probability_rows.append({
        "Model": model_name,
        **metrics
    })

    print(f"ROC-AUC: {metrics['ROC-AUC']:.9f}")
    print(f"PR-AUC (AP): {metrics['PR-AUC (AP)']:.9f}")
    print(f"Brier Score: {metrics['Brier Score']:.9f}")
    print(f"Log Loss: {metrics['Log Loss']:.9f}")
    print(f"Prediction time: {prediction_time:.4f}s")

classification_test_probability_df = pd.DataFrame(
    classification_test_probability_rows
)

print("\nHELD-OUT PROBABILITY PERFORMANCE")
print("=" * 75)

display(classification_test_probability_df.round(6))

# ============================================================
# 7. PROBABILITY BASELINE IMPROVEMENT
# ============================================================

print("\nPROBABILITY BASELINE IMPROVEMENT")
print("=" * 75)

classification_test_baseline = (
    classification_test_probability_df.iloc[0]
)

classification_test_probability_comparison_df = (
    classification_test_probability_df.copy()
)

probability_comparison = (
    classification_test_probability_comparison_df
)

probability_comparison["ROC-AUC Improvement"] = (
    probability_comparison["ROC-AUC"]
    - classification_test_baseline["ROC-AUC"]
)

probability_comparison["AP Improvement"] = (
    probability_comparison["PR-AUC (AP)"]
    - classification_test_baseline["PR-AUC (AP)"]
)

probability_comparison["AP Lift"] = (
    probability_comparison["PR-AUC (AP)"]
    / classification_test_baseline["PR-AUC (AP)"]
)

probability_comparison["Brier Skill Score"] = (
    1
    - probability_comparison["Brier Score"]
    / classification_test_baseline["Brier Score"]
)

probability_comparison["Relative Log Loss Reduction"] = (
    1
    - probability_comparison["Log Loss"]
    / classification_test_baseline["Log Loss"]
)

display(probability_comparison.round(6))

# ============================================================
# 8. REFERENCE DECISION BASELINES
# ============================================================

print("\nREFERENCE DECISION BASELINES")
print("=" * 75)

classification_test_decision_predictions = {}

classification_test_threshold_rows = []

for baseline_name, baseline_class in [
    ("Always Negative", 0),
    ("Always Positive", 1)
]:

    predicted = np.full(
        classification_test_records,
        baseline_class,
        dtype=np.int64
    )

    classification_test_decision_predictions[
        (baseline_name, "Reference")
    ] = predicted

    classification_test_threshold_rows.append({
        "Model": baseline_name,
        "Threshold Type": "Reference",
        "Threshold": None,
        **calculate_classification_decision_metrics(
            classification_test_y,
            predicted
        )
    })

classification_test_reference_df = pd.DataFrame(
    classification_test_threshold_rows
)

display(classification_test_reference_df.round(6))

# ============================================================
# 9. FROZEN THRESHOLD EVALUATION
# ============================================================

print("\nFROZEN THRESHOLD EVALUATION")
print("=" * 75)

classification_test_confusion_matrices = {}
classification_test_reports = {}

for model_name in classification_test_models:

    probabilities = classification_test_probabilities[
        model_name
    ]

    threshold_configurations = {
        "Default": 0.5,
        "Proposal": 0.65,
        "Frozen Experimental": (
            classification_test_frozen_thresholds[
                model_name
            ]
        )
    }

    print("\n" + model_name)
    print("-" * 75)

    for threshold_type, threshold_value in (
        threshold_configurations.items()
    ):

        predicted = (
            probabilities >= threshold_value
        ).astype(np.int64)

        key = (model_name, threshold_type)

        classification_test_decision_predictions[
            key
        ] = predicted

        metrics = calculate_classification_decision_metrics(
            classification_test_y,
            predicted
        )

        classification_test_threshold_rows.append({
            "Model": model_name,
            "Threshold Type": threshold_type,
            "Threshold": float(threshold_value),
            **metrics
        })

        classification_test_confusion_matrices[key] = (
            confusion_matrix(
                classification_test_y,
                predicted,
                labels=[0, 1]
            )
        )

        classification_test_reports[key] = (
            classification_report(
                classification_test_y,
                predicted,
                labels=[0, 1],
                target_names=[
                    "Negative (0)",
                    "Positive (1)"
                ],
                output_dict=True,
                zero_division=0
            )
        )

        print(
            f"{threshold_type}: "
            f"{threshold_value:.17g}"
        )

        print(f"Precision: {metrics['Precision']:.6f}")
        print(f"Recall: {metrics['Recall']:.6f}")
        print(f"F1: {metrics['F1']:.6f}")

        print(
            "Predicted positive rate:",
            f"{metrics['Predicted Positive Rate']:.6%}"
        )

classification_test_threshold_df = pd.DataFrame(
    classification_test_threshold_rows
)

print("\nCONSOLIDATED TEST THRESHOLD COMPARISON")
print("=" * 75)

display(classification_test_threshold_df.round(6))

# ============================================================
# 10. EXPERIMENTAL THRESHOLD CONFUSION MATRICES
# ============================================================

print("\nEXPERIMENTAL THRESHOLD CONFUSION MATRICES")
print("=" * 75)

for model_name in classification_test_models:

    key = (model_name, "Frozen Experimental")

    cm = classification_test_confusion_matrices[key]

    print("\n" + model_name)

    print(
        "Frozen threshold:",
        repr(
            classification_test_frozen_thresholds[
                model_name
            ]
        )
    )

    display(
        pd.DataFrame(
            cm,
            index=[
                "Actual Negative",
                "Actual Positive"
            ],
            columns=[
                "Predicted Negative",
                "Predicted Positive"
            ]
        )
    )

    display(
        pd.DataFrame(
            classification_test_reports[key]
        ).T.round(6)
    )

    fig, ax = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negative", "Positive"]
    ).plot(
        ax=ax,
        cmap="Blues",
        values_format=",d",
        colorbar=True
    )

    ax.set_title(
        f"{model_name}\n"
        "Held-Out Frozen Experimental Threshold"
    )

    plt.tight_layout()
    plt.show()

# ============================================================
# 11. EXPERIMENTAL THRESHOLD UTILITY
# ============================================================

print("\nEXPERIMENTAL THRESHOLD UTILITY")
print("=" * 75)

classification_test_experimental_df = (
    classification_test_threshold_df.loc[
        classification_test_threshold_df[
            "Threshold Type"
        ] == "Frozen Experimental"
    ].copy()
)

classification_test_always_positive_f1 = float(
    classification_test_reference_df.loc[
        classification_test_reference_df["Model"]
        == "Always Positive",
        "F1"
    ].iloc[0]
)

classification_test_experimental_df[
    "F1 Improvement vs Always Positive"
] = (
    classification_test_experimental_df["F1"]
    - classification_test_always_positive_f1
)

classification_test_experimental_df[
    "False Positives per True Positive"
] = (
    classification_test_experimental_df["FP"]
    / classification_test_experimental_df["TP"].replace(
        0, np.nan
    )
)

classification_test_experimental_df[
    "Records Not Flagged"
] = (
    classification_test_records
    - classification_test_experimental_df[
        "Predicted Positives"
    ]
)

display(
    classification_test_experimental_df.round(6)
)

# ============================================================
# 12. VALIDATION-TO-TEST COMPARISON
# ============================================================

print("\nVALIDATION-TO-TEST CLASSIFICATION COMPARISON")
print("=" * 75)

classification_probability_metrics = [
    "ROC-AUC",
    "PR-AUC (AP)",
    "Brier Score",
    "Log Loss"
]

classification_validation_reference = (
    classification_probability_comparison_df[
        ["Model"] + classification_probability_metrics
    ].copy()
)

classification_validation_reference = (
    classification_validation_reference.rename(
        columns={
            metric: f"Validation {metric}"
            for metric in classification_probability_metrics
        }
    )
)

classification_test_reference = (
    classification_test_probability_df[
        ["Model"] + classification_probability_metrics
    ].rename(
        columns={
            metric: f"Test {metric}"
            for metric in classification_probability_metrics
        }
    )
)

classification_validation_test_df = (
    classification_validation_reference.merge(
        classification_test_reference,
        on="Model",
        how="inner",
        validate="one_to_one"
    )
)

assert len(classification_validation_test_df) == 3

for metric in classification_probability_metrics:

    classification_validation_test_df[
        f"{metric} Change"
    ] = (
        classification_validation_test_df[
            f"Test {metric}"
        ]
        - classification_validation_test_df[
            f"Validation {metric}"
        ]
    )

display(classification_validation_test_df.round(6))

print("Test results will not revise frozen model selection.")

# ============================================================
# 13. RISK RANKING ANALYSIS
# ============================================================

print("\nHELD-OUT RISK RANKING ANALYSIS")
print("=" * 75)

classification_test_risk_rows = []
classification_test_decile_rows = []

for model_name in classification_test_models:

    probabilities = classification_test_probabilities[
        model_name
    ]

    ranked_indices = np.argsort(
        -probabilities,
        kind="stable"
    )

    top_group_size = int(
        np.ceil(classification_test_records * 0.10)
    )

    top_indices = ranked_indices[:top_group_size]

    bottom_indices = ranked_indices[-top_group_size:]

    top_positives = int(
        classification_test_y[top_indices].sum()
    )

    top_positive_rate = float(
        classification_test_y[top_indices].mean()
    )

    classification_test_risk_rows.append({
        "Model": model_name,
        "Highest Risk Group Records": top_group_size,
        "Highest Risk Group Positives": top_positives,
        "Highest Risk Group Positive Rate": top_positive_rate,
        "Highest Risk Group Lift": (
            top_positive_rate
            / classification_test_prevalence
        ),
        "Positive Capture Rate": (
            top_positives
            / classification_test_positives
        ),
        "Lowest Risk Group Positive Rate": float(
            classification_test_y[bottom_indices].mean()
        )
    })

    for decile, indices in enumerate(
        np.array_split(ranked_indices, 10),
        start=1
    ):

        observed_rate = float(
            classification_test_y[indices].mean()
        )

        classification_test_decile_rows.append({
            "Model": model_name,
            "Risk Decile": decile,
            "Records": int(len(indices)),
            "Positive Records": int(
                classification_test_y[indices].sum()
            ),
            "Mean Predicted Probability": float(
                probabilities[indices].mean()
            ),
            "Observed Positive Rate": observed_rate,
            "Lift": (
                observed_rate
                / classification_test_prevalence
            )
        })

classification_test_risk_df = pd.DataFrame(
    classification_test_risk_rows
)

classification_test_decile_df = pd.DataFrame(
    classification_test_decile_rows
)

display(classification_test_risk_df.round(6))
display(classification_test_decile_df.round(6))

# ============================================================
# 14. PROPOSAL TARGET ASSESSMENT
# ============================================================

print("\nPROPOSAL TARGET ASSESSMENT")
print("=" * 75)

classification_test_proposal_rows = []

for model_name in classification_test_models:

    result = classification_test_probability_df.loc[
        classification_test_probability_df["Model"]
        == model_name
    ].iloc[0]

    for metric, boundary in [
        ("ROC-AUC", 0.80),
        ("PR-AUC (AP)", 0.75)
    ]:

        value = float(result[metric])

        classification_test_proposal_rows.append({
            "Model": model_name,
            "Metric": metric,
            "Test Result": value,
            "Proposal Boundary": boundary,
            "Required Direction": ">",
            "Gap to Boundary": boundary - value,
            "Target Status": (
                "ACHIEVED"
                if value > boundary
                else "NOT ACHIEVED"
            )
        })

classification_test_proposal_df = pd.DataFrame(
    classification_test_proposal_rows
)

display(classification_test_proposal_df.round(6))

# ============================================================
# 15. ROC CURVE
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))

for model_name in classification_test_models:

    probabilities = classification_test_probabilities[
        model_name
    ]

    fpr, tpr, _ = roc_curve(
        classification_test_y,
        probabilities
    )

    auc = roc_auc_score(
        classification_test_y,
        probabilities
    )

    ax.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC={auc:.4f})"
    )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Chance reference"
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Held-Out Classification ROC Curves")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ============================================================
# 16. PRECISION-RECALL CURVE
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))

for model_name in classification_test_models:

    probabilities = classification_test_probabilities[
        model_name
    ]

    precision, recall, _ = precision_recall_curve(
        classification_test_y,
        probabilities
    )

    ap = average_precision_score(
        classification_test_y,
        probabilities
    )

    ax.plot(
        recall,
        precision,
        label=f"{model_name} (AP={ap:.4f})"
    )

ax.axhline(
    classification_test_prevalence,
    linestyle="--",
    label="Test prevalence reference"
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Held-Out Precision-Recall Curves")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ============================================================
# 17. PROBABILITY DISTRIBUTIONS
# ============================================================

for model_name in classification_test_models:

    probabilities = classification_test_probabilities[
        model_name
    ]

    fig, ax = plt.subplots(figsize=(9, 5))

    bins = np.histogram_bin_edges(
        probabilities,
        bins=50
    )

    for class_value, class_name in [
        (0, "Actual Negative"),
        (1, "Actual Positive")
    ]:

        ax.hist(
            probabilities[
                classification_test_y == class_value
            ],
            bins=bins,
            density=True,
            alpha=0.5,
            label=class_name
        )

    ax.axvline(
        classification_test_frozen_thresholds[
            model_name
        ],
        linestyle="--",
        label="Frozen experimental threshold"
    )

    ax.set_xlabel("Predicted Probability")
    ax.set_ylabel("Density")
    ax.set_title(
        f"{model_name}: Held-Out Probability Distribution"
    )

    ax.legend()
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

# ============================================================
# 18. CALIBRATION DIAGNOSTICS — NO CALIBRATOR FITTED
# ============================================================

print("\nHELD-OUT CALIBRATION DIAGNOSTICS")
print("=" * 75)

classification_test_calibration_rows = []

fig, ax = plt.subplots(figsize=(8, 6))

for model_name in classification_test_models:

    observed, predicted = calibration_curve(
        classification_test_y,
        classification_test_probabilities[model_name],
        n_bins=10,
        strategy="quantile"
    )

    for bin_number, (mean_predicted, observed_rate) in enumerate(
        zip(predicted, observed),
        start=1
    ):

        classification_test_calibration_rows.append({
            "Model": model_name,
            "Bin": bin_number,
            "Mean Predicted Probability": float(
                mean_predicted
            ),
            "Observed Positive Rate": float(
                observed_rate
            )
        })

    ax.plot(
        predicted,
        observed,
        marker="o",
        label=model_name
    )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

ax.scatter(
    classification_test_train_prevalence,
    classification_test_prevalence,
    marker="s",
    label="Constant prevalence baseline"
)

ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Positive Rate")
ax.set_title("Held-Out Calibration Diagnostics")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

classification_test_calibration_df = pd.DataFrame(
    classification_test_calibration_rows
)

display(classification_test_calibration_df.round(6))

print("Calibration diagnostic only.")
print("No probability calibrator has been fitted.")

# ============================================================
# 19. PROBABILITY METRIC VISUALIZATIONS
# ============================================================

short_classifier_names = {
    "Constant Training Prevalence": "Baseline",
    "Random Forest Classifier": "Random Forest",
    "XGBoost Classifier": "XGBoost"
}

for metric in classification_probability_metrics:

    fig, ax = plt.subplots(figsize=(8, 5))

    bars = ax.bar(
        classification_test_probability_df["Model"].map(
            short_classifier_names
        ),
        classification_test_probability_df[metric]
    )

    ax.bar_label(
        bars,
        fmt="%.6f",
        padding=3
    )

    ax.axhline(
        classification_test_baseline[metric],
        linestyle="--",
        label="Reference baseline"
    )

    if metric == "ROC-AUC":

        ax.axhline(
            0.80,
            linestyle=":",
            label="Proposal boundary"
        )

        ax.set_ylim(0, 1)

    elif metric == "PR-AUC (AP)":

        ax.axhline(
            0.75,
            linestyle=":",
            label="Proposal boundary"
        )

        ax.set_ylim(0, 1)

    else:

        ax.margins(y=0.15)

    ax.set_title(f"Held-Out Classification: {metric}")
    ax.set_ylabel(metric)
    ax.legend()
    ax.grid(axis="y", alpha=0.2)

    plt.tight_layout()
    plt.show()

# ============================================================
# 20. ALERT BURDEN VISUALIZATION
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

alert_labels = (
    classification_test_experimental_df["Model"]
    .map(short_classifier_names)
)

true_positives = (
    classification_test_experimental_df["TP"]
)

false_positives = (
    classification_test_experimental_df["FP"]
)

ax.bar(
    alert_labels,
    true_positives,
    label="True positives",
    color="seagreen"
)

ax.bar(
    alert_labels,
    false_positives,
    bottom=true_positives,
    label="False positives",
    color="indianred"
)

ax.axhline(
    classification_test_positives,
    linestyle="--",
    label="Actual positive records"
)

ax.set_ylabel("Flagged visit records")
ax.set_title("Held-Out Experimental Threshold Alert Burden")
ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# ============================================================
# 21. RISK DECILE VISUALIZATION
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

for model_name in classification_test_models:

    decile_results = classification_test_decile_df.loc[
        classification_test_decile_df["Model"]
        == model_name
    ]

    ax.plot(
        decile_results["Risk Decile"],
        decile_results["Observed Positive Rate"],
        marker="o",
        label=model_name
    )

ax.axhline(
    classification_test_prevalence,
    linestyle="--",
    label="Overall test prevalence"
)

ax.set_xticks(range(1, 11))
ax.set_xlabel("Risk Decile (1 = Highest Predicted Risk)")
ax.set_ylabel("Observed Positive Rate")
ax.set_title("Held-Out Risk Ranking by Decile")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ============================================================
# 22. PRIMARY EXPERIMENTAL CLASSIFIER ASSESSMENT
# ============================================================

print("\nPRIMARY EXPERIMENTAL CLASSIFIER ASSESSMENT")
print("=" * 75)

primary_classifier_test_result = (
    classification_test_probability_comparison_df.loc[
        classification_test_probability_comparison_df[
            "Model"
        ] == frozen_primary_classifier
    ].iloc[0]
)

primary_classifier_threshold_result = (
    classification_test_experimental_df.loc[
        classification_test_experimental_df["Model"]
        == frozen_primary_classifier
    ].iloc[0]
)

print("Selected model:", frozen_primary_classifier)
print("Selection source: Validation")

print(
    "Test ROC-AUC:",
    f"{primary_classifier_test_result['ROC-AUC']:.9f}"
)

print(
    "Test PR-AUC (AP):",
    f"{primary_classifier_test_result['PR-AUC (AP)']:.9f}"
)

print(
    "Test Brier Score:",
    f"{primary_classifier_test_result['Brier Score']:.9f}"
)

print(
    "Frozen experimental threshold:",
    repr(
        classification_test_frozen_thresholds[
            frozen_primary_classifier
        ]
    )
)

print(
    "Test F1:",
    f"{primary_classifier_threshold_result['F1']:.9f}"
)

print(
    "Predicted positive rate:",
    f"{primary_classifier_threshold_result['Predicted Positive Rate']:.6%}"
)

print("Model selection changed: FALSE")
print("Clinical deployment approved: FALSE")

# ============================================================
# 23. TEST EVALUATION METADATA
# ============================================================

print("\nCLASSIFICATION TEST EVALUATION METADATA")
print("=" * 75)

def classification_table_to_records(table):

    return json.loads(
        table.to_json(
            orient="records",
            double_precision=15
        )
    )

classification_test_evaluation_metadata = {
    "dataset_version": "V2",
    "task": "30-Day Subsequent Visit Proxy",
    "target": "SUBSEQUENT_VISIT_WITHIN_30D_PROXY",
    "evaluation_dataset": "Held-out test",
    "test_records": int(classification_test_records),
    "test_positives": classification_test_positives,
    "test_negatives": classification_test_negatives,
    "test_prevalence": classification_test_prevalence,
    "training_prevalence": classification_test_train_prevalence,
    "original_features": classification_test_features,
    "frozen_primary_model": frozen_primary_classifier,
    "selection_dataset": "Validation",
    "selection_metric": "Average Precision",
    "pr_auc_convention": "Average Precision",
    "positive_class": 1,
    "prediction_rule": "probability >= threshold",
    "frozen_experimental_thresholds": (
        classification_test_frozen_thresholds.copy()
    ),
    "threshold_role": "EXPERIMENTAL_ONLY",
    "default_threshold": 0.5,
    "proposal_threshold": 0.65,
    "proposal_roc_auc_boundary": 0.80,
    "proposal_ap_boundary": 0.75,
    "proposal_required_direction": ">",
    "baseline_source": "Training prevalence",
    "model_retraining": False,
    "threshold_reoptimization": False,
    "probability_calibration_fitted": False,
    "model_selection_changed": False,
    "test_used_for_model_selection": False,
    "clinical_threshold_approved": False,
    "clinical_deployment_approved": False,
    "prediction_times_seconds": (
        classification_test_prediction_times
    ),
    "probability_results": classification_table_to_records(
        classification_test_probability_comparison_df
    ),
    "threshold_results": classification_table_to_records(
        classification_test_threshold_df
    ),
    "experimental_threshold_utility": (
        classification_table_to_records(
            classification_test_experimental_df
        )
    ),
    "validation_test_comparison": (
        classification_table_to_records(
            classification_validation_test_df
        )
    ),
    "risk_ranking": classification_table_to_records(
        classification_test_risk_df
    ),
    "risk_deciles": classification_table_to_records(
        classification_test_decile_df
    ),
    "calibration_diagnostics": (
        classification_table_to_records(
            classification_test_calibration_df
        )
    ),
    "proposal_assessment": classification_table_to_records(
        classification_test_proposal_df
    ),
    "confusion_matrices": {
        model_name: {
            threshold_type: (
                classification_test_confusion_matrices[
                    (model_name, threshold_type)
                ].tolist()
            )
            for threshold_type in [
                "Default",
                "Proposal",
                "Frozen Experimental"
            ]
        }
        for model_name in classification_test_models
    },
    "classification_reports": {
        model_name: {
            threshold_type: (
                classification_test_reports[
                    (model_name, threshold_type)
                ]
            )
            for threshold_type in [
                "Default",
                "Proposal",
                "Frozen Experimental"
            ]
        }
        for model_name in classification_test_models
    }
}

json.dumps(
    classification_test_evaluation_metadata,
    allow_nan=False
)

print("Metadata creation: COMPLETED")
print("JSON serialization verification: PASSED")

# ============================================================
# 24. FINAL INTEGRITY CHECKS
# ============================================================

assert len(classification_test_models) == 2
assert len(classification_test_probability_df) == 3
assert len(classification_test_threshold_df) == 8
assert len(classification_test_confusion_matrices) == 6

for key, matrix in classification_test_confusion_matrices.items():

    assert matrix.sum() == classification_test_records

    assert matrix[0].sum() == classification_test_negatives

    assert matrix[1].sum() == classification_test_positives

assert selected_los_pipeline is rf_los_pipeline

for fitted_pipeline in [
    lr_pipeline,
    rf_los_pipeline,
    xgb_los_pipeline,
    rf_classifier_pipeline,
    xgb_classifier_pipeline
]:

    check_is_fitted(fitted_pipeline)

classification_test_evaluation_completed = True

print("\nCLASSIFICATION TEST EVALUATION COMPLETION SUMMARY")
print("=" * 75)

print("Existing fitted classifiers: VERIFIED")
print("Frozen experimental thresholds: VERIFIED")
print("Held-out classification dataset: VERIFIED")
print("Training-derived probability baseline: COMPLETED")
print("Always-negative reference: COMPLETED")
print("Always-positive reference: COMPLETED")
print("Random Forest test evaluation: COMPLETED")
print("XGBoost test evaluation: COMPLETED")
print("Probability baseline comparison: COMPLETED")
print("Frozen threshold evaluation: COMPLETED")
print("Confusion matrices: COMPLETED")
print("Classification reports: COMPLETED")
print("Experimental threshold utility: COMPLETED")
print("Validation-to-test comparison: COMPLETED")
print("Risk ranking analysis: COMPLETED")
print("Calibration diagnostics: COMPLETED")
print("Proposal target assessment: COMPLETED")
print("Evaluation visualizations: COMPLETED")
print("Evaluation metadata: CREATED")
print("JSON serialization verification: PASSED")
print("Final integrity checks: PASSED")

print("\nFROZEN MODEL SELECTIONS")
print("=" * 75)

print("Primary LOS model: Random Forest Regressor")
print("Primary experimental classifier: XGBoost Classifier")

print(
    "Frozen experimental threshold:",
    repr(
        classification_test_frozen_thresholds[
            frozen_primary_classifier
        ]
    )
)

print("\nRegression models retained: 3")
print("Classification models retained: 2")
print("Total fitted pipelines retained: 5")

print("\nRegression test dataset evaluated: YES")
print("Classification test dataset evaluated: YES")
print("Model selection changed: NO")
print("Models retrained: NO")
print("Thresholds reoptimized: NO")
print("Probability calibration fitted: NO")
print("Clinical deployment approved: NO")
print("Model files exported: NO")

print(
    "\nReady for Cell 47: "
    "Final Predictive Model Performance Consolidation."
)


In [ ]:

# ============================================================
# CELL 47 — FINAL PREDICTIVE MODEL PERFORMANCE CONSOLIDATION
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.utils.validation import check_is_fitted
from IPython.display import display

print("FINAL PREDICTIVE MODEL PERFORMANCE CONSOLIDATION")
print("=" * 75)

# ============================================================
# 1. REQUIRED VARIABLE VERIFICATION
# ============================================================

required_variables = [
    "lr_pipeline",
    "rf_los_pipeline",
    "xgb_los_pipeline",
    "rf_classifier_pipeline",
    "xgb_classifier_pipeline",
    "selected_los_pipeline",
    "final_regression_validation_ranking",
    "regression_test_evaluation_completed",
    "classification_test_evaluation_completed",
    "regression_test_evaluation_metadata",
    "classification_test_evaluation_metadata",
    "classification_probability_comparison_df",
    "classification_test_probability_comparison_df",
    "classification_test_experimental_df",
    "classification_test_risk_df",
    "classification_test_frozen_thresholds",
    "classification_test_confusion_matrices",
    "classification_test_threshold_df",
    "frozen_primary_classifier"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing required variables: {missing_variables}"
)

assert regression_test_evaluation_completed
assert classification_test_evaluation_completed

print("Required variables: VERIFIED")
print("Regression test evaluation: VERIFIED")
print("Classification test evaluation: VERIFIED")

# ============================================================
# 2. FINAL MODEL REGISTRY
# ============================================================

final_predictive_model_registry = {
    "Linear Regression": lr_pipeline,
    "Random Forest Regressor": rf_los_pipeline,
    "XGBoost Regressor": xgb_los_pipeline,
    "Random Forest Classifier": rf_classifier_pipeline,
    "XGBoost Classifier": xgb_classifier_pipeline
}

for model_name, fitted_pipeline in (
    final_predictive_model_registry.items()
):

    check_is_fitted(fitted_pipeline)

    print(f"{model_name}: FITTED")

assert len(final_predictive_model_registry) == 5

assert selected_los_pipeline is rf_los_pipeline

assert frozen_primary_classifier == "XGBoost Classifier"

assert len({
    id(pipeline)
    for pipeline in final_predictive_model_registry.values()
}) == 5

print("Five distinct fitted pipelines: VERIFIED")
print("Frozen model selections: VERIFIED")

# ============================================================
# 3. FROZEN MODEL CONFIGURATION
# ============================================================

final_primary_los_model = "Random Forest Regressor"

final_primary_classifier = frozen_primary_classifier

final_experimental_thresholds = (
    classification_test_frozen_thresholds.copy()
)

assert final_experimental_thresholds[
    "Random Forest Classifier"
] == 0.09556332971472799

assert final_experimental_thresholds[
    "XGBoost Classifier"
] == 0.08537115901708603

final_model_roles = {
    "Linear Regression":
        "COMPARATIVE_MODEL",

    "Random Forest Regressor":
        "PRIMARY_EXPERIMENTAL_CANDIDATE",

    "XGBoost Regressor":
        "SECONDARY_EXPERIMENTAL_CANDIDATE",

    "Random Forest Classifier":
        "SECONDARY_EXPERIMENTAL_CANDIDATE",

    "XGBoost Classifier":
        "PRIMARY_EXPERIMENTAL_CANDIDATE"
}

print("\nFROZEN MODEL CONFIGURATION")
print("=" * 75)

print("Primary LOS model:", final_primary_los_model)
print("Primary classifier:", final_primary_classifier)

for model_name, threshold_value in (
    final_experimental_thresholds.items()
):

    print(
        model_name,
        "experimental threshold:",
        repr(threshold_value)
    )

# ============================================================
# 4. RECOVER REGRESSION TEST RESULTS
# ============================================================
# Locate the existing test results table using its schema.
# Do not generate new predictions.
# ============================================================

regression_required_columns = {
    "Model",
    "Test MAE",
    "Test RMSE",
    "Test R2"
}

regression_expected_models = {
    "Linear Regression",
    "Random Forest Regressor",
    "XGBoost Regressor"
}

regression_result_candidates = {}

for variable_name, variable_value in list(globals().items()):

    if not isinstance(variable_value, pd.DataFrame):
        continue

    if not regression_required_columns.issubset(
        variable_value.columns
    ):
        continue

    if not regression_expected_models.issubset(
        set(variable_value["Model"])
    ):
        continue

    candidate = variable_value.loc[
        variable_value["Model"].isin(
            regression_expected_models
        ),
        ["Model", "Test MAE", "Test RMSE", "Test R2"]
    ].copy()

    if len(candidate) != 3:
        continue

    if not candidate["Model"].is_unique:
        continue

    regression_result_candidates[variable_name] = (
        candidate.sort_values("Model").reset_index(drop=True)
    )

assert regression_result_candidates, (
    "No compatible regression test results table found."
)

candidate_names = sorted(
    regression_result_candidates.keys()
)

regression_reference_name = candidate_names[0]

regression_reference_table = (
    regression_result_candidates[
        regression_reference_name
    ].copy()
)

for candidate_name in candidate_names:

    candidate_table = regression_result_candidates[
        candidate_name
    ]

    assert candidate_table["Model"].equals(
        regression_reference_table["Model"]
    )

    assert np.allclose(
        candidate_table[
            ["Test MAE", "Test RMSE", "Test R2"]
        ].to_numpy(dtype=float),

        regression_reference_table[
            ["Test MAE", "Test RMSE", "Test R2"]
        ].to_numpy(dtype=float),

        atol=1e-8,
        rtol=0
    ), (
        f"Inconsistent regression results: {candidate_name}"
    )

final_regression_test_reference_df = (
    regression_reference_table.copy()
)

print("\nREGRESSION TEST RESULT RECOVERY")
print("=" * 75)

print("Compatible tables:", candidate_names)
print("Reference table:", regression_reference_name)
print("Cross-table consistency: PASSED")

display(final_regression_test_reference_df.round(9))

# ============================================================
# 5. REGRESSION VALIDATION AND TEST CONSOLIDATION
# ============================================================

regression_validation_columns = [
    "Model",
    "Validation MAE",
    "Validation RMSE",
    "Validation R2"
]

assert set(regression_validation_columns).issubset(
    final_regression_validation_ranking.columns
)

final_regression_performance_df = (
    final_regression_validation_ranking[
        regression_validation_columns
    ].merge(
        final_regression_test_reference_df,
        on="Model",
        how="inner",
        validate="one_to_one"
    )
)

assert len(final_regression_performance_df) == 3

final_regression_performance_df = (
    final_regression_performance_df
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

final_regression_performance_df.insert(
    0,
    "Validation Rank",
    np.arange(1, 4)
)

final_regression_performance_df["MAE Change"] = (
    final_regression_performance_df["Test MAE"]
    - final_regression_performance_df["Validation MAE"]
)

final_regression_performance_df["RMSE Change"] = (
    final_regression_performance_df["Test RMSE"]
    - final_regression_performance_df["Validation RMSE"]
)

final_regression_performance_df["R2 Change"] = (
    final_regression_performance_df["Test R2"]
    - final_regression_performance_df["Validation R2"]
)

final_regression_performance_df["Model Role"] = (
    final_regression_performance_df["Model"].map(
        final_model_roles
    )
)

final_regression_performance_df["Proposal Achieved"] = (
    final_regression_performance_df["Test MAE"] < 1.0
)

assert final_regression_performance_df.iloc[0][
    "Model"
] == final_primary_los_model

print("\nFINAL REGRESSION PERFORMANCE")
print("=" * 75)

display(final_regression_performance_df.round(9))

# ============================================================
# 6. CLASSIFICATION VALIDATION AND TEST CONSOLIDATION
# ============================================================

classification_model_names = [
    "Random Forest Classifier",
    "XGBoost Classifier"
]

probability_metrics = [
    "ROC-AUC",
    "PR-AUC (AP)",
    "Brier Score",
    "Log Loss"
]

validation_reference = (
    classification_probability_comparison_df.loc[
        classification_probability_comparison_df[
            "Model"
        ].isin(classification_model_names),
        ["Model"] + probability_metrics
    ].copy()
)

test_reference = (
    classification_test_probability_comparison_df.loc[
        classification_test_probability_comparison_df[
            "Model"
        ].isin(classification_model_names),
        ["Model"] + probability_metrics
    ].copy()
)

validation_reference = validation_reference.rename(
    columns={
        metric: f"Validation {metric}"
        for metric in probability_metrics
    }
)

test_reference = test_reference.rename(
    columns={
        metric: f"Test {metric}"
        for metric in probability_metrics
    }
)

final_classification_performance_df = (
    validation_reference.merge(
        test_reference,
        on="Model",
        how="inner",
        validate="one_to_one"
    )
)

assert len(final_classification_performance_df) == 2

final_classification_performance_df = (
    final_classification_performance_df
    .sort_values(
        "Validation PR-AUC (AP)",
        ascending=False
    )
    .reset_index(drop=True)
)

final_classification_performance_df.insert(
    0,
    "Validation Rank",
    np.arange(1, 3)
)

for metric in probability_metrics:

    final_classification_performance_df[
        f"{metric} Change"
    ] = (
        final_classification_performance_df[
            f"Test {metric}"
        ]
        - final_classification_performance_df[
            f"Validation {metric}"
        ]
    )

final_classification_performance_df["Model Role"] = (
    final_classification_performance_df["Model"].map(
        final_model_roles
    )
)

final_classification_performance_df[
    "ROC-AUC Target Achieved"
] = (
    final_classification_performance_df[
        "Test ROC-AUC"
    ] > 0.80
)

final_classification_performance_df[
    "AP Target Achieved"
] = (
    final_classification_performance_df[
        "Test PR-AUC (AP)"
    ] > 0.75
)

assert final_classification_performance_df.iloc[0][
    "Model"
] == final_primary_classifier

print("\nFINAL CLASSIFICATION PERFORMANCE")
print("=" * 75)

display(final_classification_performance_df.round(9))

# ============================================================
# 7. CLASSIFICATION THRESHOLD CONSOLIDATION
# ============================================================

decision_columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
    "Balanced Accuracy",
    "Predicted Positives",
    "Predicted Positive Rate",
    "TN",
    "FP",
    "FN",
    "TP",
    "F1 Improvement vs Always Positive",
    "False Positives per True Positive"
]

final_classification_decision_df = (
    classification_test_experimental_df[
        decision_columns
    ].copy()
)

final_classification_decision_df.insert(
    1,
    "Experimental Threshold",
    final_classification_decision_df["Model"].map(
        final_experimental_thresholds
    )
)

final_classification_decision_df["Threshold Role"] = (
    "EXPERIMENTAL_ONLY"
)

final_classification_decision_df[
    "Clinical Threshold Approved"
] = False

for _, result in final_classification_decision_df.iterrows():

    model_name = result["Model"]

    expected_matrix = (
        classification_test_confusion_matrices[
            (model_name, "Frozen Experimental")
        ]
    )

    recovered_matrix = np.array([
        [result["TN"], result["FP"]],
        [result["FN"], result["TP"]]
    ], dtype=np.int64)

    assert np.array_equal(
        recovered_matrix,
        expected_matrix
    )

print("\nFINAL CLASSIFICATION DECISION PERFORMANCE")
print("=" * 75)

display(final_classification_decision_df.round(9))

print("Confusion matrix consistency: PASSED")

# ============================================================
# 8. BASELINE COMPARISON
# ============================================================

final_classifier_baseline_df = (
    classification_test_probability_comparison_df[
        [
            "Model",
            "ROC-AUC",
            "PR-AUC (AP)",
            "Brier Score",
            "Log Loss",
            "ROC-AUC Improvement",
            "AP Improvement",
            "AP Lift",
            "Brier Skill Score",
            "Relative Log Loss Reduction"
        ]
    ].copy()
)

print("\nFINAL CLASSIFICATION BASELINE COMPARISON")
print("=" * 75)

display(final_classifier_baseline_df.round(9))

print("\nFINAL CLASSIFICATION RISK RANKING")
print("=" * 75)

final_classifier_risk_df = (
    classification_test_risk_df.copy()
)

display(final_classifier_risk_df.round(9))

# ============================================================
# 9. CONSOLIDATED PROPOSAL TARGET ASSESSMENT
# ============================================================

final_proposal_rows = []

for _, result in final_regression_performance_df.iterrows():

    value = float(result["Test MAE"])

    final_proposal_rows.append({
        "Task": "LOS Regression",
        "Model": result["Model"],
        "Metric": "MAE",
        "Test Result": value,
        "Proposal Boundary": 1.0,
        "Required Direction": "<",
        "Gap to Boundary": value - 1.0,
        "Target Achieved": bool(value < 1.0)
    })

for _, result in (
    final_classification_performance_df.iterrows()
):

    for metric, boundary in [
        ("ROC-AUC", 0.80),
        ("PR-AUC (AP)", 0.75)
    ]:

        value = float(result[f"Test {metric}"])

        final_proposal_rows.append({
            "Task": "30-Day Subsequent Visit Proxy",
            "Model": result["Model"],
            "Metric": metric,
            "Test Result": value,
            "Proposal Boundary": boundary,
            "Required Direction": ">",
            "Gap to Boundary": boundary - value,
            "Target Achieved": bool(value > boundary)
        })

final_proposal_assessment_df = pd.DataFrame(
    final_proposal_rows
)

assert len(final_proposal_assessment_df) == 7

print("\nFINAL PROPOSAL TARGET ASSESSMENT")
print("=" * 75)

display(final_proposal_assessment_df.round(9))

primary_model_names = [
    final_primary_los_model,
    final_primary_classifier
]

final_primary_proposal_df = (
    final_proposal_assessment_df.loc[
        final_proposal_assessment_df["Model"].isin(
            primary_model_names
        )
    ].copy()
)

assert len(final_primary_proposal_df) == 3

print(
    "Primary model targets achieved:",
    int(
        final_primary_proposal_df[
            "Target Achieved"
        ].sum()
    ),
    "of",
    len(final_primary_proposal_df)
)

print(
    "All model-metric targets achieved:",
    int(
        final_proposal_assessment_df[
            "Target Achieved"
        ].sum()
    ),
    "of",
    len(final_proposal_assessment_df)
)

# ============================================================
# 10. FINAL MODEL ROLES AND DEPLOYMENT STATUS
# ============================================================

final_model_status_rows = []

for model_name in final_predictive_model_registry:

    is_regression = (
        model_name in regression_expected_models
    )

    task = (
        "LOS Regression"
        if is_regression
        else "30-Day Subsequent Visit Proxy"
    )

    model_proposal_results = (
        final_proposal_assessment_df.loc[
            final_proposal_assessment_df["Model"]
            == model_name
        ]
    )

    assert len(model_proposal_results) > 0

    all_targets_achieved = bool(
        model_proposal_results[
            "Target Achieved"
        ].all()
    )

    final_model_status_rows.append({
        "Task": task,
        "Model": model_name,
        "Model Role": final_model_roles[model_name],
        "Primary Experimental Model": (
            model_name in primary_model_names
        ),
        "Fitted": True,
        "Validation Evaluated": True,
        "Held-Out Test Evaluated": True,
        "Proposal Targets Achieved": all_targets_achieved,
        "Experimental Threshold": (
            final_experimental_thresholds.get(
                model_name
            )
        ),
        "Clinical Deployment Approved": False,
        "Deployment Status": "NOT_RECOMMENDED",
        "Research Export Approved": True,
        "Operational Alerts Approved": False
    })

final_predictive_model_status_df = pd.DataFrame(
    final_model_status_rows
)

print("\nFINAL MODEL STATUS")
print("=" * 75)

display(final_predictive_model_status_df)

# ============================================================
# 11. FEATURE CONTRACT
# ============================================================

final_predictive_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

final_numeric_features = [
    "AGE_AT_VISIT",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

final_categorical_features = [
    "GENDER",
    "VISIT_TYPE"
]

final_feature_contract_rows = []

for model_name, fitted_pipeline in (
    final_predictive_model_registry.items()
):

    assert hasattr(
        fitted_pipeline,
        "feature_names_in_"
    ), f"Missing feature names: {model_name}"

    actual_features = list(
        fitted_pipeline.feature_names_in_
    )

    assert actual_features == final_predictive_features, (
        f"Feature contract mismatch: {model_name}"
    )

    final_feature_contract_rows.append({
        "Model": model_name,
        "Input Feature Count": len(actual_features),
        "Input Features": actual_features,
        "Feature Contract Verified": True
    })

final_feature_contract_df = pd.DataFrame(
    final_feature_contract_rows
)

print("\nFINAL FEATURE CONTRACT")
print("=" * 75)

display(final_feature_contract_df)

# ============================================================
# 12. FINAL PERFORMANCE VISUALIZATIONS
# ============================================================

regression_plot_df = (
    final_regression_performance_df.copy()
)

regression_plot_df["Short Name"] = (
    regression_plot_df["Model"].map({
        "Linear Regression": "Linear Regression",
        "Random Forest Regressor": "Random Forest",
        "XGBoost Regressor": "XGBoost"
    })
)

x = np.arange(len(regression_plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width / 2,
    regression_plot_df["Validation MAE"],
    width,
    label="Validation"
)

ax.bar(
    x + width / 2,
    regression_plot_df["Test MAE"],
    width,
    label="Held-out test"
)

ax.axhline(
    1.0,
    linestyle="--",
    label="Proposal boundary"
)

ax.set_xticks(x)
ax.set_xticklabels(regression_plot_df["Short Name"])
ax.set_ylabel("MAE (days)")
ax.set_title("Final LOS Regression Performance")
ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

classification_plot_df = (
    final_classification_performance_df.copy()
)

classification_plot_df["Short Name"] = (
    classification_plot_df["Model"].map({
        "Random Forest Classifier": "Random Forest",
        "XGBoost Classifier": "XGBoost"
    })
)

for metric, boundary in [
    ("ROC-AUC", 0.80),
    ("PR-AUC (AP)", 0.75)
]:

    x = np.arange(len(classification_plot_df))

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.bar(
        x - width / 2,
        classification_plot_df[f"Validation {metric}"],
        width,
        label="Validation"
    )

    ax.bar(
        x + width / 2,
        classification_plot_df[f"Test {metric}"],
        width,
        label="Held-out test"
    )

    ax.axhline(
        boundary,
        linestyle="--",
        label="Proposal boundary"
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        classification_plot_df["Short Name"]
    )

    ax.set_ylim(0, 1)
    ax.set_ylabel(metric)
    ax.set_title(f"Final Classification Performance: {metric}")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)

    plt.tight_layout()
    plt.show()

# ============================================================
# 13. FINAL LIMITATIONS AND RECOMMENDATIONS
# ============================================================

final_predictive_limitations = [
    (
        "LOS prediction errors remain large relative "
        "to the proposal target."
    ),
    (
        "LOS predictions are concentrated around "
        "central target values."
    ),
    (
        "Both classifiers exhibit approximately "
        "chance-level discrimination."
    ),
    (
        "Classification probability errors exceed "
        "the constant training-prevalence baseline."
    ),
    (
        "Experimental thresholds flag nearly all "
        "held-out visit records."
    ),
    (
        "The default and proposal thresholds produce "
        "no positive classifications."
    ),
    (
        "The highest predicted risk deciles do not "
        "demonstrate useful positive-case enrichment."
    ),
    (
        "The classification target represents a "
        "30-day subsequent visit proxy, not a "
        "validated unplanned hospital readmission outcome."
    ),
    (
        "Patient-grouped held-out evaluation does not "
        "establish temporal or external generalizability."
    ),
    (
        "No patient-grouped confidence intervals or "
        "formal statistical significance tests have "
        "been reported."
    ),
    (
        "Random Forest probability regeneration may "
        "produce negligible floating-point differences "
        "that affect exact threshold-boundary decisions."
    )
]

final_predictive_recommendations = [
    "Retain all five fitted pipelines for research.",
    "Preserve validation-based model selections.",
    "Preserve full-precision experimental thresholds.",
    "Do not activate clinical risk alerts.",
    "Do not use LOS predictions for clinical decisions.",
    (
        "Investigate additional clinically relevant "
        "features available at prediction time."
    ),
    (
        "Verify outcome definitions and source "
        "timestamp semantics before future model development."
    ),
    (
        "Use a new independent evaluation dataset "
        "for future model revisions."
    ),
    (
        "Restrict FastAPI integration to clearly "
        "labeled research or demonstration use."
    )
]

print("\nFINAL LIMITATIONS")
print("=" * 75)

for number, limitation in enumerate(
    final_predictive_limitations,
    start=1
):

    print(f"{number}. {limitation}")

print("\nFINAL RECOMMENDATIONS")
print("=" * 75)

for number, recommendation in enumerate(
    final_predictive_recommendations,
    start=1
):

    print(f"{number}. {recommendation}")

# ============================================================
# 14. FINAL CONSOLIDATED METADATA
# ============================================================

def final_table_to_records(table):

    return json.loads(
        table.to_json(
            orient="records",
            double_precision=15
        )
    )

final_predictive_model_metadata = {
    "project": "St. Hubert Medical Center",
    "dataset_version": "V2",
    "total_fitted_models": 5,
    "regression_models": 3,
    "classification_models": 2,

    "primary_los_model": final_primary_los_model,

    "primary_experimental_classifier": (
        final_primary_classifier
    ),

    "model_selection_dataset": "Validation",

    "regression_selection_metric": "MAE",

    "classification_selection_metric": (
        "Average Precision"
    ),

    "regression_target": "TARGET_LOS_DAYS",

    "classification_target": (
        "SUBSEQUENT_VISIT_WITHIN_30D_PROXY"
    ),

    "classification_target_is_validated_readmission": (
        False
    ),

    "input_features": final_predictive_features,
    "numeric_features": final_numeric_features,
    "categorical_features": final_categorical_features,

    "patient_identifier_used_as_feature": False,

    "model_selection_frozen": True,
    "experimental_thresholds_frozen": True,

    # Preserve thresholds directly as Python floats.
    # Do not recover them from rounded display tables.

    "experimental_thresholds": (
        final_experimental_thresholds.copy()
    ),

    "experimental_thresholds_full_precision": {
        model_name: repr(threshold_value)
        for model_name, threshold_value in (
            final_experimental_thresholds.items()
        )
    },

    "threshold_prediction_rule": (
        "probability >= threshold"
    ),

    "threshold_role": "EXPERIMENTAL_ONLY",

    "default_threshold": 0.5,
    "proposal_threshold": 0.65,

    "regression_test_evaluated": True,
    "classification_test_evaluated": True,

    "test_used_for_model_selection": False,
    "test_used_for_threshold_optimization": False,

    "clinical_deployment_approved": False,
    "clinical_threshold_approved": False,

    "research_export_approved": True,
    "model_files_exported": False,

    "regression_performance": final_table_to_records(
        final_regression_performance_df
    ),

    "classification_performance": final_table_to_records(
        final_classification_performance_df
    ),

    "classification_decision_performance": (
        final_table_to_records(
            final_classification_decision_df
        )
    ),

    "classification_baseline_comparison": (
        final_table_to_records(
            final_classifier_baseline_df
        )
    ),

    "classification_risk_ranking": (
        final_table_to_records(
            final_classifier_risk_df
        )
    ),

    "proposal_assessment": final_table_to_records(
        final_proposal_assessment_df
    ),

    "model_status": final_table_to_records(
        final_predictive_model_status_df
    ),

    "feature_contract": final_table_to_records(
        final_feature_contract_df
    ),

    "limitations": final_predictive_limitations,

    "recommendations": (
        final_predictive_recommendations
    ),

    "regression_test_evaluation": (
        regression_test_evaluation_metadata
    ),

    "classification_test_evaluation": (
        classification_test_evaluation_metadata
    )
}

serialized_final_metadata = json.dumps(
    final_predictive_model_metadata,
    indent=2,
    allow_nan=False
)

recovered_final_metadata = json.loads(
    serialized_final_metadata
)

assert recovered_final_metadata[
    "experimental_thresholds"
] == final_experimental_thresholds

assert recovered_final_metadata[
    "total_fitted_models"
] == 5

print("\nFINAL CONSOLIDATED METADATA")
print("=" * 75)

print("Metadata creation: COMPLETED")
print("JSON serialization: PASSED")
print("Full-precision threshold round trip: PASSED")

# ============================================================
# 15. FINAL INTEGRITY CHECKS
# ============================================================

assert len(final_predictive_model_registry) == 5

assert len(final_regression_performance_df) == 3

assert len(final_classification_performance_df) == 2

assert len(final_classification_decision_df) == 2

assert len(final_predictive_model_status_df) == 5

assert len(final_feature_contract_df) == 5

assert len(final_proposal_assessment_df) == 7

assert final_predictive_model_status_df[
    "Held-Out Test Evaluated"
].all()

assert not final_predictive_model_status_df[
    "Clinical Deployment Approved"
].any()

assert not final_predictive_model_status_df[
    "Operational Alerts Approved"
].any()

assert final_predictive_model_status_df[
    "Research Export Approved"
].all()

assert selected_los_pipeline is rf_los_pipeline

assert final_primary_classifier == (
    "XGBoost Classifier"
)

for fitted_pipeline in (
    final_predictive_model_registry.values()
):

    check_is_fitted(fitted_pipeline)

final_predictive_consolidation_completed = True

print("\nFINAL PERFORMANCE CONSOLIDATION SUMMARY")
print("=" * 75)

print("Five fitted pipelines: VERIFIED")
print("Frozen model selections: VERIFIED")
print("Regression performance: CONSOLIDATED")
print("Classification performance: CONSOLIDATED")
print("Experimental thresholds: VERIFIED")
print("Confusion matrices: VERIFIED")
print("Baseline comparisons: CONSOLIDATED")
print("Risk ranking: CONSOLIDATED")
print("Proposal assessment: COMPLETED")
print("Model roles: FINALIZED")
print("Feature contracts: VERIFIED")
print("Deployment restrictions: DOCUMENTED")
print("Limitations: DOCUMENTED")
print("Recommendations: DOCUMENTED")
print("Final metadata: CREATED")
print("JSON serialization: PASSED")
print("Final integrity checks: PASSED")

print("\nFINAL MODEL SELECTIONS")
print("=" * 75)

print("Primary LOS model:", final_primary_los_model)

print(
    "Primary experimental classifier:",
    final_primary_classifier
)

print(
    "Primary experimental threshold:",
    repr(
        final_experimental_thresholds[
            final_primary_classifier
        ]
    )
)

print("\nRegression models retained: 3")
print("Classification models retained: 2")
print("Total fitted pipelines retained: 5")

print("\nAll five models trained: YES")
print("All five models validated: YES")
print("All five models test evaluated: YES")
print("New predictions generated: NO")
print("Model selection changed: NO")
print("Models retrained: NO")
print("Thresholds reoptimized: NO")
print("Clinical deployment approved: NO")
print("Research artifact export approved: YES")
print("Model files exported: NO")

print(
    "\nReady for Cell 48: "
    "Predictive Model Artifact Export."
)


In [ ]:

# ============================================================
# CELL 48: PREDICTIVE MODEL ARTIFACT EXPORT
# St. Hubert Medical Center - BI Project
# Dataset Version: V2
# Research / Experimental Use Only
# ============================================================

import json
import hashlib
import platform
import shutil
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

from pathlib import Path
from datetime import datetime, timezone
from sklearn.utils.validation import check_is_fitted

print("=" * 75)
print("PREDICTIVE MODEL ARTIFACT EXPORT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Canonical pipeline registry
# ------------------------------------------------------------

export_registry = {
    "Linear Regression": {
        "pipeline": lr_pipeline,
        "filename": "los_linear_regression.joblib",
        "task": "LOS Regression",
        "role": "COMPARATIVE_MODEL",
        "threshold": None
    },
    "Random Forest Regressor": {
        "pipeline": rf_los_pipeline,
        "filename": "los_random_forest.joblib",
        "task": "LOS Regression",
        "role": "PRIMARY_EXPERIMENTAL_CANDIDATE",
        "threshold": None
    },
    "XGBoost Regressor": {
        "pipeline": xgb_los_pipeline,
        "filename": "los_xgboost.joblib",
        "task": "LOS Regression",
        "role": "SECONDARY_EXPERIMENTAL_CANDIDATE",
        "threshold": None
    },
    "Random Forest Classifier": {
        "pipeline": rf_classifier_pipeline,
        "filename": "subsequent_visit_random_forest.joblib",
        "task": "30-Day Subsequent Visit Proxy",
        "role": "SECONDARY_EXPERIMENTAL_CANDIDATE",
        "threshold": 0.09556332971472799
    },
    "XGBoost Classifier": {
        "pipeline": xgb_classifier_pipeline,
        "filename": "subsequent_visit_xgboost.joblib",
        "task": "30-Day Subsequent Visit Proxy",
        "role": "PRIMARY_EXPERIMENTAL_CANDIDATE",
        "threshold": 0.08537115901708603
    }
}

expected_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

assert len(export_registry) == 5

assert len({
    id(config["pipeline"])
    for config in export_registry.values()
}) == 5

# ------------------------------------------------------------
# 2. Pre-export integrity verification
# ------------------------------------------------------------

print("\nPRE-EXPORT INTEGRITY VERIFICATION")
print("=" * 75)

for model_name, config in export_registry.items():

    pipeline = config["pipeline"]

    check_is_fitted(pipeline)

    actual_features = list(pipeline.feature_names_in_)

    assert actual_features == expected_features, (
        f"Feature contract mismatch: {model_name}"
    )

    assert pipeline.n_features_in_ == 7

    if config["threshold"] is not None:
        assert list(pipeline.classes_) == [0, 1]

    print(f"{model_name}: VERIFIED")

print("Five distinct fitted pipelines: VERIFIED")
print("Feature contracts: VERIFIED")

# ------------------------------------------------------------
# 3. Create private export directory
# ------------------------------------------------------------

export_timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

export_root = Path(
    "/content/private_model_exports"
) / f"St_Hubert_Predictive_Models_V2_{export_timestamp}"

models_dir = export_root / "models"
evaluation_dir = export_root / "evaluation"
metadata_dir = export_root / "metadata"

for directory in [
    models_dir,
    evaluation_dir,
    metadata_dir
]:
    directory.mkdir(parents=True, exist_ok=False)

print("\nEXPORT DIRECTORY")
print("=" * 75)
print(export_root)

# ------------------------------------------------------------
# 4. Utility functions
# ------------------------------------------------------------

def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


def json_safe(value):

    if value is None or value is pd.NA:
        return None

    if isinstance(value, np.generic):
        return json_safe(value.item())

    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())

    if isinstance(value, pd.DataFrame):
        return json_safe(
            value.to_dict(orient="records")
        )

    if isinstance(value, pd.Series):
        return json_safe(value.tolist())

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]

    if isinstance(value, float):
        return value if np.isfinite(value) else None

    if isinstance(value, (str, int, bool)):
        return value

    if isinstance(value, (datetime, pd.Timestamp)):
        return value.isoformat()

    raise TypeError(
        f"Unsupported metadata type: {type(value).__name__}"
    )


def write_json(data, file_path):

    with open(file_path, "w", encoding="utf-8") as file:

        json.dump(
            json_safe(data),
            file,
            indent=4,
            ensure_ascii=False,
            allow_nan=False
        )

    with open(file_path, "r", encoding="utf-8") as file:
        json.load(file)


# ------------------------------------------------------------
# 5. Export all five fitted pipelines
# ------------------------------------------------------------

print("\nFITTED PIPELINE EXPORT")
print("=" * 75)

exported_model_records = []

for model_name, config in export_registry.items():

    model_path = models_dir / config["filename"]

    joblib.dump(
        config["pipeline"],
        model_path,
        compress=3
    )

    assert model_path.exists()
    assert model_path.stat().st_size > 0

    file_size = model_path.stat().st_size

    record = {
        "model": model_name,
        "task": config["task"],
        "role": config["role"],
        "filename": config["filename"],
        "relative_path": str(
            model_path.relative_to(export_root)
        ),
        "estimator": type(
            config["pipeline"].steps[-1][1]
        ).__name__,
        "feature_count": 7,
        "input_features": expected_features,
        "experimental_threshold": config["threshold"],
        "clinical_deployment_approved": False,
        "operational_alerts_approved": False,
        "research_use_only": True,
        "file_size_bytes": file_size,
        "sha256": calculate_sha256(model_path)
    }

    exported_model_records.append(record)

    print(f"Exported: {model_name}")
    print(f"File: {config['filename']}")
    print(f"Size: {file_size / (1024 ** 2):.4f} MB")
    print("-" * 75)

assert len(list(models_dir.glob("*.joblib"))) == 5

# ------------------------------------------------------------
# 6. Export existing aggregate evaluation tables
# ------------------------------------------------------------

print("\nAGGREGATE EVALUATION TABLE EXPORT")
print("=" * 75)

# Explicit allowlist.
# Do not export raw datasets or patient-level predictions.

evaluation_table_allowlist = [
    "final_regression_validation_ranking",
    "final_classification_validation_ranking",
    "regression_comparison_df",
    "regression_test_results_df",
    "regression_test_comparison_df",
    "regression_validation_test_df",
    "classification_probability_comparison_df",
    "classification_comprehensive_comparison_df",
    "classification_threshold_comparison_df",
    "classification_threshold_optimization_summary_df",
    "classification_threshold_utility_df",
    "classification_test_probability_df",
    "classification_test_probability_comparison_df",
    "classification_test_threshold_df",
    "classification_test_threshold_comparison_df",
    "classification_validation_test_df",
    "final_regression_performance_df",
    "final_classification_performance_df",
    "final_classification_decision_df",
    "final_classification_baseline_df",
    "final_classification_risk_ranking_df",
    "final_proposal_assessment_df",
    "final_model_status_df",
    "final_feature_contract_df",
    "los_model_role_df"
]

blocked_columns = {
    "PATIENT_KEY",
    "PAT_ID",
    "REFR_NO",
    "PATIENT_ID",
    "VISIT_ID",
    "PATIENT_NAME"
}

exported_evaluation_tables = []
unavailable_evaluation_tables = []

for variable_name in evaluation_table_allowlist:

    table = globals().get(variable_name)

    if not isinstance(table, pd.DataFrame):
        unavailable_evaluation_tables.append(
            variable_name
        )
        continue

    normalized_columns = {
        str(column).strip().upper()
        for column in table.columns
    }

    assert not (
        normalized_columns & blocked_columns
    ), f"Identifier detected: {variable_name}"

    # Aggregate table safeguard.
    assert len(table) <= 1000, (
        f"Unexpected table size: {variable_name}"
    )

    table_path = evaluation_dir / (
        f"{variable_name}.csv"
    )

    table.to_csv(
        table_path,
        index=False,
        float_format="%.17g"
    )

    exported_evaluation_tables.append({
        "variable": variable_name,
        "filename": table_path.name,
        "rows": len(table),
        "columns": len(table.columns)
    })

    print(
        f"Exported: {table_path.name} "
        f"({len(table)} rows)"
    )

# ------------------------------------------------------------
# 7. Export known evaluation metadata
# ------------------------------------------------------------

print("\nEVALUATION METADATA EXPORT")
print("=" * 75)

metadata_allowlist = [
    "classification_threshold_policy",
    "classification_threshold_optimization_metadata",
    "classification_comparison_metadata",
    "final_selection_metadata",
    "final_model_selection_metadata",
    "regression_test_evaluation_metadata",
    "classification_test_evaluation_metadata",
    "final_performance_metadata",
    "final_consolidated_metadata",
    "final_predictive_model_metadata"
]

exported_metadata_variables = []
unavailable_metadata_variables = []

for variable_name in metadata_allowlist:

    value = globals().get(variable_name)

    if not isinstance(value, dict):
        unavailable_metadata_variables.append(
            variable_name
        )
        continue

    metadata_path = metadata_dir / (
        f"{variable_name}.json"
    )

    write_json(value, metadata_path)

    exported_metadata_variables.append(
        variable_name
    )

    print(f"Exported: {metadata_path.name}")

# ------------------------------------------------------------
# 8. Authoritative model configuration
# ------------------------------------------------------------

# Metrics below are taken from the verified Cell 44B,
# Cell 45, Cell 46 and Cell 47 outputs.
# They are reported results, not newly computed metrics.

reported_final_metrics = {
    "Linear Regression": {
        "validation_mae": 49.739195588,
        "test_mae": 49.816931,
        "test_rmse": 57.705647,
        "test_r2": 0.010237
    },
    "Random Forest Regressor": {
        "validation_mae": 48.997453854,
        "test_mae": 49.030185594,
        "test_rmse": 57.066287299,
        "test_r2": 0.032047517
    },
    "XGBoost Regressor": {
        "validation_mae": 49.005474790,
        "test_mae": 49.030638,
        "test_rmse": 57.065311,
        "test_r2": 0.032081
    },
    "Random Forest Classifier": {
        "validation_roc_auc": 0.498105974,
        "validation_average_precision": 0.116030067,
        "test_roc_auc": 0.503309648,
        "test_average_precision": 0.117251772,
        "test_brier_score": 0.102748095,
        "test_log_loss": 0.359500546,
        "test_confusion_matrix": [
            [253, 96296],
            [31, 12665]
        ]
    },
    "XGBoost Classifier": {
        "validation_roc_auc": 0.498337598,
        "validation_average_precision": 0.116686386,
        "test_roc_auc": 0.502348257,
        "test_average_precision": 0.117014622,
        "test_brier_score": 0.102804928,
        "test_log_loss": 0.359777504,
        "test_f1": 0.208119574,
        "test_confusion_matrix": [
            [270, 96279],
            [39, 12657]
        ]
    }
}

model_configuration = {
    "project": "St. Hubert Medical Center BI",
    "dataset_version": "V2",
    "export_timestamp_utc": export_timestamp,
    "artifact_purpose": "RESEARCH_AND_DEMONSTRATION",
    "primary_los_model": "Random Forest Regressor",
    "primary_experimental_classifier": "XGBoost Classifier",
    "selection_dataset": "Validation",
    "model_selection_frozen": True,
    "experimental_thresholds_frozen": True,
    "clinical_deployment_approved": False,
    "clinical_threshold_approved": False,
    "operational_alerts_approved": False,
    "los_target": "TARGET_LOS_DAYS",
    "classification_target":
        "SUBSEQUENT_VISIT_WITHIN_30D_PROXY",
    "classification_target_limitation":
        "Not a validated unplanned readmission outcome.",
    "input_features": expected_features,
    "excluded_model_inputs": [
        "PATIENT_KEY",
        "VISIT_STATUS_AT_EXIT",
        "LOS_DAYS"
    ],
    "classification_prediction_rule":
        "positive_class_probability >= experimental_threshold",
    "experimental_thresholds": {
        "Random Forest Classifier":
            0.09556332971472799,
        "XGBoost Classifier":
            0.08537115901708603
    },
    "evaluation_records": {
        "los_validation": 115592,
        "los_test": 115809,
        "classification_validation": 109458,
        "classification_test": 109245
    },
    "proposal_targets": {
        "los_mae_below": 1.0,
        "roc_auc_above": 0.80,
        "average_precision_above": 0.75
    },
    "proposal_targets_achieved": 0,
    "proposal_model_metric_assessments": 7,
    "reported_final_metrics": reported_final_metrics,
    "metric_source":
        "Verified Cell 44B, 45, 46 and 47 outputs",
    "models": exported_model_records
}

configuration_path = (
    metadata_dir / "model_configuration.json"
)

write_json(
    model_configuration,
    configuration_path
)

# Verify full-precision threshold serialization.

with open(
    configuration_path,
    "r",
    encoding="utf-8"
) as file:
    recovered_configuration = json.load(file)

for model_name, threshold in (
    model_configuration["experimental_thresholds"].items()
):

    assert (
        recovered_configuration[
            "experimental_thresholds"
        ][model_name] == threshold
    )

print("\nModel configuration: EXPORTED")
print("Threshold round trip: PASSED")

# ------------------------------------------------------------
# 9. Record software dependencies
# ------------------------------------------------------------

import scipy

environment_metadata = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "xgboost": xgboost.__version__,
    "joblib": joblib.__version__
}

write_json(
    environment_metadata,
    metadata_dir / "environment.json"
)

requirements_lines = [
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"scipy=={scipy.__version__}",
    f"xgboost=={xgboost.__version__}",
    f"joblib=={joblib.__version__}"
]

(export_root / "requirements.txt").write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8"
)

print("\nSOFTWARE ENVIRONMENT")
print("=" * 75)

for name, version in environment_metadata.items():
    print(f"{name}: {version}")

# ------------------------------------------------------------
# 10. Research-use README
# ------------------------------------------------------------

readme_text = """# St. Hubert Medical Center
## Predictive Model Artifacts - Dataset V2

RESEARCH AND DEMONSTRATION USE ONLY

This package contains five fitted machine learning
pipelines with their fitted preprocessing components.

Primary experimental LOS model:
Random Forest Regressor

Primary experimental subsequent visit classifier:
XGBoost Classifier

Neither task achieved the proposed performance targets.

The LOS models have large prediction errors.
The classifiers exhibit approximately chance-level
discrimination and negligible practical risk enrichment.

The classification target is a 30-day subsequent visit
proxy, not a validated unplanned hospital readmission
outcome.

Experimental thresholds must not be used to activate
clinical or operational risk alerts.

Clinical deployment is not approved.

Do not use these models for patient care decisions,
discharge planning, or clinical resource allocation.

Patient-grouped held-out evaluation does not establish
temporal or external generalizability.

Random Forest probability regeneration may introduce
negligible floating-point differences at exact
threshold boundaries.

The reported metrics were consolidated from existing
validation and held-out evaluation results.

No models were retrained during export.
No thresholds were reoptimized during export.

Load joblib artifacts only from trusted sources.
Serialized model files may execute code when loaded.

Keep this package private. Do not upload it to a public
repository.

Artifact reload and prediction-equivalence verification
must be completed before research API integration.
"""

(export_root / "README.md").write_text(
    readme_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. Export inventory and completeness report
# ------------------------------------------------------------

export_inventory = {
    "exported_model_count": len(
        exported_model_records
    ),
    "exported_evaluation_tables":
        exported_evaluation_tables,
    "unavailable_evaluation_tables":
        unavailable_evaluation_tables,
    "exported_metadata_variables":
        exported_metadata_variables,
    "unavailable_metadata_variables":
        unavailable_metadata_variables,
    "model_retraining": False,
    "threshold_reoptimization": False,
    "new_predictions_generated": False,
    "artifact_reload_verified": False,
    "clinical_deployment_approved": False
}

write_json(
    export_inventory,
    metadata_dir / "export_inventory.json"
)

export_manifest_df = pd.DataFrame(
    exported_model_records
)

export_manifest_df.to_csv(
    metadata_dir / "model_manifest.csv",
    index=False,
    float_format="%.17g"
)

# ------------------------------------------------------------
# 12. Generate file integrity manifest
# ------------------------------------------------------------

file_integrity_manifest = []

for file_path in sorted(export_root.rglob("*")):

    if not file_path.is_file():
        continue

    file_integrity_manifest.append({
        "relative_path": str(
            file_path.relative_to(export_root)
        ),
        "size_bytes": file_path.stat().st_size,
        "sha256": calculate_sha256(file_path)
    })

write_json(
    file_integrity_manifest,
    metadata_dir / "file_integrity_manifest.json"
)

# Verify checksums before packaging.

for record in file_integrity_manifest:

    file_path = (
        export_root / record["relative_path"]
    )

    assert file_path.exists()

    assert (
        calculate_sha256(file_path)
        == record["sha256"]
    )

print("\nFILE INTEGRITY VERIFICATION")
print("=" * 75)
print("All recorded SHA-256 checksums: PASSED")

# ------------------------------------------------------------
# 13. Create private ZIP package
# ------------------------------------------------------------

zip_path = Path(
    shutil.make_archive(
        str(export_root),
        "zip",
        root_dir=export_root.parent,
        base_dir=export_root.name
    )
)

assert zip_path.exists()
assert zip_path.stat().st_size > 0

zip_sha256 = calculate_sha256(zip_path)

# ------------------------------------------------------------
# 14. Display export results
# ------------------------------------------------------------

print("\nEXPORTED MODEL ARTIFACTS")
print("=" * 75)

display(
    export_manifest_df[[
        "model",
        "filename",
        "role",
        "file_size_bytes",
        "experimental_threshold",
        "clinical_deployment_approved"
    ]]
)

print("\nEXPORT COVERAGE")
print("=" * 75)

print(
    "Aggregate evaluation tables exported:",
    len(exported_evaluation_tables)
)

print(
    "Existing metadata dictionaries exported:",
    len(exported_metadata_variables)
)

print("\nOptional table names not found:")

for name in unavailable_evaluation_tables:
    print(f" - {name}")

print("\nOptional metadata names not found:")

for name in unavailable_metadata_variables:
    print(f" - {name}")

print("\nPRIVATE EXPORT PACKAGE")
print("=" * 75)
print("Export directory:", export_root)
print("ZIP package:", zip_path)
print(
    "ZIP size (MB):",
    round(zip_path.stat().st_size / (1024 ** 2), 4)
)
print("ZIP SHA-256:", zip_sha256)

# ------------------------------------------------------------
# 15. Final export state
# ------------------------------------------------------------

predictive_model_artifact_export_completed = True

predictive_model_artifact_export_path = str(
    export_root
)

predictive_model_artifact_zip_path = str(
    zip_path
)

predictive_model_artifact_manifest = (
    exported_model_records
)

print("\nARTIFACT EXPORT COMPLETION SUMMARY")
print("=" * 75)
print("Five fitted pipelines exported: YES")
print("Fitted preprocessing retained: YES")
print("Feature contracts verified: YES")
print("Experimental thresholds preserved: YES")
print("Aggregate evaluation export: COMPLETED")
print("Model configuration exported: YES")
print("Software environment recorded: YES")
print("File integrity checks: PASSED")
print("Private ZIP package created: YES")
print("Models retrained: NO")
print("Thresholds reoptimized: NO")
print("New predictions generated: NO")
print("Clinical deployment approved: NO")
print("Artifact reload verification: PENDING")

print(
    "\nReady for Cell 49: "
    "Exported Model Artifact Verification."
)


In [ ]:

# ============================================================
# CELL 49: EXPORTED MODEL ARTIFACT VERIFICATION
# St. Hubert Medical Center - BI Project
# Dataset Version: V2
# Research / Experimental Use Only
# ============================================================

import json
import hashlib
import platform
import zipfile
import joblib
import numpy as np
import pandas as pd
import sklearn
import scipy
import xgboost

from pathlib import Path
from datetime import datetime, timezone
from sklearn.utils.validation import check_is_fitted

print("=" * 75)
print("EXPORTED MODEL ARTIFACT VERIFICATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify required variables
# ------------------------------------------------------------

required_variables = [
    "predictive_model_artifact_export_completed",
    "predictive_model_artifact_export_path",
    "predictive_model_artifact_zip_path",
    "lr_pipeline",
    "rf_los_pipeline",
    "xgb_los_pipeline",
    "rf_classifier_pipeline",
    "xgb_classifier_pipeline",
    "X_los_train",
    "X_los_val",
    "X_proxy_train",
    "X_proxy_val"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing required variables: {missing_variables}"
)

assert predictive_model_artifact_export_completed

# Prevent a previous successful run from being mistaken
# for completion of the current verification.

predictive_model_artifact_verification_completed = False

export_root = Path(
    predictive_model_artifact_export_path
)

zip_path = Path(
    predictive_model_artifact_zip_path
)

assert export_root.is_dir()
assert zip_path.is_file()

print("Required variables: VERIFIED")
print("Cell 48 completion: VERIFIED")
print("Export directory: VERIFIED")
print("ZIP package: VERIFIED")

# ------------------------------------------------------------
# 2. File hashing utility
# ------------------------------------------------------------

def verification_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. Recover exported metadata
# ------------------------------------------------------------

metadata_dir = export_root / "metadata"

with open(
    metadata_dir / "model_configuration.json",
    encoding="utf-8"
) as file:

    saved_configuration = json.load(file)

with open(
    metadata_dir / "file_integrity_manifest.json",
    encoding="utf-8"
) as file:

    saved_integrity_manifest = json.load(file)

with open(
    metadata_dir / "environment.json",
    encoding="utf-8"
) as file:

    saved_environment = json.load(file)

with open(
    metadata_dir / "export_inventory.json",
    encoding="utf-8"
) as file:

    saved_export_inventory = json.load(file)

assert saved_configuration["dataset_version"] == "V2"

assert saved_configuration[
    "model_selection_frozen"
] is True

assert saved_configuration[
    "experimental_thresholds_frozen"
] is True

assert saved_configuration[
    "clinical_deployment_approved"
] is False

assert saved_configuration[
    "operational_alerts_approved"
] is False

assert saved_configuration[
    "primary_los_model"
] == "Random Forest Regressor"

assert saved_configuration[
    "primary_experimental_classifier"
] == "XGBoost Classifier"

print("\nEXPORTED CONFIGURATION VERIFICATION")
print("=" * 75)
print("Dataset version: V2")
print("Frozen model selections: VERIFIED")
print("Research-use restrictions: VERIFIED")

# ------------------------------------------------------------
# 4. Verify exported file checksums
# ------------------------------------------------------------

print("\nEXPORTED FILE INTEGRITY")
print("=" * 75)

verified_file_records = []

for record in saved_integrity_manifest:

    file_path = export_root / record["relative_path"]

    assert file_path.is_file(), (
        f"Missing file: {record['relative_path']}"
    )

    actual_size = file_path.stat().st_size
    actual_hash = verification_sha256(file_path)

    assert actual_size == record["size_bytes"], (
        f"File size mismatch: {file_path.name}"
    )

    assert actual_hash == record["sha256"], (
        f"Checksum mismatch: {file_path.name}"
    )

    verified_file_records.append({
        "File": record["relative_path"],
        "Size Bytes": actual_size,
        "SHA-256 Verified": True
    })

print(
    "Manifest-listed files verified:",
    len(verified_file_records)
)

print("Exported file integrity: PASSED")

# ------------------------------------------------------------
# 5. Verify ZIP archive
# ------------------------------------------------------------

print("\nZIP ARCHIVE VERIFICATION")
print("=" * 75)

# Hash recorded by Cell 48 in the current runtime.
expected_zip_sha256 = zip_sha256

actual_zip_sha256 = verification_sha256(zip_path)

assert actual_zip_sha256 == expected_zip_sha256, (
    "ZIP checksum differs from Cell 48 output."
)

with zipfile.ZipFile(zip_path, "r") as archive:

    assert archive.testzip() is None, (
        "ZIP archive contains a corrupted file."
    )

    archive_files = {
        name for name in archive.namelist()
        if not name.endswith("/")
    }

    expected_archive_files = {
        f"{export_root.name}/"
        f"{path.relative_to(export_root).as_posix()}"
        for path in export_root.rglob("*")
        if path.is_file()
    }

    assert archive_files == expected_archive_files, (
        "ZIP file inventory differs from export directory."
    )

    # Compare every archived file with its disk counterpart.

    for archive_name in sorted(archive_files):

        relative_name = archive_name.split("/", 1)[1]

        disk_path = export_root / relative_name

        archive_digest = hashlib.sha256()

        with archive.open(archive_name) as file:

            for chunk in iter(
                lambda: file.read(1024 * 1024),
                b""
            ):
                archive_digest.update(chunk)

        assert (
            archive_digest.hexdigest()
            == verification_sha256(disk_path)
        ), f"Archived file mismatch: {relative_name}"

print("ZIP SHA-256:", actual_zip_sha256)
print("ZIP checksum: PASSED")
print("ZIP corruption check: PASSED")
print("ZIP file inventory: PASSED")
print("ZIP content equivalence: PASSED")
print("Archived files:", len(archive_files))

# ------------------------------------------------------------
# 6. Verify software environment
# ------------------------------------------------------------

print("\nSOFTWARE ENVIRONMENT VERIFICATION")
print("=" * 75)

current_environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "xgboost": xgboost.__version__,
    "joblib": joblib.__version__
}

environment_records = []

for package, current_version in (
    current_environment.items()
):

    exported_version = saved_environment[package]

    version_matches = (
        current_version == exported_version
    )

    environment_records.append({
        "Package": package,
        "Exported Version": exported_version,
        "Current Version": current_version,
        "Match": version_matches
    })

    assert version_matches, (
        f"Environment mismatch: {package}"
    )

artifact_environment_verification_df = pd.DataFrame(
    environment_records
)

display(artifact_environment_verification_df)

print("Software environment: VERIFIED")

# ------------------------------------------------------------
# 7. Original fitted model registry
# ------------------------------------------------------------

original_artifact_registry = {
    "Linear Regression": lr_pipeline,
    "Random Forest Regressor": rf_los_pipeline,
    "XGBoost Regressor": xgb_los_pipeline,
    "Random Forest Classifier": rf_classifier_pipeline,
    "XGBoost Classifier": xgb_classifier_pipeline
}

expected_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

assert saved_configuration[
    "input_features"
] == expected_features

saved_model_records = {
    record["model"]: record
    for record in saved_configuration["models"]
}

assert set(saved_model_records) == set(
    original_artifact_registry
)

assert len(saved_model_records) == 5

# ------------------------------------------------------------
# 8. Verify full-precision thresholds
# ------------------------------------------------------------

print("\nEXPERIMENTAL THRESHOLD VERIFICATION")
print("=" * 75)

expected_thresholds = {
    "Random Forest Classifier":
        0.09556332971472799,
    "XGBoost Classifier":
        0.08537115901708603
}

for model_name, expected_threshold in (
    expected_thresholds.items()
):

    saved_threshold = saved_configuration[
        "experimental_thresholds"
    ][model_name]

    assert saved_threshold == expected_threshold

    assert saved_model_records[model_name][
        "experimental_threshold"
    ] == expected_threshold

    print(
        f"{model_name}: "
        f"{saved_threshold:.17g}"
    )

print("Full-precision thresholds: VERIFIED")

# ------------------------------------------------------------
# 9. Create reproducible verification samples
# ------------------------------------------------------------

# Training and validation features only.
# No held-out test dataset access.
# No sample records will be exported.

VERIFICATION_SAMPLE_SIZE = 2048
VERIFICATION_RANDOM_STATE = 42

def make_verification_sample(frame):

    assert list(frame.columns) == expected_features

    return frame.sample(
        n=min(
            VERIFICATION_SAMPLE_SIZE,
            len(frame)
        ),
        random_state=VERIFICATION_RANDOM_STATE
    ).reset_index(drop=True)


artifact_verification_samples = {
    "LOS Regression": {
        "Training": make_verification_sample(
            X_los_train
        ),
        "Validation": make_verification_sample(
            X_los_val
        )
    },
    "30-Day Subsequent Visit Proxy": {
        "Training": make_verification_sample(
            X_proxy_train
        ),
        "Validation": make_verification_sample(
            X_proxy_val
        )
    }
}

print("\nVERIFICATION SAMPLE INVENTORY")
print("=" * 75)

for task, samples in (
    artifact_verification_samples.items()
):

    for sample_name, sample in samples.items():

        print(
            f"{task} | {sample_name}: "
            f"{len(sample):,} records"
        )

# ------------------------------------------------------------
# 10. Reload all five exported pipelines
# ------------------------------------------------------------

print("\nSERIALIZED PIPELINE RELOAD")
print("=" * 75)

reloaded_model_registry = {}

for model_name, record in (
    saved_model_records.items()
):

    model_path = export_root / record["relative_path"]

    # Load only artifacts generated by our trusted
    # Cell 48 export and verified above.

    assert (
        verification_sha256(model_path)
        == record["sha256"]
    )

    restored_pipeline = joblib.load(model_path)

    original_pipeline = original_artifact_registry[
        model_name
    ]

    check_is_fitted(restored_pipeline)

    assert type(restored_pipeline) is type(
        original_pipeline
    )

    assert type(
        restored_pipeline.steps[-1][1]
    ) is type(
        original_pipeline.steps[-1][1]
    )

    assert [
        name for name, _ in restored_pipeline.steps
    ] == [
        name for name, _ in original_pipeline.steps
    ]

    assert list(
        restored_pipeline.feature_names_in_
    ) == expected_features

    assert restored_pipeline.n_features_in_ == 7

    assert restored_pipeline is not original_pipeline

    if model_name in expected_thresholds:

        assert list(
            restored_pipeline.classes_
        ) == [0, 1]

    reloaded_model_registry[
        model_name
    ] = restored_pipeline

    print(f"{model_name}: RELOADED AND VERIFIED")

assert len(reloaded_model_registry) == 5

print("Five serialized pipelines reloaded: YES")

# ------------------------------------------------------------
# 11. Prediction equivalence verification
# ------------------------------------------------------------

print("\nPREDICTION EQUIVALENCE VERIFICATION")
print("=" * 75)

# Explicit numerical tolerances.
# Small floating-point differences are acceptable.
# Material differences are not.

REGRESSION_ATOL = 1e-8
REGRESSION_RTOL = 1e-10

CLASSIFICATION_ATOL = 1e-12
CLASSIFICATION_RTOL = 1e-10

prediction_verification_records = []

for model_name, restored_pipeline in (
    reloaded_model_registry.items()
):

    original_pipeline = original_artifact_registry[
        model_name
    ]

    task = saved_model_records[model_name]["task"]

    is_classifier = model_name in expected_thresholds

    print(f"\n{model_name}")
    print("-" * 75)

    for sample_name, sample in (
        artifact_verification_samples[task].items()
    ):

        # Compare fitted preprocessing outputs.

        original_transformed = (
            original_pipeline[:-1].transform(sample)
        )

        restored_transformed = (
            restored_pipeline[:-1].transform(sample)
        )

        if hasattr(original_transformed, "toarray"):

            original_transformed = (
                original_transformed.toarray()
            )

        if hasattr(restored_transformed, "toarray"):

            restored_transformed = (
                restored_transformed.toarray()
            )

        original_transformed = np.asarray(
            original_transformed
        )

        restored_transformed = np.asarray(
            restored_transformed
        )

        assert (
            original_transformed.shape
            == restored_transformed.shape
        )

        assert np.isfinite(
            original_transformed
        ).all()

        assert np.isfinite(
            restored_transformed
        ).all()

        np.testing.assert_allclose(
            original_transformed,
            restored_transformed,
            atol=1e-12,
            rtol=1e-10
        )

        preprocessing_max_difference = float(
            np.max(
                np.abs(
                    original_transformed
                    - restored_transformed
                )
            )
        )

        # Compare regression predictions or
        # classification probabilities.

        if is_classifier:

            original_predictions = (
                original_pipeline.predict_proba(sample)
            )

            restored_predictions = (
                restored_pipeline.predict_proba(sample)
            )

            atol = CLASSIFICATION_ATOL
            rtol = CLASSIFICATION_RTOL

            assert (
                original_predictions.shape
                == (len(sample), 2)
            )

            assert (
                restored_predictions.shape
                == (len(sample), 2)
            )

            assert (
                (restored_predictions >= 0)
                & (restored_predictions <= 1)
            ).all()

            np.testing.assert_allclose(
                restored_predictions.sum(axis=1),
                1.0,
                atol=1e-6,
                rtol=0
            )

        else:

            original_predictions = (
                original_pipeline.predict(sample)
            )

            restored_predictions = (
                restored_pipeline.predict(sample)
            )

            atol = REGRESSION_ATOL
            rtol = REGRESSION_RTOL

        original_predictions = np.asarray(
            original_predictions,
            dtype=np.float64
        )

        restored_predictions = np.asarray(
            restored_predictions,
            dtype=np.float64
        )

        assert (
            original_predictions.shape
            == restored_predictions.shape
        )

        assert np.isfinite(
            original_predictions
        ).all()

        assert np.isfinite(
            restored_predictions
        ).all()

        np.testing.assert_allclose(
            original_predictions,
            restored_predictions,
            atol=atol,
            rtol=rtol
        )

        absolute_differences = np.abs(
            original_predictions
            - restored_predictions
        )

        maximum_difference = float(
            absolute_differences.max()
        )

        mean_difference = float(
            absolute_differences.mean()
        )

        exact_match = bool(
            np.array_equal(
                original_predictions,
                restored_predictions
            )
        )

        changed_decisions = None
        boundary_only_changes = None

        # Verify experimental threshold decisions.

        if is_classifier:

            threshold = expected_thresholds[
                model_name
            ]

            original_probabilities = (
                original_predictions[:, 1]
            )

            restored_probabilities = (
                restored_predictions[:, 1]
            )

            original_decisions = (
                original_probabilities >= threshold
            )

            restored_decisions = (
                restored_probabilities >= threshold
            )

            changed_mask = (
                original_decisions
                != restored_decisions
            )

            changed_decisions = int(
                changed_mask.sum()
            )

            # Do not alter the frozen threshold or
            # silently round probabilities.
            # Only explain numerical boundary changes.

            if changed_decisions > 0:

                boundary_tolerance = (
                    atol + rtol * abs(threshold)
                )

                boundary_only_changes = bool(
                    np.all(
                        np.abs(
                            original_probabilities[
                                changed_mask
                            ] - threshold
                        ) <= boundary_tolerance
                    )
                    and
                    np.all(
                        np.abs(
                            restored_probabilities[
                                changed_mask
                            ] - threshold
                        ) <= boundary_tolerance
                    )
                )

                assert boundary_only_changes, (
                    "Material threshold decision mismatch: "
                    f"{model_name}"
                )

            else:

                boundary_only_changes = True

        prediction_verification_records.append({
            "Model": model_name,
            "Task": task,
            "Sample": sample_name,
            "Records": len(sample),
            "Transformed Features":
                restored_transformed.shape[1],
            "Preprocessing Max Difference":
                preprocessing_max_difference,
            "Prediction Max Difference":
                maximum_difference,
            "Prediction Mean Difference":
                mean_difference,
            "Exact Numerical Match":
                exact_match,
            "Tolerance Equivalence": True,
            "Changed Threshold Decisions":
                changed_decisions,
            "Boundary-Only Changes":
                boundary_only_changes,
            "Verification Status": "PASSED"
        })

        print(f"{sample_name} sample:")
        print(f"  Records: {len(sample):,}")
        print(
            "  Preprocessing maximum difference:",
            f"{preprocessing_max_difference:.3e}"
        )
        print(
            "  Prediction maximum difference:",
            f"{maximum_difference:.3e}"
        )
        print(
            "  Exact numerical match:",
            exact_match
        )
        print("  Tolerance equivalence: PASSED")

        if is_classifier:

            print(
                "  Changed threshold decisions:",
                changed_decisions
            )

            print(
                "  Boundary-only verification:",
                "PASSED"
            )

artifact_prediction_verification_df = pd.DataFrame(
    prediction_verification_records
)

assert len(
    artifact_prediction_verification_df
) == 10

assert artifact_prediction_verification_df[
    "Tolerance Equivalence"
].all()

print("\nCONSOLIDATED PREDICTION VERIFICATION")
print("=" * 75)

display(artifact_prediction_verification_df)

# ------------------------------------------------------------
# 12. Full validation threshold reproduction
# ------------------------------------------------------------

# Specifically verify the previously documented
# Random Forest floating-point boundary issue.
# This is a reload comparison, not threshold optimization.

print("\nFULL VALIDATION THRESHOLD REPRODUCTION")
print("=" * 75)

full_validation_verification_records = []

for model_name, threshold in (
    expected_thresholds.items()
):

    original_probabilities = (
        original_artifact_registry[model_name]
        .predict_proba(X_proxy_val)[:, 1]
        .astype(np.float64)
    )

    restored_probabilities = (
        reloaded_model_registry[model_name]
        .predict_proba(X_proxy_val)[:, 1]
        .astype(np.float64)
    )

    np.testing.assert_allclose(
        original_probabilities,
        restored_probabilities,
        atol=CLASSIFICATION_ATOL,
        rtol=CLASSIFICATION_RTOL
    )

    probability_difference = np.abs(
        original_probabilities
        - restored_probabilities
    )

    original_decisions = (
        original_probabilities >= threshold
    )

    restored_decisions = (
        restored_probabilities >= threshold
    )

    changed_mask = (
        original_decisions != restored_decisions
    )

    changed_count = int(changed_mask.sum())

    boundary_tolerance = (
        CLASSIFICATION_ATOL
        + CLASSIFICATION_RTOL * abs(threshold)
    )

    boundary_only = bool(
        np.all(
            np.abs(
                original_probabilities[changed_mask]
                - threshold
            ) <= boundary_tolerance
        )
        and
        np.all(
            np.abs(
                restored_probabilities[changed_mask]
                - threshold
            ) <= boundary_tolerance
        )
    )

    assert boundary_only

    cached_comparison_status = "NOT_AVAILABLE"
    cached_changed_count = None

    # Compare against the original cached validation
    # probabilities when they remain available.

    cached_probability_registry = globals().get(
        "threshold_validation_probabilities"
    )

    if isinstance(
        cached_probability_registry, dict
    ) and model_name in cached_probability_registry:

        cached_probabilities = np.asarray(
            cached_probability_registry[model_name],
            dtype=np.float64
        )

        assert (
            cached_probabilities.shape
            == restored_probabilities.shape
        )

        np.testing.assert_allclose(
            cached_probabilities,
            restored_probabilities,
            atol=CLASSIFICATION_ATOL,
            rtol=CLASSIFICATION_RTOL
        )

        cached_decisions = (
            cached_probabilities >= threshold
        )

        cached_changed_mask = (
            cached_decisions != restored_decisions
        )

        cached_changed_count = int(
            cached_changed_mask.sum()
        )

        assert np.all(
            np.abs(
                cached_probabilities[
                    cached_changed_mask
                ] - threshold
            ) <= boundary_tolerance
        )

        assert np.all(
            np.abs(
                restored_probabilities[
                    cached_changed_mask
                ] - threshold
            ) <= boundary_tolerance
        )

        cached_comparison_status = "PASSED"

    full_validation_verification_records.append({
        "Model": model_name,
        "Validation Records":
            len(original_probabilities),
        "Frozen Threshold": threshold,
        "Maximum Probability Difference":
            float(probability_difference.max()),
        "Changed Decisions vs Original":
            changed_count,
        "Boundary-Only Changes": boundary_only,
        "Cached Comparison":
            cached_comparison_status,
        "Changed Decisions vs Cached":
            cached_changed_count,
        "Verification Status": "PASSED"
    })

    print(f"\n{model_name}")
    print(
        "Full validation records:",
        len(original_probabilities)
    )
    print(
        "Maximum probability difference:",
        f"{probability_difference.max():.3e}"
    )
    print(
        "Changed decisions vs original:",
        changed_count
    )
    print(
        "Changed decisions vs cached:",
        cached_changed_count
    )
    print(
        "Cached comparison:",
        cached_comparison_status
    )
    print("Boundary verification: PASSED")

artifact_threshold_verification_df = pd.DataFrame(
    full_validation_verification_records
)

display(artifact_threshold_verification_df)

# ------------------------------------------------------------
# 13. Audit optional export coverage
# ------------------------------------------------------------

print("\nOPTIONAL EXPORT COVERAGE AUDIT")
print("=" * 75)

missing_optional_tables = saved_export_inventory[
    "unavailable_evaluation_tables"
]

missing_optional_metadata = saved_export_inventory[
    "unavailable_metadata_variables"
]

# Identify candidate aggregate tables by their schema.
# Do not automatically export or substitute them.

coverage_candidates = {
    "final_classification_validation_ranking": {
        "Model",
        "ROC-AUC",
        "PR-AUC (AP)",
        "Rank"
    },
    "final_classification_baseline_df": {
        "Model",
        "AP Lift",
        "Brier Skill Score"
    },
    "final_classification_risk_ranking_df": {
        "Model",
        "Highest Risk Group Records",
        "Highest Risk Group Lift"
    },
    "final_model_status_df": {
        "Model",
        "Deployment Status",
        "Research Export Approved"
    },
    "classification_test_threshold_comparison_df": {
        "Model",
        "Threshold Type",
        "TN",
        "FP",
        "FN",
        "TP"
    }
}

coverage_audit_records = []

namespace_snapshot = list(globals().items())

for missing_name in missing_optional_tables:

    required_columns = coverage_candidates.get(
        missing_name,
        set()
    )

    matching_variables = []

    if required_columns:

        for variable_name, value in namespace_snapshot:

            if not isinstance(value, pd.DataFrame):
                continue

            if len(value) > 1000:
                continue

            if required_columns.issubset(
                set(value.columns)
            ):

                matching_variables.append(
                    variable_name
                )

    coverage_audit_records.append({
        "Missing Optional Table": missing_name,
        "Candidate Variables":
            ", ".join(sorted(matching_variables)),
        "Manual Reconciliation Required": True
    })

artifact_coverage_audit_df = pd.DataFrame(
    coverage_audit_records
)

display(artifact_coverage_audit_df)

print("\nUnavailable optional metadata names:")

for name in missing_optional_metadata:
    print(f" - {name}")

print(
    "\nCoverage candidates are diagnostic only."
)

print(
    "No additional notebook variables were exported."
)

# ------------------------------------------------------------
# 14. Create verification metadata
# ------------------------------------------------------------

artifact_verification_metadata = {
    "project": "St. Hubert Medical Center BI",
    "dataset_version": "V2",
    "verification_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),
    "verified_zip_sha256": actual_zip_sha256,
    "verified_model_count":
        len(reloaded_model_registry),
    "verified_manifest_file_count":
        len(verified_file_records),
    "zip_integrity_verified": True,
    "zip_content_equivalence_verified": True,
    "software_environment_verified": True,
    "fitted_pipeline_reload_verified": True,
    "feature_contract_verified": True,
    "preprocessing_equivalence_verified": True,
    "prediction_equivalence_verified": True,
    "experimental_thresholds_verified": True,
    "threshold_boundary_verification_passed": True,
    "verification_samples": {
        "sample_size_per_split":
            VERIFICATION_SAMPLE_SIZE,
        "random_state":
            VERIFICATION_RANDOM_STATE,
        "splits": ["Training", "Validation"]
    },
    "numerical_tolerances": {
        "regression_atol": REGRESSION_ATOL,
        "regression_rtol": REGRESSION_RTOL,
        "classification_atol":
            CLASSIFICATION_ATOL,
        "classification_rtol":
            CLASSIFICATION_RTOL
    },
    "prediction_verification":
        artifact_prediction_verification_df.to_dict(
            orient="records"
        ),
    "full_validation_threshold_verification":
        artifact_threshold_verification_df.to_dict(
            orient="records"
        ),
    "optional_coverage_reconciliation_pending":
        bool(
            missing_optional_tables
            or missing_optional_metadata
        ),
    "model_retraining": False,
    "threshold_reoptimization": False,
    "test_dataset_reevaluated": False,
    "model_selection_changed": False,
    "clinical_deployment_approved": False,
    "operational_alerts_approved": False
}

# Reuse the JSON conversion utility from Cell 48.
# Convert NaN values to JSON null.

verification_json = json.dumps(
    json_safe(artifact_verification_metadata),
    indent=4,
    allow_nan=False
)

json.loads(verification_json)

print("\nVERIFICATION METADATA")
print("=" * 75)
print("Metadata creation: COMPLETED")
print("JSON serialization: PASSED")

# ------------------------------------------------------------
# 15. Save verification report separately
# ------------------------------------------------------------

# Preserve the original ZIP and its recorded checksum.
# Do not modify the existing export directory yet.

verification_report_dir = (
    export_root.parent
    / f"{export_root.name}_verification"
)

verification_report_dir.mkdir(
    parents=True,
    exist_ok=True
)

verification_report_path = (
    verification_report_dir
    / "artifact_verification_report.json"
)

verification_report_path.write_text(
    verification_json,
    encoding="utf-8"
)

artifact_prediction_verification_df.to_csv(
    verification_report_dir
    / "prediction_equivalence.csv",
    index=False,
    float_format="%.17g"
)

artifact_threshold_verification_df.to_csv(
    verification_report_dir
    / "threshold_reproduction.csv",
    index=False,
    float_format="%.17g"
)

artifact_environment_verification_df.to_csv(
    verification_report_dir
    / "environment_verification.csv",
    index=False
)

artifact_coverage_audit_df.to_csv(
    verification_report_dir
    / "optional_coverage_audit.csv",
    index=False
)

# Confirm that the original archive remains unchanged.

assert (
    verification_sha256(zip_path)
    == expected_zip_sha256
)

# ------------------------------------------------------------
# 16. Final verification state
# ------------------------------------------------------------

predictive_model_artifact_verification_completed = True

predictive_model_artifact_verification_path = str(
    verification_report_dir
)

print("\nARTIFACT VERIFICATION COMPLETION SUMMARY")
print("=" * 75)
print("Original ZIP checksum: VERIFIED")
print("ZIP archive integrity: PASSED")
print("Exported file checksums: PASSED")
print("Software environment: VERIFIED")
print("Five serialized pipelines reloaded: YES")
print("Fitted preprocessing verified: YES")
print("Feature contracts verified: YES")
print("Prediction equivalence: PASSED")
print("Full validation probability equivalence: PASSED")
print("Experimental thresholds preserved: YES")
print("Threshold boundary verification: PASSED")
print("Verification metadata created: YES")
print("Verification reports saved: YES")
print("Original ZIP modified: NO")
print("Optional export coverage audited: YES")
print("Models retrained: NO")
print("Thresholds reoptimized: NO")
print("Held-out test datasets reevaluated: NO")
print("Model selection changed: NO")
print("Clinical deployment approved: NO")

print("\nVERIFICATION REPORT DIRECTORY")
print("=" * 75)
print(verification_report_dir)

print("\nFROZEN MODEL SELECTIONS")
print("=" * 75)
print("Primary LOS model: Random Forest Regressor")
print("Primary experimental classifier: XGBoost Classifier")
print(
    "Frozen experimental threshold:",
    expected_thresholds["XGBoost Classifier"]
)

print("\nReady for Cell 50:")
print("Final Artifact Package Reconciliation and Handoff.")


In [ ]:

# ============================================================
# CELL 50: FINAL ARTIFACT PACKAGE RECONCILIATION AND HANDOFF
# St. Hubert Medical Center - BI Project
# Dataset Version: V2
# Research / Experimental Use Only
# ============================================================

import json
import hashlib
import shutil
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

print("=" * 75)
print("FINAL ARTIFACT PACKAGE RECONCILIATION AND HANDOFF")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify previous cell completion
# ------------------------------------------------------------

required_variables = [
    "predictive_model_artifact_export_completed",
    "predictive_model_artifact_verification_completed",
    "predictive_model_artifact_export_path",
    "predictive_model_artifact_zip_path",
    "predictive_model_artifact_verification_path",
    "artifact_verification_metadata",
    "final_predictive_model_metadata",
    "final_regression_performance_df",
    "final_classification_performance_df",
    "final_classification_decision_df",
    "final_proposal_assessment_df",
    "json_safe"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    f"Missing required variables: {missing_variables}"
)

assert predictive_model_artifact_export_completed
assert predictive_model_artifact_verification_completed

final_predictive_model_handoff_completed = False

source_root = Path(
    predictive_model_artifact_export_path
)

source_zip = Path(
    predictive_model_artifact_zip_path
)

verification_root = Path(
    predictive_model_artifact_verification_path
)

assert source_root.is_dir()
assert source_zip.is_file()
assert verification_root.is_dir()

print("Required variables: VERIFIED")
print("Cell 48 export completion: VERIFIED")
print("Cell 49 verification completion: VERIFIED")

# ------------------------------------------------------------
# 2. Integrity and JSON utilities
# ------------------------------------------------------------

def final_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def read_final_json(path):

    with open(path, encoding="utf-8") as file:

        return json.load(file)


def write_final_json(path, payload):

    serialized = json.dumps(
        json_safe(payload),
        indent=4,
        allow_nan=False
    )

    json.loads(serialized)

    path.write_text(
        serialized,
        encoding="utf-8"
    )


source_zip_hash = final_sha256(source_zip)

assert source_zip_hash == (
    artifact_verification_metadata[
        "verified_zip_sha256"
    ]
)

assert artifact_verification_metadata[
    "prediction_equivalence_verified"
]

assert artifact_verification_metadata[
    "clinical_deployment_approved"
] is False

original_file_hashes = {
    path.relative_to(source_root).as_posix():
        final_sha256(path)
    for path in source_root.rglob("*")
    if path.is_file()
}

# Verify source directory still matches verified archive.

with zipfile.ZipFile(source_zip, "r") as archive:

    assert archive.testzip() is None

    archived_names = [
        name for name in archive.namelist()
        if not name.endswith("/")
    ]

    assert len(archived_names) == len(
        set(archived_names)
    )

    assert set(archived_names) == {
        f"{source_root.name}/{relative_path}"
        for relative_path in original_file_hashes
    }

    for relative_path, expected_hash in (
        original_file_hashes.items()
    ):

        archived_bytes = archive.read(
            f"{source_root.name}/{relative_path}"
        )

        assert hashlib.sha256(
            archived_bytes
        ).hexdigest() == expected_hash

print("Original ZIP checksum: VERIFIED")
print("Original export contents: VERIFIED")

# ------------------------------------------------------------
# 3. Create separate final package
# ------------------------------------------------------------

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S%fZ")

final_root = source_root.parent / (
    "St_Hubert_Predictive_Models_V2_FINAL_"
    + timestamp
)

assert not final_root.exists()

shutil.copytree(
    source_root,
    final_root
)

metadata_dir = final_root / "metadata"
evaluation_dir = final_root / "evaluation"
documentation_dir = final_root / "documentation"
final_verification_dir = final_root / "verification"

evaluation_dir.mkdir(exist_ok=True)
documentation_dir.mkdir(exist_ok=True)

shutil.copytree(
    verification_root,
    final_verification_dir
)

configuration = read_final_json(
    metadata_dir / "model_configuration.json"
)

environment = read_final_json(
    metadata_dir / "environment.json"
)

original_inventory = read_final_json(
    metadata_dir / "export_inventory.json"
)

assert configuration[
    "clinical_deployment_approved"
] is False

assert configuration[
    "operational_alerts_approved"
] is False

assert configuration[
    "primary_los_model"
] == "Random Forest Regressor"

assert configuration[
    "primary_experimental_classifier"
] == "XGBoost Classifier"

print("\nFINAL PACKAGE DIRECTORY")
print("=" * 75)
print(final_root)
print("Original package preserved: YES")
print("Verification reports incorporated: YES")

# ------------------------------------------------------------
# 4. Reconcile missing evaluation tables
# ------------------------------------------------------------

print("\nEVALUATION TABLE RECONCILIATION")
print("=" * 75)

reconciliation_specs = {
    "final_classification_validation_ranking": {
        "source": "final_classification_selection_df",
        "columns": {
            "Model",
            "ROC-AUC",
            "PR-AUC (AP)",
            "Rank"
        },
        "expected_rows": 2
    },
    "classification_test_threshold_comparison_df": {
        "source": "classification_test_threshold_df",
        "columns": {
            "Model",
            "Threshold Type",
            "TN",
            "FP",
            "FN",
            "TP"
        },
        "expected_rows": 8
    },
    "final_classification_baseline_df": {
        "source":
            "classification_test_probability_comparison_df",
        "columns": {
            "Model",
            "ROC-AUC",
            "PR-AUC (AP)",
            "AP Lift",
            "Brier Skill Score"
        },
        "expected_rows": 3
    },
    "final_classification_risk_ranking_df": {
        "source": "final_classifier_risk_df",
        "columns": {
            "Model",
            "Highest Risk Group Records",
            "Highest Risk Group Lift",
            "Positive Capture Rate"
        },
        "expected_rows": 2
    },
    "final_model_status_df": {
        "source": "final_predictive_model_status_df",
        "columns": {
            "Model",
            "Deployment Status",
            "Research Export Approved",
            "Clinical Deployment Approved"
        },
        "expected_rows": 5
    }
}

reconciliation_records = []

reconciled_tables = {}

for export_name, spec in (
    reconciliation_specs.items()
):

    source_name = spec["source"]

    source_table = globals().get(source_name)

    assert isinstance(
        source_table,
        pd.DataFrame
    ), f"Required reconciliation source missing: {source_name}"

    assert spec["columns"].issubset(
        set(source_table.columns)
    ), f"Unexpected schema: {source_name}"

    assert len(source_table) == (
        spec["expected_rows"]
    ), f"Unexpected row count: {source_name}"

    reconciled_tables[export_name] = (
        source_table.copy()
    )

# ------------------------------------------------------------
# 5. Verify reconciliation semantics
# ------------------------------------------------------------

expected_classifier_names = {
    "Random Forest Classifier",
    "XGBoost Classifier"
}

expected_model_names = {
    "Linear Regression",
    "Random Forest Regressor",
    "XGBoost Regressor",
    "Random Forest Classifier",
    "XGBoost Classifier"
}

ranking_table = reconciled_tables[
    "final_classification_validation_ranking"
]

assert set(
    ranking_table["Model"]
) == expected_classifier_names

assert ranking_table.sort_values(
    "Rank"
).iloc[0]["Model"] == "XGBoost Classifier"

# Ensure recovered ranking contains validation metrics.

for _, row in ranking_table.iterrows():

    reference = final_classification_performance_df.loc[
        final_classification_performance_df["Model"]
        == row["Model"]
    ].iloc[0]

    np.testing.assert_allclose(
        [
            row["ROC-AUC"],
            row["PR-AUC (AP)"]
        ],
        [
            reference["Validation ROC-AUC"],
            reference["Validation PR-AUC (AP)"]
        ],
        atol=1e-8,
        rtol=1e-8
    )

threshold_table = reconciled_tables[
    "classification_test_threshold_comparison_df"
]

assert (
    threshold_table[
        ["TN", "FP", "FN", "TP"]
    ].sum(axis=1) == 109245
).all()

assert (
    threshold_table["TP"]
    + threshold_table["FN"]
    == 12696
).all()

assert (
    threshold_table["TN"]
    + threshold_table["FP"]
    == 96549
).all()

baseline_table = reconciled_tables[
    "final_classification_baseline_df"
]

assert set(baseline_table["Model"]) == (
    expected_classifier_names
    | {"Constant Training Prevalence"}
)

for model_name in expected_classifier_names:

    baseline_row = baseline_table.loc[
        baseline_table["Model"] == model_name
    ].iloc[0]

    reference = final_classification_performance_df.loc[
        final_classification_performance_df["Model"]
        == model_name
    ].iloc[0]

    np.testing.assert_allclose(
        [
            baseline_row["ROC-AUC"],
            baseline_row["PR-AUC (AP)"]
        ],
        [
            reference["Test ROC-AUC"],
            reference["Test PR-AUC (AP)"]
        ],
        atol=1e-8,
        rtol=1e-8
    )

risk_table = reconciled_tables[
    "final_classification_risk_ranking_df"
]

assert set(
    risk_table["Model"]
) == expected_classifier_names

assert (
    risk_table["Highest Risk Group Records"]
    == 10925
).all()

assert (
    risk_table["Highest Risk Group Lift"] < 1
).all()

expected_risk_positives = {
    "Random Forest Classifier": 1251,
    "XGBoost Classifier": 1267
}

for model_name, expected_positives in (
    expected_risk_positives.items()
):

    row = risk_table.loc[
        risk_table["Model"] == model_name
    ].iloc[0]

    assert int(
        row["Highest Risk Group Positives"]
    ) == expected_positives

status_table = reconciled_tables[
    "final_model_status_df"
]

assert set(
    status_table["Model"]
) == expected_model_names

assert status_table[
    "Clinical Deployment Approved"
].eq(False).all()

assert status_table[
    "Research Export Approved"
].eq(True).all()

assert status_table[
    "Deployment Status"
].eq("NOT_RECOMMENDED").all()

print("Reconciliation schema checks: PASSED")
print("Reconciliation semantic checks: PASSED")

# ------------------------------------------------------------
# 6. Export reconciled aggregate tables
# ------------------------------------------------------------

for export_name, table in (
    reconciled_tables.items()
):

    destination = (
        evaluation_dir / f"{export_name}.csv"
    )

    assert not destination.exists()

    table.to_csv(
        destination,
        index=False,
        float_format="%.17g"
    )

    reconciliation_records.append({
        "Requested Export": export_name,
        "Actual Source":
            reconciliation_specs[export_name]["source"],
        "Relative Path":
            destination.relative_to(
                final_root
            ).as_posix(),
        "Rows": len(table),
        "Schema Verified": True,
        "Semantics Verified": True,
        "Status": "RECONCILED"
    })

    print(
        f"Reconciled: {export_name} "
        f"({len(table)} rows)"
    )

final_export_reconciliation_df = pd.DataFrame(
    reconciliation_records
)

display(final_export_reconciliation_df)

final_export_reconciliation_df.to_csv(
    final_verification_dir
    / "export_reconciliation.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Reconcile optional metadata names
# ------------------------------------------------------------

print("\nMETADATA RECONCILIATION")
print("=" * 75)

metadata_aliases = {
    "final_selection_metadata":
        "final_model_selection_metadata.json",
    "final_performance_metadata":
        "final_predictive_model_metadata.json",
    "final_consolidated_metadata":
        "final_predictive_model_metadata.json"
}

metadata_reconciliation_records = []

for missing_name, actual_file in (
    metadata_aliases.items()
):

    actual_path = metadata_dir / actual_file

    assert actual_path.is_file()

    actual_payload = read_final_json(
        actual_path
    )

    assert isinstance(actual_payload, dict)

    metadata_reconciliation_records.append({
        "unavailable_optional_name": missing_name,
        "canonical_metadata_file":
            f"metadata/{actual_file}",
        "resolution":
            "EXISTING_CANONICAL_METADATA_REFERENCE",
        "duplicate_file_created": False
    })

    print(
        f"{missing_name} -> {actual_file}"
    )

write_final_json(
    metadata_dir / "metadata_reconciliation.json",
    metadata_reconciliation_records
)

# Preserve original inventory as historical evidence.

shutil.copy2(
    metadata_dir / "export_inventory.json",
    metadata_dir / "original_export_inventory.json"
)

shutil.copy2(
    metadata_dir / "file_integrity_manifest.json",
    metadata_dir / "original_file_integrity_manifest.json"
)

# ------------------------------------------------------------
# 8. Create explicit confusion matrix exports
# ------------------------------------------------------------

print("\nCONFUSION MATRIX EXPORT")
print("=" * 75)

confusion_records = []

for _, row in threshold_table.iterrows():

    confusion_records.append({
        "Model": row["Model"],
        "Threshold Type": row["Threshold Type"],
        "Threshold": row.get("Threshold"),
        "TN": int(row["TN"]),
        "FP": int(row["FP"]),
        "FN": int(row["FN"]),
        "TP": int(row["TP"]),
        "Total Records": 109245
    })

final_confusion_matrices_df = pd.DataFrame(
    confusion_records
)

final_confusion_matrices_df.to_csv(
    evaluation_dir
    / "held_out_confusion_matrices.csv",
    index=False,
    float_format="%.17g"
)

display(final_confusion_matrices_df)

print("Confusion matrix export: COMPLETED")

# ------------------------------------------------------------
# 9. Recover frozen configuration
# ------------------------------------------------------------

features = configuration["input_features"]

expected_features = [
    "AGE_AT_VISIT",
    "GENDER",
    "VISIT_TYPE",
    "ADMISSION_MONTH",
    "ADMISSION_WEEKDAY",
    "ADMISSION_HOUR",
    "PRIOR_VISIT_COUNT"
]

assert features == expected_features

thresholds = configuration[
    "experimental_thresholds"
]

assert thresholds[
    "Random Forest Classifier"
] == 0.09556332971472799

assert thresholds[
    "XGBoost Classifier"
] == 0.08537115901708603

primary_los = (
    final_regression_performance_df.loc[
        final_regression_performance_df["Model"]
        == "Random Forest Regressor"
    ].iloc[0]
)

primary_classifier = (
    final_classification_performance_df.loc[
        final_classification_performance_df["Model"]
        == "XGBoost Classifier"
    ].iloc[0]
)

primary_decision = (
    final_classification_decision_df.loc[
        final_classification_decision_df["Model"]
        == "XGBoost Classifier"
    ].iloc[0]
)

assert not final_proposal_assessment_df[
    "Target Achieved"
].any()

# ------------------------------------------------------------
# 10. Generate package README
# ------------------------------------------------------------

readme = """# St. Hubert Medical Center
## Predictive Modeling Research Artifacts - Dataset V2

### Purpose

This package contains five fitted predictive modeling
pipelines, aggregate evaluation results, configuration
metadata, and artifact verification reports.

It is intended for academic research and controlled
technical demonstrations only.

### Models

LOS regression:
- Linear Regression
- Random Forest Regressor
- XGBoost Regressor

30-day subsequent visit proxy classification:
- Random Forest Classifier
- XGBoost Classifier

### Frozen experimental selections

- Primary LOS candidate: Random Forest Regressor
- Primary classification candidate: XGBoost Classifier

Model selections were determined using validation data
and were not revised using held-out test results.

### Important restrictions

None of the predictive performance proposal targets
were achieved.

Clinical deployment is not approved.
Operational risk alerts are not approved.
Patient-specific clinical decision support is not approved.

The classification outcome represents a subsequent
visit proxy, not a validated unplanned hospital
readmission outcome.

### Package contents

- models/: fitted preprocessing and estimator pipelines
- evaluation/: aggregate evaluation tables
- metadata/: model configuration and integrity records
- verification/: serialization and numerical checks
- documentation/: model card and integration guidance

### Security

Keep this package private. Do not publish it to a public
repository without appropriate authorization.

Load joblib files only from a trusted source after
verifying their checksums.

No raw patient-level datasets or verification sample
records were intentionally included.

### Reproducibility

Use the recorded software versions.
Artifact reload verification was performed in the
original training environment.

Compatibility with another operating system or Python
environment requires additional verification.
"""

(final_root / "README.md").write_text(
    readme,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. Generate model card
# ------------------------------------------------------------

model_card = f"""# Predictive Model Card

## Project

St. Hubert Medical Center BI Project

Dataset version: V2

Intended use: academic research and demonstration.

## Prediction tasks

1. Length of Stay (LOS) regression.
2. 30-day subsequent visit proxy classification.

## Training and evaluation

Patient-grouped training, validation and held-out test
partitions were used.

Model selection and experimental threshold selection
were performed using validation data.

Held-out test results were not used to revise selections.

## Primary experimental LOS model

Model: Random Forest Regressor

Held-out test records: 115809

- MAE: {primary_los['Test MAE']:.6f} days
- RMSE: {primary_los['Test RMSE']:.6f} days
- R2: {primary_los['Test R2']:.6f}

Proposal MAE target: below 1 day.
Target achieved: No.

The model explains only a small proportion of observed
LOS variability. Predictions concentrate around central
target values and have substantial errors for short
and long stays.

## Primary experimental classifier

Model: XGBoost Classifier

Held-out test records: 109245
Positive records: 12696
Negative records: 96549

- ROC-AUC: {primary_classifier['Test ROC-AUC']:.6f}
- Average precision: {primary_classifier['Test PR-AUC (AP)']:.6f}
- Brier score: {primary_classifier['Test Brier Score']:.6f}
- Log loss: {primary_classifier['Test Log Loss']:.6f}

Frozen experimental threshold:
{thresholds['XGBoost Classifier']:.17g}

- Precision: {primary_decision['Precision']:.6f}
- Recall: {primary_decision['Recall']:.6f}
- F1: {primary_decision['F1']:.6f}
- Predicted positive rate:
  {primary_decision['Predicted Positive Rate'] * 100:.6f}%

Proposal ROC-AUC target: above 0.80.
Proposal average precision target: above 0.75.

Targets achieved: No.

The classifier exhibits approximately chance-level
discrimination. The experimental threshold flags nearly
all held-out visit records.

## Limitations

- Large LOS prediction errors.
- Weak classification discrimination.
- Probability errors worse than the constant
  training-prevalence baseline.
- Experimental thresholds have excessive alert burden.
- Default and proposal thresholds flag no test records.
- No useful highest-risk-decile enrichment.
- Subsequent visit proxy is not validated readmission.
- No established temporal or external generalizability.
- No reported patient-grouped confidence intervals.
- Source timestamp semantics and outcome definitions
  require further verification.
- Random Forest predictions may exhibit negligible
  floating-point variation near decision boundaries.

## Deployment assessment

Clinical deployment approved: No.
Operational alerts approved: No.
Research artifact export approved: Yes.

The primary experimental candidates are comparative
research selections, not clinically validated models.
"""

(
    documentation_dir / "MODEL_CARD.md"
).write_text(
    model_card,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 12. Generate feature contract documentation
# ------------------------------------------------------------

feature_contract = """# Predictive Model Feature Contract

All five fitted pipelines expect the following seven
original input features in the recorded order.

| Feature | Input type | Meaning |
|---|---|---|
| AGE_AT_VISIT | Numeric | Age at visit |
| GENDER | Categorical | Recorded gender |
| VISIT_TYPE | Categorical | Recorded visit type |
| ADMISSION_MONTH | Integer | Admission month |
| ADMISSION_WEEKDAY | Integer | Admission weekday |
| ADMISSION_HOUR | Integer | Admission hour |
| PRIOR_VISIT_COUNT | Integer | Prior visit count |

## Observed categorical vocabulary

GENDER:
- Female
- Male
- Other

VISIT_TYPE:
- Diagnosis
- Treatment

## Fitted preprocessing

Numeric inputs:
- Median imputation
- Standard scaling

Categorical inputs:
- Most-frequent imputation
- One-hot encoding
- Unknown categories handled by the fitted encoder

The verified fitted preprocessing produces ten
transformed features.

Do not independently fit or recreate preprocessing
during inference.

Pass the seven original features to the complete
serialized pipeline.

## Excluded inputs

Patient identifiers and outcome targets are not model
input features.

Exit status and realized LOS are not inputs to the
subsequent visit classifier.

## Input semantics

Future integrations must reproduce the training
dataset's feature definitions, category encoding,
weekday convention, and prior-visit counting rules.

Do not assume that a technically valid input is a
clinically meaningful input.
"""

(
    documentation_dir / "FEATURE_CONTRACT.md"
).write_text(
    feature_contract,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 13. Generate FastAPI integration guidance
# ------------------------------------------------------------

fastapi_guide = f"""# FastAPI Research Integration Guide

## Integration status

Research demonstration only.

Do not expose these models as clinically validated
prediction services.

## Recommended artifacts

LOS experimental candidate:
models/los_random_forest.joblib

Subsequent visit experimental candidate:
models/subsequent_visit_xgboost.joblib

## Recommended integration sequence

1. Create an isolated Python environment.
2. Install the recorded model dependencies.
3. Verify the artifact SHA-256 checksums.
4. Load only trusted serialized pipelines.
5. Validate the seven original input features.
6. Run a synthetic-input smoke test.
7. Verify prediction shapes and finite outputs.
8. Compare synthetic-input results across environments.
9. Clearly label all outputs as experimental.
10. Keep clinical alerts and operational automation disabled.

## Inference methods

LOS regression:
pipeline.predict(input_dataframe)

Subsequent visit probability:
pipeline.predict_proba(input_dataframe)

The positive-class probability corresponds to class 1.

Verify classifier class ordering before selecting
the probability column.

## Experimental thresholds

Random Forest Classifier:
{thresholds['Random Forest Classifier']:.17g}

XGBoost Classifier:
{thresholds['XGBoost Classifier']:.17g}

These thresholds are retained for research
reproducibility only.

They must not be treated as approved clinical
decision thresholds.

## Suggested demonstration endpoints

POST /research/predict-los
POST /research/predict-subsequent-visit

Suggested response metadata:

- dataset_version: V2
- research_only: true
- clinical_deployment_approved: false
- operational_alerts_approved: false

Do not describe the subsequent visit probability as a
validated probability of unplanned hospital readmission.

## Environment compatibility

The models were verified in their original Linux
training environment.

Successful loading on a different local operating
system has not yet been established.

FastAPI and Uvicorn versions are not pinned here
because this package documents the model environment,
not a verified API deployment environment.
"""

(
    documentation_dir
    / "FASTAPI_INTEGRATION_GUIDE.md"
).write_text(
    fastapi_guide,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 14. Export model dependency requirements
# ------------------------------------------------------------

dependency_names = [
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "xgboost",
    "joblib"
]

requirements_text = (
    "# Model environment dependencies\n"
    f"# Python version: {environment['python']}\n"
    + "\n".join(
        f"{name}=={environment[name]}"
        for name in dependency_names
    )
    + "\n"
)

(
    documentation_dir / "requirements.txt"
).write_text(
    requirements_text,
    encoding="utf-8"
)

print("\nTECHNICAL DOCUMENTATION")
print("=" * 75)
print("README: CREATED")
print("Model card: CREATED")
print("Feature contract: CREATED")
print("FastAPI integration guide: CREATED")
print("Model requirements: CREATED")

# ------------------------------------------------------------
# 15. Verify model files remain unchanged
# ------------------------------------------------------------

print("\nFINAL MODEL FILE VERIFICATION")
print("=" * 75)

final_model_records = []

for record in configuration["models"]:

    relative_path = record["relative_path"]

    original_model_path = (
        source_root / relative_path
    )

    final_model_path = (
        final_root / relative_path
    )

    assert final_model_path.is_file()

    final_hash = final_sha256(
        final_model_path
    )

    assert final_hash == final_sha256(
        original_model_path
    )

    assert final_hash == record["sha256"]

    final_model_records.append({
        "Model": record["model"],
        "Relative Path": relative_path,
        "Size Bytes":
            final_model_path.stat().st_size,
        "SHA-256": final_hash,
        "Unchanged": True
    })

    print(
        f"{record['model']}: VERIFIED"
    )

assert len(final_model_records) == 5

final_model_inventory_df = pd.DataFrame(
    final_model_records
)

display(final_model_inventory_df)

# ------------------------------------------------------------
# 16. Create final handoff metadata
# ------------------------------------------------------------

final_handoff_metadata = {
    "project": "St. Hubert Medical Center BI",
    "dataset_version": "V2",
    "package_status":
        "VERIFIED_RESEARCH_ARTIFACT_PACKAGE",
    "created_utc":
        datetime.now(timezone.utc).isoformat(),
    "original_export_zip_sha256":
        source_zip_hash,
    "model_count": 5,
    "primary_los_model":
        "Random Forest Regressor",
    "primary_experimental_classifier":
        "XGBoost Classifier",
    "experimental_thresholds":
        thresholds,
    "model_selection_frozen": True,
    "experimental_thresholds_frozen": True,
    "model_artifacts_unchanged": True,
    "artifact_reload_verified": True,
    "prediction_equivalence_verified": True,
    "verification_environment":
        environment,
    "reconciled_evaluation_tables":
        reconciliation_records,
    "metadata_reconciliation":
        metadata_reconciliation_records,
    "proposal_targets_achieved": 0,
    "proposal_targets_assessed": 7,
    "research_export_approved": True,
    "clinical_deployment_approved": False,
    "operational_alerts_approved": False,
    "external_environment_verified": False,
    "model_retraining": False,
    "threshold_reoptimization": False,
    "new_predictions_generated": False,
    "held_out_test_reevaluated": False,
    "model_selection_changed": False,
    "raw_patient_records_intentionally_exported":
        False
}

write_final_json(
    metadata_dir / "final_handoff_metadata.json",
    final_handoff_metadata
)

# ------------------------------------------------------------
# 17. Package privacy and inventory checks
# ------------------------------------------------------------

print("\nPACKAGE CONTENT AUDIT")
print("=" * 75)

allowed_extensions = {
    ".joblib",
    ".csv",
    ".json",
    ".md",
    ".txt"
}

for path in final_root.rglob("*"):

    assert not path.is_symlink(), (
        f"Unexpected symbolic link: {path}"
    )

    if not path.is_file():
        continue

    assert path.suffix.lower() in (
        allowed_extensions
    ), f"Unexpected file type: {path.name}"

    if path.suffix.lower() == ".csv":

        table = pd.read_csv(path)

        assert len(table) <= 1000, (
            f"Unexpected non-aggregate CSV: {path.name}"
        )

        forbidden_columns = {
            "PATIENT_KEY",
            "PAT_ID",
            "REFR_NO",
            "PATIENT_ID",
            "PATIENT_NAME"
        }

        assert not (
            forbidden_columns
            & set(table.columns)
        ), f"Identifier column found: {path.name}"

    elif path.suffix.lower() == ".json":

        read_final_json(path)

assert len(
    list(final_root.rglob("*.joblib"))
) == 5

print("Allowed file types: VERIFIED")
print("Aggregate CSV row limits: VERIFIED")
print("Explicit identifier columns: NOT FOUND")
print("JSON parsing: PASSED")
print("Five model artifacts: VERIFIED")

# ------------------------------------------------------------
# 18. Update final export inventory
# ------------------------------------------------------------

final_inventory = {
    "dataset_version": "V2",
    "package_status":
        "FINAL_VERIFIED_RESEARCH_PACKAGE",
    "models": final_model_records,
    "evaluation_tables": sorted(
        path.relative_to(
            final_root
        ).as_posix()
        for path in evaluation_dir.rglob("*.csv")
    ),
    "verification_files": sorted(
        path.relative_to(
            final_root
        ).as_posix()
        for path in final_verification_dir.rglob("*")
        if path.is_file()
    ),
    "documentation_files": sorted(
        path.relative_to(
            final_root
        ).as_posix()
        for path in documentation_dir.rglob("*")
        if path.is_file()
    ),
    "unavailable_evaluation_tables": [],
    "unavailable_metadata_variables": [],
    "metadata_alias_resolution":
        metadata_reconciliation_records,
    "clinical_deployment_approved": False,
    "operational_alerts_approved": False
}

write_final_json(
    metadata_dir / "export_inventory.json",
    final_inventory
)

# ------------------------------------------------------------
# 19. Generate final SHA-256 manifest
# ------------------------------------------------------------

print("\nFINAL FILE INTEGRITY MANIFEST")
print("=" * 75)

manifest_path = (
    metadata_dir / "file_integrity_manifest.json"
)

final_manifest = []

for path in sorted(final_root.rglob("*")):

    if not path.is_file():
        continue

    # The manifest cannot contain its own checksum.

    if path == manifest_path:
        continue

    final_manifest.append({
        "relative_path":
            path.relative_to(final_root).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": final_sha256(path)
    })

write_final_json(
    manifest_path,
    final_manifest
)

for record in final_manifest:

    path = final_root / record["relative_path"]

    assert path.is_file()

    assert path.stat().st_size == (
        record["size_bytes"]
    )

    assert final_sha256(path) == (
        record["sha256"]
    )

print(
    "Manifest-listed files:",
    len(final_manifest)
)

print("Final file checksums: PASSED")

# ------------------------------------------------------------
# 20. Create final private ZIP archive
# ------------------------------------------------------------

final_zip = final_root.with_suffix(".zip")

assert not final_zip.exists()

with zipfile.ZipFile(
    final_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in sorted(final_root.rglob("*")):

        if path.is_file():

            archive.write(
                path,
                arcname=(
                    f"{final_root.name}/"
                    f"{path.relative_to(final_root).as_posix()}"
                )
            )

final_zip_hash = final_sha256(final_zip)

# ------------------------------------------------------------
# 21. Verify final ZIP content equivalence
# ------------------------------------------------------------

with zipfile.ZipFile(
    final_zip,
    mode="r"
) as archive:

    assert archive.testzip() is None

    archive_names = [
        name for name in archive.namelist()
        if not name.endswith("/")
    ]

    assert len(archive_names) == len(
        set(archive_names)
    )

    expected_archive_names = {
        f"{final_root.name}/"
        f"{path.relative_to(final_root).as_posix()}"
        for path in final_root.rglob("*")
        if path.is_file()
    }

    assert set(
        archive_names
    ) == expected_archive_names

    for archive_name in archive_names:

        relative_path = archive_name.split(
            "/", 1
        )[1]

        archived_hash = hashlib.sha256(
            archive.read(archive_name)
        ).hexdigest()

        assert archived_hash == final_sha256(
            final_root / relative_path
        )

assert len(archive_names) == (
    len(final_manifest) + 1
)

# Confirm original package was not modified.

assert final_sha256(
    source_zip
) == source_zip_hash

assert {
    path.relative_to(source_root).as_posix():
        final_sha256(path)
    for path in source_root.rglob("*")
    if path.is_file()
} == original_file_hashes

# ------------------------------------------------------------
# 22. Save external ZIP checksum
# ------------------------------------------------------------

final_checksum_path = final_zip.with_suffix(
    ".zip.sha256"
)

final_checksum_path.write_text(
    f"{final_zip_hash}  {final_zip.name}\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# 23. Final handoff state
# ------------------------------------------------------------

final_predictive_model_handoff_completed = True

final_predictive_model_handoff_path = str(
    final_root
)

final_predictive_model_handoff_zip_path = str(
    final_zip
)

final_predictive_model_handoff_sha256 = (
    final_zip_hash
)

final_predictive_model_handoff_checksum_path = str(
    final_checksum_path
)

print("\nFINAL PRIVATE HANDOFF PACKAGE")
print("=" * 75)
print("Final directory:", final_root)
print("Final ZIP:", final_zip)
print(
    "ZIP size (MiB):",
    round(
        final_zip.stat().st_size / (1024 ** 2),
        4
    )
)
print("ZIP SHA-256:", final_zip_hash)
print("Checksum file:", final_checksum_path)
print("Archived files:", len(archive_names))

print("\nFINAL PACKAGE COMPLETION SUMMARY")
print("=" * 75)
print("Five fitted pipelines retained: YES")
print("Model files unchanged: YES")
print("Fitted preprocessing retained: YES")
print("Frozen selections preserved: YES")
print("Experimental thresholds preserved: YES")
print("Optional evaluation tables reconciled: 5/5")
print("Optional metadata names reconciled: 3/3")
print("Confusion matrices exported: YES")
print("Verification reports incorporated: YES")
print("Technical documentation created: YES")
print("Model environment recorded: YES")
print("Package content audit: PASSED")
print("Final file integrity checks: PASSED")
print("Final ZIP integrity: PASSED")
print("Final ZIP content equivalence: PASSED")
print("Original package modified: NO")
print("Models retrained: NO")
print("Thresholds reoptimized: NO")
print("New predictions generated: NO")
print("Held-out test datasets reevaluated: NO")
print("Model selection changed: NO")
print("Clinical deployment approved: NO")
print("Operational alerts approved: NO")
print("Research handoff package ready: YES")

print("\nFINAL MODEL SELECTIONS")
print("=" * 75)
print("Primary LOS model: Random Forest Regressor")
print("Primary classifier: XGBoost Classifier")
print(
    "Experimental classifier threshold:",
    thresholds["XGBoost Classifier"]
)

print("\nReady for Cell 51:")
print("Private Artifact Download and Transfer.")


In [ ]:

# ============================================================
# CELL 51: PRIVATE ARTIFACT DOWNLOAD AND TRANSFER
# St. Hubert Medical Center
# Predictive Models V2
# ============================================================

from pathlib import Path
import hashlib
import zipfile
from google.colab import files

print("=" * 75)
print("PRIVATE ARTIFACT DOWNLOAD AND TRANSFER")
print("=" * 75)

# ------------------------------------------------------------
# 1. Locate the verified final handoff package
# ------------------------------------------------------------

assert final_predictive_model_handoff_completed
handoff_zip_path = Path(final_predictive_model_handoff_zip_path)
handoff_checksum_path = Path(final_predictive_model_handoff_checksum_path)
expected_handoff_sha256 = final_predictive_model_handoff_sha256

assert handoff_zip_path.is_file(), (
    "Final handoff ZIP not found."
)

assert handoff_checksum_path.is_file(), (
    "SHA-256 checksum file not found."
)

print("\nFinal ZIP: VERIFIED")
print("Checksum file: VERIFIED")

# ------------------------------------------------------------
# 2. Verify package checksum before download
# ------------------------------------------------------------

print("\nPRE-DOWNLOAD INTEGRITY VERIFICATION")
print("=" * 75)

sha256_hasher = hashlib.sha256()

with handoff_zip_path.open("rb") as file:

    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b""
    ):
        sha256_hasher.update(chunk)

actual_handoff_sha256 = sha256_hasher.hexdigest()

recorded_handoff_sha256 = (
    handoff_checksum_path
    .read_text()
    .strip()
    .split()[0]
)

assert actual_handoff_sha256 == expected_handoff_sha256, (
    "Final ZIP checksum mismatch."
)

assert recorded_handoff_sha256 == actual_handoff_sha256, (
    "Checksum sidecar mismatch."
)

print("Expected SHA-256:", expected_handoff_sha256)
print("Actual SHA-256:  ", actual_handoff_sha256)
print("ZIP checksum verification: PASSED")
print("Checksum sidecar verification: PASSED")

# ------------------------------------------------------------
# 3. Verify archive contents
# ------------------------------------------------------------

print("\nFINAL ARCHIVE VERIFICATION")
print("=" * 75)

with zipfile.ZipFile(handoff_zip_path, "r") as archive:

    assert archive.testzip() is None, (
        "ZIP corruption detected."
    )

    archived_files = [
        item.filename
        for item in archive.infolist()
        if not item.is_dir()
    ]

    archived_model_files = [
        name
        for name in archived_files
        if name.endswith(".joblib")
    ]

    assert len(archived_files) == 59
    assert len(archived_model_files) == 5

    expected_model_filenames = {
        "los_linear_regression.joblib",
        "los_random_forest.joblib",
        "los_xgboost.joblib",
        "subsequent_visit_random_forest.joblib",
        "subsequent_visit_xgboost.joblib"
    }

    actual_model_filenames = {
        Path(name).name
        for name in archived_model_files
    }

    assert actual_model_filenames == (
        expected_model_filenames
    )

print("ZIP corruption check: PASSED")
print("Archived files:", len(archived_files))
print("Fitted model artifacts:", len(archived_model_files))

for name in sorted(actual_model_filenames):
    print("-", name)

# ------------------------------------------------------------
# 4. Display final download information
# ------------------------------------------------------------

package_size_mib = (
    handoff_zip_path.stat().st_size / (1024 ** 2)
)

print("\nPRIVATE DOWNLOAD INFORMATION")
print("=" * 75)

print("Package:", handoff_zip_path.name)
print(f"Package size: {package_size_mib:.4f} MiB")
print("Dataset version: V2")
print("Research-use package: YES")
print("Clinical deployment approved: NO")
print("Operational alerts approved: NO")

# ------------------------------------------------------------
# 5. Initiate private package download
# ------------------------------------------------------------

print("\nINITIATING PRIVATE ZIP DOWNLOAD")
print("=" * 75)

files.download(str(handoff_zip_path))

print("\nZIP download requested.")
print("Confirm that the browser finishes downloading.")

# ------------------------------------------------------------
# 6. Completion summary
# ------------------------------------------------------------

private_artifact_download_requested = True

print("\nPRIVATE TRANSFER SUMMARY")
print("=" * 75)

print("Final ZIP integrity: PASSED")
print("Checksum file integrity: PASSED")
print("Archive corruption check: PASSED")
print("Five fitted pipelines: VERIFIED")
print("ZIP download requested: YES")
print("Local download completion: USER CONFIRMATION REQUIRED")

print("\nModels retrained: NO")
print("Thresholds reoptimized: NO")
print("New predictions generated: NO")
print("Package modified: NO")
print("Clinical deployment approved: NO")

print("\nReady for local checksum verification.")


In [ ]:

# ============================================================
# CELL 51A: DOWNLOAD SHA-256 CHECKSUM FILE
# ============================================================

assert handoff_checksum_path.is_file()

files.download(str(handoff_checksum_path))

print("SHA-256 checksum download requested.")
print("Confirm both files are saved locally.")
